In [ ]:
# ================================================================================
# AUTHENTICATION CELL — Google Drive Upload (Kaggle Secret PRIMARY)
# Fixes: BUG-AUTH-1, BUG-AUTH-2
# Defines globals: upload_checkpoint_to_gdrive, upload_epoch_files_to_gdrive
# ================================================================================
import os, io, sys, json, traceback
from pathlib import Path

# ── Install dependencies ─────────────────────────────────────────────────────────
import subprocess
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "google-auth", "google-auth-httplib2", "google-api-python-client"
], check=False)

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.oauth2.service_account import Credentials

# ── CONFIGURATION ────────────────────────────────────────────────────────────────
_SECRET_LABEL = "GDRIVE_SERVICE_ACCOUNT"              # Kaggle Secret label
_SA_FILENAME  = "tensile-courier-392915-24cc7a149809.json"
_FOLDER_ID    = "1SykY9HXLB9XZvxlLXAWRKZeRjLyctCI0"  # your Drive folder ID
_CHUNK_SIZE   = 50 * 1024 * 1024                       # 50 MB resumable chunks
_SCOPES       = ["https://www.googleapis.com/auth/drive"]

# Candidate local paths (fallback if secret fails)
_SA_SEARCH_PATHS = [
    f"/kaggle/input/gdrive-sa-key/{_SA_FILENAME}",
    f"/kaggle/input/gdrivesakey/{_SA_FILENAME}",
    f"/kaggle/working/{_SA_FILENAME}",
    f"./{_SA_FILENAME}",
]

# ── Module-level state ───────────────────────────────────────────────────────────
_drive_service = None
_auth_method   = None
_auth_success  = False


# ── Build Drive service ───────────────────────────────────────────────────────────
def _build_drive_service() -> bool:
    global _drive_service, _auth_method, _auth_success

    # ════════════════════════════════════════════════════════════════════════
    # METHOD 1 (PRIMARY): Kaggle Secret — label = GDRIVE_SERVICE_ACCOUNT
    # ════════════════════════════════════════════════════════════════════════
    try:
        from kaggle_secrets import UserSecretsClient
        print(f"[AUTH] Trying Kaggle Secret: '{_SECRET_LABEL}' ...")
        sa_json = UserSecretsClient().get_secret(_SECRET_LABEL)
        sa_info = json.loads(sa_json)

        # Validate it's a proper service account JSON
        assert sa_info.get("type") == "service_account", \
            "JSON type is not 'service_account'"
        assert "client_email" in sa_info, \
            "Missing 'client_email' in JSON"

        creds = Credentials.from_service_account_info(sa_info, scopes=_SCOPES)
        _drive_service = build("drive", "v3", credentials=creds)
        _auth_method   = "Kaggle Secret"
        _auth_success  = True

        print(f"[AUTH] ✅ Authenticated via Kaggle Secret")
        print(f"[AUTH]    Service account: {sa_info.get('client_email')}")
        print(f"[AUTH]    Project        : {sa_info.get('project_id')}")
        return True

    except Exception as e1:
        print(f"[AUTH] Method 1 (Kaggle Secret) failed: {type(e1).__name__}: {e1}")

    # ════════════════════════════════════════════════════════════════════════
    # METHOD 2 (FALLBACK): Service Account JSON file from Kaggle Dataset
    # ════════════════════════════════════════════════════════════════════════
    sa_path = next((p for p in _SA_SEARCH_PATHS if os.path.isfile(p)), None)
    if sa_path:
        try:
            print(f"[AUTH] Trying JSON file: {sa_path} ...")
            creds = Credentials.from_service_account_file(sa_path, scopes=_SCOPES)
            _drive_service = build("drive", "v3", credentials=creds)
            _auth_method   = f"JSON file ({sa_path})"
            _auth_success  = True
            print(f"[AUTH] ✅ Authenticated via Service Account JSON file")
            return True
        except Exception as e2:
            print(f"[AUTH] Method 2 (JSON file) failed: {type(e2).__name__}: {e2}")
    else:
        print(f"[AUTH] Method 2 (JSON file): Not found in any candidate path.")

    # ════════════════════════════════════════════════════════════════════════
    # METHOD 3 (NO-OP): Training continues; Drive uploads silently skipped
    # ════════════════════════════════════════════════════════════════════════
    print("[AUTH] ⚠️  All auth methods failed.")
    print("[AUTH]    Drive uploads will be SKIPPED. Training continues normally.")
    print("[AUTH]    Checkpoints → /kaggle/working/ (download from Output tab).")
    _auth_success = False
    return False


# ── Verify Drive folder is accessible ───────────────────────────────────────────
def _verify_drive_folder(folder_id: str = _FOLDER_ID) -> bool:
    if _drive_service is None:
        return False
    try:
        meta = _drive_service.files().get(
            fileId=folder_id,
            fields="id, name, mimeType"
        ).execute()
        print(f"[AUTH] ✅ Drive folder OK: '{meta.get('name', '?')}' (id={folder_id})")
        return True
    except Exception as e:
        print(f"[AUTH] ❌ Drive folder NOT accessible: {type(e).__name__}: {e}")
        print(f"[AUTH]    → Share folder {folder_id} with the service account email as Editor.")
        return False


# ================================================================================
# UPLOAD FUNCTIONS (called from Cell 7 training loop)
# ================================================================================

def upload_checkpoint_to_gdrive(
    local_path : str,
    epoch       = None,
    folder_id  : str = _FOLDER_ID,
) -> bool:
    """
    Upload a single .pt checkpoint to Google Drive via resumable upload.
    Returns True on success. Never raises — safe to call inside training loop.
    """
    if _drive_service is None:
        print(f"[GDRIVE] ⏭  Skipping (no auth): {os.path.basename(str(local_path))}")
        return False

    local_path = str(local_path)
    if not os.path.isfile(local_path):
        print(f"[GDRIVE] ❌ File not found: {local_path}")
        return False

    try:
        filename     = os.path.basename(local_path)
        file_size_mb = os.path.getsize(local_path) / (1024 ** 2)
        epoch_tag    = f" [epoch {epoch}]" if epoch is not None else ""

        print(f"[GDRIVE] ↑ Uploading{epoch_tag}: {filename} ({file_size_mb:.1f} MB) ...")

        # Overwrite if file already exists — avoids duplicates on re-runs
        existing = _drive_service.files().list(
            q=(f"name='{filename}' and '{folder_id}' in parents "
               f"and trashed=false"),
            spaces="drive",
            fields="files(id, name)",
        ).execute().get("files", [])

        media = MediaFileUpload(
            local_path,
            mimetype="application/octet-stream",
            resumable=True,
            chunksize=_CHUNK_SIZE,
        )

        if existing:
            req = _drive_service.files().update(
                fileId=existing[0]["id"],
                media_body=media,
            )
        else:
            req = _drive_service.files().create(
                body={"name": filename, "parents": [folder_id]},
                media_body=media,
                fields="id",
            )

        # Resumable upload with progress
        response = None
        while response is None:
            status, response = req.next_chunk()
            if status:
                print(f"[GDRIVE]    {filename}: {int(status.progress() * 100)}%",
                      end="\r")

        print(f"[GDRIVE] ✅ Done: {filename} ({file_size_mb:.1f} MB){epoch_tag}        ")
        return True

    except Exception as e:
        print(f"\n[GDRIVE] ❌ Upload failed ({os.path.basename(local_path)}): "
              f"{type(e).__name__}: {e}")
        traceback.print_exc()
        return False


def upload_epoch_files_to_gdrive(
    epoch          : int,
    checkpoint_dir : str = "/kaggle/working",
    folder_id      : str = _FOLDER_ID,
) -> bool:
    """
    Upload per-epoch bundle to Google Drive:
      - tatn_epoch{epoch}.pt
      - dscd_prototypes_epoch{epoch}.pt
    Called from Cell 7 after each epoch checkpoint save.
    Returns True if all found files uploaded successfully.
    """
    if _drive_service is None:
        print(f"[GDRIVE] ⏭  Skipping epoch {epoch} bundle (no auth)")
        return False

    print(f"[GDRIVE] {'='*50}")
    print(f"[GDRIVE] Epoch {epoch} — uploading bundle to Drive ...")
    print(f"[GDRIVE] {'='*50}")

    files_to_upload = [
        os.path.join(checkpoint_dir, f"tatn_epoch{epoch}.pt"),
        os.path.join(checkpoint_dir, f"dscd_prototypes_epoch{epoch}.pt"),
    ]

    all_ok, n_uploaded = True, 0
    for fpath in files_to_upload:
        if os.path.isfile(fpath):
            ok = upload_checkpoint_to_gdrive(fpath, epoch=epoch, folder_id=folder_id)
            all_ok = all_ok and ok
            if ok:
                n_uploaded += 1
        else:
            print(f"[GDRIVE] ⚠️  Not found (skip): {os.path.basename(fpath)}")

    status = "✅ All uploaded" if all_ok else "⚠️  Partial upload"
    print(f"[GDRIVE] {status} — {n_uploaded}/{len(files_to_upload)} files for epoch {epoch}")
    return all_ok


# ================================================================================
# RUN ON CELL EXECUTION
# ================================================================================
print("=" * 70)
print("  AUTHENTICATION CELL — Google Drive Upload Setup")
print("=" * 70)

_auth_ok   = _build_drive_service()
_folder_ok = _verify_drive_folder(_FOLDER_ID) if _auth_ok else False

print("-" * 70)
print(f"  Auth status    : {'✅ SUCCESS' if _auth_ok else '⚠️  FAILED (uploads skipped)'}")
print(f"  Auth method    : {_auth_method if _auth_ok else 'N/A'}")
print(f"  Folder access  : {'✅ OK' if _folder_ok else '⚠️  NOT VERIFIED'}")
print(f"  Folder ID      : {_FOLDER_ID}")
print(f"  Chunk size     : {_CHUNK_SIZE // (1024*1024)} MB")
print("-" * 70)

# ── Hard guard: both functions MUST exist as callables before Cell 7 runs ───────
assert callable(upload_epoch_files_to_gdrive), \
    "FATAL: upload_epoch_files_to_gdrive not callable!"
assert callable(upload_checkpoint_to_gdrive), \
    "FATAL: upload_checkpoint_to_gdrive not callable!"

print("  upload_epoch_files_to_gdrive  ✅  defined and callable")
print("  upload_checkpoint_to_gdrive   ✅  defined and callable")
print("=" * 70)
if not _auth_ok:
    print("  ℹ️  Training proceeds. Download outputs from Kaggle Output tab.")
print("=" * 70)


In [ ]:
# ============================================================================
# CELL 0: ENVIRONMENT SETUP & IMPORTS - mBART-50 Compatible (fixed)
# ============================================================================
# Note: run this cell in a Jupyter / Colab notebook. It reinstalls the specified
# transformers version and required libs, then performs a quick tokenizer check.
# ----------------------------------------------------------------------------

# Step 0: (Optional) show what's being done
print("Installing exact package versions. This may take a few minutes...")

# Step 1: Clean uninstall potentially conflicting packages (quiet)
# (Using %pip / !pip is fine in notebooks; here we use the bang form.)
!pip uninstall -y transformers tokenizers sentence-transformers huggingface-hub datasets tokenizers -q || true

# Step 2: Install required packages with the requested transformers version
# Add huggingface-hub and datasets, and sentencepiece which is required by MBART tokenizers.
!pip install -q --upgrade pip
!pip install -q transformers==4.57.6 huggingface-hub datasets tokenizers sacrebleu sacremoses sentencepiece

# Step 3: Import and verify core libraries
import importlib, sys, os, gc, time, warnings
warnings.filterwarnings("ignore")
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print("="*80)
print("IMPORT & VERSION CHECK")
print("="*80)

# Try imports and print versions with defensive guards
try:
    import transformers
    print(f"✅ transformers: {transformers.__version__}")
except Exception as e:
    print(f"❌ transformers import failed: {type(e).__name__}: {e}")
    raise

try:
    import tokenizers
    print(f"✅ tokenizers: {tokenizers.__version__}")
except Exception as e:
    # tokenizers might be part of the transformers wheel; still try to continue
    print(f"⚠️  tokenizers import issue: {type(e).__name__}: {e}")

try:
    import sacrebleu
    print(f"✅ sacrebleu: {sacrebleu.__version__}")
except Exception as e:
    print(f"⚠️  sacrebleu import issue: {type(e).__name__}: {e}")

try:
    import datasets
    print(f"✅ datasets: {datasets.__version__}")
except Exception as e:
    print(f"⚠️  datasets import issue: {type(e).__name__}: {e}")

print("="*80)
print("IMPORTS FOR TATN & mBART-50")
print("="*80)

# Core Python libs
import math, re, json, traceback
from collections import defaultdict, OrderedDict
from typing import List, Dict, Tuple, Optional, Any, Union
from datetime import datetime

# Data processing
import numpy as np
import pandas as pd

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Hugging Face - mBART-50 specific imports (names depend on transformers version)
from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
from transformers.modeling_outputs import BaseModelOutput

# Metrics
from sacrebleu.metrics import BLEU, CHRF

# Threading for TATN
import threading

print("✅ All libraries imported (attempt).")

# Step 4: Check CUDA availability
print("\n" + "="*80)
print("HARDWARE DETECTION")
print("="*80)

if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"✅ CUDA available: {gpu_count} GPU(s)")
    for i in range(gpu_count):
        try:
            gpu_name = torch.cuda.get_device_name(i)
            gpu_memory = torch.cuda.get_device_properties(i).total_memory / (1024**3)
            print(f"   GPU {i}: {gpu_name} ({gpu_memory:.1f} GB)")
        except Exception:
            print(f"   GPU {i}: (name lookup failed)")
else:
    print("⚠️  CUDA not available - using CPU")

# Step 5: Test mBART-50 tokenizer/model availability
print("\n" + "="*80)
print("TESTING mBART-50 TOKENIZER ACCESS")
print("="*80)

# Use the many-to-many model by default; many-to-one exists but many-to-many is standard.
HF_MODEL = "facebook/mbart-large-50-many-to-many-mmt"

try:
    # Load the tokenizer (this performs network access the first time)
    test_tokenizer = MBart50TokenizerFast.from_pretrained(HF_MODEL)
    # Set src/tgt language codes (some tokenizer versions expect these attributes)
    try:
        test_tokenizer.src_lang = "bn_IN"
        test_tokenizer.tgt_lang = "en_XX"
    except Exception:
        # Not all tokenizer wrappers accept direct assignment; it's OK
        pass

    # Vocab size: prefer tokenizer.vocab_size or len(tokenizer)
    vocab_size = getattr(test_tokenizer, "vocab_size", None)
    if vocab_size is None:
        try:
            vocab_size = len(test_tokenizer)
        except Exception:
            vocab_size = "unknown"

    print("✅ Tokenizer loaded successfully")
    print(f"   Model: {HF_MODEL}")
    print(f"   Vocab size: {vocab_size}")

    # Print language token IDs if available
    lang_map = getattr(test_tokenizer, "lang_code_to_id", None)
    if isinstance(lang_map, dict):
        bn_token = lang_map.get("bn_IN", lang_map.get("bn", "N/A"))
        en_token = lang_map.get("en_XX", lang_map.get("en", "N/A"))
        print(f"   lang_code_to_id keys present. bn_IN -> {bn_token}, en_XX -> {en_token}")
    else:
        # try alternative get_lang_id if available
        if hasattr(test_tokenizer, "get_lang_id"):
            try:
                bn_token = test_tokenizer.get_lang_id("bn_IN")
                en_token = test_tokenizer.get_lang_id("en_XX")
                print(f"   get_lang_id: bn_IN -> {bn_token}, en_XX -> {en_token}")
            except Exception:
                print("   No lang mapping available on tokenizer instance.")
        else:
            print("   No lang mapping available on tokenizer instance.")

    # Quick encode/decode smoke test
    sample = "আমি কল বন্ধ করেছি।"
    enc = test_tokenizer(sample, return_tensors="pt", truncation=True, padding=True)
    ids = enc["input_ids"][0][:10].tolist()
    dec = test_tokenizer.decode(ids, skip_special_tokens=True)
    print(f"   Sample encode/decode OK. Encoded IDs (first 10): {ids}")
    print(f"   Decoded sample (trimmed): {dec[:60]}")

    # Clean up
    del test_tokenizer
    torch.cuda.empty_cache()
    gc.collect()

except Exception as e:
    print(f"❌ mBART-50 tokenizer test failed: {type(e).__name__}: {e}")
    print("   Check internet access or HF model availability and retry.")
    # Re-raise so user sees the full traceback if desired
    raise

# Step 6: Final message
print("\n" + "="*80)
print("ENVIRONMENT SETUP COMPLETE ✅")
print("="*80)
print("Ready for:")
print("  ✅ PATH 1: TATN (DSCD + ASBN + TRG) - Homograph detection")
print("  ✅ PATH 2: mBART-50 M2M - Translation")
print("\nProceed to next cells...")
print("="*80)

In [ ]:
# ==============================================================================
# CELL 0: DUAL-PATH TATN CONFIGURATION — mBART-50 M2M COMPATIBLE
# [KAGGLE T4/L4 OPTIMIZED — ALL FIXES APPLIED]
# ==============================================================================


import os
import sys
import math
import random
import re
import unicodedata
import time
import threading
from pathlib import Path
from collections import deque, defaultdict
from typing import List, Dict, Tuple, Optional, Union, Set, Any
from types import SimpleNamespace


import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import warnings
import gc


try:
    import pandas as pd
    _HAS_PANDAS = True
except ImportError:
    _HAS_PANDAS = False


try:
    from transformers import MBart50TokenizerFast
    _HAS_MBART_TOKENIZER = True
except Exception:
    MBart50TokenizerFast = None
    _HAS_MBART_TOKENIZER = False


warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"]   = "3"


NUM_GPUS      = torch.cuda.device_count() if torch.cuda.is_available() else 0
USE_MULTI_GPU = False


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Cell 0] Single GPU Mode | Multi-GPU: DISABLED | DataParallel: DISABLED")
print(f"[Cell 0] Device: {DEVICE}")


# FIX-C0-1: Detect GPU capability at runtime so T4 → float16, L4/A100 → bfloat16.
# Cell 7 reads AMP_DTYPE to set autocast dtype and decide whether to enable GradScaler.
if torch.cuda.is_available():
    _major, _ = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if _major >= 8 else torch.float16
else:
    AMP_DTYPE = torch.float16

# FIX-C0-2: USE_BF16 derived from AMP_DTYPE (not hardcoded True).
# GradScaler is disabled in Cell 7 when USE_BF16=True (BF16 has wider dynamic range).
USE_BF16 = (AMP_DTYPE == torch.bfloat16)

print(f"[Cell 0] AMP_DTYPE: {AMP_DTYPE} | USE_BF16: {USE_BF16}")


# ==============================================================================
# DATASET PATHS
# ==============================================================================


DATASET_CSV_PATH = os.environ.get(
    "DATASET_PATH",
    "/kaggle/input/datasets/manas00000003/bpcc-merged/merged_bn_en_410k.csv"
)


SRC_COL = "src"
TGT_COL = "tgt"


# ==============================================================================
# CHECKPOINT & PROTOTYPE PATHS
# ==============================================================================

# FIX-C0-4: Removed trailing slash from CHECKPOINT_DIR.
# Original "/kaggle/working/" caused os.path.join to produce double-slash paths
# and mismatched Cell 7's _CHECKPOINT_DIR fallback "/kaggle/working".
# upload_epoch_files_to_gdrive scans this exact path — trailing slash mismatch
# was causing it to silently find no files and skip all uploads.
CHECKPOINT_DIR = "/kaggle/working"

CHECKPOINT_FILENAME                = "tatn_final.pt"
CHECKPOINT_BEST_FILENAME           = "tatn_best.pt"
CHECKPOINT_LATEST_FILENAME         = "tatn_latest.pt"
CHECKPOINT_EPOCH_FILENAME_TEMPLATE = "tatn_epoch{epoch}.pt"


PROTOTYPE_FILENAME                = "dscd_prototypes.pt"
FINAL_PROTOTYPE_FILENAME          = "dscd_prototypes_final.pt"
PROTOTYPE_EPOCH_FILENAME_TEMPLATE = "dscd_prototypes_epoch{epoch}.pt"


PROTOTYPE_PATH  = os.path.join(CHECKPOINT_DIR, PROTOTYPE_FILENAME)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, CHECKPOINT_FILENAME)


MBART_MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"



def _safe_int(value, default: int, name: str, min_val: int = 1) -> int:
    try:
        result = int(value)
        if result < min_val:
            return default
        return result
    except Exception:
        return default



def _safe_float(value, default: float, name: str, min_val: float = 0.0) -> float:
    try:
        result = float(value)
        if result < min_val:
            return default
        return result
    except Exception:
        return default



# ==============================================================================
# CORE TRAINING HYPERPARAMETERS
# ==============================================================================

BATCH_SIZE  = _safe_int(4,     4,     "BATCH_SIZE",  min_val=1)
NUM_SAMPLES = _safe_int(30000, 30000, "NUM_SAMPLES", min_val=1)
MAX_LENGTH  = _safe_int(256,    256,    "MAX_LENGTH",  min_val=8)

LR_NMT     = _safe_float(3e-5, 3e-5, "LR_NMT",     min_val=1e-7)
LR_TRG     = _safe_float(1e-5, 1e-5, "LR_TRG",     min_val=1e-7)
LR_PHI     = _safe_float(1e-4, 1e-4, "LR_PHI",     min_val=1e-7)
LR_ADAPTER = _safe_float(5e-5, 5e-5, "LR_ADAPTER", min_val=1e-7)


EPOCHS         = _safe_int(2,   2,   "EPOCHS",         min_val=1)
GRAD_CLIP_NORM = _safe_float(1.0, 1.0, "GRAD_CLIP_NORM", min_val=0.1)
USE_AMP        = True
PRINT_INTERVAL = _safe_int(1000, 1000, "PRINT_INTERVAL", min_val=1)
SEED           = _safe_int(42,   42,   "SEED",           min_val=0)


TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10


VALIDATION_SPLIT   = _safe_float(0.10, 0.10, "VALIDATION_SPLIT",   min_val=0.01)
ACCUMULATION_STEPS = _safe_int(4, 4, "ACCUMULATION_STEPS", min_val=1)


MC_DROPOUT_PASSES = _safe_int(3,  3,  "MC_DROPOUT_PASSES", min_val=1)
TRG_EVIDENCE_K    = _safe_int(3,  3,  "TRG_EVIDENCE_K",    min_val=1)
MAX_SILVER_BUFFER = _safe_int(50, 50, "MAX_SILVER_BUFFER", min_val=1)


NUM_WORKERS            = _safe_int(1, 1, "NUM_WORKERS", min_val=0)
PIN_MEMORY             = True
PREFETCH_FACTOR        = _safe_int(2, 2, "PREFETCH_FACTOR", min_val=1)
GRADIENT_CHECKPOINTING = True


# ==============================================================================
# DEBUG FLAGS
# ==============================================================================


DEBUG_DISCOVERY = False
DEBUG_TIMING    = True
DEBUG_VERBOSE   = False
VERBOSE_LOGGING = False


# ==============================================================================
# DSCD MODULE CONFIGURATION
# ==============================================================================


DSCD_BUFFER_SIZE                = _safe_int(30,   30,   "DSCD_BUFFER_SIZE",             min_val=1)
DSCD_MAX_PROTOS                 = _safe_int(5,    5,    "DSCD_MAX_PROTOS",              min_val=1)
DSCD_N_MIN                      = _safe_int(3,    3,    "DSCD_N_MIN",                   min_val=1)
DSCD_DISPERSION_THRESHOLD       = _safe_float(0.15, 0.15, "DSCD_DISPERSION_THRESHOLD",  min_val=0.0)
DSCD_NEWSENSE_LAMBDA            = _safe_float(1.2,  1.2,  "DSCD_NEWSENSE_LAMBDA",       min_val=0.1)
DSCD_EMBED_DIM                  = _safe_int(1024, 1024, "DSCD_EMBED_DIM",               min_val=64)
DSCD_TEMPERATURE                = _safe_float(0.7,  0.7,  "DSCD_TEMPERATURE",           min_val=0.01)
DSCD_DROPOUT                    = _safe_float(0.1,  0.1,  "DSCD_DROPOUT",               min_val=0.0)
DSCD_AUGMENT_SCALE              = _safe_float(0.05, 0.05, "DSCD_AUGMENT_SCALE",         min_val=0.0)
DSCD_ENABLE_TRAINING_CLUSTERING = True
DSCD_WARMUP_SAMPLES             = _safe_int(1000, 1000, "DSCD_WARMUP_SAMPLES",          min_val=0)
DSCD_MIN_LETTERS                = _safe_int(2,    2,    "DSCD_MIN_LETTERS",             min_val=1)
DSCD_MIN_LETTER_FRACTION        = _safe_float(0.5, 0.5,  "DSCD_MIN_LETTER_FRACTION",   min_val=0.0)
LAMBDA_DSCD_CONTRASTIVE         = _safe_float(0.1, 0.1,  "LAMBDA_DSCD_CONTRASTIVE",    min_val=0.0)


DSCDWARMUPSAMPLES = DSCD_WARMUP_SAMPLES


PERIODIC_DISCOVERY_FREQUENCY = _safe_int(300, 300, "PERIODIC_DISCOVERY_FREQUENCY", min_val=1)
MAX_TOKENS_PER_DISCOVERY      = _safe_int(100, 100, "MAX_TOKENS_PER_DISCOVERY",     min_val=1)


# ==============================================================================
# MODULE ENABLE FLAGS
# ==============================================================================


ENABLE_ASBN_TRAINING   = False
ENABLE_ASBN_INFERENCE  = False
ENABLE_TRG_TRAINING    = True
ENABLE_TRG_INFERENCE   = True
USE_DUAL_PATH_TRAINING = True


# ==============================================================================
# RUNTIME INTERVALS
# ==============================================================================


CLUSTERING_TIMEOUT        = _safe_int(3,    3,    "CLUSTERING_TIMEOUT",        min_val=1)
MEMORY_CLEANUP_FREQUENCY  = _safe_int(100,  100,  "MEMORY_CLEANUP_FREQUENCY",  min_val=1)
VALIDATION_CHECK_INTERVAL = _safe_int(2000, 2000, "VALIDATION_CHECK_INTERVAL", min_val=1)


# ==============================================================================
# CHECKPOINT CONTROL FLAGS
# ==============================================================================


CHECKPOINT_SAVE_AFTER_TRAINING = True
CHECKPOINT_INTERVAL            = 99999999
SAVE_REPLAY_BUFFER             = False
LOAD_REPLAY_BUFFER             = False
REPLAY_BUFFER_SIZE             = _safe_int(10000, 10000, "REPLAY_BUFFER_SIZE", min_val=0)
RESUME_FROM_CHECKPOINT         = False
SAVE_DSCD_PROTOTYPES           = True


# ==============================================================================
# SAVE_CHECKPOINT_STEPS — DYNAMIC: one save trigger per epoch boundary
# ==============================================================================


_train_samples             = int(NUM_SAMPLES * TRAIN_RATIO)
_optimizer_steps_per_epoch = max(1, (_train_samples // BATCH_SIZE) // ACCUMULATION_STEPS)
_total_optimizer_steps     = _optimizer_steps_per_epoch * EPOCHS


SAVE_CHECKPOINT_STEPS: List[int] = [
    _optimizer_steps_per_epoch * (ep + 1)
    for ep in range(EPOCHS)
]


optimizer_steps_per_epoch = _optimizer_steps_per_epoch
total_optimizer_steps     = _total_optimizer_steps


# ==============================================================================
# THRESHOLD PARAMETERS
# ==============================================================================


TAU_LOW    = _safe_float(0.15, 0.15, "TAU_LOW",    min_val=0.0)
TAU_HIGH   = _safe_float(0.85, 0.85, "TAU_HIGH",   min_val=0.0)
TAU_ACCEPT = _safe_float(0.70, 0.70, "TAU_ACCEPT", min_val=0.0)


TRG_MAX_GEN_LEN               = _safe_int(12, 12, "TRG_MAX_GEN_LEN",                min_val=1)
TRG_GEN_EMBED                 = _safe_int(64, 64, "TRG_GEN_EMBED",                  min_val=8)
TRG_GEN_HID                   = _safe_int(64, 64, "TRG_GEN_HID",                    min_val=8)
TRG_SPAN_THRESHOLD            = _safe_float(0.08, 0.08, "TRG_SPAN_THRESHOLD",        min_val=0.0)
TRG_UNCERTAINTY_THRESHOLD     = _safe_float(0.05, 0.05, "TRG_UNCERTAINTY_THRESHOLD", min_val=0.0)
TRG_TEMPERATURE               = _safe_float(1.0,  1.0,  "TRG_TEMPERATURE",           min_val=0.01)
MAX_EXPLANATIONS_PER_SENTENCE = _safe_int(10, 10, "MAX_EXPLANATIONS_PER_SENTENCE",    min_val=1)


SPAN_THRESHOLD        = _safe_float(0.08, 0.08, "SPAN_THRESHOLD",        min_val=0.0)
UNCERTAINTY_THRESHOLD = _safe_float(0.05, 0.05, "UNCERTAINTY_THRESHOLD", min_val=0.0)


ASBN_HIDDEN_DIM = _safe_int(64,    64,    "ASBN_HIDDEN_DIM", min_val=8)
ASBN_LAMBDA     = _safe_float(0.05, 0.05, "ASBN_LAMBDA",     min_val=0.0)
ASBN_DROPOUT    = _safe_float(0.1,  0.1,  "ASBN_DROPOUT",    min_val=0.0)


# ==============================================================================
# LR SCHEDULER
# ==============================================================================


WARMUP_STEPS             = _safe_int(500,  500,  "WARMUP_STEPS",             min_val=1)
USE_LR_SCHEDULER         = True
EARLY_STOPPING_PATIENCE  = _safe_int(10,  10,  "EARLY_STOPPING_PATIENCE",   min_val=1)
EARLY_STOPPING_MIN_DELTA = _safe_float(0.01, 0.01, "EARLY_STOPPING_MIN_DELTA", min_val=0.0)
LR_SCHEDULER_PATIENCE    = _safe_int(2,   2,   "LR_SCHEDULER_PATIENCE",     min_val=1)
LR_SCHEDULER_FACTOR      = _safe_float(0.5, 0.5, "LR_SCHEDULER_FACTOR",      min_val=0.1)
LR_SCHEDULER_MIN_LR      = _safe_float(1e-6, 1e-6, "LR_SCHEDULER_MIN_LR",   min_val=1e-8)


# ==============================================================================
# LOSS LAMBDAS
# ==============================================================================


LAMBDA_ASBN       = _safe_float(0.0,   0.0,   "LAMBDA_ASBN",       min_val=0.0)
LAMBDA_DSCD       = _safe_float(0.01,  0.01,  "LAMBDA_DSCD",       min_val=0.0)
LAMBDA_TRG        = _safe_float(0.001, 0.001, "LAMBDA_TRG",        min_val=0.0)
LAMBDA_TOKEN      = _safe_float(0.0,   0.0,   "LAMBDA_TOKEN",      min_val=0.0)
LAMBDA_CONFIDENCE = _safe_float(0.0,   0.0,   "LAMBDA_CONFIDENCE", min_val=0.0)
LAMBDA_LENGTH     = _safe_float(0.0,   0.0,   "LAMBDA_LENGTH",     min_val=0.0)
LAMBDA_BT         = _safe_float(1.0,   1.0,   "LAMBDA_BT",         min_val=0.0)


LAMBDA_PATH1_INIT       = _safe_float(10.0,  10.0,  "LAMBDA_PATH1_INIT",       min_val=0.1)
LAMBDA_PATH2_INIT       = _safe_float(1.0,   1.0,   "LAMBDA_PATH2_INIT",       min_val=0.1)
LOSS_BALANCE_EMA_ALPHA  = _safe_float(0.9,   0.9,   "LOSS_BALANCE_EMA_ALPHA",  min_val=0.0)
LOSS_BALANCE_MIN_LAMBDA = _safe_float(1.0,   1.0,   "LOSS_BALANCE_MIN_LAMBDA", min_val=0.1)
LOSS_BALANCE_MAX_LAMBDA = _safe_float(100.0, 100.0, "LOSS_BALANCE_MAX_LAMBDA", min_val=1.0)

LABEL_SMOOTHING = _safe_float(0.1,  0.1,  "LABEL_SMOOTHING", min_val=0.0)
RDROP_ALPHA     = _safe_float(0.3,  0.3,  "RDROP_ALPHA",     min_val=0.0)

# FIX-C0-3: USE_RDROP renamed from USERDROP and set to False.
# Original had USERDROP = True which is the wrong variable name — Cell 7 reads USE_RDROP.
# Must be False: Cell 7 assert crashes if USE_RDROP=True at training start.
USE_RDROP    = False
USERDROP     = USE_RDROP

WEIGHT_DECAY = _safe_float(0.01, 0.01, "WEIGHT_DECAY", min_val=0.0)


# ==============================================================================
# DOMAIN ADVERSARIAL TRAINING
# ==============================================================================


TRAIN_DOMAIN      = 0
TEST_DOMAIN       = 1
USE_DOMAIN_LABELS = True


GRL_ALPHA_START    = _safe_float(0.1, 0.1, "GRL_ALPHA_START", min_val=0.0)
GRL_ALPHA_END      = _safe_float(1.0, 1.0, "GRL_ALPHA_END",   min_val=0.0)
GRL_ALPHA_SCHEDULE = "linear"
GRL_ALPHA_STEPS    = _safe_int(500, 500, "GRL_ALPHA_STEPS", min_val=1)


# ==============================================================================
# LANGUAGE & VOCABULARY CONFIGURATION
# ==============================================================================


SOURCE_LANGUAGE = "bn_IN"
TARGET_LANGUAGE = "en_XX"


MBART50_BN_TOKEN_ID = 250028
MBART50_EN_TOKEN_ID = 250004
MBART50_VOCAB_SIZE  = 250054


FREEZE_ENCODER        = False
FREEZE_FIRST_N_LAYERS = 0


# ==============================================================================
# EVALUATION CONFIGURATION
# ==============================================================================


EVAL_BATCH_SIZE           = _safe_int(8,   8,   "EVAL_BATCH_SIZE",           min_val=1)
EVAL_NUM_BEAMS            = _safe_int(8,   8,   "EVAL_NUM_BEAMS",            min_val=1)
EVAL_LENGTH_PENALTY       = _safe_float(0.7, 0.7, "EVAL_LENGTH_PENALTY",     min_val=0.0)
EVAL_REPETITION_PENALTY   = _safe_float(1.3, 1.3, "EVAL_REPETITION_PENALTY", min_val=1.0)
EVAL_MIN_LENGTH           = _safe_int(4,   4,   "EVAL_MIN_LENGTH",           min_val=1)
EVAL_NO_REPEAT_NGRAM_SIZE = _safe_int(4,   4,   "EVAL_NO_REPEAT_NGRAM_SIZE", min_val=1)


BLEU_EVAL_SAMPLES = _safe_int(500, 500, "BLEU_EVAL_SAMPLES", min_val=100)
BLEU_SEED         = _safe_int(42,  42,  "BLEU_SEED",         min_val=0)


# ==============================================================================
# NOVEL COMPONENT FLAGS — SCEA + HFCAA
# ==============================================================================


USE_CONTEXT_GATES        = True
CONTEXT_GATE_TEMPERATURE = 0.7


USE_SENSE_CONDITIONING = True
SENSE_BLEND_ALPHA      = 0.3


USE_ATTENTION_ENTROPY_PENALTY = True
LAMBDA_ATTENTION_ENTROPY      = 0.01


USE_COVERAGE    = True
LAMBDA_COVERAGE = 0.1


USE_CONTEXT_BEAM_SEARCH = False
CONTEXT_BEAM_ALPHA      = 0.2


USE_POSITION_GUIDANCE   = True
POSITION_GUIDANCE_ALPHA = 0.3


USE_NTREX_DATASET = False


SCEA_BOTTLENECK = 128
SCEA_MAX_SENSES = 5


HAUG_GATE_QUALITY_THRESHOLD   = 0.02
HAUG_GATE_HOMOGRAPH_THRESHOLD = 0.03
PROTO_SEPARATION_THRESHOLD    = 0.30
PROTO_MIN_COUNT               = 5


# ==============================================================================
# BACK-TRANSLATION CONFIGURATION
# ==============================================================================


USE_BACK_TRANSLATION  = True
BT_MONOLINGUAL_SAMPLE = _safe_int(50000, 50000, "BT_MONOLINGUAL_SAMPLE", min_val=0)
BT_GEN_BATCH_SIZE     = _safe_int(16,    16,    "BT_GEN_BATCH_SIZE",     min_val=1)
BT_NUM_BEAMS          = _safe_int(4,     4,     "BT_NUM_BEAMS",          min_val=1)
BT_MIX_RATIO          = _safe_float(0.3, 0.3,  "BT_MIX_RATIO",          min_val=0.0)
BT_NOISE_DROPOUT      = _safe_float(0.1, 0.1,  "BT_NOISE_DROPOUT",      min_val=0.0)
BT_MIN_LENGTH         = _safe_int(4,     4,     "BT_MIN_LENGTH",         min_val=1)
BT_MAX_NEW_TOKENS     = _safe_int(128,   128,   "BT_MAX_NEW_TOKENS",     min_val=8)
BT_CACHE_ENABLED      = True
BT_CACHE_PATH         = os.path.join(CHECKPOINT_DIR, "bt_cache.pt")


# ==============================================================================
# HOMOGRAPH REFERENCE & WATCHLISTS
# ==============================================================================


HOMOGRAPH_REFERENCE_LIST_BN: Set[str] = {
    "কল", "কাল", "পাতা", "ফল",   "বার",  "হার",  "তারা",
    "পড়া", "দেখা", "চলা", "ধরা",  "অর্থ", "শব্দ", "মুখ",
    "তোলা", "বাঁচা", "মারা", "উত্তর", "পাত্র", "বেলা", "গান",
    "নাম",  "বল",   "চাল", "কলা",  "ধারা", "পত্র", "রাগ",  "রস",
    "তীর",  "জমা",  "মান", "দাবি", "আসন",  "সাড়া", "বসা",  "পদ",
    "অংশ", "মোড়",  "ঘর",  "মন",   "ব্যাংক"
}


HOMOGRAPH_WATCHLIST_BN: Set[str] = set()
HOMOGRAPH_WATCHLIST: Set[str]    = set()
USE_WATCHLIST_PRIORITIZATION     = False
WATCHLIST_ONLY_FOR_TRG           = False


_GLOBAL_VAL_PAIRS: List = []
GLOBALVALPAIRS = _GLOBAL_VAL_PAIRS


# ==============================================================================
# TEXT NORMALIZATION UTILITIES
# ==============================================================================


def normalize_bengali(t: str) -> str:
    if not t:
        return ""
    t = unicodedata.normalize("NFKC", t)
    t = t.replace("▁", "").replace("##", "").strip()
    return t



def normalize_english(t: str) -> str:
    if not t or not isinstance(t, str):
        return ""
    t = unicodedata.normalize("NFKC", t).strip()
    return t



# ==============================================================================
# GPU / CUDA UTILITIES
# ==============================================================================


def empty_cuda_cache() -> None:
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass



def safe_cuda_synchronize() -> None:
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass



def monitor_gpu_usage() -> None:
    if torch.cuda.is_available():
        visible_gpus = torch.cuda.device_count()
        print(f"\n[GPU MONITOR] Checking {visible_gpus} GPU(s):")
        for i in range(visible_gpus):
            try:
                mem_alloc    = torch.cuda.memory_allocated(i) / (1024 ** 3)
                mem_reserved = torch.cuda.memory_reserved(i) / (1024 ** 3)
                print(f"  GPU {i}: {mem_alloc:.2f}GB allocated / {mem_reserved:.2f}GB reserved")
            except Exception:
                pass



# ==============================================================================
# CHECKPOINT PATH HELPERS
# ==============================================================================


def get_checkpoint_path() -> str:
    return os.path.join(CHECKPOINT_DIR, CHECKPOINT_FILENAME)



def get_best_checkpoint_path() -> str:
    return os.path.join(CHECKPOINT_DIR, CHECKPOINT_BEST_FILENAME)



def get_latest_checkpoint_path() -> str:
    return os.path.join(CHECKPOINT_DIR, CHECKPOINT_LATEST_FILENAME)



def get_epoch_checkpoint_path(epoch: int) -> str:
    return os.path.join(CHECKPOINT_DIR, CHECKPOINT_EPOCH_FILENAME_TEMPLATE.format(epoch=epoch))



def get_epoch_prototype_path(epoch: int) -> str:
    return os.path.join(CHECKPOINT_DIR, PROTOTYPE_EPOCH_FILENAME_TEMPLATE.format(epoch=epoch))



def get_final_prototype_path() -> str:
    return os.path.join(CHECKPOINT_DIR, FINAL_PROTOTYPE_FILENAME)



def should_save_checkpoint(global_step: int, epoch: int, is_final: bool = False) -> bool:
    if is_final and CHECKPOINT_SAVE_AFTER_TRAINING:
        return True
    if global_step in SAVE_CHECKPOINT_STEPS:
        return True
    if epoch > 0 and global_step > 0 and global_step == _optimizer_steps_per_epoch * epoch:
        return True
    return False



# ==============================================================================
# THREADING TIMEOUT UTILITY
# ==============================================================================


class FunctionTimeoutError(Exception):
    pass



def with_timeout(seconds: int):
    def decorator(func):
        def wrapper(*args, **kwargs):
            result = [FunctionTimeoutError("Function timed out")]
            def target():
                try:
                    result[0] = func(*args, **kwargs)
                except Exception as e:
                    result[0] = e
            thread = threading.Thread(target=target, daemon=True)
            thread.start()
            thread.join(timeout=seconds)
            if thread.is_alive():
                return None
            if isinstance(result[0], Exception):
                if isinstance(result[0], FunctionTimeoutError):
                    return None
                raise result[0]
            return result[0]
        return wrapper
    return decorator



# ==============================================================================
# TOKEN VALIDATION UTILITIES
# ==============================================================================


def get_special_tokens(tokenizer) -> Set[str]:
    try:
        s = set(getattr(tokenizer, "all_special_tokens", []))
    except Exception:
        s = {"<pad>", "</s>", "<s>", "<unk>"}
    s.update({SOURCE_LANGUAGE, TARGET_LANGUAGE})
    return s



_token_validation_cache: Dict[Tuple[str, str], bool] = {}
_cache_lock     = threading.Lock()
_cache_max_size = 5000



def is_valid_token(
    token,
    special_tokens: Optional[Set[str]] = None,
    tokenizer=None,
    language: str = "bn",
) -> bool:
    token = "" if token is None else str(token)
    cache_key = (token, language)
    with _cache_lock:
        if cache_key in _token_validation_cache:
            return _token_validation_cache[cache_key]

    clean = token.replace("▁", "").replace("Ġ", "").replace("##", "").strip()

    if special_tokens and token in special_tokens:
        result = False
    elif len(clean) < 2:
        result = False
    else:
        has_bengali_chars = any("\u0980" <= c <= "\u09FF" for c in clean)
        if not has_bengali_chars:
            result = False
        else:
            bengali_count  = sum(1 for c in clean if "\u0980" <= c <= "\u09FF")
            alphanum_count = sum(1 for c in clean if c.isalnum())
            if alphanum_count == 0:
                result = False
            else:
                result = (bengali_count / alphanum_count) >= 0.5

    with _cache_lock:
        if len(_token_validation_cache) < _cache_max_size:
            _token_validation_cache[cache_key] = result
    return result



# ==============================================================================
# DISCOVERY TIMER
# ==============================================================================


class DiscoveryTimer:
    def __init__(self):
        self.discovery_times: List[float] = []
        self.discovery_steps: List[int]   = []


    def record(self, step: int, duration: float) -> None:
        self.discovery_times.append(duration)
        self.discovery_steps.append(step)


    def get_stats(self) -> Dict[str, float]:
        if not self.discovery_times:
            return {"count": 0, "total": 0.0, "avg": 0.0, "max": 0.0}
        total = sum(self.discovery_times)
        return {
            "count": len(self.discovery_times),
            "total": total,
            "avg":   total / len(self.discovery_times),
            "max":   max(self.discovery_times),
        }



_discovery_timer = DiscoveryTimer()
discoverytimer   = _discovery_timer


# ==============================================================================
# REPRODUCIBILITY
# ==============================================================================


torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


if hasattr(torch, "set_float32_matmul_precision"):
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass


torch.backends.cudnn.benchmark     = True
torch.backends.cudnn.deterministic = False


# ==============================================================================
# DERIVED TRAINING GEOMETRY
# ==============================================================================


effective_batch = BATCH_SIZE * ACCUMULATION_STEPS
warmup_pct      = (WARMUP_STEPS / _total_optimizer_steps * 100) if _total_optimizer_steps > 0 else 0.0


_gpu_name = "Unknown"
if torch.cuda.is_available():
    try:
        _gpu_name = torch.cuda.get_device_name(0)
    except Exception:
        pass


# ==============================================================================
# CONFIGURATION SUMMARY PRINT
# ==============================================================================


print("\n" + "=" * 80)
print("DUAL-PATH TATN CONFIGURATION — mBART-50 M2M [KAGGLE T4/L4 OPTIMIZED]")
print("=" * 80)
print(f"User:             {os.getenv('KAGGLE_USERNAME', os.getenv('USER', 'manas0003'))}")
print(f"GPU:              {_gpu_name}")
print(f"AMP dtype:        {AMP_DTYPE}")
print(f"BF16:             {'ENABLED' if USE_BF16 else 'DISABLED (T4 — using float16)'}")
print(f"GradScaler:       {'DISABLED (BF16 path)' if USE_BF16 else 'ENABLED (float16 path)'}")
print(f"Multi-GPU:        DISABLED ({NUM_GPUS} GPU)")
print(f"DataParallel:     DISABLED")
print(f"Device:           {DEVICE}")
print(f"Dataset:          {DATASET_CSV_PATH}")
print(f"CSV columns:      src={SRC_COL} (Bengali), tgt={TGT_COL} (English)")
print(f"Samples:          {NUM_SAMPLES:,} | Batch: {BATCH_SIZE} | Accum: {ACCUMULATION_STEPS}")
print(f"Effective batch:  {effective_batch}")
print(f"Split ratios:     Train={TRAIN_RATIO:.0%} / Val={VAL_RATIO:.0%} / Test={TEST_RATIO:.0%}")
print(f"Validation split: {VALIDATION_SPLIT:.0%}")
print(f"Max length:       {MAX_LENGTH} | Epochs: {EPOCHS}")
print(f"Workers:          {NUM_WORKERS} | PinMemory: {PIN_MEMORY}")
print()
print("PATH 1 (TATN — Homograph Detection):")
print(f"  - DSCD discovery freq:     {PERIODIC_DISCOVERY_FREQUENCY} steps")
print(f"  - ASBN:                    {'DISABLED' if not ENABLE_ASBN_TRAINING else 'ENABLED'}")
print(f"  - TRG:                     {'ENABLED' if ENABLE_TRG_TRAINING else 'DISABLED'}")
print(f"  - LAMBDA_DSCD:             {LAMBDA_DSCD}")
print(f"  - LAMBDA_DSCD_CONTRASTIVE: {LAMBDA_DSCD_CONTRASTIVE}")
print(f"  - LAMBDA_TRG:              {LAMBDA_TRG}")
print()
print("PATH 2 (mBART-50 M2M):")
print(f"  - Model:                   {MBART_MODEL_NAME}")
print(f"  - Vocab size:              {MBART50_VOCAB_SIZE:,}")
print(f"  - Languages:               {SOURCE_LANGUAGE} -> {TARGET_LANGUAGE}")
print(f"  - Token IDs:               BN={MBART50_BN_TOKEN_ID}, EN={MBART50_EN_TOKEN_ID}")
print(f"  - LR NMT backbone:         {LR_NMT}")
print(f"  - LR SCEA/HFCAA adapter:   {LR_ADAPTER}")
print(f"  - LR PHI (ASBN critic):    {LR_PHI}")
print(f"  - Label smoothing:         {LABEL_SMOOTHING}")
print(f"  - R-Drop:                  DISABLED (USE_RDROP={USE_RDROP}, alpha={RDROP_ALPHA})")
print(f"  - Weight decay:            {WEIGHT_DECAY}")
print(f"  - Warmup steps:            {WARMUP_STEPS} ({warmup_pct:.1f}% of {_total_optimizer_steps} total steps)")
print()
print("BACK-TRANSLATION:")
print(f"  - USE_BACK_TRANSLATION:    {'ENABLED' if USE_BACK_TRANSLATION else 'DISABLED'}")
print(f"  - BT_MONOLINGUAL_SAMPLE:   {BT_MONOLINGUAL_SAMPLE:,}")
print(f"  - BT_GEN_BATCH_SIZE:       {BT_GEN_BATCH_SIZE}")
print(f"  - BT_NUM_BEAMS:            {BT_NUM_BEAMS}")
print(f"  - BT_MIX_RATIO:            {BT_MIX_RATIO}")
print(f"  - BT_NOISE_DROPOUT:        {BT_NOISE_DROPOUT}")
print(f"  - LAMBDA_BT:               {LAMBDA_BT}")
print(f"  - BT_CACHE_ENABLED:        {BT_CACHE_ENABLED}")
print(f"  - BT_CACHE_PATH:           {BT_CACHE_PATH}")
print()
print("EXPECTED IMPROVEMENT (20hr run):")
print(f"  - Baseline BLEU:  16.44")
print(f"  - Target range:   26-34 BLEU (single-ref) | 36-41 BLEU (multi-ref)")
print(f"  - Training:       {_total_optimizer_steps:,} optimizer steps "
      f"({_optimizer_steps_per_epoch}/epoch x {EPOCHS} epochs)")
print(f"  - Epoch checkpoint steps (dynamic): {SAVE_CHECKPOINT_STEPS}")
print()
print("TOXIC PENALTIES REMOVED:")
print(f"  - LAMBDA_TOKEN:      0.25 -> {LAMBDA_TOKEN}      (DISABLED)")
print(f"  - LAMBDA_CONFIDENCE: 0.15 -> {LAMBDA_CONFIDENCE} (DISABLED)")
print(f"  - LAMBDA_LENGTH:     0.05 -> {LAMBDA_LENGTH}     (DISABLED)")
print(f"  - LAMBDA_ASBN:       0.3  -> {LAMBDA_ASBN}       (DISABLED)")
print()
print("EVALUATION SETTINGS:")
print(f"  - Batch size:          {EVAL_BATCH_SIZE}")
print(f"  - Num beams:           {EVAL_NUM_BEAMS}")
print(f"  - Length penalty:      {EVAL_LENGTH_PENALTY}")
print(f"  - No-repeat ngram:     {EVAL_NO_REPEAT_NGRAM_SIZE}")
print(f"  - Repetition penalty:  {EVAL_REPETITION_PENALTY}")
print()
print("NOVEL COMPONENTS [SCEA + HFCAA]:")
print(f"  - SCEA_BOTTLENECK:               {SCEA_BOTTLENECK}")
print(f"  - SCEA_MAX_SENSES:               {SCEA_MAX_SENSES}")
print(f"  - HAUG_GATE_QUALITY_THRESHOLD:   {HAUG_GATE_QUALITY_THRESHOLD}")
print(f"  - HAUG_GATE_HOMOGRAPH_THRESHOLD: {HAUG_GATE_HOMOGRAPH_THRESHOLD}")
print()
print("CHECKPOINT & PROTOTYPE PATHS:")
print(f"  - CHECKPOINT_DIR:                    {CHECKPOINT_DIR}")
print(f"  - CHECKPOINT_FILENAME (final):        {CHECKPOINT_FILENAME}")
print(f"  - CHECKPOINT_BEST_FILENAME:           {CHECKPOINT_BEST_FILENAME}")
print(f"  - CHECKPOINT_LATEST_FILENAME:         {CHECKPOINT_LATEST_FILENAME}")
print(f"  - CHECKPOINT_EPOCH_FILENAME_TEMPLATE: {CHECKPOINT_EPOCH_FILENAME_TEMPLATE}")
print(f"  - PROTOTYPE_FILENAME:                 {PROTOTYPE_FILENAME}")
print(f"  - FINAL_PROTOTYPE_FILENAME:           {FINAL_PROTOTYPE_FILENAME}")
print(f"  - PROTOTYPE_EPOCH_FILENAME_TEMPLATE:  {PROTOTYPE_EPOCH_FILENAME_TEMPLATE}")
print(f"  - SAVE_DSCD_PROTOTYPES:               {SAVE_DSCD_PROTOTYPES}")
print(f"  - RESUME_FROM_CHECKPOINT:             {RESUME_FROM_CHECKPOINT}")
print(f"  - CHECKPOINT_PATH (resume):           {CHECKPOINT_PATH}")
print()
print("VERIFY BEFORE RUNNING TRAINING:")
print("  core = model.module if hasattr(model,'module') else model")
print("  print(core.dscd.get_prototype_summary())")
print("  -> If 'quality_score' key exists -> HAUG_GATE fix in Cell 7 is safe")
print("  -> If key is missing -> skip Cell 7 HAUG fix, leave gate threshold only")
print()
monitor_gpu_usage()


# ==============================================================================
# DYNAMIC LANGUAGE TOKEN ID DETECTION
# ==============================================================================


if _HAS_MBART_TOKENIZER and MBart50TokenizerFast is not None:
    try:
        print("[Cell 0] Detecting language token IDs...")
        _temp_tokenizer = MBart50TokenizerFast.from_pretrained(
            MBART_MODEL_NAME,
            src_lang=SOURCE_LANGUAGE,
            tgt_lang=TARGET_LANGUAGE,
        )
        MBART50_BN_TOKEN_ID = _temp_tokenizer.convert_tokens_to_ids(f"{SOURCE_LANGUAGE}")
        MBART50_EN_TOKEN_ID = _temp_tokenizer.convert_tokens_to_ids(f"{TARGET_LANGUAGE}")
        print(f"[Cell 0] Detected language token IDs:")
        print(f"         {SOURCE_LANGUAGE}: {MBART50_BN_TOKEN_ID}")
        print(f"         {TARGET_LANGUAGE}: {MBART50_EN_TOKEN_ID}")
        del _temp_tokenizer
    except Exception as e:
        print(f"[Cell 0] Could not detect language tokens, using defaults: {e}")
        MBART50_BN_TOKEN_ID = 250028
        MBART50_EN_TOKEN_ID = 250004
else:
    print("[Cell 0] MBart50TokenizerFast not available, using hardcoded token IDs")
    MBART50_BN_TOKEN_ID = 250028
    MBART50_EN_TOKEN_ID = 250004


print("\n" + "=" * 80)
print("Cell 0: CONFIG — Ready | Run Cell 1 next")
print("=" * 80)


In [ ]:
# ===========================================================================================
# CELL 1: DUAL-PATH TOKENIZER UTILITIES + TRAINING LOSSES
# ===========================================================================================

import threading
from typing import Tuple, List, Dict, Optional, Set, Union
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F   # [IMP-1] Already present — confirmed ✅
import re

# ── Read Cell 0 constants safely ──────────────────────────────────────────────
try:
    if isinstance(MAX_LENGTH, (int, float)) and MAX_LENGTH > 0:
        SAFE_OFFSET_MAX_LEN = int(MAX_LENGTH)
    else:
        SAFE_OFFSET_MAX_LEN = 48
except (NameError, ValueError, TypeError):
    SAFE_OFFSET_MAX_LEN = 48

try:
    _SOURCE_LANG = SOURCE_LANGUAGE
except NameError:
    _SOURCE_LANG = "bn_IN"

try:
    _TARGET_LANG = TARGET_LANGUAGE
except NameError:
    _TARGET_LANG = "en_XX"

try:
    _DEBUG_VERBOSE = DEBUG_VERBOSE
except NameError:
    _DEBUG_VERBOSE = False

try:
    _DEBUG_DISCOVERY = DEBUG_DISCOVERY
except NameError:
    _DEBUG_DISCOVERY = False

try:
    _MBART_BN_TOKEN_ID = MBART50_BN_TOKEN_ID
except NameError:
    _MBART_BN_TOKEN_ID = 9

try:
    _MBART_EN_TOKEN_ID = MBART50_EN_TOKEN_ID
except NameError:
    _MBART_EN_TOKEN_ID = 2

try:
    _MBART_VOCAB_SIZE = MBART50_VOCAB_SIZE
except NameError:
    _MBART_VOCAB_SIZE = 250054

try:
    _DSCD_MIN_LETTERS = int(DSCD_MIN_LETTERS)
except NameError:
    _DSCD_MIN_LETTERS = 3

try:
    _DSCD_MIN_LETTER_FRACTION = float(DSCD_MIN_LETTER_FRACTION)
except NameError:
    _DSCD_MIN_LETTER_FRACTION = 0.6

try:
    _LABEL_SMOOTHING_EPS = float(LABEL_SMOOTHING)
except NameError:
    _LABEL_SMOOTHING_EPS = 0.1

# [BUG-C1-1] FIX: was 5.0 — Cell 0 now sets RDROP_ALPHA=0.0 and USE_RDROP=False.
# If NameError fires, 5.0 would silently enable R-Drop in RDropLoss.
# Correct fallback is 0.0 to match Cell 0 intent.
try:
    _RDROP_ALPHA = float(RDROP_ALPHA)
except NameError:
    _RDROP_ALPHA = 0.0   # was 5.0 — FIXED

# [MINOR-1] Read USE_RDROP flag from Cell 0 so RDropLoss has a gate
try:
    _USE_RDROP = bool(USE_RDROP)
except NameError:
    _USE_RDROP = False   # default disabled — matches Cell 0

# ── Module-level caches ───────────────────────────────────────────────────────
_SPECIAL_TOKENS_CACHE: Dict[str, Set[str]] = {}
_SPECIAL_TOKENS_LOCK  = threading.Lock()
_LANGUAGE_WARNING_COUNT = 0
_MAX_LANGUAGE_WARNINGS  = 3
_VOCAB_SIZE_CACHE: Dict[str, int] = {}

# [BUG-C1-4] NOTE: _token_validation_cache and _cache_lock are intentionally
# REDEFINED here from Cell 0. Cell 1's is_valid_token is the CANONICAL version
# (uses correct Unicode "\u0980" chars, not literal strings).
# _cache_max_size: Cell 0 = 5000, Cell 1 = 10000 (larger for full training).
# Cell 1's definitions WIN — this is intentional and documented here.
_token_validation_cache: Dict[Tuple[str, str], bool] = {}
_cache_lock     = threading.Lock()
_cache_max_size = 10000


# ==============================================================================
# BengaliWordTokenizer — Path 1 (word-level tokenizer)
# ==============================================================================

class BengaliWordTokenizer:
    def __init__(self, vocab_size: int = 50000):
        self.vocab_size = vocab_size
        self.word_to_id: Dict[str, int] = {"<pad>": 0, "<unk>": 1, "<s>": 2, "</s>": 3}
        self.id_to_word: Dict[int, str] = {0: "<pad>", 1: "<unk>", 2: "<s>", 3: "</s>"}
        self.next_id = 4
        self._lock = threading.Lock()

        self.pad_token = "<pad>"
        self.unk_token = "<unk>"
        self.bos_token = "<s>"
        self.eos_token = "</s>"
        self.pad_token_id = 0
        self.unk_token_id = 1
        self.bos_token_id = 2
        self.eos_token_id = 3

        self.bengali_pattern = re.compile(r'[\u0980-\u09FF]+')
        self.punct_pattern   = re.compile(r'[।॥,.;:!?"\'\-\(\)\[\]{}]')

    def tokenize(self, text: str) -> List[str]:
        if not text or not isinstance(text, str):
            return []
        text = text.strip()
        if not text:
            return []

        words  = []
        tokens = re.findall(
            r'[\u0980-\u09FF]+|[a-zA-Z]+|[0-9]+|[।॥]|[,.;:!?"\'\-\(\)\[\]{}]',
            text
        )
        for token in tokens:
            token = token.strip()
            if token:
                words.append(token)
        return words

    def encode(
        self,
        text: Union[str, List[str]],
        add_special_tokens: bool = True,
        max_length: Optional[int] = None,
        padding: bool = False,
        truncation: bool = False,
        return_tensors: Optional[str] = None,
    ) -> Dict[str, Union[List[int], torch.Tensor]]:
        if isinstance(text, str):
            texts = [text]
        else:
            texts = text

        all_input_ids      = []
        all_attention_masks = []

        for txt in texts:
            words = self.tokenize(txt)

            with self._lock:
                ids = []
                for word in words:
                    if word not in self.word_to_id:
                        if self.next_id < self.vocab_size:
                            self.word_to_id[word]          = self.next_id
                            self.id_to_word[self.next_id]  = word
                            self.next_id += 1
                            ids.append(self.word_to_id[word])
                        else:
                            ids.append(self.unk_token_id)
                    else:
                        ids.append(self.word_to_id[word])

            if add_special_tokens:
                ids = [self.bos_token_id] + ids + [self.eos_token_id]

            if truncation and max_length:
                ids = ids[:max_length]

            attention_mask = [1] * len(ids)

            all_input_ids.append(ids)
            all_attention_masks.append(attention_mask)

        if padding and max_length:
            for i in range(len(all_input_ids)):
                if len(all_input_ids[i]) < max_length:
                    pad_len = max_length - len(all_input_ids[i])
                    all_input_ids[i]       = all_input_ids[i]       + [self.pad_token_id] * pad_len
                    all_attention_masks[i] = all_attention_masks[i] + [0] * pad_len

        if return_tensors == "pt":
            max_len = max(len(ids) for ids in all_input_ids)
            for i in range(len(all_input_ids)):
                if len(all_input_ids[i]) < max_len:
                    pad_len = max_len - len(all_input_ids[i])
                    all_input_ids[i]       = all_input_ids[i]       + [self.pad_token_id] * pad_len
                    all_attention_masks[i] = all_attention_masks[i] + [0] * pad_len
            return {
                "input_ids":      torch.tensor(all_input_ids,       dtype=torch.long),
                "attention_mask": torch.tensor(all_attention_masks, dtype=torch.long),
            }

        if len(all_input_ids) == 1:
            return {
                "input_ids":      all_input_ids[0],
                "attention_mask": all_attention_masks[0],
            }

        return {
            "input_ids":      all_input_ids,
            "attention_mask": all_attention_masks,
        }

    def decode(
        self,
        token_ids: Union[List[int], torch.Tensor],
        skip_special_tokens: bool = True,
    ) -> str:
        if isinstance(token_ids, torch.Tensor):
            token_ids = token_ids.tolist()
        words = []
        for tid in token_ids:
            if tid in self.id_to_word:
                word = self.id_to_word[tid]
                if skip_special_tokens and word in {"<pad>", "<unk>", "<s>", "</s>"}:
                    continue
                words.append(word)
        return " ".join(words)

    def convert_ids_to_tokens(self, ids: Union[List[int], torch.Tensor]) -> List[str]:
        if isinstance(ids, torch.Tensor):
            ids = ids.tolist()
        return [self.id_to_word.get(tid, self.unk_token) for tid in ids]

    def convert_tokens_to_ids(self, tokens: Union[str, List[str]]) -> Union[int, List[int]]:
        if isinstance(tokens, str):
            return self.word_to_id.get(tokens, self.unk_token_id)
        return [self.word_to_id.get(tok, self.unk_token_id) for tok in tokens]

    def __call__(self, text: Union[str, List[str]], **kwargs):
        return self.encode(text, **kwargs)

    def __len__(self):
        return len(self.word_to_id)


# ==============================================================================
# LabelSmoothingLoss — Path 2 (mBART training)
# ==============================================================================

class LabelSmoothingLoss(nn.Module):
    def __init__(
        self,
        num_classes: int,
        smoothing: float = 0.1,
        ignore_index: int = -100,
    ):
        super().__init__()
        self.num_classes   = num_classes
        self.smoothing     = smoothing
        self.ignore_index  = ignore_index
        self.confidence    = 1.0 - smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if logits.dim() == 3:
            logits = logits.reshape(-1, logits.size(-1))
        if targets.dim() == 2:
            targets = targets.reshape(-1)

        mask    = targets != self.ignore_index
        targets = targets.masked_select(mask)
        logits  = logits[mask]

        if targets.numel() == 0:
            return torch.tensor(0.0, device=logits.device, requires_grad=True)

        log_probs = F.log_softmax(logits, dim=-1)

        nll_loss    = -log_probs.gather(dim=-1, index=targets.unsqueeze(1)).squeeze(1)
        smooth_loss = -log_probs.mean(dim=-1)

        loss = self.confidence * nll_loss + self.smoothing * smooth_loss
        return loss.mean()


# ==============================================================================
# RDropLoss — Path 2 (disabled; USE_RDROP=False in Cell 0)
# ==============================================================================

class RDropLoss(nn.Module):
    # [BUG-C1-2] FIX: was alpha=5.0 — default inconsistent with Cell 0's RDROP_ALPHA=0.0.
    # If instantiated without argument anywhere, it would use 5.0 silently.
    # Changed default to 0.0 to match Cell 0.
    # Note: USE_RDROP=False in Cell 0 — this class is defined but not called in training.
    def __init__(self, alpha: float = 0.0):   # was 5.0 — FIXED
        super().__init__()
        self.alpha = alpha

    def forward(
        self,
        logits1: torch.Tensor,
        logits2: torch.Tensor,
        targets: torch.Tensor,
        ignore_index: int = -100,
    ) -> torch.Tensor:
        if logits1.dim() == 3:
            logits1 = logits1.reshape(-1, logits1.size(-1))
        if logits2.dim() == 3:
            logits2 = logits2.reshape(-1, logits2.size(-1))
        if targets.dim() == 2:
            targets = targets.reshape(-1)

        mask    = targets != ignore_index
        logits1 = logits1[mask]
        logits2 = logits2[mask]

        if logits1.numel() == 0:
            return torch.tensor(0.0, device=logits1.device, requires_grad=True)

        p1 = F.log_softmax(logits1, dim=-1)
        p2 = F.log_softmax(logits2, dim=-1)

        p1_probs = F.softmax(logits1, dim=-1)
        p2_probs = F.softmax(logits2, dim=-1)

        kl_1_2 = F.kl_div(p1, p2_probs, reduction='batchmean', log_target=False)
        kl_2_1 = F.kl_div(p2, p1_probs, reduction='batchmean', log_target=False)

        kl_loss = (kl_1_2 + kl_2_1) / 2.0
        return self.alpha * kl_loss


# ==============================================================================
# Tokenizer utility helpers
# ==============================================================================

def _special_token_cache_key(tokenizer) -> str:
    name = getattr(tokenizer, "name_or_path", None) or getattr(tokenizer, "name", None)
    if not name:
        name = "unknown_tokenizer"
    vocab = None
    if hasattr(tokenizer, "vocab_size"):
        try:
            vocab = int(getattr(tokenizer, "vocab_size"))
        except Exception:
            vocab = None
    elif hasattr(tokenizer, "get_vocab") and callable(getattr(tokenizer, "get_vocab")):
        try:
            vocab = len(tokenizer.get_vocab())
        except Exception:
            vocab = None
    return f"{name}__vocab={vocab}"


def get_tokenizer_vocab_size(tokenizer) -> int:
    cache_key  = _special_token_cache_key(tokenizer)
    if cache_key in _VOCAB_SIZE_CACHE:
        return _VOCAB_SIZE_CACHE[cache_key]

    vocab_size = _MBART_VOCAB_SIZE
    try:
        if hasattr(tokenizer, "__len__"):
            vocab_size = len(tokenizer)
        elif hasattr(tokenizer, "vocab_size"):
            vocab_size = int(tokenizer.vocab_size)
        elif hasattr(tokenizer, "get_vocab"):
            vocab_size = len(tokenizer.get_vocab())
    except Exception:
        pass

    _VOCAB_SIZE_CACHE[cache_key] = vocab_size
    return vocab_size


def get_tokenizer_special_tokens(tokenizer) -> Set[str]:
    cache_key = _special_token_cache_key(tokenizer)
    with _SPECIAL_TOKENS_LOCK:
        if cache_key in _SPECIAL_TOKENS_CACHE:
            return _SPECIAL_TOKENS_CACHE[cache_key]

        special_tokens: Set[str] = set()
        try:
            if hasattr(tokenizer, "all_special_tokens"):
                try:
                    result = getattr(tokenizer, "all_special_tokens")
                    if isinstance(result, (list, tuple, set)):
                        special_tokens.update(x for x in result if x)
                except Exception:
                    pass
            if hasattr(tokenizer, "additional_special_tokens"):
                try:
                    result = getattr(tokenizer, "additional_special_tokens")
                    if isinstance(result, (list, tuple, set)):
                        special_tokens.update(x for x in result if x)
                except Exception:
                    pass
            for attr in ("pad_token", "unk_token", "bos_token", "eos_token",
                         "cls_token", "sep_token", "mask_token"):
                if hasattr(tokenizer, attr):
                    try:
                        tok = getattr(tokenizer, attr)
                        if tok:
                            special_tokens.add(tok)
                    except Exception:
                        pass
            try:
                stm = (
                    getattr(tokenizer, "special_tokens_map", None)
                    or getattr(tokenizer, "special_tokens_map_extended", None)
                )
                if isinstance(stm, dict):
                    for v in stm.values():
                        if isinstance(v, str) and v:
                            special_tokens.add(v)
            except Exception:
                pass
        except Exception:
            special_tokens = set()

        special_tokens.update({
            f"__{_SOURCE_LANG}__", f"__{_TARGET_LANG}__",
            "</s>", "<pad>", "<s>", "<unk>",
            "[PAD]", "[EOS]", "[UNK]", "[CLS]", "[SEP]", "[MASK]",
        })

        try:
            vocab = tokenizer.get_vocab() if hasattr(tokenizer, "get_vocab") else {}
            special_tokens = {
                tok
                for tok in special_tokens
                if tok in vocab or tok in {"</s>", "<pad>", "<s>", "<unk>"}
            }
        except Exception:
            pass

        _SPECIAL_TOKENS_CACHE[cache_key] = special_tokens
        return special_tokens


def get_cached_special_tokens(tokenizer) -> Set[str]:
    return get_tokenizer_special_tokens(tokenizer)


def _normalize_offset_mapping_for_batchencoding(enc):
    try:
        if "offset_mapping" in enc and enc["offset_mapping"] is not None:
            off = enc["offset_mapping"]
            try:
                if hasattr(off, "tolist"):
                    arr = off.tolist()
                    if isinstance(arr, list) and len(arr) > 0 and isinstance(arr[0], list):
                        enc["offset_mapping"] = [
                            (x[0], x[1])
                            if (isinstance(x, (list, tuple)) and len(x) >= 2)
                            else (None, None)
                            for x in arr[0]
                        ]
                        return enc
                if isinstance(off, (list, tuple)):
                    if len(off) > 0 and isinstance(off[0], (list, tuple)):
                        enc["offset_mapping"] = [
                            (x[0], x[1])
                            if (isinstance(x, (list, tuple)) and len(x) >= 2)
                            else (None, None)
                            for x in off[0]
                        ]
                        return enc
            except Exception:
                pass
    except Exception:
        pass

    try:
        data = getattr(enc, "data", None)
        if (
            data
            and isinstance(data, dict)
            and "offset_mapping" in data
            and data["offset_mapping"] is not None
        ):
            om = data["offset_mapping"]
            if isinstance(om, (list, tuple)) and len(om) > 0 and isinstance(om[0], (list, tuple)):
                enc["offset_mapping"] = [
                    (x[0], x[1])
                    if (isinstance(x, (list, tuple)) and len(x) >= 2)
                    else (None, None)
                    for x in om[0]
                ]
                return enc
    except Exception:
        pass

    try:
        seq_len = 0
        if "input_ids" in enc:
            input_ids = enc["input_ids"]
            if hasattr(input_ids, "shape") and len(input_ids.shape) > 0:
                seq_len = int(input_ids.shape[-1])
            elif (
                isinstance(input_ids, (list, tuple))
                and len(input_ids) > 0
                and isinstance(input_ids[0], (list, tuple))
            ):
                seq_len = len(input_ids[0])
        enc["offset_mapping"] = [(None, None)] * seq_len
    except Exception:
        enc["offset_mapping"] = []

    return enc


def safe_offsets_tokenize(
    tokenizer,
    text: str,
    max_length: Optional[int] = None,
    include_special_tokens: bool = False,
) -> dict:
    if max_length is None:
        max_length = SAFE_OFFSET_MAX_LEN
    eff_max = int(max_length)

    try:
        if not isinstance(text, str):
            text = "" if text is None else str(text)
    except Exception:
        if _DEBUG_VERBOSE:
            print("[WARN] Failed to convert input to string, using empty string")
        text = ""

    char_limit  = min(eff_max * 30, 8000)
    sample_text = text[:char_limit]
    is_fast     = getattr(tokenizer, "is_fast", False)
    vocab_size  = get_tokenizer_vocab_size(tokenizer)

    tokenize_kwargs = {
        "return_tensors":    "pt",
        "truncation":        True,
        "padding":           False,
        "max_length":        eff_max,
        "add_special_tokens": include_special_tokens,
    }

    try:
        if hasattr(tokenizer, 'src_lang'):
            tokenizer.src_lang = _SOURCE_LANG
    except Exception:
        pass

    if is_fast:
        try:
            tokenize_kwargs["return_offsets_mapping"] = True
            enc = tokenizer(sample_text, **tokenize_kwargs)
            enc = _normalize_offset_mapping_for_batchencoding(enc)
            if "input_ids" in enc and isinstance(enc["input_ids"], torch.Tensor):
                enc["input_ids"] = torch.clamp(enc["input_ids"], 0, vocab_size - 1)
            return enc
        except Exception:
            pass

    try:
        enc = tokenizer(sample_text, **tokenize_kwargs)
        if "input_ids" in enc and isinstance(enc["input_ids"], torch.Tensor):
            enc["input_ids"] = torch.clamp(enc["input_ids"], 0, vocab_size - 1)
    except Exception as e:
        if _DEBUG_VERBOSE:
            print(f"[WARN] Tokenization failed: {e}, returning empty encoding")
        pad_id = getattr(tokenizer, "pad_token_id", 0)
        enc = {
            "input_ids":      torch.tensor([[pad_id]], dtype=torch.long),
            "attention_mask": torch.tensor([[1]],      dtype=torch.long),
        }
        enc = _normalize_offset_mapping_for_batchencoding(enc)
        return enc

    try:
        input_ids = None
        try:
            input_ids = enc["input_ids"][0].tolist()
        except Exception:
            if hasattr(enc, "data") and "input_ids" in enc.data:
                input_ids = enc.data["input_ids"][0]

        tokens: List[str] = []
        if input_ids is not None:
            try:
                tokens = tokenizer.convert_ids_to_tokens(input_ids)
            except Exception:
                tokens = []

        offsets_list: List[Tuple[Optional[int], Optional[int]]] = []
        src     = sample_text
        cur_pos = 0
        for tok in tokens:
            token_text = (tok or "").replace("▁", "").replace("##", "").replace("Ġ", "").strip()
            if not token_text:
                offsets_list.append((None, None))
                continue
            idx = src.find(token_text, cur_pos)
            if idx == -1:
                idx = src.lower().find(token_text.lower(), cur_pos)
            if idx == -1:
                offsets_list.append((None, None))
            else:
                start = int(idx)
                end   = int(idx + len(token_text))
                offsets_list.append((start, end))
                cur_pos = end

        enc["offset_mapping"] = offsets_list
        enc = _normalize_offset_mapping_for_batchencoding(enc)
        return enc
    except Exception:
        enc = _normalize_offset_mapping_for_batchencoding(enc)
        return enc


def reconstruct_word_spans(
    tokenizer,
    text: str,
    max_length: Optional[int] = None,
) -> Tuple[Dict[int, str], List[str]]:
    global _LANGUAGE_WARNING_COUNT

    if max_length is None:
        max_length = SAFE_OFFSET_MAX_LEN
    eff_max = int(max_length)

    if not isinstance(text, str) or len(text.strip()) == 0:
        return {}, []

    has_bengali = any("\u0980" <= c <= "\u09FF" for c in text)
    has_english = any("a" <= c.lower() <= "z"   for c in text)

    if _DEBUG_VERBOSE and _DEBUG_DISCOVERY:
        bengali_pct = (
            sum(1 for c in text if "\u0980" <= c <= "\u09FF")
            / max(1, len(text))
            * 100.0
        )
        print(f"[TOKENIZER] Text sample: {text[:50]}")
        print(
            f"[TOKENIZER] Bengali: {has_bengali} ({bengali_pct:.1f}%), "
            f"English: {has_english}"
        )

    if not has_bengali and has_english and _LANGUAGE_WARNING_COUNT < _MAX_LANGUAGE_WARNINGS:
        if _DEBUG_DISCOVERY:
            print("[TOKENIZER WARNING] Text appears to be ENGLISH, not BENGALI")
            print(f"  Sample: {text[:80]}")
        _LANGUAGE_WARNING_COUNT += 1
        if _LANGUAGE_WARNING_COUNT == _MAX_LANGUAGE_WARNINGS:
            print("[TOKENIZER] Suppressing further language warnings")

    char_limit = min(eff_max * 30, 8000)
    text       = text[:char_limit]
    text_len   = len(text)

    special_tokens = get_tokenizer_special_tokens(tokenizer)
    vocab_size     = get_tokenizer_vocab_size(tokenizer)

    # [BUG-C1-3] FIX: removed `current_lang = SOURCE_LANGUAGE` block that was here.
    # `current_lang` was assigned but never used anywhere in this function — pure dead code.
    # Removed the entire try/except block to clean up.

    try:
        encoded = safe_offsets_tokenize(
            tokenizer, text, max_length=eff_max, include_special_tokens=False
        )
    except Exception:
        return {}, []

    offsets = encoded.get("offset_mapping", [])
    try:
        input_ids = encoded["input_ids"][0].tolist()
        input_ids = [min(max(0, tid), vocab_size - 1) for tid in input_ids]
    except Exception:
        input_ids = []
    try:
        tokens = tokenizer.convert_ids_to_tokens(input_ids) if input_ids else []
    except Exception:
        tokens = []

    if isinstance(offsets, list) and len(offsets) > 0 and all(
        isinstance(x, tuple) for x in offsets
    ):
        offsets_list = offsets
    elif isinstance(offsets, list) and len(offsets) > 0 and isinstance(
        offsets[0], (list, tuple)
    ):
        offsets_list = [
            (x[0], x[1])
            if (isinstance(x, (list, tuple)) and len(x) >= 2)
            else (None, None)
            for x in offsets[0]
        ]
    else:
        offsets_list = [(None, None)] * len(tokens)

    token_word_map: Dict[int, str] = {}
    words: List[str] = []

    used_any_offset = any(
        isinstance(o, tuple) and o[0] is not None and o[1] is not None
        for o in offsets_list
    )
    if used_any_offset:
        word_start: Optional[int] = None
        word_end:   Optional[int] = None
        current_accumulated_word  = ""

        for idx, (off, tok) in enumerate(zip(offsets_list, tokens)):
            try:
                off_start = int(off[0]) if off[0] is not None else None
                off_end   = int(off[1]) if off[1] is not None else None
            except Exception:
                off_start, off_end = None, None

            if off_start is not None and off_end is not None:
                if off_start < 0 or off_end < 0:
                    if _DEBUG_VERBOSE:
                        print(
                            f"[WARN] Negative offset detected: "
                            f"({off_start}, {off_end}), skipping"
                        )
                    off_start, off_end = None, None
                else:
                    off_start = max(0, min(off_start, text_len))
                    off_end   = max(off_start, min(off_end, text_len))

            if off_start is None or off_end is None:
                if current_accumulated_word:
                    token_word_map[idx] = current_accumulated_word
                if word_start is not None and word_end is not None:
                    try:
                        wtext = text[word_start:word_end].strip()
                        if wtext:
                            words.append(wtext)
                    except Exception:
                        pass
                word_start = None
                word_end   = None
                continue

            if tok in special_tokens:
                continue

            if word_start is None:
                word_start = off_start
                word_end   = off_end
            else:
                if off_start > word_end:
                    try:
                        wtext = text[word_start:word_end].strip()
                        if wtext:
                            words.append(wtext)
                    except Exception:
                        pass
                    word_start = off_start
                    word_end   = off_end
                else:
                    word_end = max(word_end, off_end)

            try:
                current_word = text[word_start:word_end].strip()
                if current_word:
                    token_word_map[idx]          = current_word
                    current_accumulated_word     = current_word
            except Exception:
                pass

        if word_start is not None and word_end is not None:
            try:
                wtext = text[word_start:word_end].strip()
                if wtext:
                    words.append(wtext)
            except Exception:
                pass

        if token_word_map:
            words = [w for w in words if isinstance(w, str) and w.strip()]
            return token_word_map, words

    # ── Fallback 1: SentencePiece ▁ / Ġ markers ──────────────────────────────
    token_word_map = {}
    assembled:      List[str] = []
    current_parts:  List[str] = []
    running_word    = ""
    max_word_len    = 100

    for i, tok in enumerate(tokens):
        if tok in special_tokens:
            continue
        clean = (tok or "").replace("▁", "").replace("Ġ", "").replace("##", "").strip()
        if not clean:
            continue

        if tok.startswith("▁") or tok.startswith("Ġ"):
            if current_parts:
                word = "".join(current_parts)
                if len(word) <= max_word_len:
                    assembled.append(word)
            current_parts = [clean]
            running_word  = clean
        else:
            current_parts.append(clean)
            running_word = "".join(current_parts)
            if len(running_word) > max_word_len:
                if current_parts[:-1]:
                    word = "".join(current_parts[:-1])
                    assembled.append(word)
                current_parts = [clean]
                running_word  = clean

        if running_word:
            token_word_map[i] = running_word

    if current_parts:
        word = "".join(current_parts)
        if len(word) <= max_word_len:
            assembled.append(word)

    if token_word_map:
        words = [w for w in assembled if w and w.strip()]
        return token_word_map, words

    # ── Fallback 2: word-boundary scan ───────────────────────────────────────
    try:
        words_from_markers: List[str] = []
        current_word_parts: List[str] = []

        for tok in tokens:
            if tok in special_tokens:
                continue
            clean = (tok or "").replace("▁", "").replace("Ġ", "").replace("##", "").strip()
            if not clean:
                continue
            if tok.startswith("▁") or tok.startswith("Ġ"):
                if current_word_parts:
                    words_from_markers.append("".join(current_word_parts))
                current_word_parts = [clean]
            else:
                current_word_parts.append(clean)

        if current_word_parts:
            words_from_markers.append("".join(current_word_parts))

        word_list = words_from_markers if words_from_markers else [
            w for w in text.split() if w.strip()
        ]

        token_word_map = {}
        if tokens and word_list:
            word_idx = 0
            for i, tok in enumerate(tokens):
                clean = (tok or "").replace("▁", "").replace("Ġ", "").replace("##", "").strip()
                if not clean or tok in special_tokens:
                    continue
                if word_idx < len(word_list):
                    current_word = word_list[word_idx]
                    if clean in current_word or current_word.startswith(clean):
                        token_word_map[i] = current_word
                    else:
                        word_idx = min(word_idx + 1, len(word_list) - 1)
                        token_word_map[i] = word_list[word_idx]
                else:
                    if word_list:
                        token_word_map[i] = word_list[-1]

        return token_word_map, word_list
    except Exception:
        return {}, []


# ==============================================================================
# Token validation — CANONICAL version (overrides Cell 0 definition)
# Uses correct Python Unicode escapes "\u0980" not literal strings
# ==============================================================================

def is_valid_token(
    token,
    special_tokens: Optional[Set[str]] = None,
    tokenizer=None,
    language: str = "bn",
) -> bool:
    token     = "" if token is None else str(token)
    cache_key = (token, language)
    with _cache_lock:
        if cache_key in _token_validation_cache:
            return _token_validation_cache[cache_key]

    # SentencePiece / WordPiece prefix cleanup — uses real Unicode chars ✅
    clean = token.replace("▁", "").replace("Ġ", "").replace("##", "").strip()

    if special_tokens and token in special_tokens:
        result = False
    else:
        if len(clean) < _DSCD_MIN_LETTERS:
            result = False
        else:
            # Proper Python Unicode escape: Bengali range U+0980–U+09FF
            has_bengali_chars = any("\u0980" <= c <= "\u09FF" for c in clean)
            if not has_bengali_chars:
                result = False
            else:
                bengali_count  = sum(1 for c in clean if "\u0980" <= c <= "\u09FF")
                alphanum_count = sum(1 for c in clean if c.isalnum())
                if alphanum_count == 0:
                    result = False
                else:
                    result = (bengali_count / alphanum_count) >= _DSCD_MIN_LETTER_FRACTION

    with _cache_lock:
        if len(_token_validation_cache) < _cache_max_size:
            _token_validation_cache[cache_key] = result
    return result


def should_track_token(
    token: str,
    special_tokens: Optional[Set[str]] = None,
    tokenizer=None,
    language: str = "bn",
) -> bool:
    return is_valid_token(token, special_tokens, tokenizer, language)


# ==============================================================================
# Tokenizer validation utilities
# ==============================================================================

def validate_tokenizer_vocab(tokenizer, expected_vocab_size: Optional[int] = None) -> bool:
    actual_vocab_size = get_tokenizer_vocab_size(tokenizer)
    print(f"[TOKENIZER-VALIDATION] Actual vocab size: {actual_vocab_size}")

    if expected_vocab_size is not None:
        if actual_vocab_size != expected_vocab_size:
            print(f"[TOKENIZER-VALIDATION] ❌ MISMATCH: Expected {expected_vocab_size}, got {actual_vocab_size}")
            return False
        else:
            print(f"[TOKENIZER-VALIDATION] ✅ Vocab size matches: {actual_vocab_size}")
            return True

    bn_token_str = f"__{_SOURCE_LANG}__"
    en_token_str = f"__{_TARGET_LANG}__"

    try:
        bn_id = tokenizer.convert_tokens_to_ids(bn_token_str)
        en_id = tokenizer.convert_tokens_to_ids(en_token_str)

        print(f"[TOKENIZER-VALIDATION] Language tokens:")
        print(f"  {bn_token_str} → {bn_id}")
        print(f"  {en_token_str} → {en_id}")

        if bn_id >= actual_vocab_size or en_id >= actual_vocab_size:
            print(f"[TOKENIZER-VALIDATION] ❌ Language token IDs exceed vocab size!")
            return False

        if expected_vocab_size is None:
            try:
                if bn_id != _MBART_BN_TOKEN_ID or en_id != _MBART_EN_TOKEN_ID:
                    print(f"[TOKENIZER-VALIDATION] ⚠️  Language token IDs differ from Cell 0:")
                    print(f"  Expected: bn={_MBART_BN_TOKEN_ID}, en={_MBART_EN_TOKEN_ID}")
                    print(f"  Got: bn={bn_id}, en={en_id}")
                    print(f"  → Update Cell 0 with correct values")
            except NameError:
                pass

        print(f"[TOKENIZER-VALIDATION] ✅ Language tokens valid")
        return True

    except Exception as e:
        print(f"[TOKENIZER-VALIDATION] ❌ Language token validation failed: {e}")
        return False


def test_tokenizer_utilities_quick(tokenizer=None) -> bool:
    sample_bn = "কাল আমি বাজারে যাব।"
    sample_en = "Tomorrow I will go to the market."

    print("\n" + "=" * 60)
    print("TOKENIZER UTILITIES TEST")
    print("=" * 60)

    try:
        if tokenizer is None:
            print("No tokenizer provided: skipping test")
            return True

        print("\n[TEST 0] Vocabulary validation:")
        validate_tokenizer_vocab(tokenizer)

        print("\n[TEST 1] Bengali text processing:")
        print(f"  Input: {sample_bn}")
        enc_bn = safe_offsets_tokenize(
            tokenizer, sample_bn, max_length=32, include_special_tokens=False
        )
        enc_len = (
            int(enc_bn["input_ids"].shape[-1])
            if isinstance(enc_bn, dict) and "input_ids" in enc_bn
            else "N/A"
        )
        print(f"  Encoded length: {enc_len}")
        offsets_bn = enc_bn.get("offset_mapping") or []
        print(f"  Offsets (first 5): {offsets_bn[:5]}")

        token_map_bn, words_bn = reconstruct_word_spans(tokenizer, sample_bn, max_length=32)
        print(f"  Reconstructed words: {words_bn}")
        print(f"  Token map sample: {dict(list(token_map_bn.items())[:3])}")

        has_bengali_words = any(
            any("\u0980" <= c <= "\u09FF" for c in w) for w in words_bn
        )
        print(f"  Contains Bengali words: {has_bengali_words}")

        print("\n[TEST 2] English text processing (should show warning):")
        print(f"  Input: {sample_en}")
        token_map_en, words_en = reconstruct_word_spans(tokenizer, sample_en, max_length=32)
        print(f"  Reconstructed words: {words_en}")
        has_english_words = any(
            any("a" <= c.lower() <= "z" for c in w) for w in words_en
        )
        print(f"  Contains English words: {has_english_words}")

        print("\n[TEST 3] Token validation:")
        special_tokens = get_tokenizer_special_tokens(tokenizer)
        test_tokens = ["কাল", "▁আমি", "</s>", "##ing", "a"]
        for tok in test_tokens:
            valid = is_valid_token(tok, special_tokens, tokenizer, "bn")
            print(f"  '{tok}': {'valid' if valid else 'invalid'}")

        print("\n[TEST 4] mBART-50 language setting:")
        try:
            if hasattr(tokenizer, 'src_lang'):
                tokenizer.src_lang = "bn_IN"
                print("  ✅ tokenizer.src_lang = 'bn_IN' successful")
            else:
                print("  ⚠️  tokenizer.src_lang attribute not found")
        except Exception as e:
            print(f"  ❌ Language setting failed: {e}")

        if has_bengali_words and not any(
            "a" <= c.lower() <= "z" for c in "".join(words_bn)
        ):
            print("\nTest PASSED: Bengali processing works correctly")
            return True
        else:
            print("\nTest WARNING: Check language detection logic")
            return False

    except Exception as e:
        print(f"\nTest FAILED: {repr(e)}")
        import traceback
        traceback.print_exc()
        return False
    finally:
        print("=" * 60 + "\n")


# ── Aliases (snake_case ↔ camelCase compatibility) ────────────────────────────
safeoffsetstokenize       = safe_offsets_tokenize
reconstructwordspans      = reconstruct_word_spans
gettokenizerspecialtokens = get_tokenizer_special_tokens
getcachedspecialtokens    = get_cached_special_tokens
isvalidtoken              = is_valid_token
shouldtracktoken          = should_track_token
gettokenizervocabsize     = get_tokenizer_vocab_size
validatetokenizervocab    = validate_tokenizer_vocab

# ── Summary ───────────────────────────────────────────────────────────────────
print("=" * 80)
print("Cell 1: DUAL-PATH Tokenizer Utilities + Training Losses - READY")
print("=" * 80)
print("VERIFICATION:")
print(f"  ✅ DSCD_MIN_LETTERS:         {_DSCD_MIN_LETTERS}")
print(f"  ✅ DSCD_MIN_LETTER_FRACTION: {_DSCD_MIN_LETTER_FRACTION}")
print(f"  ✅ Token validation cache:   {_cache_max_size} entries max")
print(f"  ✅ mBART-50 language tokens: bn_IN={_MBART_BN_TOKEN_ID}, en_XX={_MBART_EN_TOKEN_ID}")
print(f"  ✅ mBART-50 vocab size:      {_MBART_VOCAB_SIZE:,}")
print(f"  ✅ Label Smoothing epsilon:  {_LABEL_SMOOTHING_EPS}")
# [MINOR-2] Shows DISABLED label when USE_RDROP=False — was misleading before
print(f"  ✅ R-Drop alpha:             {_RDROP_ALPHA}  "
      f"({'DISABLED' if not _USE_RDROP else 'ENABLED'})")
print("\nDUAL-PATH COMPONENTS:")
print("  ✅ BengaliWordTokenizer     — Path 1 (word-level)")
print("  ✅ mBART-50 utilities       — Path 2 (subword)")
print("  ✅ LabelSmoothingLoss       — Path 2 training")
print(f"  ✅ RDropLoss                — Path 2 regularization [{'DISABLED' if not _USE_RDROP else 'ENABLED'}]")
print("=" * 80 + "\n")


In [ ]:
# ==============================================================================
# CELL 2: DUAL-PATH DATA LOADING - WORD + SUBWORD TOKENIZATION
# ==============================================================================

from typing import Optional, List, Tuple, Dict, Any
from collections import defaultdict
import os
import time
import random
import traceback
import re

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, get_worker_info
from tqdm import tqdm

try:
    import pandas as pd
    _HAS_PANDAS = True
except ImportError:
    pd = None
    _HAS_PANDAS = False
    print("[CELL2] WARNING: pandas not available; CSV loading will fail!")

try:
    from datasets import load_dataset
    _HAS_DATASETS = True
except Exception:
    load_dataset = None
    _HAS_DATASETS = False

try:
    _VERBOSE_LOGGING = bool(VERBOSE_LOGGING)
except NameError:
    _VERBOSE_LOGGING = False

try:
    _DEBUG_VERBOSE = bool(DEBUG_VERBOSE)
except NameError:
    _DEBUG_VERBOSE = False

try:
    _DEBUG_DISCOVERY = bool(DEBUG_DISCOVERY)
except NameError:
    _DEBUG_DISCOVERY = False

DEBUG_CELL2 = bool(_VERBOSE_LOGGING) or bool(_DEBUG_VERBOSE) or bool(_DEBUG_DISCOVERY)
DEBUG_LIMIT  = 10
_cell2_dbg_counts: Dict[str, int] = defaultdict(int)


def cell2_dbg(key: str, msg: str, limit: int = DEBUG_LIMIT) -> None:
    if not DEBUG_CELL2:
        return
    _cell2_dbg_counts[key] += 1
    if _cell2_dbg_counts[key] <= limit:
        print(f"[CELL2-DBG] {msg}")


try:
    _NUM_SAMPLES = int(NUM_SAMPLES)
except Exception:
    _NUM_SAMPLES = 70000
    print(f"[CELL2] WARNING: NUM_SAMPLES not defined, using default {_NUM_SAMPLES}")

try:
    _MAX_LENGTH = int(MAX_LENGTH)
except Exception:
    _MAX_LENGTH = 64
    print("[CELL2] WARNING: MAX_LENGTH not defined, using default 64")

try:
    _SOURCE_LANG = str(SOURCE_LANGUAGE)
    _TARGET_LANG = str(TARGET_LANGUAGE)
except NameError:
    _SOURCE_LANG = "bn_IN"
    _TARGET_LANG = "en_XX"
    print("[CELL2] WARNING: SOURCE_LANGUAGE/TARGET_LANGUAGE not defined, using defaults bn_IN/en_XX")

try:
    _MBART_BN_TOKEN_ID = int(MBART50_BN_TOKEN_ID)
    _MBART_EN_TOKEN_ID = int(MBART50_EN_TOKEN_ID)
except NameError:
    _MBART_BN_TOKEN_ID = 250028
    _MBART_EN_TOKEN_ID = 250004
    print("[CELL2] WARNING: mBART-50 token IDs not defined, using defaults")

try:
    _MBART_VOCAB_SIZE = int(MBART50_VOCAB_SIZE)
except NameError:
    _MBART_VOCAB_SIZE = 250054
    print("[CELL2] WARNING: mBART-50 vocab size not defined, using default 250054")

try:
    _NUM_GPUS      = int(NUM_GPUS)
    _USE_MULTI_GPU = bool(USE_MULTI_GPU)
except NameError:
    _NUM_GPUS      = 0
    _USE_MULTI_GPU = False
    print("[CELL2] WARNING: GPU config not defined, defaulting to single GPU mode")

try:
    _NUM_WORKERS = int(NUM_WORKERS)
except NameError:
    _NUM_WORKERS = 0
    print("[CELL2] WARNING: NUM_WORKERS not defined, using 0")

try:
    _PIN_MEMORY = bool(PIN_MEMORY)
except NameError:
    _PIN_MEMORY = False

try:
    _PREFETCH_FACTOR = int(PREFETCH_FACTOR)
except NameError:
    _PREFETCH_FACTOR = 1

try:
    _DATASET_CSV_PATH = str(DATASET_CSV_PATH)
except NameError:
    _DATASET_CSV_PATH = "/kaggle/input/datasets/manas00000003/bpcc-dataset/bpcc_ben_eng.csv"
    print(f"[CELL2] WARNING: DATASET_CSV_PATH not defined, using default: {_DATASET_CSV_PATH}")

# CSV schema: src=English, tgt=Bengali
try:
    _SRC_COL = str(SRC_COL)
except NameError:
    _SRC_COL = "src"

try:
    _TGT_COL = str(TGT_COL)
except NameError:
    _TGT_COL = "tgt"

try:
    _TRAIN_DOMAIN      = int(TRAIN_DOMAIN)
    _TEST_DOMAIN       = int(TEST_DOMAIN)
    _USE_DOMAIN_LABELS = bool(USE_DOMAIN_LABELS)
except NameError:
    _TRAIN_DOMAIN      = 0
    _TEST_DOMAIN       = 1
    _USE_DOMAIN_LABELS = True
    print("[CELL2] WARNING: Domain label config not found, using defaults (train=0, test=1)")

try:
    _TRAIN_RATIO = float(TRAIN_RATIO)
    _VAL_RATIO   = float(VAL_RATIO)
    _TEST_RATIO  = float(TEST_RATIO)
except NameError:
    _TRAIN_RATIO = 0.80
    _VAL_RATIO   = 0.10
    _TEST_RATIO  = 0.10
    print("[CELL2] WARNING: Split ratios not defined, using 80/10/10")

try:
    _SEED = int(SEED)
except NameError:
    _SEED = 42

# FIX-WARN-1: Warn only when PERIODIC_DISCOVERY_FREQUENCY is TOO SMALL (< 10),
# not when it is large. Cell 0 intentionally sets 300 for the full 70K dataset.
try:
    _PERIODIC_DISCOVERY_FREQUENCY = int(PERIODIC_DISCOVERY_FREQUENCY)
    if _PERIODIC_DISCOVERY_FREQUENCY < 10:
        print(
            f"[CELL2] WARNING: PERIODIC_DISCOVERY_FREQUENCY={_PERIODIC_DISCOVERY_FREQUENCY} "
            f"is very small — discovery will run every {_PERIODIC_DISCOVERY_FREQUENCY} steps "
            f"which may significantly slow training. Consider raising to 50–300 in Cell 0."
        )
except NameError:
    _PERIODIC_DISCOVERY_FREQUENCY = 20
    print(f"[CELL2] WARNING: PERIODIC_DISCOVERY_FREQUENCY not defined, using default {_PERIODIC_DISCOVERY_FREQUENCY}")

_has_normalize              = ("normalize_bengali" in globals()) and ("normalize_english" in globals())
_has_reconstruct_word_spans = "reconstruct_word_spans" in globals()
_has_safe_offsets_tokenize  = "safe_offsets_tokenize" in globals()

if not _has_normalize:
    print("[CELL2] WARNING: normalize_bengali/normalize_english not found; using simple .strip()")

_BENGALI_CHAR_RE   = re.compile(r"[\u0980-\u09FF]")
_BENGALI_PUNCT_SET = set(['।', '॥'])
_COMMON_PUNCT_SET  = set(['.', ',', ';', ':', '!', '?', '"', "'", '-', '(', ')', '[', ']', '{', '}'])


def is_bengali_text(s: str) -> bool:
    if s is None:
        return False
    if not isinstance(s, str) or not s:
        return False
    return bool(_BENGALI_CHAR_RE.search(s))


def separate_bengali_punctuation(text: str, language: str = "bn") -> str:
    if not text or not isinstance(text, str):
        return ""
    text = text.strip()
    if language in ["bn", "hi", "te", "ta", "ml", "mr", "gu", "pa"]:
        text = re.sub(r'([।॥])', r' \1 ', text)
    text = re.sub(r'([,;:!?()\[\]{}])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def clean_and_normalize_text(text: str, language: str = "bn") -> str:
    if not text or not isinstance(text, str):
        return ""
    text = text.strip()
    if not text:
        return ""
    text = separate_bengali_punctuation(text, language)
    if _has_normalize:
        if language in ["bn", "bn_IN"]:
            text = normalize_bengali(text)
        else:
            text = normalize_english(text)
    else:
        text = text.strip()
    return text


def is_punctuation_only(token: str) -> bool:
    if not token or not isinstance(token, str):
        return False
    clean = token.replace("▁", "").replace("Ġ", "").replace("##", "").strip()
    if not clean:
        return False
    if clean in _BENGALI_PUNCT_SET:
        return True
    if clean in _COMMON_PUNCT_SET:
        return True
    if len(clean) == 1 and not clean.isalnum():
        return True
    return all(c in _BENGALI_PUNCT_SET or c in _COMMON_PUNCT_SET for c in clean)


def _dataloader_worker_init_fn(worker_id: int) -> None:
    worker_info = get_worker_info()
    dataset     = worker_info.dataset if worker_info is not None else None
    try:
        if dataset is not None and hasattr(dataset, "_tokenizer_name_or_path"):
            name_or_path = dataset._tokenizer_name_or_path
            if name_or_path:
                try:
                    from transformers import MBart50TokenizerFast
                    dataset.tokenizer  = MBart50TokenizerFast.from_pretrained(name_or_path)
                    dataset.is_fast    = getattr(dataset.tokenizer, "is_fast", False)
                    dataset.vocab_size = len(dataset.tokenizer)
                    try:
                        dataset.tokenizer.src_lang = _SOURCE_LANG
                    except Exception:
                        pass
                    if DEBUG_CELL2:
                        print(f"[CELL2-WORKER-{worker_id}] Tokenizer reloaded (vocab={dataset.vocab_size})")
                except Exception as e:
                    cell2_dbg("worker_tokenizer_reload", f"Worker {worker_id} tokenizer reload failed: {e}")
                    dataset.tokenizer  = None
                    dataset.is_fast    = False
                    dataset.vocab_size = _MBART_VOCAB_SIZE
    except Exception:
        if DEBUG_CELL2:
            print(f"[CELL2-WORKER-INIT] Tokenizer rebind failed in worker {worker_id}")

    try:
        base = int(os.environ.get("PYTHONHASHSEED", "0"))
        seed = (base ^ (worker_id + 1) ^ int(time.time())) & 0xFFFFFFFF
        random.seed(seed)
        np.random.seed(seed % (2**31 - 1))
        torch.manual_seed(seed % (2**31 - 1))
    except Exception:
        pass


def _detect_columns(df) -> Tuple[Optional[str], Optional[str]]:
    """
    Returns (bn_col, en_col).
    CSV schema: src=English, tgt=Bengali → bn_col="tgt", en_col="src"
    """
    bn_candidates = ["tgt", "bn", "ben", "bengali", "source_bn", "text_bn", "text_bengali", "sentence_bn"]
    en_candidates = ["src", "en", "eng", "english", "source_en", "text_en", "text_english", "sentence_en",
                     "target", "output"]

    cols   = list(df.columns)
    bn_col = next((c for c in bn_candidates if c in cols), None)
    en_col = next((c for c in en_candidates if c in cols), None)

    if bn_col is None or en_col is None or bn_col == en_col:
        obj_cols = [c for c in cols if df[c].dtype == object]
        if len(obj_cols) >= 2:
            sample_size = min(10, len(df))
            c0_bengali  = sum(1 for v in df[obj_cols[0]].head(sample_size) if is_bengali_text(str(v)))
            c1_bengali  = sum(1 for v in df[obj_cols[1]].head(sample_size) if is_bengali_text(str(v)))
            if c0_bengali > c1_bengali:
                bn_col = obj_cols[0]
                en_col = obj_cols[1]
            else:
                bn_col = obj_cols[1]
                en_col = obj_cols[0]
            print(f"[CELL2] Auto-detected columns via heuristic: bn='{bn_col}', en='{en_col}'")
        else:
            print(f"[CELL2] ERROR: Cannot detect columns. Available columns: {cols}")
            return None, None

    print(f"[CELL2] Using columns: bn='{bn_col}', en='{en_col}'")
    return bn_col, en_col


def _get_fallback_dataset(n: int = 70) -> List[Tuple[str, str]]:
    print(f"[CELL2] Using fallback dataset (requested n={n})")
    base_pairs = [
        ("আমি কল বন্ধ করেছি",                "I turned off the tap"),
        ("সে আমাকে পরে কল করবে",              "He will call me later"),
        ("আমরা প্রতিদিন তাজা ফল খাই",          "We eat fresh fruits every day"),
        ("তার কঠোর পরিশ্রমের ভালো ফল হয়েছে",  "His hard work has brought good results"),
        ("গাছে নতুন পাতাগুলো গজিয়েছে",         "New leaves have sprouted on the tree"),
        ("আমি বইয়ের পাতা উল্টাচ্ছি",           "I am turning the pages of the book"),
        ("কাল আমি বাজারে গিয়েছিলাম",           "Yesterday I went to the market"),
        ("কাল আমি তোমার সাথে দেখা করব",        "Tomorrow I will meet you"),
        ("তারা আকাশে উজ্জ্বল",                 "The stars are bright in the sky"),
        ("তারা বাড়িতে নেই",                    "They are not at home"),
        ("ব্যাংক নদীর ধারে ভেঙে গেছে",          "The bank by the river has collapsed"),
        ("আমি ব্যাংকে টাকা জমা দিয়েছি",         "I deposited money in the bank"),
        ("বার বার চেষ্টা করতে হবে",             "You have to try again and again"),
        ("আমি বার খুলে ভিতরে ঢুকলাম",           "I opened the bar and entered"),
        ("তার মাথা ব্যথা করছে",                 "His head is hurting"),
        ("আমি মাথা নেড়ে সম্মতি দিলাম",          "I nodded my head in agreement"),
        ("সে হার মেনে নিয়েছে",                  "He accepted defeat"),
        ("আমি গলায় সোনার হার পরেছি",            "I am wearing a gold necklace"),
        ("পানি খুব ঠান্ডা",                     "The water is very cold"),
        ("আমি পানি খাচ্ছি",                     "I am drinking water"),
        ("দল খেলায় জিতেছে",                    "The team won the game"),
        ("বাজার থেকে সবজি কিনলাম",              "I bought vegetables from the market"),
        ("বাজার অনেক ভিড় ছিল",                 "The market was very crowded"),
        ("তার নাম আহমেদ",                       "His name is Ahmed"),
        ("কথা বলা বন্ধ করো",                   "Stop talking"),
        ("তার কথা শুনে ভালো লাগল",              "I felt good hearing his words"),
        ("বই পড়তে ভালো লাগে",                  "I like reading books"),
        ("আমি একটি নতুন বই কিনেছি",             "I bought a new book"),
        ("ঘর পরিষ্কার করা হয়েছে",               "The house has been cleaned"),
        ("আমি ঘরে বসে আছি",                    "I am sitting at home"),
        ("মন ভালো নেই",                         "My mind is not good"),
        ("আমার মন চায় বেড়াতে যেতে",            "My mind wants to go for a walk"),
        ("হাত ধুয়ে নাও",                        "Wash your hands"),
        ("আমি তার হাত ধরলাম",                   "I held his hand"),
        ("দিন কেটে যাচ্ছে",                     "The day is passing by"),
        ("আজ কি দিন",                           "What day is today"),
        ("রাত হয়ে এসেছে",                       "Night has come"),
        ("আমি রাত জেগে পড়েছি",                 "I studied staying up at night"),
        ("জল খুব গরম",                          "The water is very hot"),
        ("আমি জল দিয়ে গাছ সিঞ্চন করেছি",        "I watered the plants"),
        ("বাড়ি যাচ্ছি",                          "I am going home"),
        ("আমার বাড়ি ঢাকায়",                     "My house is in Dhaka"),
        ("পার্কে অনেক মানুষ",                    "There are many people in the park"),
        ("আমি প্রতিদিন পার্কে হাঁটি",             "I walk in the park every day"),
        ("নদী বইছে",                             "The river is flowing"),
        ("আমি নদীর ধারে দাঁড়িয়ে আছি",           "I am standing by the river"),
        ("বন খুব সুন্দর",                        "The forest is very beautiful"),
        ("আমি বন দেখতে গিয়েছিলাম",              "I went to see the forest"),
        ("আকাশ নীল",                             "The sky is blue"),
        ("আমি আকাশের দিকে তাকালাম",              "I looked up at the sky"),
    ]
    processed: List[Tuple[str, str]] = []
    for bn, en in base_pairs:
        bn_c = clean_and_normalize_text(bn, "bn_IN")
        en_c = clean_and_normalize_text(en, "en_XX")
        if bn_c and en_c:
            processed.append((bn_c, en_c))
    if not processed:
        return [("আমি", "I")]
    replicated: List[Tuple[str, str]] = []
    while len(replicated) < n:
        replicated.extend(processed)
    print(f"[CELL2] Fallback dataset: returning {n} pairs (replicated from {len(processed)} base pairs)")
    return replicated[:n]


def _load_csv_pairs(num_samples: int) -> List[Tuple[str, str]]:
    if not _HAS_PANDAS:
        print("[CELL2] ERROR: pandas not available; cannot load CSV!")
        return _get_fallback_dataset(num_samples)

    if not os.path.exists(_DATASET_CSV_PATH):
        print(f"[CELL2] ERROR: CSV file not found at: {_DATASET_CSV_PATH}")
        return _get_fallback_dataset(num_samples)

    try:
        print(f"[CELL2] Reading CSV: {_DATASET_CSV_PATH}")
        df = pd.read_csv(_DATASET_CSV_PATH, on_bad_lines="skip", encoding="utf-8")
        if df.empty:
            print("[CELL2] ERROR: CSV file is empty")
            return _get_fallback_dataset(num_samples)

        print(f"[CELL2] CSV columns found: {list(df.columns)}")
        print(f"[CELL2] CSV total rows: {len(df)}")

        if _SRC_COL in df.columns and _TGT_COL in df.columns:
            # CSV: src=English, tgt=Bengali → swap so model gets bn as input, en as target
            df = df[[_TGT_COL, _SRC_COL]].rename(columns={_TGT_COL: "bn", _SRC_COL: "en"})
            print(f"[CELL2] COLUMN SWAP APPLIED: CSV '{_TGT_COL}'(Bengali) → model_src | CSV '{_SRC_COL}'(English) → model_tgt")
            print(f"[CELL2] Sample after swap → bn: '{str(df['bn'].iloc[0])[:80]}' | en: '{str(df['en'].iloc[0])[:80]}'")
        else:
            bn_col, en_col = _detect_columns(df)
            if bn_col is None or en_col is None:
                print(f"[CELL2] ERROR: Could not detect bn/en columns. Found: {list(df.columns)}")
                return _get_fallback_dataset(num_samples)
            df = df[[bn_col, en_col]].rename(columns={bn_col: "bn", en_col: "en"})
            print(f"[CELL2] COLUMN SWAP APPLIED (auto-detect): '{bn_col}'(Bengali) → model_src | '{en_col}'(English) → model_tgt")
            print(f"[CELL2] Sample after swap → bn: '{str(df['bn'].iloc[0])[:80]}' | en: '{str(df['en'].iloc[0])[:80]}'")

        sample_size       = min(10, len(df))
        src_bengali_count = sum(1 for v in df["bn"].head(sample_size) if is_bengali_text(str(v)))
        if src_bengali_count <= sample_size * 0.5:
            print("[CELL2] Bengali/English columns appear swapped; correcting.")
            df = df.rename(columns={"bn": "_tmp", "en": "bn"}).rename(columns={"_tmp": "en"})

        df = df.drop_duplicates(subset=["bn", "en"])
        df = df.dropna(subset=["bn", "en"])

        max_words = _MAX_LENGTH * 3

        def _within_length(row) -> bool:
            return (
                len(str(row["bn"]).split()) <= max_words
                and len(str(row["en"]).split()) <= max_words
            )

        before_len  = len(df)
        df          = df[df.apply(_within_length, axis=1)]
        dropped_len = before_len - len(df)
        if dropped_len > 0:
            print(f"[CELL2] Length filter dropped {dropped_len} rows (>{max_words} words)")

        df = df.head(num_samples).reset_index(drop=True)
        print(f"[CELL2] Processing {len(df)} rows from CSV...")

        pairs: List[Tuple[str, str]] = []
        skipped = 0

        for row_tuple in tqdm(df.itertuples(index=False), total=len(df), desc="Loading dataset"):
            try:
                src_val = getattr(row_tuple, "bn", None)
                tgt_val = getattr(row_tuple, "en", None)
                if src_val is None or tgt_val is None:
                    skipped += 1
                    cell2_dbg("missing_attr", "Missing bn/en attribute in row_tuple")
                    continue
                if pd.isna(src_val) or pd.isna(tgt_val):
                    skipped += 1
                    cell2_dbg("nan_value", "NaN value detected")
                    continue
                bn = str(src_val).strip()
                en = str(tgt_val).strip()
                if not bn or not en:
                    skipped += 1
                    cell2_dbg("empty_field", "Empty bn/en field")
                    continue
                if not is_bengali_text(bn):
                    skipped += 1
                    cell2_dbg("not_bengali_src", "bn field not Bengali")
                    continue
                if not re.search(r"[a-zA-Z]", en):
                    skipped += 1
                    cell2_dbg("not_english_tgt", "en field not English")
                    continue
                bn_norm = clean_and_normalize_text(bn, language="bn_IN")
                en_norm = clean_and_normalize_text(en, language="en_XX")
                if not bn_norm or not en_norm:
                    skipped += 1
                    cell2_dbg("empty_after_norm", "Empty after normalization")
                    continue
                pairs.append((bn_norm, en_norm))
            except Exception as e:
                skipped += 1
                cell2_dbg("row_exception", f"Row load exception: {type(e).__name__}")
                continue

        print(f"[CELL2] Loaded {len(pairs)} pairs, skipped {skipped} rows")

        if len(pairs) == 0:
            print("[CELL2] ERROR: No valid pairs loaded from CSV!")
            return _get_fallback_dataset(num_samples)

        return pairs

    except pd.errors.EmptyDataError:
        print(f"[CELL2] ERROR: CSV file is empty: {_DATASET_CSV_PATH}")
        return _get_fallback_dataset(num_samples)
    except Exception as e:
        print(f"[CELL2] ERROR loading CSV: {type(e).__name__}: {str(e)}")
        traceback.print_exc()
        return _get_fallback_dataset(num_samples)


def load_and_split_dataset(
    num_samples: Optional[int] = None,
) -> Tuple[List[Tuple[str, str]], List[Tuple[str, str]], List[Tuple[str, str]]]:
    if num_samples is None:
        num_samples = _NUM_SAMPLES

    pairs = _load_csv_pairs(num_samples)
    rng   = random.Random(_SEED)
    rng.shuffle(pairs)

    n       = len(pairs)
    n_train = int(n * _TRAIN_RATIO)
    n_val   = int(n * _VAL_RATIO)

    train_pairs = pairs[:n_train]
    val_pairs   = pairs[n_train : n_train + n_val]
    test_pairs  = pairs[n_train + n_val :]

    print(f"[CELL2] Split: train={len(train_pairs)}, val={len(val_pairs)}, test={len(test_pairs)}")
    return train_pairs, val_pairs, test_pairs


def load_and_preprocess_optimized(
    num_samples: Optional[int] = None,
) -> List[Tuple[str, str]]:
    """
    Called by Cell 10 main_pipeline() via globals()['load_and_preprocess_optimized'](NUM_SAMPLES).
    Returns all pairs (unsplit) as List[Tuple[bn_text, en_text]].
    Splitting into train/val/test is handled inside Cell 10 main_pipeline().
    """
    if num_samples is None:
        num_samples = _NUM_SAMPLES

    print(f"[CELL2] load_and_preprocess_optimized called with num_samples={num_samples}")
    pairs = _load_csv_pairs(num_samples)

    if not pairs:
        print(f"[CELL2] load_and_preprocess_optimized: CSV load returned 0 pairs, using fallback (n={num_samples})")
        pairs = _get_fallback_dataset(num_samples)
    elif len(pairs) < num_samples * 0.5:
        print(
            f"[CELL2] WARNING: load_and_preprocess_optimized got only {len(pairs)} pairs "
            f"but {num_samples} were requested. Check CSV file at: {_DATASET_CSV_PATH}"
        )

    print(f"[CELL2] load_and_preprocess_optimized: returning {len(pairs)} pairs")
    return pairs


# ===========================================================================
# DATASET CLASS
# ===========================================================================
class DualPathTranslationDataset(Dataset):
    def __init__(
        self,
        pairs:          List[Tuple[str, str]],
        tokenizer:      Any = None,
        word_tokenizer: Any = None,
        max_length:     Optional[int] = None,
        split:          str = "train",
        domain:         Optional[int] = None,
    ):
        if max_length is None:
            max_length = _MAX_LENGTH
        self.max_length     = int(max_length)
        self.tokenizer      = tokenizer
        self.split          = split

        if domain is not None:
            self.domain_label = int(domain)
        elif split == "train":
            self.domain_label = _TRAIN_DOMAIN
        else:
            self.domain_label = _TEST_DOMAIN

        if word_tokenizer is None:
            if "BengaliWordTokenizer" in globals():
                try:
                    self.word_tokenizer = BengaliWordTokenizer()
                except Exception:
                    self.word_tokenizer = None
            else:
                self.word_tokenizer = None
        else:
            self.word_tokenizer = word_tokenizer

        self.vocab_size = len(tokenizer) if tokenizer is not None else _MBART_VOCAB_SIZE
        print(f"[CELL2] Dataset vocab size: {self.vocab_size}")

        try:
            self._tokenizer_name_or_path = getattr(tokenizer, "name_or_path", None)
        except Exception:
            self._tokenizer_name_or_path = None

        try:
            self.is_fast = getattr(self.tokenizer, "is_fast", False)
        except Exception:
            self.is_fast = False

        self.pairs: List[Tuple[str, str]] = []
        invalid = 0

        for i, p in enumerate(pairs):
            try:
                if not isinstance(p, (list, tuple)) or len(p) != 2:
                    invalid += 1
                    cell2_dbg("init_badpair", f"Bad pair structure at idx={i}")
                    continue
                src, tgt = p
                if not isinstance(src, str) or not isinstance(tgt, str):
                    invalid += 1
                    cell2_dbg("init_badtype", f"Non-string src/tgt at idx={i}")
                    continue
                if not src or not tgt:
                    invalid += 1
                    cell2_dbg("init_empty", f"Empty src/tgt at idx={i}")
                    continue
                if len(src) > self.max_length * 20 or len(tgt) > self.max_length * 20:
                    invalid += 1
                    cell2_dbg("init_long", f"Extremely long text at idx={i}")
                    continue
                self.pairs.append((src, tgt))
            except Exception as e:
                invalid += 1
                cell2_dbg("init_exc", f"Init pair exception idx={i}: {type(e).__name__}")

        print(f"[CELL2] Dataset initialized: {len(self.pairs)} valid pairs, {invalid} invalid, split={self.split}, domain={self.domain_label}")

        try:
            if "get_tokenizer_special_tokens" in globals():
                self.special_tokens = get_tokenizer_special_tokens(self.tokenizer)
            else:
                self.special_tokens = (
                    set(getattr(self.tokenizer, "all_special_tokens", []))
                    if self.tokenizer is not None
                    else set()
                )
        except Exception:
            self.special_tokens = {
                f"__{_SOURCE_LANG}__",
                f"__{_TARGET_LANG}__",
                "</s>",
                "<pad>",
                "<s>",
                "<unk>",
            }

    def __getstate__(self):
        state = self.__dict__.copy()
        state["tokenizer"]               = None
        state["word_tokenizer"]          = None
        state["_tokenizer_name_or_path"] = getattr(self, "_tokenizer_name_or_path", None)
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        self.tokenizer      = None
        self.word_tokenizer = None
        self.is_fast        = False

    def __len__(self) -> int:
        return len(self.pairs)

    def _encode_src(self, src_text: str):
        src_text = src_text if isinstance(src_text, str) else str(src_text)
        try:
            if self.tokenizer is None:
                self.tokenizer  = globals().get("tokenizer", None)
                self.is_fast    = getattr(self.tokenizer, "is_fast", False) if self.tokenizer is not None else False
                self.vocab_size = len(self.tokenizer) if self.tokenizer is not None else _MBART_VOCAB_SIZE
            if self.tokenizer is None:
                raise RuntimeError("Tokenizer not available")

            try:
                self.tokenizer.src_lang = _SOURCE_LANG
            except Exception:
                pass

            if _has_safe_offsets_tokenize:
                enc = safe_offsets_tokenize(
                    self.tokenizer,
                    src_text,
                    max_length=self.max_length,
                    include_special_tokens=True,
                )
                try:
                    if isinstance(enc["input_ids"], torch.Tensor):
                        input_ids = enc["input_ids"].squeeze(0) if enc["input_ids"].dim() > 1 else enc["input_ids"]
                    else:
                        input_ids = (
                            torch.tensor(enc["input_ids"][0])
                            if isinstance(enc["input_ids"], list) and len(enc["input_ids"]) > 0
                            else torch.tensor(enc["input_ids"])
                        )
                except Exception:
                    input_ids = torch.tensor(enc.get("input_ids", [[1]])[0] if enc.get("input_ids") else [1])

                attention_mask = enc.get("attention_mask", None)
                if attention_mask is None:
                    attention_mask = torch.ones_like(input_ids)
                elif isinstance(attention_mask, list):
                    attention_mask = torch.tensor(attention_mask[0]) if attention_mask else torch.ones_like(input_ids)
                elif isinstance(attention_mask, torch.Tensor):
                    attention_mask = attention_mask.squeeze(0) if attention_mask.dim() > 1 else attention_mask

                try:
                    ids_list = input_ids.tolist() if isinstance(input_ids, torch.Tensor) else list(input_ids)
                    tokens   = self.tokenizer.convert_ids_to_tokens(ids_list)
                except Exception:
                    tokens = []
            else:
                enc = self.tokenizer(
                    src_text,
                    max_length=self.max_length,
                    padding="max_length",
                    truncation=True,
                    return_tensors="pt",
                    add_special_tokens=True,
                )
                input_ids      = enc["input_ids"].squeeze(0)
                attention_mask = enc.get("attention_mask", torch.ones_like(input_ids)).squeeze(0)
                try:
                    tokens = self.tokenizer.convert_ids_to_tokens(input_ids.tolist())
                except Exception:
                    tokens = []

            input_ids = torch.clamp(input_ids, 0, self.vocab_size - 1)

            token_word_map: Dict[int, str] = {}
            if _has_reconstruct_word_spans:
                try:
                    wm, words = reconstruct_word_spans(self.tokenizer, src_text, max_length=self.max_length)
                    if isinstance(wm, dict) and wm:
                        token_word_map = wm
                except Exception as e:
                    cell2_dbg("wm_exc", f"reconstruct_word_spans failed: {e}")

            # FIX-BUG-2: Assign token_word_map only at word-boundary tokens (▁/Ġ prefix),
            # not on every continuation subword token. Flush completed word at each new
            # word-start, then flush the final word after the loop ends.
            if not token_word_map and tokens:
                try:
                    current_word:      List[str] = []
                    current_start_idx: int       = 0
                    for tok_idx, tok in enumerate(tokens):
                        if isinstance(tok, str) and tok not in self.special_tokens:
                            if is_punctuation_only(tok):
                                continue
                            clean = tok.replace("▁", "").replace("Ġ", "").replace("##", "").strip()
                            if clean:
                                if tok.startswith("▁") or tok.startswith("Ġ"):
                                    if current_word:
                                        word_str = "".join(current_word)
                                        if not is_punctuation_only(word_str):
                                            token_word_map[current_start_idx] = word_str
                                    current_word      = [clean]
                                    current_start_idx = tok_idx
                                else:
                                    current_word.append(clean)
                    if current_word:
                        word_str = "".join(current_word)
                        if not is_punctuation_only(word_str):
                            token_word_map[current_start_idx] = word_str
                except Exception as e:
                    cell2_dbg("fallback_wm", f"Fallback word map failed: {e}")

            return input_ids, attention_mask, tokens, token_word_map

        except Exception as e:
            cell2_dbg("encode_src_exc", f"Encoding source failed: {type(e).__name__}")
            pad_id         = getattr(self.tokenizer, "pad_token_id", 1) if self.tokenizer is not None else 1
            input_ids      = torch.full((self.max_length,), int(pad_id), dtype=torch.long)
            attention_mask = torch.zeros(self.max_length, dtype=torch.long)
            return input_ids, attention_mask, [], {}

    def _encode_tgt(self, tgt_text: str) -> torch.Tensor:
        """
        Encodes English target text (en_XX) for decoder labels.
        Uses direct tgt_lang assignment instead of deprecated as_target_tokenizer().
        FIX-BUG-7: All src_lang assignments are now guarded with try/except.
        """
        tgt_text = tgt_text if isinstance(tgt_text, str) else str(tgt_text)
        try:
            if self.tokenizer is None:
                self.tokenizer  = globals().get("tokenizer", None)
                self.vocab_size = len(self.tokenizer) if self.tokenizer is not None else _MBART_VOCAB_SIZE
            if self.tokenizer is None:
                raise RuntimeError("Tokenizer not available")

            prev_src_lang = getattr(self.tokenizer, "src_lang", _SOURCE_LANG)
            try:
                self.tokenizer.src_lang = _TARGET_LANG
            except Exception:
                pass

            dec = self.tokenizer(
                tgt_text,
                max_length=self.max_length,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
                add_special_tokens=True,
            )

            try:
                self.tokenizer.src_lang = prev_src_lang
            except Exception:
                pass

            labels = dec["input_ids"].squeeze(0)
            labels = torch.clamp(labels, 0, self.vocab_size - 1)
            pad_id = getattr(self.tokenizer, "pad_token_id", 1) if self.tokenizer is not None else 1

            valid_before_mask = (labels != int(pad_id)).sum().item()
            labels[labels == int(pad_id)] = -100
            valid_after_mask  = (labels != -100).sum().item()

            if _DEBUG_DISCOVERY and valid_after_mask == 0:
                cell2_dbg("encode_tgt_all_masked", f"[ENCODE_TGT] WARNING: All labels masked as -100! (before={valid_before_mask}, after={valid_after_mask})")
            elif _DEBUG_DISCOVERY and valid_after_mask < 3:
                cell2_dbg("encode_tgt_few_valid", f"[ENCODE_TGT] Only {valid_after_mask} valid labels")

            return labels

        except Exception as e:
            cell2_dbg("encode_tgt_exc", f"Encoding tgt failed: {type(e).__name__}")
            return torch.full((self.max_length,), -100, dtype=torch.long)

    def _encode_word(self, src_text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        try:
            wt = self.word_tokenizer
            if wt is None:
                wt = globals().get("word_tokenizer", None)
            if wt is None:
                raise RuntimeError("word_tokenizer not available")

            enc = wt(
                src_text,
                max_length=self.max_length,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            word_input_ids      = enc["input_ids"].squeeze(0)
            word_attention_mask = enc.get("attention_mask", torch.ones_like(word_input_ids)).squeeze(0)
            return word_input_ids, word_attention_mask
        except Exception as e:
            cell2_dbg("encode_word_exc", f"Word encoding failed: {type(e).__name__}")
            word_input_ids      = torch.zeros(self.max_length, dtype=torch.long)
            word_attention_mask = torch.zeros(self.max_length, dtype=torch.long)
            return word_input_ids, word_attention_mask

    def _make_safe_sample(self, reason: str = "fallback") -> Dict[str, Any]:
        try:
            src = "আমি"
            tgt = "I"
            input_ids, attention_mask, tokens, token_word_map = self._encode_src(src)
            labels = self._encode_tgt(tgt)
            word_input_ids, word_attention_mask = self._encode_word(src)
            return {
                "input_ids":           input_ids,
                "attention_mask":      attention_mask,
                "labels":              labels,
                "word_input_ids":      word_input_ids,
                "word_attention_mask": word_attention_mask,
                "token_word_map":      token_word_map,
                "src_text":            src,
                "tgt_text":            tgt,
                "tokens":              tokens,
                "domain_label":        self.domain_label,
            }
        except Exception:
            pad_id = 1
            return {
                "input_ids":           torch.full((self.max_length,), int(pad_id), dtype=torch.long),
                "attention_mask":      torch.zeros(self.max_length, dtype=torch.long),
                "labels":              torch.full((self.max_length,), -100, dtype=torch.long),
                "word_input_ids":      torch.zeros(self.max_length, dtype=torch.long),
                "word_attention_mask": torch.zeros(self.max_length, dtype=torch.long),
                "token_word_map":      {},
                "src_text":            "",
                "tgt_text":            "",
                "tokens":              [],
                "domain_label":        self.domain_label,
            }

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        try:
            if idx < 0 or idx >= len(self.pairs):
                cell2_dbg("getitem_oob", f"Index out of range idx={idx}")
                return self._make_safe_sample("oob")

            src, tgt = self.pairs[idx]
            if not isinstance(src, str) or not isinstance(tgt, str):
                cell2_dbg("getitem_bad_types", f"Bad types at idx={idx}")
                return self._make_safe_sample("bad_types")

            if DEBUG_CELL2 and idx < 3:
                has_bengali = is_bengali_text(src)
                print(f"[CELL2-GETITEM-{idx}] src sample: {src[:50]}")
                print(f"[CELL2-GETITEM-{idx}] Bengali: {has_bengali}")
                if not has_bengali:
                    print(f"[CELL2] WARNING: src_text is NOT Bengali at idx={idx}!")

            input_ids, attention_mask, tokens, token_word_map = self._encode_src(src)
            labels = self._encode_tgt(tgt)
            word_input_ids, word_attention_mask = self._encode_word(src)

            if _DEBUG_DISCOVERY and idx < 5:
                valid_labels = (labels != -100).sum().item()
                if valid_labels == 0:
                    print(f"[CELL2-GETITEM] WARNING: idx={idx} has ALL labels = -100!")
                elif valid_labels < 3:
                    print(f"[CELL2-GETITEM] idx={idx} has only {valid_labels} valid labels")

            return {
                "input_ids":           input_ids,
                "attention_mask":      attention_mask,
                "labels":              labels,
                "word_input_ids":      word_input_ids,
                "word_attention_mask": word_attention_mask,
                "token_word_map":      token_word_map,
                "src_text":            src,
                "tgt_text":            tgt,
                "tokens":              tokens,
                "domain_label":        self.domain_label,
            }
        except Exception as e:
            cell2_dbg("getitem_exc", f"Unhandled __getitem__ exception idx={idx}: {type(e).__name__}")
            return self._make_safe_sample("unhandled")


MemoryEfficientDataset = DualPathTranslationDataset


def _infer_pad_id_from_sample(sample: Dict[str, Any], default_pad_id: int = 1) -> int:
    try:
        tk = globals().get("tokenizer", None)
        if tk is not None:
            pad = getattr(tk, "pad_token_id", None)
            if pad is not None:
                return int(pad)
    except Exception:
        cell2_dbg("infer_pad_exc", "infer pad id failed")
    return int(default_pad_id)


def _pad_or_truncate_array(tensor: torch.Tensor, length: int, pad_value: int) -> torch.Tensor:
    if tensor is None:
        return torch.full((length,), int(pad_value), dtype=torch.long)
    t = tensor.view(-1).long()
    L = t.size(0)
    if L == length:
        return t
    if L < length:
        pad = torch.full((length - L,), int(pad_value), dtype=t.dtype)
        return torch.cat([t, pad], dim=0)
    return t[:length]


def safe_collate(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    valid = [
        b for b in batch
        if isinstance(b, dict) and "input_ids" in b and isinstance(b["input_ids"], torch.Tensor)
    ]

    default_domain = _TRAIN_DOMAIN

    # FIX-CRITICAL-1: Return key is "src_text" (singular) to match Cell 7 batch.get("src_text").
    if not valid:
        pad = _infer_pad_id_from_sample({}, default_pad_id=1)
        return {
            "input_ids":           torch.full((1, _MAX_LENGTH), pad, dtype=torch.long),
            "attention_mask":      torch.zeros(1, _MAX_LENGTH, dtype=torch.long),
            "labels":              torch.full((1, _MAX_LENGTH), -100, dtype=torch.long),
            "word_input_ids":      torch.zeros(1, _MAX_LENGTH, dtype=torch.long),
            "word_attention_mask": torch.zeros(1, _MAX_LENGTH, dtype=torch.long),
            "token_word_map":      [{}],
            "src_text":            [""],
            "tgt_text":            [""],
            "tokens":              [[]],
            "domain_labels":       torch.tensor([default_domain], dtype=torch.long),
        }

    pad_id = _infer_pad_id_from_sample(valid[0], default_pad_id=1)

    raw_inputs     = []
    raw_masks      = []
    raw_labs       = []
    raw_word_ids   = []
    raw_word_masks = []
    twmaps         = []
    srcs           = []
    tgt_txts       = []
    toks           = []
    domains        = []

    for i, s in enumerate(valid):
        try:
            in_ids = s["input_ids"]
            att    = s.get("attention_mask", None)
            lab    = s["labels"]
            w_ids  = s.get("word_input_ids",      torch.zeros(_MAX_LENGTH, dtype=torch.long))
            w_mask = s.get("word_attention_mask",  torch.zeros(_MAX_LENGTH, dtype=torch.long))
            domain = s.get("domain_label", default_domain)

            if att is None:
                att = (in_ids != pad_id).long()
            else:
                try:
                    att = att.view(-1).long()
                except Exception:
                    att = (in_ids != pad_id).long()

            try:
                in_ids = in_ids.view(-1)
            except Exception:
                in_ids = in_ids.flatten()

            try:
                lab = lab.view(-1)
            except Exception:
                lab = lab.flatten()

            try:
                w_ids  = w_ids.view(-1)
                w_mask = w_mask.view(-1)
            except Exception:
                w_ids  = torch.zeros(_MAX_LENGTH, dtype=torch.long)
                w_mask = torch.zeros(_MAX_LENGTH, dtype=torch.long)

            raw_inputs.append(in_ids)
            raw_masks.append(att)
            raw_labs.append(lab)
            raw_word_ids.append(w_ids)
            raw_word_masks.append(w_mask)
            twmaps.append(s.get("token_word_map", {}))
            srcs.append(s.get("src_text", ""))
            tgt_txts.append(s.get("tgt_text", ""))
            toks.append(s.get("tokens", []))
            domains.append(domain)
        except Exception as e:
            cell2_dbg("collate_item_exc", f"Collate item exception idx={i}: {type(e).__name__}")
            continue

    # FIX-CRITICAL-1: Return key is "src_text" (singular) to match Cell 7 batch.get("src_text").
    if not raw_inputs:
        pad = _infer_pad_id_from_sample({}, default_pad_id=1)
        return {
            "input_ids":           torch.full((1, _MAX_LENGTH), pad, dtype=torch.long),
            "attention_mask":      torch.zeros(1, _MAX_LENGTH, dtype=torch.long),
            "labels":              torch.full((1, _MAX_LENGTH), -100, dtype=torch.long),
            "word_input_ids":      torch.zeros(1, _MAX_LENGTH, dtype=torch.long),
            "word_attention_mask": torch.zeros(1, _MAX_LENGTH, dtype=torch.long),
            "token_word_map":      [{}],
            "src_text":            [""],
            "tgt_text":            [""],
            "tokens":              [[]],
            "domain_labels":       torch.tensor([default_domain], dtype=torch.long),
        }

    max_input_len  = max(t.size(0) for t in raw_inputs)
    max_label_len  = max(t.size(0) for t in raw_labs)
    actual_max_len = min(max(max_input_len, max_label_len), _MAX_LENGTH)

    inputs     = []
    masks      = []
    labs       = []
    word_ids   = []
    word_masks = []

    for in_ids, att, lab, w_ids, w_mask in zip(raw_inputs, raw_masks, raw_labs, raw_word_ids, raw_word_masks):
        inputs.append(_pad_or_truncate_array(in_ids, actual_max_len, pad_id))
        masks.append(_pad_or_truncate_array(att,     actual_max_len, 0))
        labs.append(_pad_or_truncate_array(lab,      actual_max_len, -100))
        word_ids.append(_pad_or_truncate_array(w_ids,    actual_max_len, 0))
        word_masks.append(_pad_or_truncate_array(w_mask,  actual_max_len, 0))

    input_ids             = torch.stack(inputs,     dim=0)
    attention_mask        = torch.stack(masks,      dim=0)
    labels                = torch.stack(labs,       dim=0)
    word_input_ids_t      = torch.stack(word_ids,   dim=0)
    word_attention_mask_t = torch.stack(word_masks, dim=0)

    try:
        domain_labels = torch.tensor(domains, dtype=torch.long)
    except Exception:
        domain_labels = torch.full((len(inputs),), default_domain, dtype=torch.long)

    if _DEBUG_DISCOVERY:
        batch_size            = labels.size(0)
        total_label_positions = labels.numel()
        valid_labels          = (labels != -100).sum().item()
        padding_labels        = total_label_positions - valid_labels
        if valid_labels == 0:
            print(f"[COLLATE] CRITICAL WARNING: ALL labels are -100! Decoder won't train!")
            print(f"[COLLATE]   batch_size={batch_size}, total_positions={total_label_positions}")
        elif valid_labels < batch_size * 2:
            print(f"[COLLATE] WARNING: Very few valid labels!")
            print(f"[COLLATE]   valid_labels={valid_labels}, padding={padding_labels}, avg={valid_labels/batch_size:.1f}")
        else:
            print(f"[COLLATE] Labels OK: {valid_labels}/{total_label_positions} valid ({valid_labels/batch_size:.1f} per sample)")

    # FIX-CRITICAL-1: "src_text" and "tgt_text" (singular) — matches Cell 7 batch.get("src_text").
    return {
        "input_ids":           input_ids,
        "attention_mask":      attention_mask,
        "labels":              labels,
        "word_input_ids":      word_input_ids_t,
        "word_attention_mask": word_attention_mask_t,
        "token_word_map":      twmaps,
        "src_text":            srcs,
        "tgt_text":            tgt_txts,
        "tokens":              toks,
        "domain_labels":       domain_labels,
    }


def create_optimized_dataloader(
    dataset:    Dataset,
    batch_size: Optional[int] = None,
    shuffle:    bool = True,
    split:      str = "train",
) -> DataLoader:
    if batch_size is None:
        try:
            batch_size = int(BATCH_SIZE)
        except NameError:
            batch_size = 8
    batch_size = int(batch_size)

    num_workers = _NUM_WORKERS if isinstance(_NUM_WORKERS, int) and _NUM_WORKERS >= 0 else 0
    try:
        max_possible = max(0, (os.cpu_count() or 1) - 1)
        if num_workers > max_possible:
            num_workers = max_possible
    except Exception:
        pass

    loader_kwargs: Dict[str, Any] = {
        "dataset":     dataset,
        "batch_size":  batch_size,
        "shuffle":     shuffle,
        "num_workers": num_workers,
        "pin_memory":  bool(_PIN_MEMORY and torch.cuda.is_available()),
        "collate_fn":  safe_collate,
        "drop_last":   False,
    }

    if num_workers > 0:
        loader_kwargs["worker_init_fn"]     = _dataloader_worker_init_fn
        loader_kwargs["prefetch_factor"]    = _PREFETCH_FACTOR
        loader_kwargs["persistent_workers"] = False

    try:
        dataloader = DataLoader(**loader_kwargs)
    except Exception as e:
        print(f"[CELL2] DataLoader init failed with num_workers={num_workers}: {type(e).__name__}")
        print("[CELL2] Retrying with num_workers=0")
        loader_kwargs["num_workers"] = 0
        loader_kwargs.pop("prefetch_factor",    None)
        loader_kwargs.pop("persistent_workers", None)
        loader_kwargs.pop("worker_init_fn",     None)
        dataloader = DataLoader(**loader_kwargs)

    print(f"[CELL2] DataLoader created ({split}): batch_size={batch_size}, workers={loader_kwargs.get('num_workers', 0)}")
    return dataloader


def build_dataloaders(
    train_pairs:     List[Tuple[str, str]],
    val_pairs:       List[Tuple[str, str]],
    test_pairs:      List[Tuple[str, str]],
    mbart_tokenizer: Any = None,
    word_tokenizer:  Any = None,
    batch_size:      Optional[int] = None,
    eval_batch_size: Optional[int] = None,
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    if batch_size is None:
        try:
            batch_size = int(BATCH_SIZE)
        except NameError:
            batch_size = 4

    if eval_batch_size is None:
        try:
            eval_batch_size = int(EVAL_BATCH_SIZE)
        except NameError:
            eval_batch_size = batch_size

    train_dataset = DualPathTranslationDataset(
        pairs=train_pairs,
        tokenizer=mbart_tokenizer,
        word_tokenizer=word_tokenizer,
        split="train",
        domain=_TRAIN_DOMAIN,
    )
    val_dataset = DualPathTranslationDataset(
        pairs=val_pairs,
        tokenizer=mbart_tokenizer,
        word_tokenizer=word_tokenizer,
        split="val",
        domain=_TEST_DOMAIN,
    )
    test_dataset = DualPathTranslationDataset(
        pairs=test_pairs,
        tokenizer=mbart_tokenizer,
        word_tokenizer=word_tokenizer,
        split="test",
        domain=_TEST_DOMAIN,
    )

    train_loader = create_optimized_dataloader(train_dataset, batch_size=batch_size,      shuffle=True,  split="train")
    val_loader   = create_optimized_dataloader(val_dataset,   batch_size=eval_batch_size, shuffle=False, split="val")
    test_loader  = create_optimized_dataloader(test_dataset,  batch_size=eval_batch_size, shuffle=False, split="test")

    print(f"[CELL2] build_dataloaders complete: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")
    return train_loader, val_loader, test_loader


globals()["load_and_preprocess_optimized"]   = load_and_preprocess_optimized
globals()["loadandpreprocessoptimized"]      = load_and_preprocess_optimized
globals()["load_and_split_dataset"]          = load_and_split_dataset
globals()["build_dataloaders"]               = build_dataloaders
globals()["create_optimized_dataloader"]     = create_optimized_dataloader
globals()["safe_collate"]                    = safe_collate
globals()["DualPathTranslationDataset"]      = DualPathTranslationDataset
globals()["MemoryEfficientDataset"]          = MemoryEfficientDataset
globals()["_PERIODIC_DISCOVERY_FREQUENCY"]   = _PERIODIC_DISCOVERY_FREQUENCY

print("=" * 80)
print("Cell 2: Dual-path data loading ready - WORD + SUBWORD TOKENIZATION")
print("=" * 80)
print("CONFIGURATION:")
print(f"  mBART-50 tokenizer (subword path + word path)")
print(f"  Language codes: {_SOURCE_LANG} → {_TARGET_LANG}")
print(f"  Token IDs: bn={_MBART_BN_TOKEN_ID}, en={_MBART_EN_TOKEN_ID}")
print(f"  Vocab size: {_MBART_VOCAB_SIZE:,}")
print(f"  CSV path: {_DATASET_CSV_PATH}")
print(f"  CSV columns: tgt(bn)='{_TGT_COL}', src(en)='{_SRC_COL}'")
print(f"  Split ratios: train={_TRAIN_RATIO:.0%} / val={_VAL_RATIO:.0%} / test={_TEST_RATIO:.0%}")
print(f"  Domain labels: train={_TRAIN_DOMAIN}, val/test={_TEST_DOMAIN}")
print(f"  Multi-GPU: DISABLED | DataParallel: DISABLED")
print(f"  PERIODIC_DISCOVERY_FREQUENCY: {_PERIODIC_DISCOVERY_FREQUENCY} steps")
print(f"  load_and_preprocess_optimized: REGISTERED in globals() ✅")
print(f"  loadandpreprocessoptimized (alias): REGISTERED in globals() ✅")
print("=" * 80 + "\n")

In [ ]:
# ==============================================================================
# CELL 3: DSCD MODULE — WORD-LEVEL HOMOGRAPH DISAMBIGUATION
# ==============================================================================

import os
import threading
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gc
from collections import deque
import unicodedata
from typing import Optional, Dict, List, Any, Set, Tuple

PRINT_INTERVAL = 200

try:
    from scipy.cluster.hierarchy import linkage, fcluster
    from scipy.spatial.distance import pdist
    HAS_CLUSTERING = True
except Exception:
    HAS_CLUSTERING = False
    print("[CELL3] WARNING: scipy not available")

try:
    from sklearn.cluster import KMeans
    HAS_KMEANS = True
except Exception:
    HAS_KMEANS = False
    print("[CELL3] WARNING: sklearn not available")

try:
    DSCD_MAX_PROTOS                 = int(DSCD_MAX_PROTOS)
    DSCD_BUFFER_SIZE                = int(DSCD_BUFFER_SIZE)
    DSCD_N_MIN                      = int(DSCD_N_MIN)
    DSCD_DISPERSION_THRESHOLD       = float(DSCD_DISPERSION_THRESHOLD)
    DSCD_NEWSENSE_LAMBDA            = float(DSCD_NEWSENSE_LAMBDA)
    DSCD_EMBED_DIM                  = int(DSCD_EMBED_DIM)
    VERBOSE_LOGGING                 = bool(VERBOSE_LOGGING)
    DSCD_ENABLE_TRAINING_CLUSTERING = bool(DSCD_ENABLE_TRAINING_CLUSTERING)
    DSCD_MIN_LETTERS                = int(DSCD_MIN_LETTERS)
    DSCD_MIN_LETTER_FRACTION        = float(DSCD_MIN_LETTER_FRACTION)
except (NameError, ValueError, TypeError):
    DSCD_MAX_PROTOS                 = 3
    DSCD_BUFFER_SIZE                = 20
    DSCD_N_MIN                      = 3
    DSCD_DISPERSION_THRESHOLD       = 0.25
    DSCD_NEWSENSE_LAMBDA            = 1.2
    DSCD_EMBED_DIM                  = 1024
    VERBOSE_LOGGING                 = False
    DSCD_ENABLE_TRAINING_CLUSTERING = True
    DSCD_MIN_LETTERS                = 3
    DSCD_MIN_LETTER_FRACTION        = 0.6
    print("[CELL3] WARNING: Using default DSCD config")

try:
    DEBUG_DISCOVERY = bool(DEBUG_DISCOVERY)
except NameError:
    DEBUG_DISCOVERY = False

try:
    MAX_TOKENS_PER_DISCOVERY = int(globals().get('MAX_TOKENS_PER_DISCOVERY', 150))
except Exception:
    MAX_TOKENS_PER_DISCOVERY = 150

try:
    HOMOGRAPH_REFERENCE_LIST_BN = set(HOMOGRAPH_REFERENCE_LIST_BN)
    print(f"[CELL3] Loaded reference list for evaluation: {len(HOMOGRAPH_REFERENCE_LIST_BN)} words")
except (NameError, TypeError):
    HOMOGRAPH_REFERENCE_LIST_BN = {
        'কল', 'কাল', 'পাতা', 'ফল', 'বার', 'হার', 'তারা',
        'পড়া', 'দেখা', 'চলা', 'ধরা', 'অর্থ', 'শব্দ', 'মুখ',
        'তোলা', 'বাঁচা', 'মারা', 'উত্তর', 'পাত্র', 'বেলা', 'গান',
        'নাম', 'বল', 'চাল', 'কলা', 'ধারা', 'পত্র', 'রাগ', 'রস',
        'তীর', 'জমা', 'মান', 'দাবি', 'আসন', 'সাড়া', 'বসা', 'পদ',
        'অংশ', 'মোড়', 'ঘর', 'মন', 'ব্যাংক'
    }
    print("[CELL3] Using default reference list")

DSCD_MAX_CLUSTERING_POINTS = 500

BENGALI_PUNCT_SET = set(['।', '॥'])
COMMON_PUNCT_SET  = set(['.', ',', '!', '?', ';', ':', '-', '—',
                          '"', "'", '(', ')', '[', ']', '{', '}'])
PUNCT_SET = BENGALI_PUNCT_SET | COMMON_PUNCT_SET

# ==============================================================================
# [3B] BENGALI_FUNCTION_WORDS — extended set, defined BEFORE class
# ==============================================================================
BENGALI_FUNCTION_WORDS: Set[str] = {
    'আমি', 'তুমি', 'সে', 'আমরা', 'তোমরা', 'আপনি', 'আপনারা',
    'তার', 'তাকে', 'আমাকে', 'তোমাকে', 'তাদের', 'তাদেরকে',
    'এটা', 'এটি', 'ওটা', 'সেটা', 'এই', 'ওই', 'সেই', 'তাই', 'যা', 'যে',
    'এবং', 'ও', 'কিন্তু', 'তবে', 'অথবা', 'বা', 'যদি', 'তাহলে',
    'নাহলে', 'কিংবা',
    'কারণ', 'যখন', 'তখন', 'যদিও', 'তথাপি', 'সুতরাং', 'অতএব',
    'থেকে', 'দিয়ে', 'সাথে', 'জন্য', 'কাছে', 'উপর', 'নিচে',
    'পাশে', 'ভেতরে', 'ওপরে', 'নিকট',
    'মধ্যে', 'বাইরে', 'পরে', 'আগে', 'ছাড়া', 'পর্যন্ত', 'দিকে',
    'হয়', 'হলো', 'ছিল', 'আছে', 'ছিলেন', 'হবে', 'হচ্ছে',
    'হতো', 'হত', 'হয়েছিল',
    'করা', 'করে', 'করি', 'করেন', 'করেছে', 'করছে', 'হয়েছে',
    'না', 'নয়', 'নি', 'হয়নি', 'নেই',
    'কি', 'কে', 'কোথায়', 'কেন', 'কীভাবে',
    'এক', 'দুই', 'তিন', 'চার', 'পাঁচ', 'ছয়', 'সাত', 'আট', 'নয়', 'দশ',
}


# ==============================================================================
# Module-level token helpers
# ==============================================================================

def is_punctuation_only(token: str) -> bool:
    if not token or not isinstance(token, str):
        return False
    clean = token.replace("▁", "").replace("Ġ", "").replace("##", "").strip()
    if not clean:
        return False
    if clean in BENGALI_PUNCT_SET:
        return True
    if clean in COMMON_PUNCT_SET:
        return True
    if len(clean) == 1 and not clean.isalnum():
        return True
    return all(c in PUNCT_SET for c in clean)


def clean_token_for_dscd(token: str) -> str:
    if not token or not isinstance(token, str):
        return ""
    cleaned = token.replace("▁", "").replace("Ġ", "").replace("##", "").strip()
    for punct in list(PUNCT_SET):
        cleaned = cleaned.replace(punct, "")
    return cleaned.lower()


def normalize_token_key(token: str) -> str:
    return clean_token_for_dscd(token)


def is_word_token(
    token: str,
    min_letters: int = 2,
    min_letter_fraction: float = 0.6,
) -> bool:
    if not token or not isinstance(token, str):
        return False
    token = token.strip()
    if not token:
        return False
    letters = 0
    total   = 0
    for ch in token:
        cat = unicodedata.category(ch)
        if cat.startswith('L'):
            letters += 1
        if not ch.isspace():
            total += 1
    if total == 0:
        return False
    if letters < min_letters:
        return False
    if (letters / total) < min_letter_fraction:
        return False
    return True


# ==============================================================================
# [3A] is_indic_subword_fragment — fixed flag logic + added Odia vowel range
# ==============================================================================
def is_indic_subword_fragment(token: str) -> bool:
    if not token or not isinstance(token, str):
        return False
    token = token.strip()
    if not token:
        return False

    only_combining_marks = True
    has_virama           = False
    letter_count         = 0

    virama_chars = [
        '\u094D', '\u09CD', '\u0A4D', '\u0ACD',
        '\u0B4D', '\u0BCD', '\u0C4D', '\u0CCD', '\u0D4D',
    ]

    for ch in token:
        cat = unicodedata.category(ch)
        if cat.startswith('L'):
            letter_count         += 1
            only_combining_marks  = False
        elif cat not in ('Mn', 'Mc'):
            only_combining_marks  = False
        if ch in virama_chars:
            has_virama = True

    if only_combining_marks:
        return True
    if has_virama and len(token) <= 2:
        return True
    if letter_count == 0:
        return True

    vowel_modifier_ranges = [
        ('\u093E', '\u094C'),
        ('\u09BE', '\u09CC'),
        ('\u0ABE', '\u0ACC'),
        ('\u0B3E', '\u0B4C'),
        ('\u0BBE', '\u0BCC'),
        ('\u0C3E', '\u0C4C'),
        ('\u0CBE', '\u0CCC'),
    ]
    modifier_count = 0
    for ch in token:
        for start, end in vowel_modifier_ranges:
            if start <= ch <= end:
                modifier_count += 1
                break

    if modifier_count > 0 and modifier_count == len(token):
        return True
    if len(token) <= 2 and modifier_count > 0:
        return True
    return False


# ==============================================================================
# MemoryEfficientPrototypeStore
# ==============================================================================

class MemoryEfficientPrototypeStore:
    def __init__(
        self,
        embed_dim: int,
        max_protos: Optional[int] = None,
        buffer_size: Optional[int] = None,
    ):
        if max_protos is None:
            max_protos = DSCD_MAX_PROTOS
        if buffer_size is None:
            buffer_size = DSCD_BUFFER_SIZE
        self.embed_dim              = embed_dim
        self.max_protos             = int(max_protos)
        self.buffer_size            = int(buffer_size)
        self.centroids: List[torch.Tensor] = []
        self.counts: List[int]             = []
        self.creation_time: List[float]    = []
        self.distances: List[float]        = []
        self.mu                            = 0.0
        self.tau                           = 1e-6
        self.alpha                         = 0.1
        self.n_updates: int                = 0
        self.labels: Optional[torch.Tensor] = None

    def state_dict(self) -> dict:
        return {
            "centroids": [c.cpu().clone() for c in self.centroids],
            "counts":    list(self.counts),
            "mu":        float(self.mu)  if self.mu  is not None else None,
            "tau":       float(self.tau) if self.tau is not None else None,
            "n_updates": int(getattr(self, "n_updates", 0)),
        }

    def load_state_dict(self, state: dict, device=None) -> None:
        if device is None:
            try:
                device = DEVICE
            except NameError:
                device = torch.device("cpu")
        self.centroids = [c.to(device) for c in state.get("centroids", [])]
        self.counts    = list(state.get("counts", []))
        self.mu        = state.get("mu",  None)
        self.tau       = state.get("tau", None)
        self.n_updates = int(state.get("n_updates", 0))

    def add_prototype(
        self,
        vector: torch.Tensor,
        current_time: Optional[float] = None,
        count: int = 1,
    ) -> None:
        if current_time is None:
            current_time = time.time()
        v = vector.detach().cpu().clone()
        if len(self.centroids) < self.max_protos:
            self.centroids.append(v)
            self.counts.append(int(count))
            self.creation_time.append(float(current_time))
        else:
            min_idx = int(np.argmin(self.counts)) if self.counts else 0
            self.centroids[min_idx]     = v
            self.counts[min_idx]        = int(count)
            self.creation_time[min_idx] = float(current_time)

    def update_prototype(
        self,
        idx: int,
        vector: torch.Tensor,
        eta: float = 0.05,
        assignment_distance: Optional[float] = None,
    ) -> None:
        if idx < 0 or idx >= len(self.centroids):
            self.add_prototype(vector, time.time(), count=1)
            return
        old_centroid        = self.centroids[idx]
        new_vector          = vector.detach().cpu()
        self.centroids[idx] = (1.0 - eta) * old_centroid + eta * new_vector
        self.counts[idx]    = int(self.counts[idx]) + 1
        self.n_updates      += 1
        if assignment_distance is not None:
            self.update_rolling_stats(float(assignment_distance))

    def update_rolling_stats(self, d: float) -> None:
        if not self.distances:
            self.mu        = float(d)
            self.tau       = max(1e-6, float(d) * 0.1)
            self.distances = [float(d)]
            return
        prev_mu  = self.mu
        self.mu  = (1 - self.alpha) * self.mu  + self.alpha * float(d)
        self.tau = (1 - self.alpha) * self.tau  + self.alpha * abs(float(d) - prev_mu)
        self.distances.append(float(d))
        if len(self.distances) > 50:
            self.distances.pop(0)

    def get_adaptive_threshold(self, lam: float = 1.0) -> float:
        return float(self.mu + lam * max(self.tau, 1e-4))

    def size(self) -> int:
        return len(self.centroids)

    def ensure_consistency(self) -> None:
        n = len(self.centroids)
        if len(self.counts) != n:
            self.counts = (
                self.counts[:n]
                if len(self.counts) > n
                else self.counts + [1] * (n - len(self.counts))
            )
        if len(self.creation_time) != n:
            self.creation_time = (
                self.creation_time[:n]
                if len(self.creation_time) > n
                else self.creation_time + [time.time()] * (n - len(self.creation_time))
            )


# ==============================================================================
# MemoryEfficientDSCDOnline — main DSCD class
# ==============================================================================

class MemoryEfficientDSCDOnline(nn.Module):
    def __init__(
        self,
        embed_dim: int,
        tokenizer=None,
        buffer_size: Optional[int]                = None,
        max_protos: Optional[int]                 = None,
        n_min: Optional[int]                      = None,
        dispersion_threshold: Optional[float]     = None,
        language: str                             = "bn",
        enable_training_clustering: Optional[bool] = None,
        max_clustering_points: Optional[int]      = None,
        max_candidates_per_step: int              = 2,
        dscd_min_letters: int                     = 3,
        dscd_min_letter_fraction: float           = 0.6,
    ):
        super().__init__()
        if buffer_size is None:
            buffer_size = DSCD_BUFFER_SIZE
        if max_protos is None:
            max_protos = DSCD_MAX_PROTOS
        if n_min is None:
            n_min = DSCD_N_MIN
        if dispersion_threshold is None:
            dispersion_threshold = DSCD_DISPERSION_THRESHOLD
        if max_clustering_points is None:
            max_clustering_points = DSCD_MAX_CLUSTERING_POINTS
        if enable_training_clustering is None:
            enable_training_clustering = DSCD_ENABLE_TRAINING_CLUSTERING

        self.embed_dim                = int(embed_dim)
        self.buffer_size              = int(buffer_size)
        self.max_protos               = int(max_protos)
        self.n_min                    = int(n_min)
        self.dispersion_threshold     = float(dispersion_threshold)
        self.language                 = language
        self.tokenizer                = tokenizer
        self.dscd_min_letters         = int(dscd_min_letters)
        self.dscd_min_letter_fraction = float(dscd_min_letter_fraction)

        try:
            if tokenizer is not None and 'get_tokenizer_special_tokens' in globals():
                self.special_tokens = get_tokenizer_special_tokens(tokenizer)
            else:
                self.special_tokens = (
                    set(getattr(tokenizer, 'all_special_tokens', []))
                    if tokenizer is not None
                    else set()
                )
        except Exception:
            self.special_tokens = set()

        self.dscd_allowed_tokens: Set[str] = set()
        self.dscd_ignored_tokens: Set[str] = set()
        self.dscd_cache_max_size           = 10000

        self.prototype_stores: Dict[str, MemoryEfficientPrototypeStore] = {}
        self.buffers: Dict[str, deque]            = {}
        self.discovered_log: List[Dict[str, Any]] = []
        self.discovered_homographs: Set[str]      = set()

        # [3H] Sense registry: token -> {sense_id -> {centroid, count, label}}
        self.homograph_sense_registry: Dict[str, Dict[str, Any]] = {}

        self.last_periodic_check = 0
        self.cleanup_counter     = 0

        # [3E] Step counter for force_observe throttle
        self._force_observe_step_counter: int = 0

        self.dispersion_cache: Dict[str, float]        = {}
        self.dispersion_last_updated: Dict[str, float] = {}
        self.dispersion_lock  = threading.Lock()
        self.clustering_lock  = threading.Lock()
        self.buffer_lock      = threading.Lock()

        from collections import deque as thread_deque
        self.active_threads = thread_deque(maxlen=100)
        self.thread_lock    = threading.Lock()

        self.last_cluster_time: Dict[str, float] = {}
        self.cluster_cooldown_seconds            = 5.0

        self.enable_training_clustering = bool(enable_training_clustering)
        self.discovery_count            = 0
        self.discovery_times: List[float] = []
        self.clustered_tokens: Set[str]   = set()

        self.cluster_stats: Dict[str, Dict[str, Any]] = {}

        self.span_head = nn.Sequential(
            nn.Linear(self.embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
        )
        self.sigma_net = nn.Sequential(
            nn.Linear(self.embed_dim, 16),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(16, 1),
        )

        self.gate_w = nn.Parameter(torch.tensor(1.0))
        self.gate_b = nn.Parameter(torch.tensor(0.4))
        self.gamma  = nn.Parameter(torch.tensor(0.3))

        self.max_clustering_points   = int(max_clustering_points)
        self.max_candidates_per_step = int(max_candidates_per_step)

        try:
            self.homograph_reference_list = set(
                str(w).lower() for w in HOMOGRAPH_REFERENCE_LIST_BN
            )
        except Exception:
            self.homograph_reference_list = set()

    # ── Serialisation ──────────────────────────────────────────────────────────

    def state_dict(self, destination=None, prefix='', keep_vars=False):
        state = super().state_dict(
            destination=destination, prefix=prefix, keep_vars=keep_vars
        )
        plain_stores = {}
        for token, store in self.prototype_stores.items():
            plain_stores[token] = {
                'centroids':     [c.cpu() for c in store.centroids]   if hasattr(store, 'centroids')     else [],
                'counts':        list(store.counts)                    if hasattr(store, 'counts')         else [],
                'creation_time': list(store.creation_time)             if hasattr(store, 'creation_time')  else [],
                'mu':            float(store.mu)                       if hasattr(store, 'mu')             else 0.0,
                'tau':           float(store.tau)                      if hasattr(store, 'tau')            else 0.0,
                'size':          int(store.size())                     if hasattr(store, 'size')           else 0,
                'n_updates':     int(getattr(store, 'n_updates', 0)),
            }

        state['prototype_stores_data'] = plain_stores
        state['discovered_homographs'] = list(self.discovered_homographs)
        return state

    def load_state_dict(self, state_dict, strict=True):
        state_dict = dict(state_dict)

        plain_stores = state_dict.pop('prototype_stores_data', {})
        discovered   = state_dict.pop('discovered_homographs', [])

        super().load_state_dict(state_dict, strict=strict)

        if not plain_stores:
            print("[DSCD] WARNING: Empty prototype_stores in checkpoint")
            return

        self.prototype_stores      = {}
        self.discovered_homographs = set(discovered)

        for token, store_dict in plain_stores.items():
            store = MemoryEfficientPrototypeStore(
                embed_dim=self.embed_dim,
                max_protos=self.max_protos,
                buffer_size=self.buffer_size,
            )
            centroids_data  = store_dict.get('centroids', [])
            store.centroids = []
            for c in centroids_data:
                if isinstance(c, torch.Tensor):
                    store.centroids.append(c)
                else:
                    store.centroids.append(torch.tensor(c))
            store.counts        = store_dict.get('counts',        [])
            store.creation_time = store_dict.get('creation_time', [])
            store.mu            = store_dict.get('mu',       0.0)
            store.tau           = store_dict.get('tau',      0.0)
            store.n_updates     = int(store_dict.get('n_updates', 0))
            store.ensure_consistency()
            self.prototype_stores[token] = store

        print(
            f"[DSCD] Loaded {len(self.prototype_stores)} tokens, "
            f"{sum(s.size() for s in self.prototype_stores.values())} prototypes"
        )

    # ── Static helpers ─────────────────────────────────────────────────────────

    @staticmethod
    def clean_token(token):
        return clean_token_for_dscd(str(token))

    # ── Sense-validity checks ──────────────────────────────────────────────────

    def is_valid_multi_sense(self, token) -> bool:
        if token not in self.prototype_stores:
            return False
        store             = self.prototype_stores[token]
        total_occurrences = sum(store.counts) if hasattr(store, 'counts') else 0
        min_per_proto     = (
            min(store.counts)
            if hasattr(store, 'counts') and store.counts
            else 0
        )
        return store.size() >= 2 and total_occurrences >= 10 and min_per_proto >= 2

    def is_multi_sense_store(self, store: MemoryEfficientPrototypeStore) -> bool:
        if store is None:
            return False
        k = store.size()
        if k < 2:
            return False
        counts = store.counts if store.counts else [1] * k
        strong = sum(1 for c in counts if c >= max(2, self.n_min // 2))
        if strong < 2:
            return False
        try:
            cents = []
            for c in store.centroids:
                if isinstance(c, torch.Tensor):
                    cents.append(c.cpu().numpy())
                else:
                    cents.append(np.asarray(c, dtype=np.float32))
            if len(cents) < 2:
                return False
            cents = np.stack(cents, axis=0)
            dists = np.linalg.norm(
                cents[:, None, :] - cents[None, :, :], axis=-1
            )
            tri = dists[np.triu_indices(len(cents), k=1)]
            if tri.size == 0:
                return False
            min_dist = float(tri.min())
            base     = max(store.tau, 1e-3)
            return min_dist > base * DSCD_NEWSENSE_LAMBDA
        except Exception:
            return True

    # ── [3B] Function-word clustering gate ────────────────────────────────────

    def should_cluster_token(self, token_str: str) -> bool:
        if not token_str or not isinstance(token_str, str):
            return False
        clean = clean_token_for_dscd(token_str)
        if clean in BENGALI_FUNCTION_WORDS:
            return False
        return True

    # ── Discovery helpers ──────────────────────────────────────────────────────

    def discover_homographs_for_tokens(
        self,
        token_names: List[str],
        min_cluster_samples: int,
        dispersion_threshold: float,
        global_step: int,
    ) -> int:
        discovered_in_run: List[str] = []

        for idx, token in enumerate(token_names):
            try:
                if is_punctuation_only(token):
                    continue

                if not self.should_cluster_token(token):
                    continue

                success = self.cluster_buffer_to_prototypes_hierarchical(token)
                if success:
                    store = self.prototype_stores.get(token)
                    if store and store.size() >= 2:
                        clean_token = normalize_token_key(token)
                        self.discovered_homographs.add(clean_token)
                        discovered_in_run.append(clean_token)
            except Exception:
                continue

        try:
            self.discovered_log.append({
                'timestamp':            time.time(),
                'global_step':          global_step,
                'candidates_processed': len(token_names),
                'discovered_count':     len(discovered_in_run),
                'homographs':           discovered_in_run,
                'total_discovered':     len(self.discovered_homographs),
            })
        except Exception:
            pass

        return len(discovered_in_run)

    def discover_homographs(
        self,
        min_cluster_samples: Optional[int]    = None,
        dispersion_threshold: Optional[float] = None,
        max_candidates: int                   = 500,
    ) -> int:
        if min_cluster_samples is None:
            min_cluster_samples = self.n_min
        if dispersion_threshold is None:
            dispersion_threshold = self.dispersion_threshold

        candidates: List[Tuple[str, float, int, float]] = []

        with self.buffer_lock:
            for token, buffer in self.buffers.items():
                if is_punctuation_only(token):
                    continue

                if not self.should_cluster_token(token):
                    continue

                buffer_size = len(buffer)
                if buffer_size >= max(min_cluster_samples + 2, 10):
                    clean_token = clean_token_for_dscd(token)
                    if clean_token in HOMOGRAPH_REFERENCE_LIST_BN:
                        dispersion = max(
                            self.get_dispersion(token),
                            dispersion_threshold * 1.15,
                        )
                        if DEBUG_DISCOVERY:
                            print(
                                f"[DSCD-PRIORITY] Boosting reference homograph "
                                f"'{token}' dispersion to {dispersion:.3f}"
                            )
                    else:
                        dispersion = self.get_dispersion(token)

                    if dispersion >= dispersion_threshold:
                        candidates.append((token, dispersion * buffer_size, buffer_size, dispersion))

        if not candidates:
            return 0

        candidates.sort(key=lambda x: x[1], reverse=True)
        candidates = candidates[:max_candidates]

        discovered: List[str] = []

        for token, score, buf_size, disp in candidates:
            try:
                with self.clustering_lock:
                    success = self.cluster_buffer_to_prototypes_hierarchical(token)
                    if success:
                        store = self.prototype_stores.get(token)
                        if store and store.size() >= 2:
                            clean_token = normalize_token_key(token)
                            self.discovered_homographs.add(clean_token)
                            discovered.append(clean_token)
            except Exception:
                continue

        try:
            self.discovered_log.append({
                'timestamp':  time.time(),
                'candidates': len(candidates),
                'discovered': len(discovered),
                'homographs': discovered[:20],
            })
        except Exception:
            pass

        return len(discovered)

    # ── [DSCD-4] Prototype separation (= ambiguity score) ─────────────────────

    def compute_prototype_separation(self, token_id: str) -> float:
        store = self.prototype_stores.get(token_id)
        if store is None or store.size() < 2:
            return 0.0
        try:
            pairs = sorted(
                zip(store.counts, store.centroids),
                key=lambda x: x[0],
                reverse=True,
            )
            def _to_tensor(c):
                if isinstance(c, torch.Tensor):
                    return c.float()
                return torch.from_numpy(np.asarray(c, dtype=np.float32))

            v1      = _to_tensor(pairs[0][1])
            v2      = _to_tensor(pairs[1][1])
            cos_sim = F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()
            return float(np.clip(1.0 - cos_sim, 0.0, 2.0))
        except Exception:
            return 0.0

    # ── [DSCD-5] Automatic homograph detection ────────────────────────────────

    def is_automatic_homograph(self, token_id: str, token_str: str) -> bool:
        if not self.should_cluster_token(token_str):
            return False
        store = self.prototype_stores.get(token_id)
        if store is None or store.size() < 2:
            return False
        min_count = min(store.counts) if store.counts else 0
        if min_count < 5:
            return False
        return self.compute_prototype_separation(token_id) > 0.30

    # ── [DSCD-6] Auto-split prototype ─────────────────────────────────────────

    def maybe_split_prototype(self, token_id: str, proto_index: int) -> bool:
        store = self.prototype_stores.get(token_id)
        if store is None:
            return False
        if proto_index < 0 or proto_index >= store.size():
            return False
        if store.size() >= self.max_protos:
            return False
        count = store.counts[proto_index] if store.counts else 0
        if count <= 20:
            return False
        tau = getattr(store, 'tau', 0.0)
        if tau <= 0.40:
            return False
        try:
            centroid = store.centroids[proto_index]
            v = (
                centroid.float().clone()
                if isinstance(centroid, torch.Tensor)
                else torch.from_numpy(np.asarray(centroid, dtype=np.float32))
            )
            noise        = torch.randn_like(v) * (tau * 0.1)
            new_centroid = v + noise
            v_norm       = v.norm() + 1e-8
            new_centroid = F.normalize(new_centroid, dim=0) * v_norm

            half_count                    = max(1, count // 2)
            store.counts[proto_index]     = half_count
            store.add_prototype(new_centroid, time.time(), count=half_count)

            if DEBUG_DISCOVERY:
                print(
                    f"[DSCD] Auto-split proto {proto_index} of '{token_id}': "
                    f"tau={tau:.3f}, original_count={count}"
                )
            return True
        except Exception:
            return False

    # ── [DSCD-7] Contrastive prototype loss ───────────────────────────────────

    def compute_dscd_contrastive_loss(self, token_id: str) -> torch.Tensor:
        store = self.prototype_stores.get(token_id)
        if store is None or store.size() < 2:
            return torch.tensor(0.0)
        try:
            centroids = []
            for c in store.centroids:
                if isinstance(c, torch.Tensor):
                    centroids.append(c.float())
                else:
                    centroids.append(
                        torch.from_numpy(np.asarray(c, dtype=np.float32))
                    )
            C      = torch.stack(centroids, dim=0)
            C_norm = F.normalize(C, dim=-1)
            sim    = torch.mm(C_norm, C_norm.t())
            K      = C_norm.size(0)
            margin = 0.70
            loss   = torch.tensor(0.0)
            count  = 0
            for i in range(K):
                for j in range(i + 1, K):
                    hinge = torch.clamp(sim[i, j] - margin, min=0.0)
                    loss  = loss + hinge
                    count += 1
            if count > 0:
                loss = loss / count
            return loss
        except Exception:
            return torch.tensor(0.0)

    # ── [DSCD-9] Quality score ─────────────────────────────────────────────────

    def get_quality_score(self) -> float:
        try:
            total_tokens = len(self.prototype_stores)
            if total_tokens == 0:
                return 0.0

            multi_sense = sum(
                1 for s in self.prototype_stores.values()
                if self.is_multi_sense_store(s)
            )
            multi_sense_ratio = multi_sense / total_tokens

            if self.homograph_reference_list:
                ref_found = 0
                for h in self.homograph_reference_list:
                    for k in self.prototype_stores:
                        if clean_token_for_dscd(str(k)) == h:
                            if self.is_multi_sense_store(self.prototype_stores[k]):
                                ref_found += 1
                            break
                coverage = ref_found / max(1, len(self.homograph_reference_list))
            else:
                coverage = multi_sense_ratio

            return float(np.clip(0.6 * coverage + 0.4 * multi_sense_ratio, 0.0, 1.0))
        except Exception:
            return 0.0

    # ── [DSCD-10] Homograph rate ───────────────────────────────────────────────

    def get_homograph_rate(self) -> float:
        try:
            total_tokens = len(self.prototype_stores)
            if total_tokens == 0:
                return 0.0
            homograph_count = sum(
                1
                for token_id in self.prototype_stores
                if self.is_automatic_homograph(token_id, str(token_id))
            )
            return float(homograph_count / total_tokens)
        except Exception:
            return 0.0

    # ── Dispersion computation ─────────────────────────────────────────────────

    def get_dispersion(self, token_type: str) -> float:
        with self.dispersion_lock:
            if token_type in self.dispersion_cache:
                try:
                    last_update = self.dispersion_last_updated.get(token_type, 0.0)
                    if time.time() - last_update < 3600:
                        return self.dispersion_cache[token_type]
                except Exception:
                    pass

        embeddings: List[np.ndarray] = []
        with self.buffer_lock:
            if token_type not in self.buffers:
                return 0.0
            buf_len = len(self.buffers[token_type])
            if buf_len < 2:
                return 0.05 if buf_len == 1 else 0.0
            for emb in self.buffers[token_type]:
                try:
                    if isinstance(emb, torch.Tensor):
                        embeddings.append(emb.cpu().numpy())
                    else:
                        embeddings.append(np.asarray(emb, dtype=np.float32))
                except Exception:
                    continue

        if len(embeddings) < 2:
            return 0.05 if len(embeddings) == 1 else 0.0

        try:
            embeddings_np = np.stack(embeddings, axis=0)
            centroid      = embeddings_np.mean(axis=0)
            distances     = np.linalg.norm(embeddings_np - centroid[None, :], axis=1)
            dispersion    = float(distances.std())
        except Exception:
            return 0.0

        with self.dispersion_lock:
            self.dispersion_cache[token_type]        = dispersion
            self.dispersion_last_updated[token_type] = time.time()

        return dispersion

    # ── Validation ────────────────────────────────────────────────────────────

    def validate_prototypes(
        self,
        homograph_list: Optional[List[str]] = None,
        cluster_missing: bool = True,
    ) -> Dict[str, Any]:
        if homograph_list is None:
            try:
                homograph_list = list(HOMOGRAPH_REFERENCE_LIST_BN)
            except Exception:
                homograph_list = ['কল', 'পাতা', 'ফল', 'মান']

        print("=" * 80)
        print("DSCD-VALIDATION: Prototype Quality Check")
        print("=" * 80)

        validation_results: Dict[str, Any] = {
            'total_tokens':              len(self.prototype_stores),
            'total_prototypes':          0,
            'multi_sense_tokens':        0,
            'homographs_found':          0,
            'homographs_missing':        [],
            'avg_prototypes_per_token':  0.0,
            'avg_samples_per_prototype': 0.0,
            'quality_score':             0.0,
        }

        total_samples = 0
        for token, store in self.prototype_stores.items():
            validation_results['total_prototypes'] += len(store.centroids)
            if self.is_multi_sense_store(store):
                validation_results['multi_sense_tokens'] += 1
            try:
                total_samples += sum(store.counts)
            except Exception:
                pass

        if validation_results['total_tokens'] > 0:
            validation_results['avg_prototypes_per_token'] = (
                validation_results['total_prototypes'] /
                validation_results['total_tokens']
            )
        if validation_results['total_prototypes'] > 0:
            validation_results['avg_samples_per_prototype'] = (
                total_samples / validation_results['total_prototypes']
            )

        print("VALIDATION: Reference Homograph Coverage")
        print("-" * 80)

        missing_tokens_to_cluster: List[str] = []

        for homograph in homograph_list:
            clean_h      = clean_token_for_dscd(homograph)
            found        = False
            found_key    = None
            found_protos = 0

            for key in self.prototype_stores.keys():
                if clean_token_for_dscd(str(key)) == clean_h:
                    found        = True
                    found_key    = key
                    found_protos = len(self.prototype_stores[key].centroids)
                    break

            if found and self.is_multi_sense_store(self.prototype_stores[found_key]):
                validation_results['homographs_found'] += 1
                try:
                    counts = self.prototype_stores[found_key].counts
                    print(f"  OK {homograph} - {found_protos} prototypes (counts={counts})")
                except Exception:
                    print(f"  OK {homograph} - {found_protos} prototypes")
            elif found and found_protos == 1:
                validation_results['homographs_missing'].append(homograph)
                print(f"  WARN {homograph} - Only 1 prototype")
                if cluster_missing:
                    missing_tokens_to_cluster.append(found_key)
            else:
                validation_results['homographs_missing'].append(homograph)
                print(f"  MISS {homograph} - NOT FOUND")
                if cluster_missing:
                    for buf_key in self.buffers.keys():
                        if clean_token_for_dscd(str(buf_key)) == clean_h:
                            if len(self.buffers[buf_key]) >= max(self.n_min + 2, 10):
                                print(f"      - Found in buffer, will cluster")
                                missing_tokens_to_cluster.append(buf_key)
                            break

        if cluster_missing and missing_tokens_to_cluster:
            print(f"\nVALIDATION: Clustering {len(missing_tokens_to_cluster)} missing tokens...")
            for token in missing_tokens_to_cluster:
                try:
                    with self.clustering_lock:
                        self.cluster_buffer_to_prototypes_hierarchical(token)
                        if (
                            token in self.prototype_stores
                            and self.is_multi_sense_store(self.prototype_stores[token])
                        ):
                            print(f"  OK Successfully clustered: {token}")
                except Exception as e:
                    print(f"  FAIL Failed to cluster {token}: {e}")

        homograph_coverage = (
            validation_results['homographs_found'] / len(homograph_list)
            if homograph_list else 0.0
        )
        multi_sense_ratio = (
            validation_results['multi_sense_tokens'] /
            validation_results['total_tokens']
            if validation_results['total_tokens'] > 0 else 0.0
        )
        validation_results['quality_score'] = (
            homograph_coverage * 0.6 + multi_sense_ratio * 0.4
        )

        print("-" * 80)
        print("VALIDATION: Summary")
        print(f"  - Total tokens:      {validation_results['total_tokens']}")
        print(f"  - Total prototypes:  {validation_results['total_prototypes']}")
        print(f"  - Multi-sense tokens:{validation_results['multi_sense_tokens']}")
        print(f"  - Reference found:   {validation_results['homographs_found']}/{len(homograph_list)}")
        print(f"  - Quality Score:     {validation_results['quality_score']*100:.2f}%")
        print("=" * 80)
        return validation_results

    # ── Token tracking / key resolution ───────────────────────────────────────

    def should_track_token(self, token_text: str) -> bool:
        if not token_text or not isinstance(token_text, str):
            return False

        if len(self.dscd_allowed_tokens) > self.dscd_cache_max_size:
            self.dscd_allowed_tokens.clear()
        if len(self.dscd_ignored_tokens) > self.dscd_cache_max_size:
            self.dscd_ignored_tokens.clear()

        if token_text in self.dscd_allowed_tokens:
            return True
        if token_text in self.dscd_ignored_tokens:
            return False

        if not getattr(self, 'training', False):
            if token_text in self.prototype_stores:
                self.dscd_allowed_tokens.add(token_text)
                return True
            clean = clean_token_for_dscd(token_text)
            if clean and clean in self.prototype_stores:
                self.dscd_allowed_tokens.add(token_text)
                return True

        if token_text in self.special_tokens:
            self.dscd_ignored_tokens.add(token_text)
            return False

        if is_punctuation_only(token_text):
            self.dscd_ignored_tokens.add(token_text)
            return False

        clean = clean_token_for_dscd(token_text)
        if not clean:
            self.dscd_ignored_tokens.add(token_text)
            return False

        if len(clean) < self.dscd_min_letters:
            self.dscd_ignored_tokens.add(token_text)
            return False

        if not any(c.isalpha() for c in clean):
            self.dscd_ignored_tokens.add(token_text)
            return False

        if clean.isdigit():
            self.dscd_ignored_tokens.add(token_text)
            return False

        try:
            has_indic = (
                any('\u0900' <= c <= '\u0DFF' for c in clean)
                or any('\u0980' <= c <= '\u09FF' for c in clean)
            )
            if has_indic:
                if len(clean) >= self.dscd_min_letters:
                    self.dscd_allowed_tokens.add(token_text)
                    return True
                else:
                    self.dscd_ignored_tokens.add(token_text)
                    return False
        except Exception:
            pass

        if is_word_token(
            clean,
            min_letters=self.dscd_min_letters,
            min_letter_fraction=self.dscd_min_letter_fraction,
        ):
            self.dscd_allowed_tokens.add(token_text)
            return True

        self.dscd_ignored_tokens.add(token_text)
        return False

    # ── [3C] canonical_token_key — fragment check on cleaned form only ─────────

    def canonical_token_key(
        self,
        raw_token: str,
        token_word_map: Optional[Dict[int, Optional[str]]],
        idx: int,
    ) -> Optional[str]:
        canonical: Optional[str] = None
        try:
            if (
                token_word_map
                and isinstance(token_word_map, dict)
                and idx in token_word_map
                and token_word_map[idx]
            ):
                word      = str(token_word_map[idx]).strip()
                canonical = clean_token_for_dscd(word)
                if canonical and len(canonical) >= self.dscd_min_letters:
                    if is_indic_subword_fragment(canonical):
                        canonical = None
                    else:
                        has_indic = (
                            any('\u0900' <= c <= '\u0DFF' for c in canonical)
                            or any('\u0980' <= c <= '\u09FF' for c in canonical)
                        )
                        if has_indic:
                            return canonical
        except Exception:
            pass

        canonical = clean_token_for_dscd(raw_token)
        if not canonical or len(canonical) < self.dscd_min_letters:
            return None

        has_indic = (
            any('\u0900' <= c <= '\u0DFF' for c in canonical)
            or any('\u0980' <= c <= '\u09FF' for c in canonical)
        )
        if not has_indic:
            return None

        if is_indic_subword_fragment(canonical):
            return None

        return canonical

    # ── [3D] force_observe_reference_homographs — new method ──────────────────

    def force_observe_reference_homographs(self, global_step: int) -> int:
        forced_count = 0
        try:
            buffer_keys_snapshot: List[str] = []
            with self.buffer_lock:
                buffer_keys_snapshot = list(self.buffers.keys())

            for token in buffer_keys_snapshot:
                try:
                    clean_tok = clean_token_for_dscd(token)
                    if clean_tok not in self.homograph_reference_list:
                        continue
                    if not self.should_cluster_token(token):
                        continue

                    buf_len = 0
                    with self.buffer_lock:
                        buf_len = len(self.buffers.get(token, []))

                    if buf_len < max(self.n_min + 2, 10):
                        continue

                    store = self.prototype_stores.get(token)
                    if store is not None and store.size() >= 2:
                        continue

                    with self.clustering_lock:
                        success = self.cluster_buffer_to_prototypes_hierarchical(token)
                        if success:
                            updated_store = self.prototype_stores.get(token)
                            if updated_store and updated_store.size() >= 2:
                                self.discovered_homographs.add(clean_tok)
                                forced_count += 1
                                if DEBUG_DISCOVERY:
                                    print(
                                        f"[DSCD-FORCE] Reference homograph '{token}' "
                                        f"force-clustered at step {global_step}: "
                                        f"{updated_store.size()} prototypes"
                                    )
                except Exception:
                    continue
        except Exception:
            pass
        return forced_count

    # ── [3H] Sense registry accessor ──────────────────────────────────────────

    def get_sense_info(self, token: str) -> Dict[str, Any]:
        clean_tok = clean_token_for_dscd(token)
        if clean_tok in self.homograph_sense_registry:
            return self.homograph_sense_registry[clean_tok]
        if token in self.homograph_sense_registry:
            return self.homograph_sense_registry[token]
        return {}

    # ── Memory / thread management ─────────────────────────────────────────────

    def cleanup_threads(self) -> None:
        try:
            with self.thread_lock:
                alive = [th for th in list(self.active_threads) if th.is_alive()]
                self.active_threads.clear()
                self.active_threads.extend(alive)
        except Exception:
            pass

    def cleanup_memory(self) -> None:
        try:
            for token_type, buffer in list(self.buffers.items()):
                if len(buffer) > int(self.buffer_size * 1.5):
                    while len(buffer) > self.buffer_size:
                        buffer.popleft()
            try:
                now     = time.time()
                expired = [
                    k for k, v in self.dispersion_last_updated.items()
                    if now - v > 3600
                ]
                for k in expired:
                    self.dispersion_cache.pop(k, None)
                    self.dispersion_last_updated.pop(k, None)
            except Exception:
                pass
            if gc.isenabled():
                gc.collect()
        except Exception:
            pass

    # ── Forward / process_sequence ─────────────────────────────────────────────

    def forward(
        self,
        token_embeddings=None,
        token_types=None,
        train_mode: bool = True,
        token_word_map=None,
        h_all=None,
        input_ids=None,
        attention_mask=None,
    ):
        if token_embeddings is None and h_all is not None:
            token_embeddings = h_all

        if token_embeddings is None:
            raise ValueError(
                "MemoryEfficientDSCDOnline.forward requires token_embeddings or h_all"
            )

        if input_ids is not None and token_types is None:
            batch_size, seq_len = input_ids.shape
            token_types = []
            for b in range(batch_size):
                if self.tokenizer is not None:
                    try:
                        token_types.append(
                            self.tokenizer.convert_ids_to_tokens(input_ids[b].tolist())
                        )
                    except Exception:
                        token_types.append([f"tok{i}" for i in range(seq_len)])
                else:
                    token_types.append([f"tok{i}" for i in range(seq_len)])

        self.cleanup_counter += 1
        if self.cleanup_counter % 50 == 0:
            self.cleanup_counter = 0
            self.cleanup_memory()
            self.cleanup_threads()

        device     = token_embeddings.device
        batch_size = int(token_embeddings.size(0))
        seq_len    = int(token_embeddings.size(1))

        all_outputs: Dict[str, List[Any]] = {
            'proto_assignments': [],
            'proto_probs':       [],
            'uncertainties':     [],
            'span_preds':        [],
            'gates':             [],
            'h_augmented':       [],
        }

        for b in range(batch_size):
            word_map = (
                token_word_map[b]
                if token_word_map and len(token_word_map) > b
                else None
            )
            batch_outputs = self.process_sequence(
                token_embeddings[b],
                (
                    token_types[b]
                    if token_types and len(token_types) > b
                    else [f"tok{i}" for i in range(seq_len)]
                ),
                device,
                word_map=word_map,
                train_mode=train_mode,
            )
            for k in all_outputs:
                all_outputs[k].append(batch_outputs[k])

        try:
            h_aug_list: List[torch.Tensor] = []
            for b in range(batch_size):
                h_batch_list = all_outputs['h_augmented'][b]
                if len(h_batch_list) > 0 and isinstance(h_batch_list[0], torch.Tensor):
                    h_batch = torch.stack(h_batch_list, dim=0)
                    if h_batch.size(0) < seq_len:
                        h_batch = F.pad(h_batch, (0, 0, 0, seq_len - h_batch.size(0)))
                    elif h_batch.size(0) > seq_len:
                        h_batch = h_batch[:seq_len]
                else:
                    h_batch = token_embeddings[b].clone()
                h_aug_list.append(h_batch)
            all_outputs['h_augmented'] = torch.stack(h_aug_list, dim=0)
        except Exception:
            all_outputs['h_augmented'] = token_embeddings

        try:
            proto_assign_tensor = []
            for row in all_outputs['proto_assignments']:
                try:
                    stacked = torch.stack(
                        [x if isinstance(x, torch.Tensor) else torch.tensor(x) for x in row],
                        dim=0,
                    )
                    proto_assign_tensor.append(stacked)
                except Exception:
                    proto_assign_tensor.append(
                        torch.tensor(
                            [int(x) if not isinstance(x, torch.Tensor) else int(x.item()) for x in row],
                            dtype=torch.long,
                        )
                    )
            all_outputs['proto_assignments'] = proto_assign_tensor
        except Exception:
            pass

        return all_outputs

    # ── [3E] process_sequence — force_observe call added at top ───────────────

    def process_sequence(
        self,
        token_embeddings: torch.Tensor,
        token_types: List[Any],
        device: torch.device,
        word_map: Optional[Dict[int, Optional[str]]] = None,
        train_mode: bool = True,
    ) -> Dict[str, List[Any]]:
        seq_len = int(token_embeddings.size(0))
        outputs: Dict[str, List[Any]] = {
            'proto_assignments': [],
            'proto_probs':       [],
            'uncertainties':     [],
            'span_preds':        [],
            'gates':             [],
            'h_augmented':       [],
        }

        # [3E] Periodically force-observe reference homographs
        self._force_observe_step_counter += 1
        if train_mode and self._force_observe_step_counter % PRINT_INTERVAL == 0:
            try:
                self.force_observe_reference_homographs(
                    global_step=self._force_observe_step_counter
                )
            except Exception:
                pass

        for j in range(seq_len):
            raw_tok = token_types[j] if j < len(token_types) else f"tok{j}"
            if not isinstance(raw_tok, str):
                raw_tok = str(raw_tok) if raw_tok is not None else f"tok{j}"

            token_key = self.canonical_token_key(raw_tok, word_map, j)
            h_j       = token_embeddings[j]

            if not token_key:
                outputs['proto_assignments'].append(torch.tensor(-1))
                outputs['proto_probs'].append([])
                outputs['uncertainties'].append(0.0)
                outputs['span_preds'].append(0.0)
                outputs['gates'].append(0.0)
                outputs['h_augmented'].append(h_j)
                continue

            if not self.should_track_token(token_key):
                outputs['proto_assignments'].append(torch.tensor(-1))
                outputs['proto_probs'].append([])
                outputs['uncertainties'].append(0.0)
                outputs['span_preds'].append(0.0)
                outputs['gates'].append(0.0)
                outputs['h_augmented'].append(h_j)
                continue

            with self.buffer_lock:
                if token_key not in self.buffers:
                    self.buffers[token_key] = deque(maxlen=self.buffer_size)
                    self.prototype_stores[token_key] = MemoryEfficientPrototypeStore(
                        embed_dim=self.embed_dim,
                        max_protos=self.max_protos,
                        buffer_size=self.buffer_size,
                    )
                try:
                    self.buffers[token_key].append(h_j.detach().clone().cpu())
                except Exception:
                    try:
                        self.buffers[token_key].append(h_j.cpu())
                    except Exception:
                        pass

            buffer_len = len(self.buffers[token_key])

            try:
                if (
                    self.enable_training_clustering
                    and buffer_len >= max(self.n_min + 2, 10)
                    and self.should_cluster_token(token_key)
                ):
                    now    = time.time()
                    last_t = self.last_cluster_time.get(token_key, 0.0)
                    if now - last_t >= self.cluster_cooldown_seconds:
                        self.last_cluster_time[token_key] = now

                        def bg_cluster(tok: str = token_key) -> None:
                            try:
                                with self.clustering_lock:
                                    self.cluster_buffer_to_prototypes_hierarchical(tok)
                            except Exception:
                                pass

                        th = threading.Thread(target=bg_cluster, daemon=True)
                        th.start()
                        with self.thread_lock:
                            self.active_threads.append(th)
            except Exception:
                pass

            store = self.prototype_stores[token_key]

            centroids_snapshot: Optional[List[torch.Tensor]] = None
            with self.clustering_lock:
                try:
                    if hasattr(store, 'centroids') and len(store.centroids) > 0:
                        centroids_snapshot = []
                        for c in store.centroids:
                            try:
                                if isinstance(c, torch.Tensor):
                                    centroids_snapshot.append(c.clone().cpu())
                                else:
                                    centroids_snapshot.append(
                                        torch.from_numpy(
                                            np.asarray(c, dtype=np.float32)
                                        ).cpu()
                                    )
                            except Exception:
                                continue
                        if not centroids_snapshot:
                            centroids_snapshot = None
                except Exception:
                    centroids_snapshot = None

            assignment  = -1
            prob_list: List[float] = []
            uncertainty = 0.0
            span_pred   = 0.0
            gate_val    = 0.0
            h_aug       = h_j

            if centroids_snapshot and len(centroids_snapshot) >= 1:
                try:
                    try:
                        h_cpu = h_j.detach().cpu().numpy()
                    except Exception:
                        h_cpu = h_j.cpu().numpy()

                    try:
                        cents_np = np.stack([c.numpy() for c in centroids_snapshot], axis=0)
                    except Exception:
                        cents_np = np.stack(
                            [np.asarray(c, dtype=np.float32) for c in centroids_snapshot],
                            axis=0,
                        )

                    dists_np = np.linalg.norm(cents_np - h_cpu[None, :], axis=1)

                    if dists_np.size > 0:
                        min_dist = float(dists_np.min())
                        min_idx  = int(np.argmin(dists_np))

                        if len(centroids_snapshot) >= 2:
                            mean_dist = float(np.mean(dists_np))
                            std_dist  = float(np.std(dists_np))
                            span_pred = float(np.clip(std_dist / (mean_dist + 1e-6), 0.0, 1.0))
                        else:
                            span_pred = float(
                                np.clip((min_dist - store.mu) / (1e-3), 0.0, 1.0)
                            )

                        base_threshold   = max(store.tau, 1e-3) if store.size() > 0 else 0.3
                        uncertainty_dist = float(
                            np.clip(min_dist / (base_threshold * 2), 0.0, 1.0)
                        )

                        if len(centroids_snapshot) >= 2:
                            precisions   = 1.0 / (dists_np ** 2 + 1e-6)
                            gate_weights = precisions / (np.sum(precisions) + 1e-6)
                            gate_val     = float(np.max(gate_weights))
                        else:
                            gate_val = float(
                                np.clip(1.0 - (min_dist - store.mu) / (1e-3), 0.0, 1.0)
                            )

                        if (
                            store.size() < self.max_protos
                            and min_dist > store.get_adaptive_threshold(DSCD_NEWSENSE_LAMBDA)
                        ):
                            store.add_prototype(h_j, time.time(), count=1)
                            assignment = store.size() - 1
                            centroids_snapshot.append(h_j.cpu())
                            cents_np = np.vstack([cents_np, h_cpu[None, :]])
                        else:
                            assignment = min_idx
                            try:
                                store.update_rolling_stats(min_dist)
                            except Exception:
                                pass

                        if train_mode and 0 <= assignment < store.size():
                            try:
                                self.maybe_split_prototype(token_key, assignment)
                            except Exception:
                                pass

                        try:
                            dist_tensor  = torch.from_numpy(dists_np).to(device)
                            probs_tensor = F.softmax(-dist_tensor, dim=0)
                            prob_list    = probs_tensor.tolist()
                            entropy      = -torch.sum(
                                probs_tensor * torch.log(probs_tensor + 1e-10)
                            )
                            max_entropy         = np.log(len(dists_np))
                            uncertainty_entropy = float(
                                entropy.item() / max_entropy
                            ) if max_entropy > 0 else 0.0
                        except Exception:
                            exps = (
                                np.exp(-dists_np - np.max(-dists_np))
                                if dists_np.size > 0
                                else np.array([1.0])
                            )
                            if exps.size > 0:
                                probs               = exps / (exps.sum() + 1e-12)
                                prob_list           = probs.tolist()
                                entropy_val         = -np.sum(probs * np.log(probs + 1e-10))
                                max_entropy         = np.log(len(dists_np))
                                uncertainty_entropy = float(entropy_val / max_entropy) if max_entropy > 0 else 0.0
                            else:
                                prob_list           = []
                                uncertainty_entropy = 0.0

                        if len(centroids_snapshot) >= 2:
                            uncertainty = 0.4 * uncertainty_dist + 0.6 * uncertainty_entropy
                        else:
                            uncertainty = uncertainty_dist

                        if gate_val > 0.3 and 0 <= assignment < len(centroids_snapshot):
                            try:
                                centroid_t = centroids_snapshot[assignment]
                                if device != torch.device('cpu'):
                                    try:
                                        centroid_t = centroid_t.to(device)
                                    except Exception:
                                        pass
                                blend_weight = 0.3 if gate_val > 0.7 else 0.15
                                h_aug        = h_j + blend_weight * (centroid_t - h_j)
                            except Exception:
                                h_aug = h_j

                except Exception as e:
                    if DEBUG_DISCOVERY:
                        print(f"[DSCD] Assignment error for {token_key}: {str(e)[:200]}")

            outputs['proto_assignments'].append(torch.tensor(assignment))
            outputs['proto_probs'].append(prob_list)
            outputs['uncertainties'].append(uncertainty)
            outputs['span_preds'].append(span_pred)
            outputs['gates'].append(gate_val)
            outputs['h_augmented'].append(h_aug)

        try:
            if not train_mode and len(self.prototype_stores) > 0 and VERBOSE_LOGGING:
                if self.last_periodic_check % PRINT_INTERVAL == 0:
                    self.print_clusters_summary()
                self.last_periodic_check += 1
        except Exception:
            pass

        return outputs

    # ── Cluster summary printer ────────────────────────────────────────────────

    def print_clusters_summary(self) -> None:
        try:
            items: List[Tuple[str, int, int, float, float, int]] = []
            for token, store in self.prototype_stores.items():
                if is_punctuation_only(token):
                    continue
                try:
                    proto_sample_count = sum(getattr(store, 'counts', []) or [])
                except Exception:
                    proto_sample_count = 0
                buffer_len  = len(self.buffers.get(token, [])) if token in self.buffers else 0
                total_count = proto_sample_count if proto_sample_count > 0 else buffer_len
                protos      = store.size()
                mu          = getattr(store, 'mu', 0.0)
                tau         = getattr(store, 'tau', 0.0)
                items.append((token, total_count, protos, mu, tau, buffer_len))

            items.sort(key=lambda x: x[1], reverse=True)
            top5 = items[:5]

            if VERBOSE_LOGGING:
                print("CLUSTER: Top 5 clusters")
                print("-" * 90)
                print(f"{'Rank':6} {'Token':14} {'Count':12} {'Protos':10} {'Mu':14} {'Tau':12}")
                print("-" * 90)
                for rank, (tok, cnt, prot, mu, tau, buflen) in enumerate(top5, 1):
                    tok_str = str(tok)[:14]
                    print(f"{rank:6} {tok_str:14} {cnt:12} {prot:10} {mu:14.6f} {tau:12.6f}")
                print("-" * 90)
        except Exception as e:
            try:
                if VERBOSE_LOGGING:
                    print(f"[CLUSTER] Error printing summary: {str(e)[:200]}")
            except Exception:
                pass

    # ── [3G] cluster_buffer_to_prototypes_hierarchical — fixed 3 issues ───────

    def cluster_buffer_to_prototypes_hierarchical(self, token_type: str) -> bool:
        try:
            if is_punctuation_only(token_type):
                if DEBUG_DISCOVERY:
                    print(f"[DSCD-CLUSTER] Skipping punctuation token: {token_type}")
                return False

            if not self.should_track_token(token_type):
                if DEBUG_DISCOVERY:
                    print(f"[DSCD-CLUSTER] Skipping non-word token: {token_type}")
                return False

            if not self.should_cluster_token(token_type):
                if DEBUG_DISCOVERY:
                    print(f"[DSCD-CLUSTER] Skipping function word: {token_type}")
                return False

            with self.buffer_lock:
                if token_type not in self.buffers:
                    return False
                buf_snapshot = [
                    e.clone() if isinstance(e, torch.Tensor) else e
                    for e in self.buffers[token_type]
                ]

            if len(buf_snapshot) < max(self.n_min + 2, 10):
                if DEBUG_DISCOVERY:
                    print(
                        f"[DSCD-CLUSTER] {token_type}: buffer={len(buf_snapshot)} "
                        f"< min={max(self.n_min + 2, 10)}"
                    )
                return False

            emb_list: List[np.ndarray] = []
            for e in buf_snapshot:
                try:
                    if isinstance(e, torch.Tensor):
                        try:
                            emb_list.append(e.numpy())
                        except Exception:
                            emb_list.append(e.cpu().numpy())
                    else:
                        emb_list.append(np.asarray(e, dtype=np.float32))
                except Exception:
                    continue

            if len(emb_list) == 0:
                return False

            if len(emb_list) > self.max_clustering_points:
                idxs       = np.random.choice(len(emb_list), size=self.max_clustering_points, replace=False)
                embeddings = np.stack([emb_list[i] for i in idxs], axis=0)
            else:
                embeddings = np.stack(emb_list, axis=0)

            if embeddings.shape[0] < 2:
                return False

            norms = np.linalg.norm(embeddings, axis=1)
            if np.all(norms < 1e-6):
                if DEBUG_DISCOVERY:
                    print(f"[DSCD-CLUSTER] {token_type}: all zero vectors, skipping")
                return False

            if DEBUG_DISCOVERY:
                print(
                    f"[DSCD-CLUSTER] {token_type}: buf={len(buf_snapshot)}, "
                    f"sampled={embeddings.shape[0]}, "
                    f"mean_norm={norms.mean():.4f}"
                )

            store        = self.prototype_stores[token_type]
            protos_added = 0
            new_centroids: List[torch.Tensor] = []
            new_counts:    List[int]           = []
            new_times:     List[float]         = []

            if HAS_CLUSTERING:
                try:
                    condensed = pdist(embeddings, metric='euclidean')
                    if condensed.size > 0:
                        Z        = linkage(condensed, method='average')
                        max_dist = condensed.max() if condensed.size > 0 else 1.0
                        rel_t    = self.dispersion_threshold
                        abs_t    = rel_t * max_dist
                        clusters = fcluster(Z, t=abs_t, criterion='distance') - 1

                        if clusters.size > 0:
                            max_c = int(clusters.max())
                            for cid in range(max_c + 1):
                                mask         = clusters == cid
                                cluster_size = int(mask.sum())
                                if cluster_size >= self.n_min:
                                    centroid        = embeddings[mask].mean(axis=0).astype(np.float32)
                                    centroid_tensor = torch.from_numpy(centroid)
                                    new_centroids.append(centroid_tensor)
                                    new_counts.append(cluster_size)
                                    new_times.append(time.time())
                                    protos_added += 1

                            if len(new_centroids) > self.max_protos:
                                sorted_indices = np.argsort(new_counts)[::-1][:self.max_protos]
                                new_centroids  = [new_centroids[i] for i in sorted_indices]
                                new_counts     = [new_counts[i]    for i in sorted_indices]
                                new_times      = [new_times[i]     for i in sorted_indices]
                                protos_added   = len(new_centroids)

                            store.centroids     = new_centroids
                            store.counts        = new_counts
                            store.creation_time = new_times
                            # [3G-Fix1] guard against unbound clusters variable
                            if clusters is not None:
                                store.labels = torch.tensor(clusters)

                            if DEBUG_DISCOVERY and protos_added > 0:
                                print(
                                    f"[DSCD-CLUSTER] Hierarchical created {protos_added} "
                                    f"prototypes for {token_type}"
                                )
                except Exception as e:
                    if DEBUG_DISCOVERY:
                        print(
                            f"[DSCD-CLUSTER] Hierarchical failed for {token_type}: "
                            f"{type(e).__name__}: {str(e)[:200]}"
                        )

            if protos_added == 0 and HAS_KMEANS:
                try:
                    min_k = 1
                    max_k = min(self.max_protos, len(embeddings) // self.n_min)
                    if max_k < min_k:
                        max_k = min_k

                    if len(embeddings) >= 20:
                        k_guess = min(max_k, max(2, int(np.sqrt(len(embeddings) / 2))))
                    elif len(embeddings) >= 10:
                        k_guess = min(max_k, 2)
                    else:
                        k_guess = 1

                    k_guess = max(min_k, min(k_guess, len(embeddings)))

                    if k_guess >= 1 and len(embeddings) >= k_guess:
                        km     = KMeans(n_clusters=k_guess, random_state=0, n_init=10).fit(embeddings)
                        labels = km.labels_
                        new_centroids = []
                        new_counts    = []
                        new_times     = []
                        for c in range(k_guess):
                            mask         = labels == c
                            cluster_size = int(mask.sum())
                            if cluster_size >= self.n_min:
                                centroid        = embeddings[mask].mean(axis=0).astype(np.float32)
                                centroid_tensor = torch.from_numpy(centroid)
                                new_centroids.append(centroid_tensor)
                                new_counts.append(cluster_size)
                                new_times.append(time.time())
                                protos_added += 1

                        store.centroids     = new_centroids
                        store.counts        = new_counts
                        store.creation_time = new_times
                        # [3G-Fix2] store cluster count only, not the full per-sample label array
                        store.labels = torch.tensor(new_counts, dtype=torch.long)

                        if DEBUG_DISCOVERY and protos_added > 0:
                            print(
                                f"[DSCD-CLUSTER] KMeans created {protos_added} "
                                f"prototypes for {token_type}"
                            )
                except Exception as e:
                    if DEBUG_DISCOVERY:
                        print(
                            f"[DSCD-CLUSTER] KMeans failed for {token_type}: "
                            f"{type(e).__name__}: {str(e)[:200]}"
                        )

            if DEBUG_DISCOVERY:
                print(
                    f"[DSCD-CLUSTER] {token_type}: final={store.size()} protos, "
                    f"counts={store.counts}"
                )

            try:
                if store.centroids:
                    counts      = store.counts if store.counts else [1] * len(store.centroids)
                    total_count = sum(counts)
                    mean_count  = float(total_count) / max(1, len(counts))
                    # [3G-Fix3] use token_type directly as key, not str(token_type)
                    self.cluster_stats[token_type] = {
                        'num_prototypes': len(store.centroids),
                        'counts':         [int(c) for c in counts],
                        'total_samples':  int(total_count),
                        'mean_count':     float(mean_count),
                        'mu':             float(store.mu),
                        'tau':            float(store.tau),
                    }
            except Exception:
                pass

            # [3H] Populate sense registry after successful multi-sense clustering
            if store.size() >= 2 and self.is_multi_sense_store(store):
                try:
                    clean_tok = clean_token_for_dscd(token_type)
                    sense_info: Dict[str, Any] = {
                        'num_senses':  store.size(),
                        'counts':      list(store.counts),
                        'centroids':   [c.cpu().clone() for c in store.centroids],
                        'mu':          float(store.mu),
                        'tau':         float(store.tau),
                        'updated_at':  time.time(),
                    }
                    self.homograph_sense_registry[clean_tok] = sense_info
                except Exception:
                    pass

            return store.size() > 0

        except Exception as e:
            if DEBUG_DISCOVERY:
                print(
                    f"[DSCD-ERROR] Clustering error for {token_type}: "
                    f"{type(e).__name__}: {str(e)[:200]}"
                )
            return False

    # ── Public accessors ──────────────────────────────────────────────────────

    def get_explanations(self, threshold_span: float = 0.3) -> List[Dict[str, Any]]:
        expl: List[Dict[str, Any]] = []
        for token_type, store in self.prototype_stores.items():
            if store.size() >= 2:
                expl.append({'token': str(token_type), 'protos': store.size()})
        return expl

    # ── [3F] periodic_discovery_check — buffer_lock fix + already_clustered fix ─

    def periodic_discovery_check(self, global_step: int, frequency: int = 200) -> int:
        if frequency > 0 and global_step % frequency != 0:
            return 0

        try:
            candidates: List[Tuple[str, float, int]] = []
            buffer_snapshot:   Dict[str, int]        = {}
            already_clustered: Set[str]              = set()

            with self.buffer_lock:
                for token in list(self.buffers.keys()):
                    buffer_snapshot[token] = len(self.buffers.get(token, []))

            with self.clustering_lock:
                for token in self.prototype_stores.keys():
                    if self.prototype_stores[token].size() >= 2:
                        if normalize_token_key(token) in self.discovered_homographs:
                            already_clustered.add(token)

            for token, buffer_size in buffer_snapshot.items():
                if is_punctuation_only(token):
                    continue
                if not self.should_cluster_token(token):
                    continue
                if token in already_clustered:
                    continue
                if buffer_size >= max(self.n_min + 2, 10):
                    try:
                        dispersion = self.get_dispersion(token)
                        if dispersion >= self.dispersion_threshold:
                            rank_score = dispersion * buffer_size
                            candidates.append((token, rank_score, buffer_size))
                    except Exception:
                        continue

            if not candidates:
                return 0

            candidates.sort(key=lambda x: x[1], reverse=True)
            candidates_to_process = candidates[:min(MAX_TOKENS_PER_DISCOVERY, len(candidates))]

            return self.discover_homographs_for_tokens(
                [c[0] for c in candidates_to_process],
                self.n_min,
                self.dispersion_threshold,
                global_step,
            )

        except Exception as e:
            if DEBUG_DISCOVERY:
                print(f"[DSCD] periodic_discovery_check failed: {e}")
            return 0

    def get_prototype_summary(self) -> Dict[str, Any]:
        try:
            total_tokens     = len(self.prototype_stores)
            total_prototypes = sum(s.size() for s in self.prototype_stores.values())
            homographs       = sum(
                1 for s in self.prototype_stores.values() if s.size() >= 2
            )
            return {
                'total_tokens':          total_tokens,
                'total_prototypes':      total_prototypes,
                'num_homographs':        homographs,
                'discovered_homographs': len(self.discovered_homographs),
                'quality_score':         self.get_quality_score(),
            }
        except Exception:
            return {
                'total_tokens':          0,
                'total_prototypes':      0,
                'num_homographs':        0,
                'discovered_homographs': 0,
                'quality_score':         0.0,
            }

    def get_discovered_homographs(self) -> Set[str]:
        return self.discovered_homographs.copy()


# ==============================================================================
# DSCD prototype save / load helpers
# ==============================================================================

def save_dscd_prototypes(dscd_module, path: str) -> None:
    try:
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        proto_state = {}
        stores      = getattr(dscd_module, "prototype_stores", {})
        for word, store in stores.items():
            if store.size() > 0:
                proto_state[word] = store.state_dict()
        torch.save(proto_state, path)
        print(f"[DSCD] Prototypes saved: {len(proto_state)} words -> {path}")
    except Exception as e:
        print(f"[DSCD] Prototype save failed: {e}")


def load_dscd_prototypes(dscd_module, path: str) -> None:
    try:
        if not os.path.exists(path):
            print(f"[DSCD] Prototype file not found: {path}")
            return
        proto_state = torch.load(path, map_location="cpu")
        stores      = getattr(dscd_module, "prototype_stores", {})
        restored    = 0
        for word, state in proto_state.items():
            if word in stores:
                stores[word].load_state_dict(state, device=torch.device("cpu"))
                restored += 1
            else:
                new_store = MemoryEfficientPrototypeStore(
                    embed_dim=dscd_module.embed_dim,
                    max_protos=dscd_module.max_protos,
                    buffer_size=dscd_module.buffer_size,
                )
                new_store.load_state_dict(state, device=torch.device("cpu"))
                stores[word] = new_store
                restored    += 1
        print(f"[DSCD] Prototypes restored: {restored} words from {path}")
    except Exception as e:
        print(f"[DSCD] Prototype load failed: {e}")


# ==============================================================================
# Cell 3 ready
# ==============================================================================
print("=" * 80)
print("Cell 3: DSCD Word-Level Homograph Disambiguation - READY")
print("=" * 80)
print("CONFIGURATION:")
print(f"  Model-agnostic: works with any encoder")
print(f"  Operates on token embeddings / hidden states")
print(f"  Cache size: 10000 (aligned with Cell 1)")
print(f"  Parameters: max_protos={DSCD_MAX_PROTOS}, n_min={DSCD_N_MIN}")
print(f"  Thread-safe clustering: hierarchical + KMeans")
print(f"  Bengali Unicode validation (language-agnostic)")
print("=" * 80)

In [ ]:
# ==============================================================================
# CELL 4: ASBN MODULE - ADVERSARIAL SELECTIVE BATCH NORMALIZATION
# ==============================================================================

import traceback
from typing import Any, List, Tuple, Optional, Dict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    _MAX_LENGTH = int(MAX_LENGTH)
except Exception:
    _MAX_LENGTH = 48

try:
    _ENABLE_ASBN_TRAINING = bool(ENABLE_ASBN_TRAINING)
except Exception:
    _ENABLE_ASBN_TRAINING = True

try:
    _VERBOSE_LOGGING = bool(VERBOSE_LOGGING)
except Exception:
    _VERBOSE_LOGGING = False

try:
    _DEBUG_DISCOVERY = bool(DEBUG_DISCOVERY)
except Exception:
    _DEBUG_DISCOVERY = False

try:
    _DEBUG_TIMING = bool(DEBUG_TIMING)
except Exception:
    _DEBUG_TIMING = False

try:
    _SOURCE_LANGUAGE = str(SOURCE_LANGUAGE)
except Exception:
    _SOURCE_LANGUAGE = "bn"

try:
    _GRL_ALPHA_START = float(GRL_ALPHA_START)
    _GRL_ALPHA_END = float(GRL_ALPHA_END)
    _GRL_ALPHA_SCHEDULE = str(GRL_ALPHA_SCHEDULE)
    try:
        _GRL_ALPHA_STEPS = int(GRL_ALPHA_STEPS)
    except Exception:
        _GRL_ALPHA_STEPS = 10000
except Exception:
    _GRL_ALPHA_START = 0.1
    _GRL_ALPHA_END = 1.0
    _GRL_ALPHA_SCHEDULE = "linear"
    _GRL_ALPHA_STEPS = 10000

_has_is_valid_token = False
_has_get_tokenizer_special_tokens = False
_has_should_track_token = False
_is_valid_token_fn = None
_get_tokenizer_special_tokens_fn = None
_should_track_token_fn = None

try:
    if 'is_valid_token' in dir():
        _is_valid_token_fn = is_valid_token
        _has_is_valid_token = True
    elif 'is_valid_token' in globals():
        _is_valid_token_fn = globals()['is_valid_token']
        _has_is_valid_token = True
except Exception:
    pass

try:
    if 'get_tokenizer_special_tokens' in dir():
        _get_tokenizer_special_tokens_fn = get_tokenizer_special_tokens
        _has_get_tokenizer_special_tokens = True
    elif 'get_tokenizer_special_tokens' in globals():
        _get_tokenizer_special_tokens_fn = globals()['get_tokenizer_special_tokens']
        _has_get_tokenizer_special_tokens = True
except Exception:
    pass

try:
    if 'should_track_token' in dir():
        _should_track_token_fn = should_track_token
        _has_should_track_token = True
    elif 'should_track_token' in globals():
        _should_track_token_fn = globals()['should_track_token']
        _has_should_track_token = True
except Exception:
    pass


class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = float(alpha)
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


def gradient_reversal(x, alpha: float = 1.0):
    return GradientReversalFunction.apply(x, alpha)


class LightweightDiscriminator(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(x)


class DomainDiscriminator(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(x)


class MemoryEfficientASBNModule(nn.Module):
    def __init__(
        self,
        embed_dim: int,
        tokenizer=None,
        language: str = "bn",
        freq_threshold: float = 0.7,
        uncertainty_threshold: float = 0.3,
        gate_threshold: float = 0.5,
        warmup_steps: int = 50,
        encoder_grl_scale: float = 1.0,
    ):
        super().__init__()
        self.language = language
        self.tokenizer = tokenizer
        self.embed_dim = int(embed_dim)

        self.bn_source = nn.BatchNorm1d(self.embed_dim, track_running_stats=True)
        self.bn_target = nn.BatchNorm1d(self.embed_dim, track_running_stats=True)

        self.d_domain = DomainDiscriminator(self.embed_dim)
        self.d_freq = LightweightDiscriminator(self.embed_dim + 2)
        self.d_ctx = LightweightDiscriminator(self.embed_dim + 2)
        self.d_xl = LightweightDiscriminator(self.embed_dim)

        self.freq_threshold = float(freq_threshold)
        self.uncertainty_threshold = float(uncertainty_threshold)
        self.gate_threshold = float(gate_threshold)
        self.warmup_steps = int(warmup_steps)
        self.current_step = 0

        self.lambda_base = {"freq": 1.0, "ctx": 1.0, "xl": 1.0, "domain": 1.0}
        self.lambda_max = 2.0
        self.encoder_grl_scale = float(encoder_grl_scale)

        self.stats_reset_interval = 100
        self.stats = {
            "domain_loss": 0.0,
            "domain_accuracy": 0.0,
            "source_accuracy": 0.0,
            "target_accuracy": 0.0,
            "asbn_loss": 0.0,
            "num_updates": 0,
        }

        try:
            if tokenizer is not None:
                if _has_get_tokenizer_special_tokens and _get_tokenizer_special_tokens_fn is not None:
                    self.special_tokens = _get_tokenizer_special_tokens_fn(tokenizer)
                else:
                    self.special_tokens = set(getattr(tokenizer, "all_special_tokens", []))
            else:
                self.special_tokens = set()
        except Exception:
            self.special_tokens = set()

        if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
            print("[ASBN-INIT] Initialized MemoryEfficientASBNModule:")
            print(f"  - embed_dim: {self.embed_dim}")
            print(f"  - warmup_steps: {self.warmup_steps}")
            print(f"  - encoder_grl_scale: {self.encoder_grl_scale}")
            print(f"  - GRL_ALPHA: {_GRL_ALPHA_START} → {_GRL_ALPHA_END} over {_GRL_ALPHA_STEPS} steps")
            print(f"  - thresholds: freq={self.freq_threshold}, uncert={self.uncertainty_threshold}, gate={self.gate_threshold}")
            print(f"  - Function availability: should_track={_has_should_track_token}, is_valid={_has_is_valid_token}")

    def get_grl_alpha(self, global_step: Optional[int] = None) -> float:
        if global_step is None:
            global_step = self.current_step
        step = max(0, int(global_step))

        if _GRL_ALPHA_SCHEDULE == "linear":
            progress = min(1.0, float(step) / float(_GRL_ALPHA_STEPS))
            alpha = _GRL_ALPHA_START + progress * (_GRL_ALPHA_END - _GRL_ALPHA_START)
        elif _GRL_ALPHA_SCHEDULE == "exponential":
            progress = min(1.0, float(step) / float(_GRL_ALPHA_STEPS))
            ratio = _GRL_ALPHA_END / max(1e-8, _GRL_ALPHA_START if _GRL_ALPHA_START > 0 else 1e-3)
            alpha = _GRL_ALPHA_START * (ratio ** progress)
        else:
            alpha = _GRL_ALPHA_END

        return float(alpha)

    def get_asbn_stats(self) -> Dict[str, float]:
        return self.get_detailed_stats()

    def get_detailed_stats(self) -> Dict[str, float]:
        if self.stats["num_updates"] > 0:
            n = float(self.stats["num_updates"])
            return {
                "domain_loss": self.stats["domain_loss"] / n,
                "domain_accuracy": self.stats["domain_accuracy"] / n,
                "source_accuracy": self.stats["source_accuracy"] / n,
                "target_accuracy": self.stats["target_accuracy"] / n,
                "asbn_loss": self.stats["asbn_loss"] / n,
                "num_updates": self.stats["num_updates"],
            }
        return {
            "domain_loss": 0.0,
            "domain_accuracy": 0.0,
            "source_accuracy": 0.0,
            "target_accuracy": 0.0,
            "asbn_loss": 0.0,
            "num_updates": 0,
        }

    def reset_stats(self) -> None:
        self.stats = {
            "domain_loss": 0.0,
            "domain_accuracy": 0.0,
            "source_accuracy": 0.0,
            "target_accuracy": 0.0,
            "asbn_loss": 0.0,
            "num_updates": 0,
        }

    def critic_parameters(self):
        return (
            list(self.d_domain.parameters())
            + list(self.d_freq.parameters())
            + list(self.d_ctx.parameters())
            + list(self.d_xl.parameters())
        )

    def _ensure_discriminators_on_device(self, device: torch.device) -> None:
        try:
            for mod in (
                self.d_domain,
                self.d_freq,
                self.d_ctx,
                self.d_xl,
                self.bn_source,
                self.bn_target,
            ):
                try:
                    p = next(mod.parameters())
                    if p.device != device:
                        mod.to(device)
                except StopIteration:
                    mod.to(device)
                except Exception:
                    pass
        except Exception:
            if _VERBOSE_LOGGING:
                try:
                    print("[ASBN] Device migration failed:", traceback.format_exc().splitlines()[-1])
                except Exception:
                    print("[ASBN] Device migration failed")

    def _expand_domain_labels(self, domain_labels: Optional[torch.Tensor], batch_size: int) -> Optional[torch.Tensor]:
        if domain_labels is None:
            return None

        if domain_labels.dim() == 0:
            domain_labels = domain_labels.unsqueeze(0)

        if domain_labels.size(0) == 1 and batch_size > 1:
            domain_labels = domain_labels.expand(batch_size).contiguous()
        elif domain_labels.size(0) != batch_size:
            if _DEBUG_DISCOVERY:
                print(f"[ASBN] Domain label size mismatch: {domain_labels.size(0)} vs batch {batch_size}, using first label")
            domain_labels = domain_labels[0].unsqueeze(0).expand(batch_size).contiguous()

        return domain_labels

    def _parse_proto_probs_matrix(self, proto_probs: Any, batch_size: int, seq_len: int, device: torch.device) -> torch.Tensor:
        pmax = torch.full((batch_size, seq_len), 0.5, dtype=torch.float32, device=device)

        try:
            if proto_probs is None:
                return pmax

            if isinstance(proto_probs, torch.Tensor):
                if proto_probs.dim() == 3:
                    B, T, K = proto_probs.shape
                    p = proto_probs.detach().to(device)
                    b_max = min(batch_size, B)
                    t_max = min(seq_len, T)
                    pmax[:b_max, :t_max] = p[:b_max, :t_max].max(dim=2)[0]
                    return pmax

                if proto_probs.dim() == 2:
                    p = proto_probs.detach().to(device)
                    if batch_size >= 1:
                        t_max = min(seq_len, p.size(0))
                        pmax[0, :t_max] = p[:t_max].max(dim=1)[0]
                        return pmax

            if isinstance(proto_probs, (list, tuple)):
                if len(proto_probs) == batch_size:
                    for b in range(batch_size):
                        row = proto_probs[b]
                        if isinstance(row, torch.Tensor) and row.dim() == 2:
                            t_max = min(seq_len, row.size(0))
                            pmax[b, :t_max] = row[:t_max].max(dim=1)[0].to(device)
                        elif isinstance(row, (list, tuple)):
                            for t in range(min(seq_len, len(row))):
                                try:
                                    val = row[t]
                                    if isinstance(val, torch.Tensor):
                                        pmax[b, t] = float(val.max().item())
                                    else:
                                        arr = np.asarray(val, dtype=np.float32)
                                        pmax[b, t] = float(np.max(arr))
                                except Exception:
                                    pmax[b, t] = 0.5
                else:
                    if batch_size == 1:
                        row = proto_probs
                        for t in range(min(seq_len, len(row))):
                            try:
                                val = row[t]
                                if isinstance(val, torch.Tensor):
                                    pmax[0, t] = float(val.max().item())
                                else:
                                    pmax[0, t] = float(np.max(np.asarray(val, dtype=np.float32)))
                            except Exception:
                                pmax[0, t] = 0.5

        except Exception as e:
            if _VERBOSE_LOGGING:
                print(f"[ASBN] parse_proto_probs exception: {e}")

        return pmax

    def _parse_scalar_matrix(self, mat: Any, batch_size: int, seq_len: int, device: torch.device,
                            default: float = 0.0) -> torch.Tensor:
        out = torch.full((batch_size, seq_len), float(default), dtype=torch.float32, device=device)

        try:
            if mat is None:
                return out

            if isinstance(mat, torch.Tensor):
                if mat.dim() == 3:
                    B, T, _ = mat.shape
                    b_max = min(batch_size, B)
                    t_max = min(seq_len, T)
                    out[:b_max, :t_max] = mat[:b_max, :t_max, 0].to(device)
                elif mat.dim() == 2:
                    if mat.size(0) == batch_size:
                        t_max = min(seq_len, mat.size(1))
                        out[:, :t_max] = mat[:, :t_max].to(device)
                    elif batch_size == 1:
                        t_max = min(seq_len, mat.size(0))
                        out[0, :t_max] = mat[:t_max].to(device)
                elif mat.dim() == 1 and batch_size == 1:
                    t_max = min(seq_len, mat.size(0))
                    out[0, :t_max] = mat[:t_max].to(device)

            elif isinstance(mat, (list, tuple)):
                if len(mat) == batch_size:
                    for b in range(batch_size):
                        row = mat[b]
                        if isinstance(row, torch.Tensor) and row.dim() >= 1:
                            t_max = min(seq_len, row.size(0))
                            for t in range(t_max):
                                out[b, t] = float(row[t].item())
                        elif isinstance(row, (list, tuple, np.ndarray)):
                            t_max = min(seq_len, len(row))
                            for t in range(t_max):
                                try:
                                    v = row[t]
                                    out[b, t] = (float(v.item()) if isinstance(v, torch.Tensor) else float(v))
                                except Exception:
                                    out[b, t] = float(default)
                elif batch_size == 1:
                    row = mat
                    t_max = min(seq_len, len(row))
                    for t in range(t_max):
                        try:
                            v = row[t]
                            out[0, t] = (float(v.item()) if isinstance(v, torch.Tensor) else float(v))
                        except Exception:
                            out[0, t] = float(default)

        except Exception:
            if _VERBOSE_LOGGING:
                try:
                    print("[ASBN] parse_scalar_matrix exception:", traceback.format_exc().splitlines()[-1])
                except Exception:
                    pass

        return out

    def compute_lambda_scaled_tensor(self, pmax: torch.Tensor, uncertainty: torch.Tensor,
                                    gate: torch.Tensor, lambda_type: str) -> torch.Tensor:
        base = float(self.lambda_base.get(lambda_type, 1.0))
        lam = base * torch.ones_like(pmax)
        lam = torch.clamp(lam, min=0.1, max=float(self.lambda_max))
        lam = lam.contiguous()
        lam = torch.where(torch.isfinite(lam), lam, torch.ones_like(lam))
        return lam

    def forward(
        self,
        h: torch.Tensor,
        proto_probs: Any = None,
        uncertainties: Any = None,
        gates: Any = None,
        token_word_map: Optional[List[Dict[int, str]]] = None,
        domain_labels: Optional[torch.Tensor] = None,
        global_step: Optional[int] = None,
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:

        if global_step is not None:
            self.current_step = int(global_step)

        if not isinstance(h, torch.Tensor) or h.dim() != 3:
            dev = h.device if isinstance(h, torch.Tensor) else torch.device("cpu")
            zero = torch.tensor(0.0, device=dev)
            return h, {
                "encoder_loss": zero,
                "adversarial_loss": zero,
                "domain_loss": zero,
                "domain_accuracy": zero,
            }

        B, T, H = h.size()
        device = h.device

        domain_labels = self._expand_domain_labels(domain_labels, B)

        h_normalized = h.clone()

        if domain_labels is not None and B * T >= 2:
            try:
                self._ensure_discriminators_on_device(device)
                h_flat = h.view(B * T, H)
                domain_expanded = domain_labels.unsqueeze(1).expand(B, T).reshape(-1)

                source_mask = domain_expanded == 0
                target_mask = domain_expanded == 1

                h_norm_flat = h_flat.clone()

                source_count = source_mask.sum().item()
                target_count = target_mask.sum().item()

                if source_count >= 2:
                    self.bn_source.train(self.training)
                    h_norm_flat[source_mask] = self.bn_source(h_flat[source_mask])
                elif source_count == 1:
                    self.bn_source.eval()
                    with torch.no_grad():
                        h_norm_flat[source_mask] = self.bn_source(h_flat[source_mask])

                if target_count >= 2:
                    self.bn_target.train(self.training)
                    h_norm_flat[target_mask] = self.bn_target(h_flat[target_mask])
                elif target_count == 1:
                    self.bn_target.eval()
                    with torch.no_grad():
                        h_norm_flat[target_mask] = self.bn_target(h_flat[target_mask])

                h_normalized = h_norm_flat.view(B, T, H)

                if _DEBUG_DISCOVERY and self.current_step % 500 == 0:
                    print(f"[ASBN-BN] Applied BN: {source_count} source, {target_count} target tokens")

            except Exception as e:
                if _VERBOSE_LOGGING:
                    print(f"[ASBN] BN failed: {e}")
                h_normalized = h

        if self.current_step < self.warmup_steps:
            if _DEBUG_DISCOVERY and self.current_step % 50 == 0:
                print(f"[ASBN] Warmup: {self.current_step}/{self.warmup_steps}")
            zero = torch.tensor(0.0, device=device)
            return h_normalized, {
                "encoder_loss": zero,
                "adversarial_loss": zero,
                "domain_loss": zero,
                "domain_accuracy": zero,
            }

        if not self.training or not _ENABLE_ASBN_TRAINING:
            zero = torch.tensor(0.0, device=device)
            return h_normalized, {
                "encoder_loss": zero,
                "adversarial_loss": zero,
                "domain_loss": zero,
                "domain_accuracy": zero,
            }

        self._ensure_discriminators_on_device(device)
        self.d_domain.train()
        self.d_freq.train()
        self.d_ctx.train()
        self.d_xl.train()

        pmax_mat = self._parse_proto_probs_matrix(proto_probs, B, T, device)
        U_mat = self._parse_scalar_matrix(uncertainties, B, T, device, default=0.1)
        G_mat = self._parse_scalar_matrix(gates, B, T, device, default=0.0)

        sel_mask = torch.ones((B, T), dtype=torch.bool, device=device)
        batch_indices = torch.arange(B, device=device).unsqueeze(1).expand(B, T)

        if token_word_map:
            try:
                for b in range(min(B, len(token_word_map))):
                    wm = token_word_map[b] or {}
                    for t in range(T):
                        if t in wm:
                            try:
                                token_str = wm[t]
                                tracked = True

                                if _has_should_track_token and _should_track_token_fn is not None:
                                    tracked = bool(_should_track_token_fn(token_str, self.special_tokens, self.tokenizer, self.language))
                                elif _has_is_valid_token and _is_valid_token_fn is not None:
                                    tracked = bool(_is_valid_token_fn(token_str, self.special_tokens, self.tokenizer, self.language))

                                if not tracked:
                                    sel_mask[b, t] = False
                            except Exception:
                                pass
            except Exception:
                if _VERBOSE_LOGGING:
                    try:
                        print("[ASBN] Token filtering failed:", traceback.format_exc().splitlines()[-1])
                    except Exception:
                        pass

        sel_idx = sel_mask.view(-1).nonzero(as_tuple=False).squeeze(1)
        batch_idx = batch_indices.view(-1)[sel_idx]

        if sel_idx.numel() == 0:
            if _DEBUG_DISCOVERY:
                print("[ASBN] No valid tokens after filtering")
            zero = torch.tensor(0.0, device=device)
            return h_normalized, {
                "encoder_loss": zero,
                "adversarial_loss": zero,
                "domain_loss": zero,
                "domain_accuracy": zero,
            }

        h_flat = h_normalized.view(B * T, H)
        sel_emb = h_flat[sel_idx]

        pmax_flat = pmax_mat.view(-1)[sel_idx]
        U_flat = U_mat.view(-1)[sel_idx]
        G_flat = G_mat.view(-1)[sel_idx]

        seq_len_feature = float(T) / max(int(_MAX_LENGTH), 1)
        freq_feature = torch.stack([pmax_flat, U_flat], dim=1).to(device)
        ctx_feature = torch.stack([G_flat, torch.full_like(G_flat, seq_len_feature)], dim=1).to(device)
        xl_input = sel_emb

        grl_alpha = self.get_grl_alpha(global_step)

        freq_input = torch.cat([sel_emb, freq_feature], dim=1)
        ctx_input = torch.cat([sel_emb, ctx_feature], dim=1)

        xl_input_grl = gradient_reversal(xl_input, alpha=grl_alpha)
        freq_input_grl = gradient_reversal(freq_input, alpha=grl_alpha)
        ctx_input_grl = gradient_reversal(ctx_input, alpha=grl_alpha)

        freq_logits = self.d_freq(freq_input_grl)
        ctx_logits = self.d_ctx(ctx_input_grl)
        xl_logits = self.d_xl(xl_input_grl)

        freq_label = (pmax_flat > self.freq_threshold).long().to(device)
        ctx_label = (U_flat < self.uncertainty_threshold).long().to(device)
        xl_label = (G_flat > self.gate_threshold).long().to(device)

        loss_freq = F.cross_entropy(freq_logits, freq_label, reduction="none")
        loss_ctx = F.cross_entropy(ctx_logits, ctx_label, reduction="none")
        loss_xl = F.cross_entropy(xl_logits, xl_label, reduction="none")

        lam_freq = self.compute_lambda_scaled_tensor(pmax_flat, U_flat, G_flat, "freq")
        lam_ctx = self.compute_lambda_scaled_tensor(pmax_flat, U_flat, G_flat, "ctx")
        lam_xl = self.compute_lambda_scaled_tensor(pmax_flat, U_flat, G_flat, "xl")

        weighted = lam_freq * loss_freq + lam_ctx * loss_ctx + lam_xl * loss_xl
        mean_weighted = torch.mean(weighted)

        domain_loss = torch.tensor(0.0, device=device)
        domain_accuracy = torch.tensor(0.0, device=device)

        if domain_labels is not None:
            try:
                domain_flat = domain_labels[batch_idx]

                domain_input = gradient_reversal(sel_emb, alpha=grl_alpha)
                domain_logits = self.d_domain(domain_input)

                domain_loss = F.cross_entropy(domain_logits, domain_flat)

                with torch.no_grad():
                    domain_preds = torch.argmax(domain_logits, dim=1)
                    domain_accuracy = (domain_preds == domain_flat).float().mean()

                    source_mask = domain_flat == 0
                    target_mask = domain_flat == 1

                    if source_mask.any():
                        source_acc = ((domain_preds[source_mask] == domain_flat[source_mask]).float().mean())
                        self.stats["source_accuracy"] += float(source_acc.item())

                    if target_mask.any():
                        target_acc = ((domain_preds[target_mask] == domain_flat[target_mask]).float().mean())
                        self.stats["target_accuracy"] += float(target_acc.item())

            except Exception as e:
                if _VERBOSE_LOGGING:
                    print(f"[ASBN] Domain loss failed: {e}")

        encoder_loss = self.encoder_grl_scale * (mean_weighted + domain_loss)

        try:
            with torch.no_grad():
                self.stats["domain_loss"] += float(domain_loss.item())
                self.stats["domain_accuracy"] += float(domain_accuracy.item())
                self.stats["asbn_loss"] += float(encoder_loss.item())
                self.stats["num_updates"] += 1

                if self.stats["num_updates"] >= self.stats_reset_interval:
                    if _DEBUG_DISCOVERY:
                        stats = self.get_detailed_stats()
                        print(f"\n[ASBN-STATS] After {stats['num_updates']} updates:")
                        print(f"  Domain loss: {stats['domain_loss']:.4f}")
                        print(f"  Domain acc: {stats['domain_accuracy']:.2%}")
                        print(f"  Source acc: {stats['source_accuracy']:.2%}")
                        print(f"  Target acc: {stats['target_accuracy']:.2%}")
                        print(f"  ASBN loss: {stats['asbn_loss']:.4f}")
                    self.reset_stats()
        except Exception:
            pass

        if _DEBUG_DISCOVERY and self.current_step % 500 == 0:
            print(f"\n[ASBN-STEP-{self.current_step}]")
            print(f"  GRL alpha: {grl_alpha:.3f}")
            print(f"  Encoder loss: {encoder_loss.item():.4f}")
            print(f"  Domain loss: {domain_loss.item():.4f}")
            print(f"  Domain acc: {domain_accuracy.item():.2%}")

        return h_normalized, {
            "encoder_loss": encoder_loss,
            "adversarial_loss": mean_weighted,
            "domain_loss": domain_loss,
            "domain_accuracy": domain_accuracy,
        }

    def test_asbn(self, batch_size: int = 2, seq_len: int = 10) -> bool:
        print("\n" + "=" * 60)
        print("[ASBN-TEST] Testing ASBN module")
        print("=" * 60)

        try:
            try:
                device = next(self.parameters()).device
            except StopIteration:
                device = torch.device("cpu")

            h = torch.randn(batch_size, seq_len, self.embed_dim, device=device)
            domain_labels = torch.randint(0, 2, (batch_size,), device=device)

            h_out, losses = self.forward(h, domain_labels=domain_labels)
            assert h_out.shape == h.shape, "Forward output shape mismatch"
            assert "domain_loss" in losses, "Missing domain_loss"
            print("  ✓ forward() with domain_labels passed")

            proto_probs = torch.rand(batch_size, seq_len, 3, device=device)
            uncertainties = torch.rand(batch_size, seq_len, device=device)
            gates = torch.rand(batch_size, seq_len, device=device)

            self.train()
            self.current_step = self.warmup_steps + 1

            h_out, losses = self.forward(
                h,
                proto_probs=proto_probs,
                uncertainties=uncertainties,
                gates=gates,
                domain_labels=domain_labels,
                global_step=self.current_step,
            )

            assert losses["encoder_loss"].item() >= 0.0, "Encoder loss negative"
            assert 0.0 <= losses["domain_accuracy"].item() <= 1.0, "Domain accuracy out of range"
            print("  ✓ forward() with full inputs passed")

            stats = self.get_detailed_stats()
            assert "domain_loss" in stats, "Missing domain_loss in stats"
            print("  ✓ Statistics tracking passed")

            print("\n✓ All ASBN tests passed")
            print("=" * 60 + "\n")
            return True

        except Exception as e:
            print(f"\n✗ ASBN test failed: {e}")
            traceback.print_exc()
            print("=" * 60 + "\n")
            return False


print("\n" + "=" * 80)
print("Cell 4: ASBN Module - VERIFIED CORRECT")
print("=" * 80)
print("Configuration:")
print(f"  - Warmup Steps: 50")
print(f"  - GRL Alpha: {_GRL_ALPHA_START:.3f} → {_GRL_ALPHA_END:.3f} over {_GRL_ALPHA_STEPS} steps")
print(f"  - GRL Schedule: {_GRL_ALPHA_SCHEDULE}")
print(f"  - Encoder GRL Scale: 1.0")
print(f"  - Stats Reset Interval: 100")
print(f"  - ASBN Training: {'ENABLED' if _ENABLE_ASBN_TRAINING else 'DISABLED'}")
print("=" * 80 + "\n")


In [ ]:
# ==============================================================================
# CELL 5: TRG MODULE - TRANSPARENT RATIONALE GENERATION
# ==============================================================================

from typing import List, Dict, Tuple, Optional, Set, Any
from collections import deque
import traceback
import numpy as np
import torch
import torch.nn as nn
import threading
import time

try:
    _TRG_EVIDENCE_K = int(TRG_EVIDENCE_K)
except (NameError, ValueError, TypeError):
    _TRG_EVIDENCE_K = 3

try:
    _TRG_GEN_EMBED = int(TRG_GEN_EMBED)
except (NameError, ValueError, TypeError):
    _TRG_GEN_EMBED = 64

try:
    _MAX_SILVER_BUFFER = int(MAX_SILVER_BUFFER)
except (NameError, ValueError, TypeError):
    _MAX_SILVER_BUFFER = 50

try:
    _VERBOSE_LOGGING = bool(VERBOSE_LOGGING)
except NameError:
    _VERBOSE_LOGGING = False

try:
    _DEBUG_DISCOVERY = bool(DEBUG_DISCOVERY)
except NameError:
    _DEBUG_DISCOVERY = False

try:
    _DEBUG_TIMING = bool(DEBUG_TIMING)
except NameError:
    _DEBUG_TIMING = False

try:
    _ENABLE_TRG_INFERENCE = bool(ENABLE_TRG_INFERENCE)
except NameError:
    _ENABLE_TRG_INFERENCE = True

try:
    _SOURCE_LANGUAGE = str(SOURCE_LANGUAGE)
except (NameError, TypeError):
    _SOURCE_LANGUAGE = "bn"

# [BUG-C5-1 / TRG-2 FIX]
# Original read from TAU_LOW (a dynamic tau threshold, ~0.15).
# TRG-2 requires reading config.uncertainty_threshold (CFG-1 = 0.05).
# Priority chain: config.uncertainty_threshold → TAU_LOW → 0.05
try:
    _TRG_UNCERTAINTY_THRESHOLD = float(config.uncertainty_threshold)
except (NameError, AttributeError, ValueError, TypeError):
    try:
        _TRG_UNCERTAINTY_THRESHOLD = float(TAU_LOW)
    except (NameError, ValueError, TypeError):
        _TRG_UNCERTAINTY_THRESHOLD = 0.05

# [BUG-C5-2 / TRG-3 FIX]
# Original read from SPAN_THRESHOLD global (default 0.20).
# TRG-3 requires reading config.span_threshold (CFG-2 = 0.08).
# Priority chain: config.span_threshold → SPAN_THRESHOLD → 0.08
try:
    _TRG_SPAN_THRESHOLD = float(config.span_threshold)
except (NameError, AttributeError, ValueError, TypeError):
    try:
        _TRG_SPAN_THRESHOLD = float(SPAN_THRESHOLD)
    except (NameError, ValueError, TypeError):
        _TRG_SPAN_THRESHOLD = 0.08

try:
    _TAU_HIGH = float(TAU_HIGH)
except (NameError, ValueError, TypeError):
    _TAU_HIGH = 0.85

try:
    _TAU_LOW = float(TAU_LOW)
except (NameError, ValueError, TypeError):
    _TAU_LOW = 0.15

try:
    _TAU_ACCEPT = float(TAU_ACCEPT)
except (NameError, ValueError, TypeError):
    _TAU_ACCEPT = 0.80

try:
    _TRG_TEMPERATURE = float(TRG_TEMPERATURE)
except (NameError, ValueError, TypeError):
    _TRG_TEMPERATURE = 1.0

try:
    _MAX_EXPLANATIONS_PER_SENTENCE = (
        int(MAX_EXPLANATIONS_PER_SENTENCE)
        if "MAX_EXPLANATIONS_PER_SENTENCE" in globals()
        else 10
    )
except Exception:
    _MAX_EXPLANATIONS_PER_SENTENCE = 10

# ── Late-binding helpers (loaded from Cell 1/2) ───────────────────────────────
_has_is_valid_token                = False
_has_get_tokenizer_special_tokens  = False
_has_get_cached_special_tokens     = False
_is_valid_token_fn                 = None
_get_tokenizer_special_tokens_fn   = None
_get_cached_special_tokens_fn      = None

try:
    if 'is_valid_token' in dir():
        _is_valid_token_fn  = is_valid_token
        _has_is_valid_token = True
    elif 'is_valid_token' in globals():
        _is_valid_token_fn  = globals()['is_valid_token']
        _has_is_valid_token = True
except Exception:
    pass

try:
    if 'get_tokenizer_special_tokens' in dir():
        _get_tokenizer_special_tokens_fn  = get_tokenizer_special_tokens
        _has_get_tokenizer_special_tokens = True
    elif 'get_tokenizer_special_tokens' in globals():
        _get_tokenizer_special_tokens_fn  = globals()['get_tokenizer_special_tokens']
        _has_get_tokenizer_special_tokens = True
except Exception:
    pass

try:
    if 'get_cached_special_tokens' in dir():
        _get_cached_special_tokens_fn  = get_cached_special_tokens
        _has_get_cached_special_tokens = True
    elif 'get_cached_special_tokens' in globals():
        _get_cached_special_tokens_fn  = globals()['get_cached_special_tokens']
        _has_get_cached_special_tokens = True
except Exception:
    pass

# ── Punctuation / function word sets ─────────────────────────────────────────
_BENGALI_PUNCT_SET = set(['।', '॥'])
_COMMON_PUNCT_SET  = set(['.', ',', ';', ':', '!', '?', '"', "'",
                           '-', '(', ')', '[', ']', '{', '}', '/', '\\'])
_TRG_PUNCT_SET     = _BENGALI_PUNCT_SET | _COMMON_PUNCT_SET

# [BUG-C5-8 FIX] 'কোথায' → 'কোথায়' (missing ় vowel mark was a typo)
_FUNCTION_WORDS: Set[str] = {
    'এবং', 'ও', 'কিন্তু', 'তবে', 'যদি', 'তাহলে', 'কারণ', 'যেমন',
    'যখন', 'তখন', 'যেহেতু', 'সেহেতু', 'অথবা', 'কিংবা', 'বা',
    'এই', 'সেই', 'ঐ', 'ওই', 'কোন', 'কোনো', 'যে', 'যা', 'যিনি',
    'একটি', 'একজন', 'কয়েকটি', 'অনেক', 'সব', 'সকল', 'কিছু', 'সবকিছু',
    'আমি', 'তুমি', 'সে', 'তিনি', 'আমরা', 'তোমরা', 'তারা', 'আপনি', 'আপনারা',
    'আমার', 'তোমার', 'তার', 'আমাদের', 'তোমাদের', 'তাদের', 'আপনার', 'আপনাদের',
    'কি', 'কী', 'কে', 'কেন', 'কখন', 'কোথায়',   # ← BUG-C5-8 fixed
    'কীভাবে', 'কতটা',
    'না', 'নয়', 'নেই', 'নি', 'আছে', 'ছিল', 'হবে', 'হয়',
    'থেকে', 'পর্যন্ত', 'জন্য', 'সঙ্গে', 'সাথে', 'দিয়ে', 'মধ্যে', 'উপর',
    'করা', 'করে', 'করেন', 'করছে', 'করবে', 'করলে', 'হওয়া', 'হয়ে', 'হয়েছে',
}


# ==============================================================================
# Module-level token helpers
# ==============================================================================

def _is_punctuation_only(token: str) -> bool:
    if not token or not isinstance(token, str):
        return False
    clean = (
        token.replace("▁", "")
             .replace("Ġ", "")
             .replace("##", "")
             .replace("</w>", "")
             .strip()
    )
    if not clean:
        return False
    if clean in _BENGALI_PUNCT_SET:
        return True
    if clean in _COMMON_PUNCT_SET:
        return True
    if len(clean) == 1 and not clean.isalnum():
        return True
    return all(c in _TRG_PUNCT_SET for c in clean)


def _fallback_is_valid_token(
    token: str,
    special_tokens: set,
    tokenizer=None,
    language: str = "bn",
) -> bool:
    if token is None:
        return False
    if not isinstance(token, str):
        try:
            token = str(token)
        except Exception:
            return False
    token = token.strip()
    if not token:
        return False
    if token in special_tokens:
        return False
    clean = (
        token.replace("▁", "")
             .replace("Ġ", "")
             .replace("##", "")
             .replace("</w>", "")
             .strip()
    )
    if len(clean) < 2:
        return False
    if not any(c.isalpha() for c in clean):
        return False
    if _is_punctuation_only(token):
        return False
    if clean.isdigit():
        return False
    return True


def _is_word_start(
    raw_token: str,
    token_word_map: Optional[dict],
    idx: int,
) -> bool:
    if not isinstance(raw_token, str):
        return False
    try:
        if token_word_map is not None and isinstance(token_word_map, dict):
            if idx in token_word_map:
                w = token_word_map[idx]
                if isinstance(w, str) and w.strip():
                    return True
        if raw_token.startswith("▁") or raw_token.startswith("Ġ"):
            return True
        clean = (
            raw_token.replace("▁", "")
                     .replace("Ġ", "")
                     .replace("##", "")
                     .replace("</w>", "")
                     .strip()
        )
        if len(clean) < 2:
            return False
        if _is_punctuation_only(raw_token):
            return False
        if token_word_map is None and any(c.isalpha() for c in clean):
            return True
        return False
    except Exception:
        return False


# ==============================================================================
# ComprehensiveTRGExplanationTemplate
# ==============================================================================

class ComprehensiveTRGExplanationTemplate:
    def __init__(self):
        self.explanation_templates = {
            "high_confidence": (
                "Chose '{sense}' with high confidence ({confidence:.1%}) based on: '{evidence}'.   "
                "Pattern matches learned data.   {alternatives_text}"
            ),
            "medium_confidence": (
                "Selected '{sense}' with moderate confidence ({confidence:.1%}). "
                "Evidence: '{evidence}'. Some uncertainty ({uncertainty:.1%}).   {alternatives_text}"
            ),
            "low_confidence": (
                "Uncertain; chose '{sense}' ({confidence:.1%}). "
                "Evidence: '{evidence}'.   {alternatives_text} Review recommended."
            ),
            "fallback": ("Token '{token}' analyzed.   Context: '{evidence}'."),
        }

    def generate_explanation(self, evidence: Dict) -> str:
        if not evidence or not isinstance(evidence, dict):
            return ""

        token = (
            str(evidence.get("token", "unknown"))
            .replace("▁", "")
            .replace("Ġ", "")
        )
        sense_info = evidence.get("chosen_sense", ("unknown", 0.5))

        if isinstance(sense_info, (tuple, list)) and len(sense_info) >= 2:
            sense_name, confidence = str(sense_info[0]), float(sense_info[1])
        else:
            sense_name, confidence = "unknown", 0.5

        uncertainty = float(evidence.get("uncertainty", 0.5))

        evidence_tokens = evidence.get("evidence_tokens", [])
        evidence_str = (
            ", ".join(
                [
                    str(tok).replace("▁", "").replace("Ġ", "")
                    for tok in evidence_tokens[:_TRG_EVIDENCE_K]
                ]
            )
            or "limited context"
        )

        alternatives      = evidence.get("alternatives", [])
        alternatives_text = ""
        if isinstance(alternatives, list) and len(alternatives) > 0:
            alt_parts = []
            for alt in alternatives[:2]:
                if isinstance(alt, (tuple, list)) and len(alt) >= 2:
                    alt_name, alt_conf = str(alt[0]), float(alt[1])
                    alt_parts.append(f"'{alt_name}' ({alt_conf:.1%})")
            if alt_parts:
                alternatives_text = f"Alternatives: {', '.join(alt_parts)}."

        if confidence >= _TAU_ACCEPT:
            template_key = "high_confidence"
        elif confidence >= _TRG_UNCERTAINTY_THRESHOLD:
            template_key = "medium_confidence"
        else:
            template_key = "low_confidence"

        template = self.explanation_templates.get(
            template_key, self.explanation_templates["fallback"]
        )

        try:
            return template.format(
                sense=sense_name,
                confidence=confidence,
                uncertainty=uncertainty,
                evidence=evidence_str,
                alternatives_text=alternatives_text,
                token=token,
            )
        except Exception:
            return f"Token '{token}' -> '{sense_name}' ({confidence:.1%})."


# ==============================================================================
# MemoryEfficientTRGExtractor
# ==============================================================================

class MemoryEfficientTRGExtractor:
    def __init__(self, tokenizer=None, language: str = "bn", dscd_module=None):
        self.tokenizer   = tokenizer
        self.language    = language
        self.dscd_module = dscd_module
        self.span_clamp_warnings = 0
        self.last_warning_time   = 0.0

        if tokenizer is not None:
            try:
                if _has_get_tokenizer_special_tokens and _get_tokenizer_special_tokens_fn is not None:
                    self.special_tokens = _get_tokenizer_special_tokens_fn(tokenizer)
                elif _has_get_cached_special_tokens and _get_cached_special_tokens_fn is not None:
                    self.special_tokens = _get_cached_special_tokens_fn(tokenizer)
                else:
                    self.special_tokens = set(tokenizer.all_special_tokens)
            except Exception:
                self.special_tokens = set()
        else:
            self.special_tokens = set()

    def extract_evidence_from_target(
        self,
        token_idx: int,
        span_start: int,
        span_end: int,
        tgt_preds: torch.Tensor,
    ) -> Optional[List[str]]:
        if not isinstance(token_idx, int) or token_idx < 0:
            return None
        if not isinstance(span_start, int) or not isinstance(span_end, int):
            return None
        if span_start < 0:
            return None
        if not isinstance(tgt_preds, (torch.Tensor, list)):
            return None
        seq_len = (
            len(tgt_preds)
            if isinstance(tgt_preds, list)
            else int(tgt_preds.size(0))
        )
        if span_end > seq_len:
            return None
        if span_start >= span_end:
            return None
        if token_idx < span_start or token_idx >= span_end:
            return None
        if token_idx >= seq_len:
            return None
        try:
            evidence_tokens: List[str] = []
            for i in range(span_start, span_end):
                if i == token_idx:
                    continue
                if isinstance(tgt_preds, list):
                    evidence_tokens.append(str(tgt_preds[i]))
                else:
                    try:
                        evidence_tokens.append(str(int(tgt_preds[i].item())))
                    except Exception:
                        evidence_tokens.append(f"token_{i}")
            return evidence_tokens if evidence_tokens else None
        except Exception:
            return None

    def extract_evidence_efficiently(
        self,
        token_idx: int,
        tokens: List[str],
        dscd_outputs: Dict,
        token_word_map: Optional[dict] = None,
        decoder_attention: Optional[torch.Tensor] = None,
    ) -> Dict:
        if not isinstance(tokens, list):
            return self._create_fallback_evidence(token_idx, [])
        if not isinstance(token_idx, int):
            return self._create_fallback_evidence(0, tokens)
        if token_idx < 0 or token_idx >= len(tokens):
            return self._create_fallback_evidence(
                max(0, min(token_idx, len(tokens) - 1)), tokens
            )

        raw_token = tokens[token_idx]

        if _has_is_valid_token and _is_valid_token_fn is not None:
            try:
                is_valid = _is_valid_token_fn(
                    raw_token,
                    self.special_tokens,
                    self.tokenizer,
                    language=self.language,
                )
            except Exception:
                is_valid = _fallback_is_valid_token(
                    raw_token, self.special_tokens, self.tokenizer, self.language
                )
        else:
            is_valid = _fallback_is_valid_token(
                raw_token, self.special_tokens, self.tokenizer, self.language
            )

        if not is_valid:
            return self._create_fallback_evidence(token_idx, tokens)

        try:
            proto_probs = self._safe_extract_proto_probs(token_idx, dscd_outputs)
            uncertainty = self._safe_extract_uncertainty(token_idx, dscd_outputs)
            gate        = self._safe_extract_gate(token_idx, dscd_outputs)
            span        = self._safe_extract_span(token_idx, dscd_outputs)

            # [TRG-4] Compute separation_score via DSCD-4 method
            separation_score = 0.0
            if self.dscd_module is not None:
                try:
                    token_key = (
                        raw_token.replace("▁", "")
                                 .replace("Ġ", "")
                                 .replace("##", "")
                                 .strip()
                                 .lower()
                    )
                    if hasattr(self.dscd_module, "compute_prototype_separation"):
                        separation_score = float(
                            self.dscd_module.compute_prototype_separation(token_key)
                        )
                except Exception:
                    separation_score = 0.0

            evidence_tokens: Optional[List[str]] = None
            if decoder_attention is not None and isinstance(decoder_attention, torch.Tensor):
                try:
                    if decoder_attention.dim() == 4:
                        if (
                            decoder_attention.size(0) > 1
                            and decoder_attention.size(1) > 1
                        ):
                            attn_avg = decoder_attention.mean(dim=(0, 1))
                        elif decoder_attention.size(0) > 1:
                            attn_avg = decoder_attention.mean(dim=1)
                        else:
                            attn_avg = decoder_attention.mean(dim=0)
                        if attn_avg.dim() == 2 and token_idx < attn_avg.size(0):
                            vec = attn_avg[token_idx]
                        else:
                            vec = attn_avg.reshape(-1)
                    elif decoder_attention.dim() == 3:
                        attn_avg = decoder_attention.mean(dim=0)
                        if attn_avg.dim() == 2 and token_idx < attn_avg.size(0):
                            vec = attn_avg[token_idx]
                        else:
                            vec = attn_avg.reshape(-1)
                    elif decoder_attention.dim() == 2:
                        if token_idx < decoder_attention.size(0):
                            vec = decoder_attention[token_idx]
                        else:
                            vec = decoder_attention.reshape(-1)
                    elif decoder_attention.dim() == 1:
                        vec = decoder_attention
                    else:
                        vec = None

                    if vec is not None and vec.numel() > 0:
                        k            = min(5, int(vec.size(0)))
                        top_k_indices = torch.topk(vec, k=k).indices.cpu().numpy()
                        evidence_tokens = []
                        for i in top_k_indices:
                            if i < len(tokens) and i != token_idx:
                                evidence_tokens.append(tokens[int(i)])
                except Exception:
                    evidence_tokens = None

            if evidence_tokens is None:
                evidence_tokens = self._extract_context_window(
                    token_idx, tokens, token_word_map
                )

            seen: Dict[str, bool] = {}
            dedup_evidence: List[str] = []
            for t in evidence_tokens:
                if t not in seen:
                    seen[t] = True
                    dedup_evidence.append(t)
            evidence_tokens = dedup_evidence[:_TRG_EVIDENCE_K]

            top_senses   = self._compute_sense_alternatives_fast(
                proto_probs, temperature=_TRG_TEMPERATURE
            )
            chosen_sense = top_senses[0] if len(top_senses) > 0 else ("unknown", 0.5)
            alternatives = top_senses[1:3] if len(top_senses) > 1 else []

            if (
                token_word_map
                and token_idx in token_word_map
                and isinstance(token_word_map[token_idx], str)
                and token_word_map[token_idx].strip()
            ):
                token_value = token_word_map[token_idx]
            else:
                token_value = raw_token

            # [TRG-4] separation_score added to return dict
            return {
                "token":            token_value,
                "token_idx":        token_idx,
                "evidence_tokens":  evidence_tokens,
                "chosen_sense":     chosen_sense,
                "alternatives":     alternatives,
                "uncertainty":      float(uncertainty),
                "gate":             float(gate),
                "span":             float(span),
                "separation_score": float(separation_score),
            }

        except Exception as e:
            if _VERBOSE_LOGGING or _DEBUG_DISCOVERY:
                print(f"[TRG] Evidence error @ {token_idx}: {e}")
            return self._create_fallback_evidence(token_idx, tokens)

    def _extract_context_window(
        self,
        token_idx: int,
        tokens: List[str],
        token_word_map: Optional[dict],
    ) -> List[str]:
        context_window = 2
        start_idx       = max(0, token_idx - context_window)
        end_idx         = min(len(tokens), token_idx + context_window + 1)
        evidence_tokens: List[str] = []

        for i in range(start_idx, end_idx):
            if i == token_idx or i >= len(tokens):
                continue
            rtok       = tokens[i]
            clean_token = (
                str(rtok)
                .replace("▁", "")
                .replace("Ġ", "")
                .replace("</w>", "")
                .strip()
            )

            if not _is_word_start(rtok, token_word_map, i):
                if (
                    token_word_map is None
                    and len(clean_token) >= 2
                    and any(c.isalpha() for c in clean_token)
                ):
                    pass
                else:
                    continue

            if _has_is_valid_token and _is_valid_token_fn is not None:
                try:
                    ok = _is_valid_token_fn(
                        rtok,
                        self.special_tokens,
                        self.tokenizer,
                        language=self.language,
                    )
                except Exception:
                    ok = _fallback_is_valid_token(
                        rtok, self.special_tokens, self.tokenizer, self.language
                    )
            else:
                ok = _fallback_is_valid_token(
                    rtok, self.special_tokens, self.tokenizer, self.language
                )

            if ok and len(clean_token) > 0:
                if (
                    token_word_map
                    and isinstance(token_word_map.get(i, ""), str)
                    and token_word_map[i].strip()
                ):
                    evidence_tokens.append(token_word_map[i].strip())
                else:
                    evidence_tokens.append(clean_token)

        return evidence_tokens

    # ── Safe DSCD output extractors ───────────────────────────────────────────

    def _safe_extract_proto_probs(
        self, token_idx: int, dscd_outputs: Dict
    ) -> torch.Tensor:
        try:
            if not isinstance(dscd_outputs, dict):
                return torch.tensor([1.0], dtype=torch.float32)
            pp_all = dscd_outputs.get("proto_probs", None)
            if pp_all and len(pp_all) > 0:
                row = pp_all[0]
                if isinstance(row, torch.Tensor):
                    if row.ndim == 2 and token_idx < row.shape[0]:
                        return row[token_idx].detach().cpu().flatten()
                    return row.detach().cpu().flatten()
                if isinstance(row, (list, tuple)):
                    if token_idx < len(row):
                        val = row[token_idx]
                        if isinstance(val, torch.Tensor):
                            return val.detach().cpu().flatten()
                        if isinstance(val, (list, tuple, np.ndarray)):
                            return torch.as_tensor(val, dtype=torch.float32).flatten()
                        return torch.tensor([float(val)], dtype=torch.float32)
                    if len(row) > 0:
                        maybe = row[0]
                        if isinstance(maybe, torch.Tensor):
                            return maybe.detach().cpu().flatten()
        except Exception:
            if _VERBOSE_LOGGING:
                print(
                    f"[TRG] Proto_probs extraction failed for token {token_idx}, "
                    f"using default [1.0]"
                )
        return torch.tensor([1.0], dtype=torch.float32)

    def _safe_extract_uncertainty(
        self, token_idx: int, dscd_outputs: Dict
    ) -> float:
        try:
            if not isinstance(dscd_outputs, dict):
                return 0.5
            U_all = dscd_outputs.get("uncertainties", None)
            if U_all and len(U_all) > 0:
                row = U_all[0]
                if isinstance(row, torch.Tensor):
                    if row.ndim == 2 and token_idx < row.shape[0]:
                        return float(row[token_idx].item())
                    if row.ndim == 1 and token_idx < row.shape[0]:
                        return float(row[token_idx].item())
                if isinstance(row, (list, tuple)) and token_idx < len(row):
                    val = row[token_idx]
                    return (
                        float(val.item())
                        if isinstance(val, torch.Tensor)
                        else float(val)
                    )
        except Exception:
            pass
        return 0.5

    def _safe_extract_gate(self, token_idx: int, dscd_outputs: Dict) -> float:
        try:
            if not isinstance(dscd_outputs, dict):
                return 0.0
            G_all = dscd_outputs.get("gates", None)
            if G_all and len(G_all) > 0:
                row = G_all[0]
                if isinstance(row, torch.Tensor):
                    if row.ndim == 2 and token_idx < row.shape[0]:
                        return float(row[token_idx].item())
                    if row.ndim == 1 and token_idx < row.shape[0]:
                        return float(row[token_idx].item())
                if isinstance(row, (list, tuple)) and token_idx < len(row):
                    val = row[token_idx]
                    return (
                        float(val.item())
                        if isinstance(val, torch.Tensor)
                        else float(val)
                    )
        except Exception:
            pass
        return 0.0

    def _safe_extract_span(self, token_idx: int, dscd_outputs: Dict) -> float:
        try:
            if not isinstance(dscd_outputs, dict):
                return 0.0
            S_all = dscd_outputs.get("span_preds", None)
            if S_all and len(S_all) > 0:
                row = S_all[0]
                if isinstance(row, torch.Tensor):
                    if row.ndim == 2 and token_idx < row.shape[0]:
                        span_val = float(row[token_idx].item())
                    elif row.ndim == 1 and token_idx < row.shape[0]:
                        span_val = float(row[token_idx].item())
                    else:
                        return 0.0
                elif isinstance(row, (list, tuple)) and token_idx < len(row):
                    val      = row[token_idx]
                    span_val = (
                        float(val.item())
                        if isinstance(val, torch.Tensor)
                        else float(val)
                    )
                else:
                    return 0.0

                if span_val < 0.0 or span_val > 1.0:
                    current_time = time.time()
                    if self.span_clamp_warnings < 10 or (
                        current_time - self.last_warning_time
                    ) > 60.0:
                        if _VERBOSE_LOGGING or _DEBUG_DISCOVERY:
                            print(f"[TRG] Clamping span {span_val:.3f} -> [0.0, 1.0]")
                        self.span_clamp_warnings  += 1
                        self.last_warning_time     = current_time
                span_val = max(0.0, min(1.0, float(span_val)))
                return span_val
        except Exception:
            pass
        return 0.0

    def compute_span(self, sense_probs) -> float:
        try:
            if isinstance(sense_probs, dict):
                probs = list(sense_probs.values())
            else:
                probs = sense_probs
            if isinstance(probs, torch.Tensor):
                probs = probs.cpu().numpy().flatten().tolist()
            if isinstance(probs, (np.ndarray, list)):
                probs = list(probs)
            if len(probs) < 2:
                return 0.0
            sorted_probs = sorted([float(p) for p in probs], reverse=True)
            span         = float(sorted_probs[0]) - float(sorted_probs[1])
            return max(0.0, min(1.0, span))
        except Exception:
            return 0.0

    def _compute_sense_alternatives_fast(
        self, proto_probs: torch.Tensor, temperature: float = 1.0
    ) -> List[Tuple[str, float]]:
        try:
            if not isinstance(proto_probs, torch.Tensor):
                proto_probs = torch.as_tensor(proto_probs, dtype=torch.float32)
            probs = proto_probs.flatten().float()
            if probs.numel() == 0:
                return [("unknown", 0.5)]
            probs = torch.clamp(probs, min=1e-10, max=1.0)
            if temperature != 1.0 and probs.numel() > 1:
                log_probs        = torch.log(probs)
                scaled_log_probs = log_probs / float(temperature)
                probs            = torch.softmax(scaled_log_probs, dim=0)
            if probs.numel() > 1:
                probs_sorted, indices = torch.sort(probs, descending=True)
                top_k = min(3, int(indices.numel()))
                return [
                    (
                        f"sense_{int(indices[i].item())}",
                        float(probs_sorted[i].item()),
                    )
                    for i in range(top_k)
                ]
            else:
                return [("sense_0", float(probs[0].item()))]
        except Exception:
            return [("unknown", 0.5)]

    def _create_fallback_evidence(
        self, token_idx: int, tokens: List[str]
    ) -> Dict:
        if isinstance(tokens, list) and 0 <= token_idx < len(tokens):
            token = tokens[token_idx]
        else:
            token = "UNK"
        # [BUG-C5-6 FIX] separation_score added so callers never get a KeyError
        return {
            "token":            token,
            "token_idx":        token_idx,
            "evidence_tokens":  [],
            "chosen_sense":     ("unknown", 0.5),
            "alternatives":     [],
            "uncertainty":      0.5,
            "gate":             0.0,
            "span":             0.0,
            "separation_score": 0.0,
        }

    def get_homograph_tokens_from_dscd(self) -> Set[str]:
        homograph_tokens: Set[str] = set()
        try:
            if self.dscd_module is not None:
                if hasattr(self.dscd_module, "get_discovered_homographs"):
                    homograph_tokens = set(
                        self.dscd_module.get_discovered_homographs()
                    )
                elif hasattr(self.dscd_module, "prototype_stores"):
                    for token, store in self.dscd_module.prototype_stores.items():
                        if hasattr(store, "size") and store.size() >= 2:
                            clean = (
                                str(token)
                                .replace("▁", "")
                                .replace("Ġ", "")
                                .replace("##", "")
                                .strip()
                            )
                            homograph_tokens.add(clean)
        except Exception:
            pass
        return homograph_tokens


# ==============================================================================
# CompleteTRGWithExplanations
# ==============================================================================

class CompleteTRGWithExplanations(nn.Module):
    def __init__(
        self,
        embed_dim: Optional[int] = None,
        tokenizer=None,
        language: str = "bn",
        dscd_module=None,
    ):
        super().__init__()
        self.embed_dim   = int(embed_dim) if embed_dim is not None else int(_TRG_GEN_EMBED)
        self.tokenizer   = tokenizer
        self.language    = language
        self.dscd_module = dscd_module

        if dscd_module is None:
            if _VERBOSE_LOGGING or _DEBUG_DISCOVERY:
                print("[TRG] No DSCD module - homograph detection disabled")

        if tokenizer is not None:
            try:
                if _has_get_tokenizer_special_tokens and _get_tokenizer_special_tokens_fn is not None:
                    self.special_tokens = _get_tokenizer_special_tokens_fn(tokenizer)
                elif _has_get_cached_special_tokens and _get_cached_special_tokens_fn is not None:
                    self.special_tokens = _get_cached_special_tokens_fn(tokenizer)
                else:
                    self.special_tokens = set(tokenizer.all_special_tokens)
            except Exception:
                self.special_tokens = set()
        else:
            self.special_tokens = set()

        # [BUG-C5-3 / TRG-2 / TRG-3 FIX]
        # Store config-derived thresholds as instance attrs so
        # process_sentence_for_explanations uses the correct values
        # rather than the stale module-level globals.
        self.uncertainty_threshold = float(_TRG_UNCERTAINTY_THRESHOLD)
        self.span_threshold        = float(_TRG_SPAN_THRESHOLD)

        self.template_system   = ComprehensiveTRGExplanationTemplate()
        self.evidence_extractor = MemoryEfficientTRGExtractor(
            tokenizer, language=language, dscd_module=dscd_module
        )

        self.silver_buffer = deque(maxlen=int(_MAX_SILVER_BUFFER))
        self._silver_lock  = threading.Lock()

        self.stats_reset_interval = 1000
        self.stats = {
            "explanations_generated":       0,
            "high_confidence_explanations": 0,
            "low_confidence_explanations":  0,
            "empty_evidence_count":         0,
            "total_evidence_tokens":        0,
            "tokens_filtered_word_start":   0,
            "tokens_filtered_validity":     0,
            "tokens_filtered_ambiguity":    0,
            "dscd_homographs_explained":    0,
        }
        self._stats_lock = threading.Lock()

        if _VERBOSE_LOGGING or _DEBUG_DISCOVERY:
            print("[TRG] Initialized:")
            print(f"  - Uncertainty threshold: {self.uncertainty_threshold:.4f} (from config)")
            print(f"  - Span threshold:        {self.span_threshold:.4f} (from config)")
            print(f"  - Temperature:           {_TRG_TEMPERATURE:.2f}")
            print(f"  - Mode: DATA-DRIVEN + ADAPTIVE THRESHOLDS")
            print(
                f"  - Function availability: "
                f"is_valid={_has_is_valid_token}, "
                f"get_special={_has_get_tokenizer_special_tokens}, "
                f"get_cached={_has_get_cached_special_tokens}"
            )

    # ── Statistics helpers ────────────────────────────────────────────────────

    def _update_stats(self, evidence: Dict, is_dscd_homograph: bool = False) -> None:
        with self._stats_lock:
            self.stats["explanations_generated"] += 1
            if is_dscd_homograph:
                self.stats["dscd_homographs_explained"] += 1
            if not evidence.get("evidence_tokens"):
                self.stats["empty_evidence_count"] += 1
            else:
                self.stats["total_evidence_tokens"] += len(evidence["evidence_tokens"])

            confidence = 0.5
            chosen     = evidence.get("chosen_sense")
            if isinstance(chosen, (tuple, list)) and len(chosen) >= 2:
                try:
                    confidence = float(chosen[1])
                except Exception:
                    confidence = 0.5

            if confidence >= _TAU_ACCEPT:
                self.stats["high_confidence_explanations"] += 1
            elif confidence < self.uncertainty_threshold:   # [BUG-C5-3] use instance attr
                self.stats["low_confidence_explanations"] += 1

            if self.stats["explanations_generated"] >= self.stats_reset_interval:
                if _DEBUG_DISCOVERY:
                    current_stats = self.get_statistics()
                    print(
                        f"\n[TRG-STATS] After {self.stats['explanations_generated']}:"
                    )
                    print(f"  High conf: {current_stats['high_confidence_rate']:.2%}")
                    print(f"  DSCD: {current_stats['dscd_homograph_rate']:.2%}")
                self.reset_statistics()

    def _add_to_silver_buffer(
        self, evidence: Dict, explanation: str, tokens: List[str]
    ) -> None:
        try:
            conf   = 0.5
            chosen = evidence.get("chosen_sense")
            if isinstance(chosen, (tuple, list)) and len(chosen) >= 2:
                conf = float(chosen[1])
            entry = {
                "token":       str(evidence.get("token", "UNK"))[:20],
                "explanation": str(explanation)[:150],
                "confidence":  conf,
            }
            with self._silver_lock:
                self.silver_buffer.append(entry)
        except Exception:
            pass

    # ── Per-token explanation ─────────────────────────────────────────────────

    def generate_explanation_for_token(
        self,
        token_idx: int,
        tokens: List[str],
        dscd_outputs: Dict,
        token_word_map: Optional[dict]       = None,
        decoder_attention: Optional[torch.Tensor] = None,
        is_dscd_homograph: bool              = False,
    ) -> Tuple[str, Dict]:
        if self.training or not _ENABLE_TRG_INFERENCE:
            return "", {}
        if not isinstance(tokens, list) or not isinstance(token_idx, int):
            return "", {}
        if token_idx < 0 or token_idx >= len(tokens):
            return "", {}

        raw_token = tokens[token_idx]

        if _has_is_valid_token and _is_valid_token_fn is not None:
            try:
                is_valid = _is_valid_token_fn(
                    raw_token,
                    self.special_tokens,
                    self.tokenizer,
                    language=self.language,
                )
            except Exception:
                is_valid = _fallback_is_valid_token(
                    raw_token, self.special_tokens, self.tokenizer, self.language
                )
        else:
            is_valid = _fallback_is_valid_token(
                raw_token, self.special_tokens, self.tokenizer, self.language
            )

        if not is_valid:
            return "", {}

        try:
            evidence         = self.evidence_extractor.extract_evidence_efficiently(
                token_idx,
                tokens,
                dscd_outputs,
                token_word_map=token_word_map,
                decoder_attention=decoder_attention,
            )
            explanation_text = self.template_system.generate_explanation(evidence)
            self._update_stats(evidence, is_dscd_homograph=is_dscd_homograph)
            self._add_to_silver_buffer(evidence, explanation_text, tokens)
            return explanation_text, evidence
        except Exception:
            return "", {}

    # ── Tensor list helper ────────────────────────────────────────────────────

    @staticmethod
    def _to_list_helper(x: Any) -> List[float]:
        if x is None:
            return []
        try:
            if isinstance(x, torch.Tensor):
                x = x.detach().cpu()
                if x.ndim == 0:
                    return [float(x.item())]
                if x.ndim == 1:
                    return [float(v.item()) for v in x]
                if x.ndim == 2:
                    if x.size(0) == 1:
                        return [float(v.item()) for v in x[0]]
                    else:
                        return [float(v.item()) for v in x.flatten()]
                if x.ndim >= 3:
                    return [float(v.item()) for v in x.flatten()]
            if isinstance(x, (list, tuple)):
                out: List[float] = []
                for v in x:
                    if isinstance(v, torch.Tensor):
                        v = v.detach().cpu()
                        if v.ndim == 0:
                            out.append(float(v.item()))
                        elif v.numel() > 0:
                            out.append(float(v.flatten()[0].item()))
                        else:
                            out.append(0.0)
                    elif isinstance(v, (int, float, np.number)):
                        out.append(float(v))
                    else:
                        try:
                            out.append(float(v))
                        except Exception:
                            out.append(0.0)
                return out
            if isinstance(x, (int, float, np.number)):
                return [float(x)]
            return [float(x)]
        except Exception:
            return []

    # ── Adaptive threshold computation ────────────────────────────────────────

    def compute_uncertainty_adaptive(
        self, proto_probs: Any, uncertainties: Any
    ) -> Tuple[float, float]:
        try:
            U = self._to_list_helper(uncertainties)
            if not U or len(U) == 0:
                return self.uncertainty_threshold, self.uncertainty_threshold
            U_arr = np.array(U, dtype=np.float32)
            U_arr = U_arr[np.isfinite(U_arr)]
            if len(U_arr) == 0:
                return self.uncertainty_threshold, self.uncertainty_threshold
            median_u           = float(np.median(U_arr))
            std_u              = float(np.std(U_arr))
            adaptive_threshold = median_u + 0.5 * std_u
            adaptive_threshold = max(0.05, min(0.50, adaptive_threshold))
            return float(adaptive_threshold), float(median_u)
        except Exception:
            return self.uncertainty_threshold, self.uncertainty_threshold

    def compute_span_adaptive(self, span_preds: Any) -> Tuple[float, float]:
        try:
            S = self._to_list_helper(span_preds)
            if not S or len(S) == 0:
                return self.span_threshold, self.span_threshold
            S_arr = np.array(S, dtype=np.float32)
            S_arr = S_arr[np.isfinite(S_arr)]
            if len(S_arr) == 0:
                return self.span_threshold, self.span_threshold
            median_s           = float(np.median(S_arr))
            percentile_75      = float(np.percentile(S_arr, 75))
            adaptive_threshold = 0.5 * median_s + 0.5 * percentile_75
            adaptive_threshold = max(0.02, min(0.30, adaptive_threshold))
            return float(adaptive_threshold), float(median_s)
        except Exception:
            return self.span_threshold, self.span_threshold

    # ── Sentence-level explanation pipeline ───────────────────────────────────

    def process_sentence_for_explanations(
        self,
        tokens: List[str],
        dscd_outputs: Dict,
        token_word_map: Optional[dict]            = None,
        span_threshold: Optional[float]           = None,
        uncertainty_threshold: Optional[float]    = None,
        decoder_attention: Optional[torch.Tensor] = None,
        max_explanations: int                     = _MAX_EXPLANATIONS_PER_SENTENCE,
    ) -> List[Dict]:
        if self.training or not _ENABLE_TRG_INFERENCE:
            return []

        # [BUG-C5-3 FIX] Use self.span_threshold / self.uncertainty_threshold
        # as defaults instead of the stale module-level globals.
        if span_threshold is None:
            span_threshold        = self.span_threshold
        if uncertainty_threshold is None:
            uncertainty_threshold = self.uncertainty_threshold

        explanations: List[Dict] = []

        try:
            if not tokens or not isinstance(tokens, list):
                return explanations
            if not isinstance(dscd_outputs, dict) or not dscd_outputs:
                return explanations

            U_all = dscd_outputs.get("uncertainties", [])
            S_all = dscd_outputs.get("span_preds",    [])

            if not U_all or not U_all[0]:
                return explanations

            U   = self._to_list_helper(U_all[0])
            S   = (
                self._to_list_helper(S_all[0])
                if S_all and S_all[0]
                else [0.0] * len(tokens)
            )

            seq_len = len(tokens)
            if len(U) < seq_len:
                U.extend([0.5] * (seq_len - len(U)))
            elif len(U) > seq_len:
                U = U[:seq_len]
            if len(S) < seq_len:
                S.extend([0.0] * (seq_len - len(S)))
            elif len(S) > seq_len:
                S = S[:seq_len]

            if not U:
                return explanations

            adaptive_u_threshold, median_u = self.compute_uncertainty_adaptive(
                dscd_outputs.get("proto_probs", None), U_all[0]
            )
            adaptive_s_threshold, median_s = self.compute_span_adaptive(
                S_all[0] if S_all else None
            )

            strict_uncertainty = max(adaptive_u_threshold, uncertainty_threshold)
            strict_span        = max(adaptive_s_threshold, span_threshold)

            if _DEBUG_DISCOVERY:
                print(
                    f"[TRG-ADAPTIVE] U: median={median_u:.3f}, "
                    f"thresh={strict_uncertainty:.3f}"
                )
                print(
                    f"[TRG-ADAPTIVE] S: median={median_s:.3f}, "
                    f"thresh={strict_span:.3f}"
                )

            dscd_homographs = self.evidence_extractor.get_homograph_tokens_from_dscd()

            candidates: List[Tuple[int, float, float, str, int, int]] = []
            local_stats = {
                "tokens_filtered_word_start": 0,
                "tokens_filtered_validity":   0,
                "tokens_filtered_ambiguity":  0,
            }

            for idx in range(seq_len):
                tok       = tokens[idx]
                clean_tok = tok.replace("▁", "").replace("Ġ", "").strip()

                if _is_punctuation_only(tok):
                    local_stats["tokens_filtered_validity"] += 1
                    continue

                if not _is_word_start(tok, token_word_map, idx):
                    local_stats["tokens_filtered_word_start"] += 1
                    continue

                if _has_is_valid_token and _is_valid_token_fn is not None:
                    try:
                        valid = _is_valid_token_fn(
                            tok,
                            self.special_tokens,
                            self.tokenizer,
                            language=self.language,
                        )
                    except Exception:
                        valid = _fallback_is_valid_token(
                            tok, self.special_tokens, self.tokenizer, self.language
                        )
                else:
                    valid = _fallback_is_valid_token(
                        tok, self.special_tokens, self.tokenizer, self.language
                    )

                if not valid:
                    local_stats["tokens_filtered_validity"] += 1
                    continue

                if clean_tok in _FUNCTION_WORDS:
                    local_stats["tokens_filtered_validity"] += 1
                    continue

                if len(clean_tok) < 3 and not any(
                    '\u0980' <= c <= '\u09FF' for c in clean_tok
                ):
                    local_stats["tokens_filtered_validity"] += 1
                    continue

                u = float(U[idx])
                s = float(S[idx])

                # [TRG-1 FIX] Replace hardcoded string-set lookup with the
                # data-driven is_automatic_homograph() call from DSCD-5.
                # Falls back to legacy set-check if dscd_module is absent.
                in_dscd = False
                if self.dscd_module is not None and hasattr(
                    self.dscd_module, "is_automatic_homograph"
                ):
                    try:
                        in_dscd = self.dscd_module.is_automatic_homograph(
                            clean_tok, clean_tok
                        )
                    except Exception:
                        in_dscd = clean_tok in dscd_homographs
                else:
                    in_dscd = clean_tok in dscd_homographs

                if in_dscd:
                    priority = 1
                elif s >= strict_span and u >= strict_uncertainty:
                    priority = 2
                elif s >= strict_span:
                    priority = 3
                elif u >= strict_uncertainty:
                    priority = 4
                else:
                    local_stats["tokens_filtered_ambiguity"] += 1
                    continue

                candidates.append((idx, u, s, clean_tok, priority, idx))

            with self._stats_lock:
                self.stats["tokens_filtered_word_start"] += local_stats[
                    "tokens_filtered_word_start"
                ]
                self.stats["tokens_filtered_validity"]   += local_stats[
                    "tokens_filtered_validity"
                ]
                self.stats["tokens_filtered_ambiguity"]  += local_stats[
                    "tokens_filtered_ambiguity"
                ]

            if not candidates:
                return explanations

            candidates.sort(key=lambda t: (t[4], -t[2], -t[1], t[5]))

            for (token_idx, u, s, clean_tok, priority, _) in candidates[
                :max_explanations
            ]:
                try:
                    explanation_text, evidence = self.generate_explanation_for_token(
                        token_idx,
                        tokens,
                        dscd_outputs,
                        token_word_map=token_word_map,
                        decoder_attention=decoder_attention,
                        is_dscd_homograph=(priority == 1),
                    )
                    if explanation_text and evidence:
                        # [BUG-C5-7 / TRG-4 FIX] Propagate separation_score
                        # into the per-sentence explanation dict for SCEA.
                        explanations.append(
                            {
                                "token_idx":        token_idx,
                                "token": (
                                    token_word_map[token_idx]
                                    if token_word_map
                                    and token_idx in token_word_map
                                    else tokens[token_idx]
                                    .replace("▁", "")
                                    .replace("Ġ", "")
                                ),
                                "explanation":      explanation_text,
                                "uncertainty":      u,
                                "span":             s,
                                "dscd_discovered":  (priority == 1),
                                "priority":         priority,
                                "separation_score": float(
                                    evidence.get("separation_score", 0.0)
                                ),
                            }
                        )
                except Exception:
                    continue

        except Exception:
            pass

        return explanations

    # ── Statistics accessors ──────────────────────────────────────────────────

    def get_statistics(self) -> Dict:
        with self._stats_lock:
            total = max(self.stats["explanations_generated"], 1)
            avg_evidence_tokens = (
                self.stats["total_evidence_tokens"] / total
                if self.stats["explanations_generated"] > 0
                else 0.0
            )
            return {
                **self.stats.copy(),
                "high_confidence_rate": self.stats["high_confidence_explanations"] / total,
                "low_confidence_rate":  self.stats["low_confidence_explanations"]  / total,
                "empty_evidence_rate":  self.stats["empty_evidence_count"]         / total,
                "avg_evidence_tokens":  avg_evidence_tokens,
                "silver_buffer_size":   len(self.silver_buffer),
                "dscd_homograph_rate":  self.stats["dscd_homographs_explained"]    / total,
            }

    def reset_statistics(self) -> None:
        with self._stats_lock:
            self.stats = {
                "explanations_generated":       0,
                "high_confidence_explanations": 0,
                "low_confidence_explanations":  0,
                "empty_evidence_count":         0,
                "total_evidence_tokens":        0,
                "tokens_filtered_word_start":   0,
                "tokens_filtered_validity":     0,
                "tokens_filtered_ambiguity":    0,
                "dscd_homographs_explained":    0,
            }

    def clear_silver_buffer(self) -> None:
        with self._silver_lock:
            self.silver_buffer.clear()

    # ── Smoke-test ────────────────────────────────────────────────────────────

    def test_trg(self, tokenizer=None) -> bool:
        print("\n" + "=" * 60)
        print("[TRG-TEST] Testing")
        print("=" * 60)

        if not _ENABLE_TRG_INFERENCE:
            print("TRG inference disabled, enabling for test...")

        try:
            tokens = ["▁আমি", "▁কল", "▁বন্ধ", "▁করেছি", "।"]

            dscd_outputs = {
                "proto_probs":  [[torch.tensor([0.6, 0.4]) for _ in tokens]],
                "uncertainties": [[0.1, 0.5, 0.2, 0.1, 0.0]],
                "span_preds":   [[0.05, 0.3, 0.1, 0.05, 0.0]],
                "gates":        [[0.2, 0.8, 0.3, 0.2, 0.0]],
            }
            token_word_map = {
                0: "আমি",
                1: "কল",
                2: "বন্ধ",
                3: "করেছি",
                4: "।",
            }

            self.eval()

            explanations = self.process_sentence_for_explanations(
                tokens=tokens,
                dscd_outputs=dscd_outputs,
                token_word_map=token_word_map,
                max_explanations=3,
            )

            print(f"  ✓ Generated {len(explanations)} explanations")
            if len(explanations) > 0:
                for i, expl in enumerate(explanations, 1):
                    sep = expl.get("separation_score", 0.0)
                    print(
                        f"    {i}. '{expl['token']}' "
                        f"(u={expl['uncertainty']:.2f}, sep={sep:.3f})"
                    )

            stats = self.get_statistics()
            print(f"  ✓ Stats: {stats['explanations_generated']} total")

            self.reset_statistics()
            stats_after = self.get_statistics()
            assert stats_after["explanations_generated"] == 0
            print("  ✓ Reset OK")

            print("\n✓ All TRG tests passed")
            print("=" * 60 + "\n")
            return True

        except Exception as e:
            print(f"\n✗ Test failed: {e}")
            try:
                traceback.print_exc()
            except Exception:
                pass
            print("=" * 60 + "\n")
            return False


# ==============================================================================
print("\n" + "=" * 80)
print("Cell 5: TRG Module - READY")
print("=" * 80)
print("Configuration:")
print(f"  - Uncertainty threshold : {_TRG_UNCERTAINTY_THRESHOLD:.4f}  ← config.uncertainty_threshold (TRG-2)")
print(f"  - Span threshold        : {_TRG_SPAN_THRESHOLD:.4f}  ← config.span_threshold (TRG-3)")
print(f"  - Temperature           : {_TRG_TEMPERATURE:.2f}")
print(f"  - TAU_HIGH / LOW / ACCEPT: {_TAU_HIGH:.2f} / {_TAU_LOW:.2f} / {_TAU_ACCEPT:.2f}")
print(f"  - Evidence K            : {_TRG_EVIDENCE_K}")
print(f"  - Max explanations      : {_MAX_EXPLANATIONS_PER_SENTENCE}")
print("Fixes applied:")
print("  ✅ TRG-1  is_automatic_homograph() replaces string-set homograph lookup")
print("  ✅ TRG-2  uncertainty_threshold ← config.uncertainty_threshold (was TAU_LOW)")
print("  ✅ TRG-3  span_threshold ← config.span_threshold (was SPAN_THRESHOLD global)")
print("  ✅ TRG-4  separation_score field added to all evidence/explanation dicts")
print("  ✅ BUG-C5-3  self.uncertainty_threshold / self.span_threshold stored as attrs")
print("  ✅ BUG-C5-6  separation_score:0.0 in _create_fallback_evidence")
print("  ✅ BUG-C5-7  separation_score propagated into process_sentence output dicts")
print("  ✅ BUG-C5-8  'কোথায' typo fixed → 'কোথায়' in _FUNCTION_WORDS")
print("=" * 80 + "\n")


In [ ]:
# ===========================================================================================
# CELL 6: DUAL-PATH TATN MODEL - PATH 1 (WORD-LEVEL) + PATH 2 (SUBWORD-LEVEL)
# ===========================================================================================

from typing import List, Dict, Optional, Any, Tuple, Union
import traceback
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import MBartForConditionalGeneration
from transformers.modeling_outputs import BaseModelOutput
import threading
import gc
import time
import re

try:
    _SOURCE_LANGUAGE = str(SOURCE_LANGUAGE)
    _TARGET_LANGUAGE = str(TARGET_LANGUAGE)
except (NameError, TypeError):
    _SOURCE_LANGUAGE = "bn_IN"
    _TARGET_LANGUAGE = "en_XX"


def _get_int_global(name: str, default: int) -> int:
    try:
        val = globals().get(name)
        if val is not None:
            return int(val)
    except (ValueError, TypeError):
        pass
    return default


def _get_float_global(name: str, default: float) -> float:
    try:
        val = globals().get(name)
        if val is not None:
            return float(val)
    except (ValueError, TypeError):
        pass
    return default


def _get_bool_global(name: str, default: bool) -> bool:
    try:
        val = globals().get(name)
        if val is not None:
            return bool(val)
    except (ValueError, TypeError):
        pass
    return default


_DSCD_BUFFER_SIZE = _get_int_global("DSCD_BUFFER_SIZE", 20)
_DSCD_MAX_PROTOS = _get_int_global("DSCD_MAX_PROTOS", 3)
_DSCD_N_MIN = _get_int_global("DSCD_N_MIN", 3)
_DSCD_DISPERSION_THRESHOLD = _get_float_global("DSCD_DISPERSION_THRESHOLD", 0.20)

_ENABLE_ASBN_TRAINING = _get_bool_global("ENABLE_ASBN_TRAINING", False)
_ENABLE_ASBN_INFERENCE = _get_bool_global("ENABLE_ASBN_INFERENCE", False)
_ENABLE_TRG_INFERENCE = _get_bool_global("ENABLE_TRG_INFERENCE", True)
_MEMORY_CLEANUP_FREQUENCY = _get_int_global("MEMORY_CLEANUP_FREQUENCY", 100)

_NUM_GPUS = _get_int_global(
    "NUM_GPUS",
    torch.cuda.device_count() if torch.cuda.is_available() else 1,
)
_USE_GC = _get_bool_global("GRADIENT_CHECKPOINTING", True)
_DSCD_ENABLE_TRAINING_CLUSTERING = _get_bool_global(
    "DSCD_ENABLE_TRAINING_CLUSTERING", True
)

_LAMBDA_ASBN = _get_float_global("LAMBDA_ASBN", 0.0)
_LAMBDA_DSCD = _get_float_global("LAMBDA_DSCD", 0.15)
_LAMBDA_TOKEN = _get_float_global("LAMBDA_TOKEN", 0.25)
_LAMBDA_CONFIDENCE = _get_float_global("LAMBDA_CONFIDENCE", 0.15)
_LAMBDA_LENGTH = _get_float_global("LAMBDA_LENGTH", 0.05)

_VERBOSE_LOGGING = _get_bool_global("VERBOSE_LOGGING", False)

try:
    _DEBUG_DISCOVERY = bool(DEBUG_DISCOVERY)
except (NameError, TypeError):
    _DEBUG_DISCOVERY = False

try:
    _DEBUG_TIMING = bool(DEBUG_TIMING)
except (NameError, TypeError):
    _DEBUG_TIMING = False

_PERIODIC_DISCOVERY_FREQUENCY = _get_int_global(
    "PERIODIC_DISCOVERY_FREQUENCY", 50
)
_VALIDATION_CHECK_INTERVAL = _get_int_global("VALIDATION_CHECK_INTERVAL", 200)

_SPAN_THRESHOLD = _get_float_global("SPAN_THRESHOLD", 0.20)
_UNCERTAINTY_THRESHOLD = _get_float_global("UNCERTAINTY_THRESHOLD", 0.15)

try:
    _TRG_UNCERTAINTY_THRESHOLD = float(config.uncertainty_threshold)
except (NameError, AttributeError, ValueError, TypeError):
    _TRG_UNCERTAINTY_THRESHOLD = _get_float_global(
        "TRG_UNCERTAINTY_THRESHOLD", _get_float_global("TAU_LOW", 0.15)
    )

_TAU_LOW = _get_float_global("TAU_LOW", 0.15)

_TRAIN_DOMAIN = _get_int_global("TRAIN_DOMAIN", 0)
_TEST_DOMAIN = _get_int_global("TEST_DOMAIN", 1)
_USE_DOMAIN_LABELS = _get_bool_global("USE_DOMAIN_LABELS", True)

try:
    _MBART50_EN_TOKEN_ID = int(MBART50_EN_TOKEN_ID)
except (NameError, ValueError, TypeError):
    _MBART50_EN_TOKEN_ID = 250004

try:
    _MBART50_BN_TOKEN_ID = int(MBART50_BN_TOKEN_ID)
except (NameError, ValueError, TypeError):
    _MBART50_BN_TOKEN_ID = 250028

try:
    _LABEL_SMOOTHING_EPS = float(LABEL_SMOOTHING)
except (NameError, ValueError, TypeError):
    _LABEL_SMOOTHING_EPS = 0.1

try:
    _RDROP_ALPHA = float(RDROP_ALPHA)
except (NameError, ValueError, TypeError):
    _RDROP_ALPHA = 5.0

_has_reconstruct_word_spans = "reconstruct_word_spans" in globals()

_BENGALI_PUNCT_SET = set(['।', '॥'])
_COMMON_PUNCT_SET = set([
    '.', ',', ';', ':', '!', '?', '"', "'",
    '-', '(', ')', '[', ']', '{', '}', '/', '\\'
])
_PUNCT_SET = _BENGALI_PUNCT_SET | _COMMON_PUNCT_SET


def _is_punctuation_only(token: str) -> bool:
    if not token or not isinstance(token, str):
        return False
    clean = (
        token.replace("▁", "")
        .replace("Ġ", "")
        .replace("##", "")
        .replace("</w>", "")
        .strip()
    )
    if not clean:
        return False
    if clean in _BENGALI_PUNCT_SET:
        return True
    if clean in _COMMON_PUNCT_SET:
        return True
    if len(clean) == 1 and not clean.isalnum():
        return True
    return all(c in _PUNCT_SET for c in clean)


def _safe_get_last_hidden_state(enc_output):
    if enc_output is None:
        return None
    if hasattr(enc_output, "last_hidden_state"):
        return enc_output.last_hidden_state
    if isinstance(enc_output, (list, tuple)) and len(enc_output) > 0:
        return enc_output[0]
    return None


def build_token_word_map_sentencepiece(
    token_strings: List[str], fallback: bool = True
) -> Dict[int, str]:
    word_map: Dict[int, str] = {}
    current_word = ""
    start_idx = None

    for i, token in enumerate(token_strings):
        if not token or token.startswith("<") or token.startswith("["):
            continue
        if token.startswith("▁"):
            if current_word and start_idx is not None:
                clean = current_word.replace("▁", "").strip()
                if clean and len(clean) >= 2 and not _is_punctuation_only(current_word):
                    word_map[start_idx] = clean
            current_word = token
            start_idx = i
        else:
            current_word += token

    if current_word and start_idx is not None:
        clean = current_word.replace("▁", "").strip()
        if clean and len(clean) >= 2 and not _is_punctuation_only(current_word):
            word_map[start_idx] = clean

    if fallback and not word_map:
        for i, tok in enumerate(token_strings):
            clean = tok.replace("▁", "").strip()
            if clean and len(clean) >= 2 and not _is_punctuation_only(tok):
                word_map[i] = clean

    return word_map


def _normalize_dscd_outputs(
    raw: Dict[str, Any],
    batch_size: int,
    seq_len: int,
    device: torch.device,
    embed_dim: int,
    fallback_h: Optional[torch.Tensor] = None,
) -> Dict[str, Any]:
    if fallback_h is None:
        fallback_h_augmented = torch.zeros(
            batch_size, seq_len, embed_dim, device=device, dtype=torch.float32
        )
    else:
        fallback_h_augmented = fallback_h.detach().clone()

    defaults = {
        "h_augmented": fallback_h_augmented,
        "proto_probs": [
            [
                torch.tensor([1.0], device=device, dtype=torch.float32)
                for _ in range(seq_len)
            ]
            for _ in range(batch_size)
        ],
        "uncertainties": [
            [
                torch.tensor(0.0, device=device, dtype=torch.float32)
                for _ in range(seq_len)
            ]
            for _ in range(batch_size)
        ],
        "gates": [
            [
                torch.tensor(0.0, device=device, dtype=torch.float32)
                for _ in range(seq_len)
            ]
            for _ in range(batch_size)
        ],
        "span_preds": [
            [
                torch.tensor(0.0, device=device, dtype=torch.float32)
                for _ in range(seq_len)
            ]
            for _ in range(batch_size)
        ],
        "proto_assignments": [
            torch.zeros(seq_len, dtype=torch.long, device=device)
            for _ in range(batch_size)
        ],
    }

    if not isinstance(raw, dict):
        return defaults

    out = defaults.copy()

    try:
        if "h_augmented" in raw and raw["h_augmented"] is not None:
            h = raw["h_augmented"]
            if isinstance(h, torch.Tensor) and h.shape == (
                batch_size,
                seq_len,
                embed_dim,
            ):
                out["h_augmented"] = h.to(device)
            else:
                try:
                    out["h_augmented"] = (
                        h.to(device).reshape(batch_size, seq_len, embed_dim)
                    )
                except Exception:
                    out["h_augmented"] = fallback_h_augmented
    except Exception:
        out["h_augmented"] = fallback_h_augmented

    for list_key in ("proto_probs", "uncertainties", "gates", "span_preds"):
        if list_key in raw and raw[list_key] is not None:
            try:
                val = raw[list_key]
                if isinstance(val, list) and len(val) == batch_size:
                    safe_batch = []
                    for b_row in val:
                        if isinstance(b_row, list):
                            safe_row = []
                            for t_idx in range(seq_len):
                                try:
                                    if t_idx < len(b_row):
                                        v = b_row[t_idx]
                                        if isinstance(v, torch.Tensor):
                                            safe_row.append(v.detach().to(device))
                                        else:
                                            safe_row.append(
                                                torch.as_tensor(
                                                    v,
                                                    device=device,
                                                    dtype=torch.float32,
                                                )
                                            )
                                    else:
                                        if list_key == "proto_probs":
                                            safe_row.append(
                                                torch.tensor(
                                                    [1.0],
                                                    device=device,
                                                    dtype=torch.float32,
                                                )
                                            )
                                        else:
                                            safe_row.append(
                                                torch.tensor(
                                                    0.0,
                                                    device=device,
                                                    dtype=torch.float32,
                                                )
                                            )
                                except Exception:
                                    safe_row.append(
                                        torch.tensor(
                                            0.0,
                                            device=device,
                                            dtype=torch.float32,
                                        )
                                    )
                            safe_batch.append(safe_row)
                        else:
                            if list_key == "proto_probs":
                                safe_batch.append(
                                    [
                                        torch.tensor(
                                            [1.0],
                                            device=device,
                                            dtype=torch.float32,
                                        )
                                        for _ in range(seq_len)
                                    ]
                                )
                            else:
                                safe_batch.append(
                                    [
                                        torch.tensor(
                                            0.0,
                                            device=device,
                                            dtype=torch.float32,
                                        )
                                        for _ in range(seq_len)
                                    ]
                                )
                    out[list_key] = safe_batch
            except Exception:
                pass

    try:
        if "proto_assignments" in raw and raw["proto_assignments"] is not None:
            pa = raw["proto_assignments"]
            if isinstance(pa, list) and len(pa) == batch_size:
                safe_pa = []
                for b_row in pa:
                    try:
                        if isinstance(b_row, torch.Tensor):
                            safe_pa.append(b_row.detach().to(device).long())
                        else:
                            safe_pa.append(
                                torch.tensor(
                                    b_row, dtype=torch.long, device=device
                                )
                            )
                    except Exception:
                        safe_pa.append(
                            torch.zeros(seq_len, dtype=torch.long, device=device)
                        )
                out["proto_assignments"] = safe_pa
    except Exception:
        pass

    return out


def _norm_scalar_matrix(uncertainties, gates, gate_threshold=0.01):
    final_normalized = []
    batch_size = len(uncertainties)
    for b in range(batch_size):
        u_row = uncertainties[b]
        g_row = gates[b]
        seq_len = len(u_row)
        safe_row = []
        for t in range(seq_len):
            try:
                u_val = float(u_row[t]) if t < len(u_row) else 0.0
                g_val = float(g_row[t]) if t < len(g_row) else 0.0
                if g_val < gate_threshold:
                    norm_val = 0.0
                else:
                    norm_val = max(0.0, min(1.0, u_val))
                safe_row.append(norm_val)
            except Exception:
                safe_row.append(0.0)
        final_normalized.append(safe_row)
    return final_normalized


def _norm_proto_probs(proto_probs):
    return [
        [pp if isinstance(pp, torch.Tensor) else torch.tensor([1.0]) for pp in row]
        for row in proto_probs
    ]


def _to_vec(x):
    if isinstance(x, torch.Tensor):
        return x.flatten().tolist()
    elif isinstance(x, (list, tuple)):
        return list(x)
    elif isinstance(x, (int, float)):
        return [float(x)]
    else:
        return [0.0]


def _extract_words_from_text(text: str) -> List[str]:
    if not text or not isinstance(text, str):
        return []
    text = text.strip()
    if not text:
        return []
    try:
        words = re.findall(r'[\u0980-\u09FF]+|[a-zA-Z]+|\d+', text)
        words = [w for w in words if w and len(w) > 0 and not _is_punctuation_only(w)]
        return words if words else []
    except Exception:
        return []


def _project_word_to_subword(
    h_word: torch.Tensor,
    target_seq_len: int,
    device: torch.device,
) -> torch.Tensor:
    """
    Resizes word-level hidden states [batch, word_len, dim] to
    subword sequence length [batch, target_seq_len, dim] via adaptive avg pool.
    """
    batch_size, word_len, embed_dim = h_word.shape
    if word_len == target_seq_len:
        return h_word.to(device)
    h_word_float = h_word.float().to(device)
    h_permuted = h_word_float.permute(0, 2, 1)
    h_pooled = F.adaptive_avg_pool1d(h_permuted, target_seq_len)
    return h_pooled.permute(0, 2, 1)


# ==============================================================================
# [SCEA-1] SenseConditionedEncoderAdapter
# ==============================================================================

class SenseConditionedEncoderAdapter(nn.Module):
    """
    SenseConditionedEncoderAdapter (SCEA)
    """
    def __init__(self, embed_dim: int = 1024, num_senses: int = 5, bottleneck: int = 128):
        super().__init__()
        self.embed_dim   = embed_dim
        self.num_senses  = num_senses
        self.bottleneck  = bottleneck

        self.down_proj = nn.ModuleList(
            [nn.Linear(embed_dim, bottleneck, bias=True) for _ in range(num_senses)]
        )
        self.up_proj = nn.ModuleList(
            [nn.Linear(bottleneck, embed_dim, bias=True) for _ in range(num_senses)]
        )
        self.sense_gates = nn.Parameter(
            torch.full((num_senses,), -3.0, dtype=torch.float32)
        )
        self._init_weights()

    def _init_weights(self):
        for layer in self.down_proj:
            nn.init.normal_(layer.weight, mean=0.0, std=1e-4)
            nn.init.zeros_(layer.bias)
        for layer in self.up_proj:
            nn.init.zeros_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(
        self,
        enc_out: torch.Tensor,
        sense_assignments: torch.Tensor,
        ambiguity_scores: torch.Tensor,
    ) -> torch.Tensor:
        if enc_out is None or not isinstance(enc_out, torch.Tensor):
            return enc_out

        B, T, D = enc_out.shape
        device   = enc_out.device
        out      = enc_out.clone()

        if sense_assignments is None or not isinstance(sense_assignments, torch.Tensor):
            sense_assignments = torch.zeros(B, T, dtype=torch.long, device=device)
        sense_assignments = sense_assignments.to(device).clamp(0, self.num_senses - 1)

        if ambiguity_scores is None or not isinstance(ambiguity_scores, torch.Tensor):
            ambiguity_scores = torch.zeros(B, T, dtype=torch.float32, device=device)
        ambiguity_scores = ambiguity_scores.to(device).float().clamp(0.0, 1.0)

        gates = torch.sigmoid(self.sense_gates)

        try:
            for s in range(self.num_senses):
                mask = (sense_assignments == s)
                if not mask.any():
                    continue

                gate_s    = gates[s]
                flat_mask = mask.view(-1)
                flat_enc  = enc_out.view(-1, D)
                tokens_s  = flat_enc[flat_mask]

                h_down = F.relu(self.down_proj[s](tokens_s))
                delta  = self.up_proj[s](h_down)

                flat_amb = ambiguity_scores.view(-1)
                amb_s    = flat_amb[flat_mask].unsqueeze(-1)

                flat_out = out.view(-1, D)
                flat_out[flat_mask] = flat_out[flat_mask] + gate_s * amb_s * delta
                out = flat_out.view(B, T, D)

        except Exception:
            return enc_out

        return out


# ==============================================================================
# [HFCAA-1] HomographFocusedCrossAttentionAdapter
# ==============================================================================

class HomographFocusedCrossAttentionAdapter(nn.Module):
    """
    HomographFocusedCrossAttentionAdapter (HFCAA)
    """
    def __init__(self, embed_dim: int = 1024, num_heads: int = 8):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        assert embed_dim % num_heads == 0, \
            f"embed_dim {embed_dim} must be divisible by num_heads {num_heads}"

        self.q_proj   = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj   = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj   = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

        self.gate_linear = nn.Linear(embed_dim, 1, bias=True)
        self.layer_norm  = nn.LayerNorm(embed_dim)

        self._init_weights()

    def _init_weights(self):
        nn.init.zeros_(self.out_proj.weight)
        nn.init.normal_(self.q_proj.weight, std=0.02)
        nn.init.normal_(self.k_proj.weight, std=0.02)
        nn.init.normal_(self.v_proj.weight, std=0.02)
        nn.init.constant_(self.gate_linear.bias, -4.0)
        nn.init.normal_(self.gate_linear.weight, std=0.01)

    def forward(
        self,
        dec_hidden: torch.Tensor,
        sense_adapted_enc: torch.Tensor,
        ambiguity_scores: torch.Tensor,
    ) -> torch.Tensor:
        if dec_hidden is None or sense_adapted_enc is None:
            return dec_hidden

        if not isinstance(dec_hidden, torch.Tensor) or \
           not isinstance(sense_adapted_enc, torch.Tensor):
            return dec_hidden

        B, T_dec, D  = dec_hidden.shape
        _,  T_enc, _ = sense_adapted_enc.shape
        device       = dec_hidden.device

        if ambiguity_scores is None or not isinstance(ambiguity_scores, torch.Tensor):
            ambiguity_scores = torch.zeros(B, T_enc, dtype=torch.float32, device=device)
        ambiguity_scores = ambiguity_scores.to(device).float().clamp(0.0, 1.0)

        try:
            Q = self.q_proj(dec_hidden)
            K = self.k_proj(sense_adapted_enc)
            V = self.v_proj(sense_adapted_enc)

            def _split_heads(x, t):
                return x.view(B, t, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

            Q = _split_heads(Q, T_dec)
            K = _split_heads(K, T_enc)
            V = _split_heads(V, T_enc)

            scale       = float(self.head_dim) ** -0.5
            attn_logits = torch.matmul(Q, K.transpose(-2, -1)) * scale

            amb_bias    = ambiguity_scores.unsqueeze(1).unsqueeze(2)
            attn_logits = attn_logits + amb_bias

            attn_weights = F.softmax(attn_logits, dim=-1)
            attn_out     = torch.matmul(attn_weights, V)

            attn_out = (
                attn_out.permute(0, 2, 1, 3)
                        .contiguous()
                        .view(B, T_dec, D)
            )

            proj_out = self.out_proj(attn_out)
            gate     = torch.sigmoid(self.gate_linear(dec_hidden))
            refined  = self.layer_norm(dec_hidden + gate * proj_out)

        except Exception:
            return dec_hidden

        return refined


# ==============================================================================
# MemoryOptimizedTATNWithExplanations (main TATN model)
# ==============================================================================

class MemoryOptimizedTATNWithExplanations(nn.Module):
    def __init__(self, tokenizer):
        super().__init__()
        self.tokenizer = tokenizer

        self.global_step           = 0
        self._step_lock            = threading.Lock()
        self.last_discovery_step   = 0
        self.last_validation_step  = 0

        self._last_path1_h_aug: Optional[torch.Tensor] = None

        # initialize tracking attributes
        self._last_translation_loss   = 0.0
        self._last_asbn_loss          = 0.0
        self._last_domain_loss        = 0.0
        self._last_domain_accuracy    = 0.0
        self._last_dscd_loss          = 0.0
        self._last_token_penalty      = 0.0
        self._last_confidence_penalty = 0.0
        self._last_length_penalty     = 0.0
        self._last_path1_asbn_loss    = 0.0
        self._last_path1_dscd_loss    = 0.0

        self.mbart = MBartForConditionalGeneration.from_pretrained(
            "facebook/mbart-large-50-many-to-many-mmt",
            torch_dtype=torch.float32,
            use_cache=False,
        )
        try:
            self.mbart.config.use_cache = False
        except Exception:
            pass

        tokenizer_vocab_size = (
            len(self.tokenizer)
            if hasattr(self.tokenizer, "__len__")
            else getattr(self.tokenizer, "vocab_size", 128112)
        )
        model_vocab_size = int(getattr(self.mbart.config, "vocab_size", 128112))

        if tokenizer_vocab_size != model_vocab_size:
            print(f"[TATN-INIT] Vocab size mismatch detected!")
            print(f"  Tokenizer: {tokenizer_vocab_size}")
            print(f"  Model: {model_vocab_size}")
            if tokenizer_vocab_size < model_vocab_size:
                print(f"[TATN-INIT] Using model's vocab size ({model_vocab_size})")
                print(f"[TATN-INIT] Note: Tokenizer has {model_vocab_size - tokenizer_vocab_size} fewer tokens")
                print(f"[TATN-INIT] Model embeddings preserved (no resize)")
                self.vocab_size = model_vocab_size
            else:
                print(f"[TATN-INIT] ERROR: Tokenizer vocab ({tokenizer_vocab_size}) > Model vocab ({model_vocab_size})")
                raise RuntimeError(
                    f"Tokenizer has more tokens than model!\n"
                    f"  Tokenizer: {tokenizer_vocab_size}\n"
                    f"  Model: {model_vocab_size}"
                )
        else:
            print(f"[TATN-INIT] Vocab sizes match: {model_vocab_size}")
            self.vocab_size = model_vocab_size

        try:
            if hasattr(self.tokenizer, "get_lang_id"):
                en_token_id = self.tokenizer.get_lang_id(_TARGET_LANGUAGE)
                bn_token_id = self.tokenizer.get_lang_id(_SOURCE_LANGUAGE)
            elif hasattr(self.tokenizer, "lang_code_to_id"):
                en_token_id = self.tokenizer.lang_code_to_id.get(
                    _TARGET_LANGUAGE, _MBART50_EN_TOKEN_ID
                )
                bn_token_id = self.tokenizer.lang_code_to_id.get(
                    _SOURCE_LANGUAGE, _MBART50_BN_TOKEN_ID
                )
            else:
                en_token_id = _MBART50_EN_TOKEN_ID
                bn_token_id = _MBART50_BN_TOKEN_ID

            if en_token_id >= self.vocab_size or bn_token_id >= self.vocab_size:
                raise ValueError(
                    f"Language token IDs out of vocabulary bounds!\n"
                    f"  EN token: {en_token_id} (vocab: {self.vocab_size})\n"
                    f"  BN token: {bn_token_id} (vocab: {self.vocab_size})"
                )

            self.mbart.config.decoder_start_token_id = int(en_token_id)
            self.mbart.config.forced_bos_token_id    = None
            self.en_token_id = int(en_token_id)
            self.bn_token_id = int(bn_token_id)

            if _DEBUG_DISCOVERY:
                print(f"[TATN-INIT] Language tokens: BN={bn_token_id}, EN={en_token_id}")
                print(f"[TATN-INIT] Disabled forced_bos_token_id to prevent double BOS")
                if bn_token_id != _MBART50_BN_TOKEN_ID or en_token_id != _MBART50_EN_TOKEN_ID:
                    print(f"[TATN-INIT] Token IDs differ from Cell 0 defaults:")
                    print(f"  Expected: BN={_MBART50_BN_TOKEN_ID}, EN={_MBART50_EN_TOKEN_ID}")
                    print(f"  Got: BN={bn_token_id}, EN={en_token_id}")

        except Exception as e:
            if _DEBUG_DISCOVERY:
                print(f"[TATN-INIT] Failed to set language tokens: {e}")
            raise RuntimeError(f"Language token setup failed: {e}")

        try:
            if _USE_GC and hasattr(self.mbart, "gradient_checkpointing_enable"):
                self.mbart.gradient_checkpointing_enable()
        except Exception:
            pass

        embed_dim = int(getattr(self.mbart.config, "d_model", 1024))

        if _DEBUG_DISCOVERY:
            print(f"[TATN-INIT] Model embed_dim: {embed_dim}")

        dscd_cls = globals().get("MemoryEfficientDSCDOnline", None)
        if callable(dscd_cls):
            try:
                self.dscd = dscd_cls(
                    embed_dim=embed_dim,
                    tokenizer=tokenizer,
                    buffer_size=_DSCD_BUFFER_SIZE,
                    max_protos=_DSCD_MAX_PROTOS,
                    n_min=_DSCD_N_MIN,
                    language=_SOURCE_LANGUAGE,
                    dispersion_threshold=_DSCD_DISPERSION_THRESHOLD,
                    enable_training_clustering=_DSCD_ENABLE_TRAINING_CLUSTERING,
                    max_clustering_points=500,
                    max_candidates_per_step=1,
                )
                dscd_embed_dim = getattr(self.dscd, "embed_dim", None)
                if dscd_embed_dim is not None and dscd_embed_dim != embed_dim:
                    raise RuntimeError(
                        f"DSCD embed_dim mismatch! Expected {embed_dim}, got {dscd_embed_dim}"
                    )
            except Exception as e:
                raise RuntimeError(
                    f"Failed to instantiate MemoryEfficientDSCDOnline: {e}"
                )
        else:
            raise RuntimeError("MemoryEfficientDSCDOnline not found in globals()")

        asbn_cls = globals().get("MemoryEfficientASBNModule", None)
        if callable(asbn_cls):
            try:
                self.asbn = asbn_cls(
                    embed_dim, tokenizer, language=_SOURCE_LANGUAGE
                )
                asbn_embed_dim = getattr(self.asbn, "embed_dim", None)
                if asbn_embed_dim is not None and asbn_embed_dim != embed_dim:
                    raise RuntimeError(
                        f"ASBN embed_dim mismatch! Expected {embed_dim}, got {asbn_embed_dim}"
                    )
            except Exception as e:
                print(f"[TATN-INIT] ASBN init failed: {e}, using stub")
                self.asbn = self._build_stub_asbn()
        else:
            self.asbn = self._build_stub_asbn()

        trg_cls = globals().get("CompleteTRGWithExplanations", None)
        if callable(trg_cls):
            try:
                self.trg = trg_cls(
                    embed_dim,
                    tokenizer,
                    language=_SOURCE_LANGUAGE,
                    dscd_module=self.dscd,
                )
            except Exception as e:
                print(f"[TATN-INIT] TRG init failed: {e}, using stub")
                self.trg = self._build_stub_trg()
        else:
            self.trg = self._build_stub_trg()

        self.scea  = SenseConditionedEncoderAdapter(embed_dim, 5, 128)
        self.hfcaa = HomographFocusedCrossAttentionAdapter(embed_dim, 8)

        label_smoothing_cls = globals().get("LabelSmoothingLoss", None)
        if callable(label_smoothing_cls):
            try:
                self.label_smoothing_loss = label_smoothing_cls(
                    num_classes=self.vocab_size,
                    smoothing=_LABEL_SMOOTHING_EPS,
                    ignore_index=-100,
                )
            except Exception as e:
                print(f"[TATN-INIT] LabelSmoothingLoss init failed: {e}, using None")
                self.label_smoothing_loss = None
        else:
            self.label_smoothing_loss = None

        rdrop_cls = globals().get("RDropLoss", None)
        if callable(rdrop_cls):
            try:
                self.rdrop_loss = rdrop_cls(alpha=_RDROP_ALPHA)
            except Exception as e:
                print(f"[TATN-INIT] RDropLoss init failed: {e}, using None")
                self.rdrop_loss = None
        else:
            self.rdrop_loss = None

        embedding_layer = self.mbart.get_input_embeddings()
        if embedding_layer is None:
            raise RuntimeError("Model has no input embeddings layer!")

        actual_embed_dim = embedding_layer.embedding_dim
        if actual_embed_dim != embed_dim:
            raise RuntimeError(
                f"Embedding dimension mismatch! Config says {embed_dim}, "
                f"but embedding layer has {actual_embed_dim}"
            )

        if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
            print("[TATN-INIT] Initialized DUAL-PATH MemoryOptimizedTATNWithExplanations:")
            print(f"  - Embed dim: {embed_dim}")
            print(f"  - Vocab size: {self.vocab_size}")
            print(f"  - Tokenizer vocab: {tokenizer_vocab_size}")
            print(f"  - BN token: {self.bn_token_id}")
            print(f"  - EN token: {self.en_token_id}")
            print(f"  - DSCD buffer: {_DSCD_BUFFER_SIZE}")
            print(f"  - DSCD n_min: {_DSCD_N_MIN}")
            print(f"  - DSCD threshold: {_DSCD_DISPERSION_THRESHOLD}")
            print(f"  - Discovery frequency: {_PERIODIC_DISCOVERY_FREQUENCY}")
            print(f"  - Validation interval: {_VALIDATION_CHECK_INTERVAL}")
            print(f"  - Lambda ASBN: {_LAMBDA_ASBN}")
            print(f"  - Lambda DSCD: {_LAMBDA_DSCD}")
            print(f"  - Lambda Token: {_LAMBDA_TOKEN}")
            print(f"  - Lambda Confidence: {_LAMBDA_CONFIDENCE}")
            print(f"  - Lambda Length: {_LAMBDA_LENGTH}")
            print(f"  - Label Smoothing eps: {_LABEL_SMOOTHING_EPS}")
            print(f"  - R-Drop alpha: {_RDROP_ALPHA}")
            print(f"  - ASBN Training: {'DISABLED' if not _ENABLE_ASBN_TRAINING else 'ENABLED'}")
            print(f"  - ASBN Inference: {'DISABLED' if not _ENABLE_ASBN_INFERENCE else 'ENABLED'}")
            print(f"  - SCEA: SenseConditionedEncoderAdapter(embed={embed_dim}, senses=5, bottleneck=128)")
            print(f"  - HFCAA: HomographFocusedCrossAttentionAdapter(embed={embed_dim}, heads=8)")
            print(f"  - Path 1 (DSCD): WORD-level -> DSCD -> ASBN -> exposes h_aug")
            print(f"  - Path 2 (Translation): SUBWORD -> Full MBART + Path1 h_aug injection + LabelSmoothing + R-Drop")

    # -------------------------------------------------------------------------
    # get_sense_assignments_from_dscd
    # -------------------------------------------------------------------------
    def get_sense_assignments_from_dscd(
        self,
        dscd_out: Dict[str, Any],
        encoder_output: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        device     = encoder_output.device
        batch_size = encoder_output.size(0)
        seq_len    = encoder_output.size(1)

        proto_assignments_raw = dscd_out.get("proto_assignments", None)
        sense_assignments = torch.zeros(
            batch_size, seq_len, dtype=torch.long, device=device
        )
        try:
            if (
                isinstance(proto_assignments_raw, list)
                and len(proto_assignments_raw) == batch_size
            ):
                for b, pa in enumerate(proto_assignments_raw):
                    try:
                        if isinstance(pa, torch.Tensor):
                            t = min(int(pa.size(0)), seq_len)
                            sense_assignments[b, :t] = pa[:t].to(device).long()
                        else:
                            t_pa = torch.tensor(
                                pa, dtype=torch.long, device=device
                            )
                            t = min(int(t_pa.size(0)), seq_len)
                            sense_assignments[b, :t] = t_pa[:t]
                    except Exception:
                        pass
            else:
                proto_probs_raw = dscd_out.get("proto_probs", [])
                if (
                    isinstance(proto_probs_raw, list)
                    and len(proto_probs_raw) == batch_size
                ):
                    for b, row in enumerate(proto_probs_raw):
                        for t, pp in enumerate(row):
                            if t >= seq_len:
                                break
                            try:
                                if isinstance(pp, torch.Tensor) and pp.numel() > 1:
                                    sense_assignments[b, t] = int(
                                        pp.argmax().item()
                                    )
                            except Exception:
                                pass
        except Exception:
            pass

        ambiguity_scores = torch.zeros(
            batch_size, seq_len, dtype=torch.float32, device=device
        )
        try:
            uncertainties_raw = dscd_out.get("uncertainties", [])
            gates_raw         = dscd_out.get("gates", [])
            if (
                isinstance(uncertainties_raw, list)
                and len(uncertainties_raw) == batch_size
            ):
                for b, u_row in enumerate(uncertainties_raw):
                    g_row = (
                        gates_raw[b]
                        if isinstance(gates_raw, list) and b < len(gates_raw)
                        else []
                    )
                    for t, u_val in enumerate(u_row):
                        if t >= seq_len:
                            break
                        try:
                            u_f = (
                                float(u_val.item())
                                if isinstance(u_val, torch.Tensor)
                                else float(u_val)
                            )
                            g_f = 0.0
                            if t < len(g_row):
                                gv  = g_row[t]
                                g_f = (
                                    float(gv.item())
                                    if isinstance(gv, torch.Tensor)
                                    else float(gv)
                                )
                            score = max(0.0, min(1.0, u_f)) if g_f > 0.01 else 0.0
                            ambiguity_scores[b, t] = score
                        except Exception:
                            pass
        except Exception:
            pass

        return sense_assignments, ambiguity_scores

    # -------------------------------------------------------------------------
    # Stub builders
    # -------------------------------------------------------------------------
    def _build_stub_asbn(self):
        class _StubASBN(nn.Module):
            def forward(self, h, **kwargs):
                dev  = h.device if isinstance(h, torch.Tensor) else torch.device("cpu")
                zero = torch.tensor(0.0, device=dev, requires_grad=True)
                return h, {
                    "encoder_loss":    zero,
                    "adversarial_loss": zero,
                    "domain_loss":     zero,
                    "domain_accuracy": zero,
                }

            def critic_parameters(self):
                return []

            def reset_stats(self):
                pass

            def get_detailed_stats(self):
                return {
                    "domain_loss":      0.0,
                    "domain_accuracy":  0.0,
                    "source_accuracy":  0.0,
                    "target_accuracy":  0.0,
                    "asbn_loss":        0.0,
                    "num_updates":      0,
                }

            def get_asbn_stats(self):
                return self.get_detailed_stats()

        return _StubASBN()

    def _build_stub_trg(self):
        class _StubTRG:
            def process_sentence_for_explanations(self, *args, **kwargs):
                return []

            def get_statistics(self):
                return {}

            def reset_statistics(self):
                pass

        return _StubTRG()

    # -------------------------------------------------------------------------
    # Static helpers
    # -------------------------------------------------------------------------
    @staticmethod
    def _entropy_reg_from_proto_probs_static(
        proto_probs_list, gates_list=None, min_gate: float = 0.01
    ) -> torch.Tensor:
        if not proto_probs_list or not isinstance(proto_probs_list, list):
            return torch.tensor(0.0)

        dev = None
        for row in proto_probs_list:
            if isinstance(row, list):
                for p in row:
                    if isinstance(p, torch.Tensor):
                        dev = p.device
                        break
            if dev is not None:
                break

        if dev is None:
            return torch.tensor(0.0)

        total = torch.tensor(0.0, device=dev)
        count = 0

        for b, row in enumerate(proto_probs_list):
            if not isinstance(row, list):
                continue
            gl = gates_list[b] if (gates_list and b < len(gates_list)) else None
            for j, probs in enumerate(row):
                if not isinstance(probs, torch.Tensor) or probs.numel() == 0:
                    continue
                if gl and j < len(gl):
                    try:
                        if float(gl[j]) < min_gate:
                            continue
                    except Exception:
                        pass
                try:
                    p = torch.clamp(probs.to(dev).float(), 1e-8, 1.0)
                    H = -torch.sum(p * torch.log(p))
                    if torch.isfinite(H):
                        total = total + H
                        count += 1
                except Exception:
                    continue

        if count == 0:
            return torch.tensor(0.0, device=dev)
        return total / count

    # -------------------------------------------------------------------------
    # Word embedding extraction
    # -------------------------------------------------------------------------
    def _extract_word_embeddings(
        self,
        src_texts: List[str],
        device: torch.device,
        embed_dim: int,
    ) -> Tuple[torch.Tensor, List[Dict[int, str]], List[List[str]]]:
        batch_size          = len(src_texts)
        word_embeddings_batch = []
        token_word_map_batch  = []
        words_batch           = []
        max_words             = 0

        try:
            embedding_layer = self.mbart.get_input_embeddings()
        except Exception:
            fallback_embs  = torch.zeros(batch_size, 1, embed_dim, device=device)
            fallback_maps  = [{0: "UNK"} for _ in range(batch_size)]
            fallback_words = [["UNK"] for _ in range(batch_size)]
            return fallback_embs, fallback_maps, fallback_words

        for batch_idx, text in enumerate(src_texts):
            if not text or not isinstance(text, str):
                text = "UNK"
            text = text.strip()
            if not text:
                text = "UNK"

            words = _extract_words_from_text(text)
            if not words or len(words) == 0:
                words = ["UNK"]

            words_batch.append(words)
            word_embeddings = []
            word_map        = {}

            for idx, word in enumerate(words):
                try:
                    if not word or len(word) == 0:
                        word = "UNK"
                    word_ids = self.tokenizer.encode(word, add_special_tokens=False)
                    if not word_ids or len(word_ids) == 0:
                        word_ids = [3]
                    word_ids = [wid for wid in word_ids if 0 <= wid < self.vocab_size]
                    if not word_ids:
                        word_ids = [3]
                    word_ids_tensor = torch.tensor(
                        [word_ids], dtype=torch.long, device=device
                    )
                    subword_embs = embedding_layer(word_ids_tensor)
                    word_emb     = subword_embs.mean(dim=1).squeeze(0)
                    if torch.isnan(word_emb).any() or torch.isinf(word_emb).any():
                        word_emb = torch.zeros(embed_dim, device=device)
                    word_embeddings.append(word_emb)
                    word_map[idx] = word
                except Exception:
                    fallback_emb = torch.zeros(embed_dim, device=device)
                    word_embeddings.append(fallback_emb)
                    word_map[idx] = word if word else "UNK"

            if word_embeddings and len(word_embeddings) > 0:
                try:
                    word_embs_tensor = torch.stack(word_embeddings, dim=0)
                    word_embeddings_batch.append(word_embs_tensor)
                    token_word_map_batch.append(word_map)
                    max_words = max(max_words, len(word_embeddings))
                except Exception:
                    fallback_emb = torch.zeros(1, embed_dim, device=device)
                    word_embeddings_batch.append(fallback_emb)
                    token_word_map_batch.append({0: "UNK"})
                    max_words = max(max_words, 1)
            else:
                fallback_emb = torch.zeros(1, embed_dim, device=device)
                word_embeddings_batch.append(fallback_emb)
                token_word_map_batch.append({0: "UNK"})
                max_words = max(max_words, 1)

        if max_words == 0:
            max_words = 1

        try:
            padded_word_embs = torch.zeros(
                batch_size, max_words, embed_dim, device=device
            )
            for i, word_embs in enumerate(word_embeddings_batch):
                try:
                    length = word_embs.size(0)
                    if length > max_words:
                        length = max_words
                    padded_word_embs[i, :length] = word_embs[:length]
                except Exception:
                    pass
        except Exception:
            padded_word_embs = torch.zeros(batch_size, 1, embed_dim, device=device)

        return padded_word_embs, token_word_map_batch, words_batch

    # -------------------------------------------------------------------------
    # Word map reconstruction
    # -------------------------------------------------------------------------
    def _reconstruct_word_maps_before_dscd(
        self,
        input_ids: torch.Tensor,
        batch_size: int,
        seq_len: int,
        src_texts: Optional[List[str]] = None,
        token_word_map: Optional[List[dict]] = None,
    ) -> List[dict]:
        if token_word_map is not None and len(token_word_map) == batch_size:
            valid_count = sum(
                1 for m in token_word_map if isinstance(m, dict) and len(m) > 0
            )
            if valid_count == batch_size:
                if _DEBUG_DISCOVERY:
                    total_words = sum(len(m) for m in token_word_map)
                    print(
                        f"[TATN-WORDMAP] Using provided word maps: {total_words} words"
                    )
                return token_word_map

        word_maps_batch: List[dict] = []

        for b in range(batch_size):
            try:
                ids_b  = input_ids[b].detach().cpu().tolist()
                tokens = self.tokenizer.convert_ids_to_tokens(ids_b)
                wm     = build_token_word_map_sentencepiece(tokens, fallback=True)
                if wm:
                    word_maps_batch.append(wm)
                else:
                    word_maps_batch.append(
                        {i: f"tok{i}" for i in range(min(5, seq_len))}
                    )
            except Exception:
                word_maps_batch.append(
                    {i: f"tok{i}" for i in range(min(5, seq_len))}
                )

        total_words = sum(len(m) for m in word_maps_batch)
        if _DEBUG_DISCOVERY:
            print(f"[TATN-WORDMAP] Reconstructed {total_words} words")

        return word_maps_batch

    # -------------------------------------------------------------------------
    # Domain label extraction
    # -------------------------------------------------------------------------
    def _extract_domain_labels(
        self,
        batch_size: int,
        device: torch.device,
        src_texts: Optional[List[str]] = None,
    ) -> Optional[torch.Tensor]:
        if not _USE_DOMAIN_LABELS:
            return None
        try:
            if self.training:
                return torch.full(
                    (batch_size,),
                    _TRAIN_DOMAIN,
                    dtype=torch.long,
                    device=device,
                )
            else:
                return torch.full(
                    (batch_size,),
                    _TEST_DOMAIN,
                    dtype=torch.long,
                    device=device,
                )
        except Exception:
            return None

    # -------------------------------------------------------------------------
    # Safe DSCD key extractor
    # -------------------------------------------------------------------------
    @staticmethod
    def _safe_take_key_static(
        dscd_struct: Dict[str, Any],
        key: str,
        b_index: int,
        seq_len: int,
        device: torch.device,
    ):
        if key == "proto_probs":
            out = [
                torch.tensor([1.0], dtype=torch.float32, device=device)
                for _ in range(seq_len)
            ]
        else:
            out = [
                torch.tensor(0.0, dtype=torch.float32, device=device)
                for _ in range(seq_len)
            ]

        try:
            val = dscd_struct.get(key, None)
            if val is None:
                return out

            if key == "proto_probs":
                if isinstance(val, list) and len(val) > b_index:
                    row = val[b_index]
                    if isinstance(row, list):
                        for t in range(min(seq_len, len(row))):
                            v = row[t]
                            if isinstance(v, torch.Tensor):
                                out[t] = v.detach().to(device)
                            else:
                                try:
                                    out[t] = torch.as_tensor(
                                        v,
                                        dtype=torch.float32,
                                        device=device,
                                    ).flatten()
                                except Exception:
                                    pass
                return out

            if isinstance(val, list) and len(val) > b_index:
                row = val[b_index]
                if isinstance(row, list):
                    for t in range(min(seq_len, len(row))):
                        v = row[t]
                        try:
                            if isinstance(v, torch.Tensor):
                                out[t] = v.detach().to(device)
                            else:
                                out[t] = torch.tensor(float(v), device=device)
                        except Exception:
                            pass
                elif isinstance(row, torch.Tensor):
                    if row.dim() == 1:
                        for t in range(min(seq_len, int(row.size(0)))):
                            try:
                                out[t] = torch.tensor(
                                    float(row[t].item()), device=device
                                )
                            except Exception:
                                pass
                return out

            if isinstance(val, torch.Tensor):
                if val.dim() >= 2 and int(val.size(0)) > b_index:
                    for t in range(min(seq_len, int(val.size(1)))):
                        try:
                            if val.dim() == 3:
                                v = val[b_index, t]
                                if v.numel() == 1:
                                    out[t] = torch.tensor(
                                        float(v.item()), device=device
                                    )
                                else:
                                    out[t] = v.detach().to(device)
                            else:
                                v = val[b_index, t]
                                out[t] = torch.tensor(
                                    float(v.item()), device=device
                                )
                        except Exception:
                            pass
        except Exception:
            pass

        return out

    # -------------------------------------------------------------------------
    # forward_path1
    # -------------------------------------------------------------------------
    def forward_path1(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        src_texts: Optional[List[str]] = None,
        token_word_map: Optional[List[dict]] = None,
        domain_labels: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        with self._step_lock:
            self.global_step += 1
            current_step = self.global_step

        if input_ids is None or attention_mask is None:
            raise ValueError("input_ids and attention_mask cannot be None")

        batch_size, seq_len = int(input_ids.size(0)), int(input_ids.size(1))
        device    = input_ids.device
        embed_dim = int(getattr(self.mbart.config, "d_model", 1024))

        if src_texts is None or not isinstance(src_texts, list) or len(src_texts) != batch_size:
            src_texts_extracted = []
            for b in range(batch_size):
                try:
                    ids_b = input_ids[b].detach().cpu().tolist()
                    text  = self.tokenizer.decode(ids_b, skip_special_tokens=True)
                    if not text or not text.strip():
                        text = "UNK"
                    src_texts_extracted.append(text.strip())
                except Exception:
                    src_texts_extracted.append("UNK")
            src_texts = src_texts_extracted

        for i in range(len(src_texts)):
            if not src_texts[i] or not isinstance(src_texts[i], str) or not src_texts[i].strip():
                src_texts[i] = "UNK"

        try:
            h, token_word_map, words_batch = self._extract_word_embeddings(
                src_texts, device, embed_dim
            )
        except Exception:
            h              = torch.zeros(batch_size, 1, embed_dim, device=device)
            token_word_map = [{0: "UNK"} for _ in range(batch_size)]
            words_batch    = [["UNK"] for _ in range(batch_size)]

        max_words = h.size(1)

        try:
            h_detached = h.detach()
            raw_dscd   = self.dscd.forward(
                h_detached,
                token_types=None,
                train_mode=self.training,
                input_ids=None,
                attention_mask=None,
                token_word_map=token_word_map,
            )
        except Exception:
            raw_dscd = {
                "h_augmented": h.detach().clone(),
                "proto_probs": [
                    [
                        torch.tensor([1.0], dtype=torch.float32, device=device)
                        for _ in range(max_words)
                    ]
                    for _ in range(batch_size)
                ],
                "uncertainties": [
                    [torch.tensor(0.0, device=device) for _ in range(max_words)]
                    for _ in range(batch_size)
                ],
                "gates": [
                    [torch.tensor(0.0, device=device) for _ in range(max_words)]
                    for _ in range(batch_size)
                ],
            }

        try:
            dscd = _normalize_dscd_outputs(
                raw_dscd, batch_size, max_words, device, embed_dim, fallback_h=h
            )
        except Exception:
            dscd = {
                "h_augmented": h.detach().clone(),
                "proto_probs": [
                    [torch.tensor([1.0], device=device) for _ in range(max_words)]
                    for _ in range(batch_size)
                ],
                "uncertainties": [
                    [torch.tensor(0.0, device=device) for _ in range(max_words)]
                    for _ in range(batch_size)
                ],
                "gates": [
                    [torch.tensor(0.0, device=device) for _ in range(max_words)]
                    for _ in range(batch_size)
                ],
                "proto_assignments": [
                    torch.zeros(max_words, dtype=torch.long, device=device)
                    for _ in range(batch_size)
                ],
            }

        h_aug = dscd.get("h_augmented", h)

        self._last_path1_h_aug = h_aug.detach().clone()

        if domain_labels is None:
            domain_labels = self._extract_domain_labels(
                batch_size=batch_size, device=device, src_texts=src_texts
            )

        asbn_loss = torch.zeros(1, device=device, requires_grad=True)
        if self.training and _ENABLE_ASBN_TRAINING and domain_labels is not None:
            try:
                h_asbn, asbn_losses = self.asbn.forward(
                    h_aug,
                    proto_probs=dscd.get("proto_probs", None),
                    uncertainties=dscd.get("uncertainties", None),
                    gates=dscd.get("gates", None),
                    token_word_map=token_word_map,
                    domain_labels=domain_labels,
                    global_step=current_step,
                )
                if isinstance(asbn_losses, dict):
                    encoder_loss = asbn_losses.get(
                        "encoder_loss",
                        torch.zeros(1, device=device, requires_grad=True),
                    )
                    if isinstance(encoder_loss, torch.Tensor) and torch.isfinite(encoder_loss):
                        if encoder_loss.requires_grad:
                            asbn_loss = encoder_loss
                        else:
                            asbn_loss = torch.tensor(
                                float(encoder_loss.item()),
                                device=device,
                                requires_grad=True,
                            )
            except Exception:
                asbn_loss = torch.zeros(1, device=device, requires_grad=True)

        dscd_reg = torch.zeros(1, device=device, requires_grad=True)
        try:
            dscd_reg_raw = self._entropy_reg_from_proto_probs_static(
                dscd.get("proto_probs", []),
                gates_list=dscd.get("gates", []),
                min_gate=0.01,
            )
            if isinstance(dscd_reg_raw, torch.Tensor):
                if torch.isfinite(dscd_reg_raw):
                    if dscd_reg_raw.requires_grad:
                        dscd_reg = torch.clamp(dscd_reg_raw.to(device), 0.0, 5.0)
                    else:
                        dscd_reg = torch.tensor(
                            float(dscd_reg_raw.item()),
                            device=device,
                            requires_grad=True,
                        )
                        dscd_reg = torch.clamp(dscd_reg, 0.0, 5.0)
        except Exception:
            dscd_reg = torch.zeros(1, device=device, requires_grad=True)

        total_loss = _LAMBDA_ASBN * asbn_loss + _LAMBDA_DSCD * dscd_reg

        if not isinstance(total_loss, torch.Tensor):
            total_loss = torch.tensor(float(total_loss), device=device, requires_grad=True)

        if not torch.isfinite(total_loss):
            total_loss = torch.tensor(0.01, device=device, requires_grad=True)

        if not total_loss.requires_grad:
            total_loss = torch.tensor(
                float(total_loss.item()), device=device, requires_grad=True
            )

        self._last_path1_asbn_loss = (
            float(asbn_loss.item()) if isinstance(asbn_loss, torch.Tensor) else 0.0
        )
        self._last_path1_dscd_loss = (
            float(dscd_reg.item()) if isinstance(dscd_reg, torch.Tensor) else 0.0
        )

        return {"loss": total_loss, "h_aug": h_aug}

    # -------------------------------------------------------------------------
    # forward_path2
    # -------------------------------------------------------------------------
    def forward_path2(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: torch.Tensor,
        src_texts: Optional[List[str]] = None,
        token_word_map: Optional[List[dict]] = None,
        use_rdrop: bool = True,
        path1_h_aug: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        with self._step_lock:
            current_step = self.global_step

        if input_ids is None or attention_mask is None or labels is None:
            raise ValueError("input_ids, attention_mask, and labels cannot be None")

        batch_size, seq_len = int(input_ids.size(0)), int(input_ids.size(1))
        device    = input_ids.device
        embed_dim = int(getattr(self.mbart.config, "d_model", 1024))

        if torch.any(input_ids >= self.vocab_size) or torch.any(input_ids < 0):
            input_ids = torch.clamp(input_ids, 0, self.vocab_size - 1)

        pad_id             = getattr(self.tokenizer, "pad_token_id", 1)
        decoder_input_ids  = labels.clone()
        decoder_input_ids  = torch.where(
            decoder_input_ids == -100,
            torch.full_like(decoder_input_ids, pad_id),
            decoder_input_ids,
        )
        bos_column = torch.full(
            (batch_size, 1),
            int(self.mbart.config.decoder_start_token_id),
            dtype=torch.long,
            device=device,
        )
        decoder_input_ids       = torch.cat([bos_column, decoder_input_ids[:, :-1]], dim=1)
        decoder_attention_mask  = (decoder_input_ids != pad_id).long()

        enc_outputs_p2 = None
        h_subword      = None
        try:
            enc_outputs_p2 = self.mbart.model.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            h_subword = _safe_get_last_hidden_state(enc_outputs_p2)
        except Exception:
            h_subword = None

        dscd_quality   = 0.0
        homograph_rate = 0.0
        try:
            if hasattr(self.dscd, "get_prototype_summary"):
                summary        = self.dscd.get_prototype_summary()
                total_protos   = int(summary.get("total_prototypes", 0))
                num_homos      = int(summary.get("num_homographs", 0))
                total_tokens   = int(summary.get("total_tokens", 1))
                dscd_quality   = float(min(total_protos, total_tokens)) / max(total_tokens, 1)
                homograph_rate = float(num_homos) / max(total_tokens, 1)
        except Exception:
            dscd_quality   = 0.0
            homograph_rate = 0.0

        enc_for_decoder_p2 = None

        if path1_h_aug is not None and h_subword is not None:
            try:
                raw_dscd_p2 = self.dscd.forward(
                    h_subword.detach(),
                    token_types=None,
                    train_mode=self.training,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_word_map=None,
                )
                dscd_p2 = _normalize_dscd_outputs(
                    raw_dscd_p2, batch_size, seq_len, device, embed_dim,
                    fallback_h=h_subword,
                )
                sense_assign_p2, amb_scores_p2 = self.get_sense_assignments_from_dscd(
                    dscd_p2, h_subword
                )
                sense_adapted_enc_p2 = self.scea(h_subword, sense_assign_p2, amb_scores_p2)
            except Exception:
                sense_adapted_enc_p2 = h_subword

            try:
                if (
                    path1_h_aug.dim() == 3
                    and path1_h_aug.size(0) == batch_size
                    and path1_h_aug.size(2) == embed_dim
                    and h_subword.dim() == 3
                    and h_subword.size(0) == batch_size
                    and h_subword.size(2) == embed_dim
                ):
                    h_aug_projected = _project_word_to_subword(
                        path1_h_aug.to(device),
                        target_seq_len=seq_len,
                        device=device,
                    )
                    h_fused = 0.7 * sense_adapted_enc_p2 + 0.3 * h_aug_projected
                    enc_for_decoder_p2 = BaseModelOutput(
                        last_hidden_state=h_fused,
                        hidden_states=getattr(enc_outputs_p2, "hidden_states", None),
                        attentions=getattr(enc_outputs_p2, "attentions", None),
                    )
                    if _DEBUG_DISCOVERY:
                        print(
                            f"[PATH2] SCEA+h_aug injected (gate removed): "
                            f"dscd_quality={dscd_quality:.3f}, "
                            f"homograph_rate={homograph_rate:.3f}, "
                            f"fused={h_fused.shape}"
                        )
                else:
                    if _DEBUG_DISCOVERY:
                        print(
                            f"[PATH2] Path1 h_aug shape mismatch, using SCEA-only encoder. "
                            f"path1_h_aug={getattr(path1_h_aug, 'shape', None)}, h_subword={getattr(h_subword, 'shape', None)}"
                        )
                    enc_for_decoder_p2 = BaseModelOutput(
                        last_hidden_state=sense_adapted_enc_p2,
                        hidden_states=getattr(enc_outputs_p2, "hidden_states", None),
                        attentions=getattr(enc_outputs_p2, "attentions", None),
                    )
            except Exception as e:
                if _DEBUG_DISCOVERY:
                    print(f"[PATH2] SCEA+fusion failed: {e}, falling back to h_subword")
                enc_for_decoder_p2 = None

        elif path1_h_aug is None and h_subword is not None:
            try:
                raw_dscd_p2 = self.dscd.forward(
                    h_subword.detach(),
                    token_types=None,
                    train_mode=self.training,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_word_map=None,
                )
                dscd_p2 = _normalize_dscd_outputs(
                    raw_dscd_p2, batch_size, seq_len, device, embed_dim,
                    fallback_h=h_subword,
                )
                sense_assign_p2, amb_scores_p2 = self.get_sense_assignments_from_dscd(
                    dscd_p2, h_subword
                )
                sense_adapted_enc_p2 = self.scea(h_subword, sense_assign_p2, amb_scores_p2)
                enc_for_decoder_p2   = BaseModelOutput(
                    last_hidden_state=sense_adapted_enc_p2,
                    hidden_states=getattr(enc_outputs_p2, "hidden_states", None) if enc_outputs_p2 is not None else None,
                    attentions=getattr(enc_outputs_p2, "attentions", None) if enc_outputs_p2 is not None else None,
                )
                if _DEBUG_DISCOVERY:
                    print(
                        f"[PATH2] path1_h_aug=None (gate removed behavior): SCEA-only encoder applied. "
                        f"h_subword={h_subword.shape}"
                    )
            except Exception as e:
                if _DEBUG_DISCOVERY:
                    print(f"[PATH2] SCEA (gate-closed path) failed: {e}, enc_for_decoder_p2=None")
                enc_for_decoder_p2 = None

        if enc_for_decoder_p2 is None:
            if h_subword is not None:
                enc_for_decoder_p2 = BaseModelOutput(
                    last_hidden_state=h_subword,
                    hidden_states=(
                        getattr(enc_outputs_p2, "hidden_states", None)
                        if enc_outputs_p2 is not None
                        else None
                    ),
                    attentions=(
                        getattr(enc_outputs_p2, "attentions", None)
                        if enc_outputs_p2 is not None
                        else None
                    ),
                )

        if enc_for_decoder_p2 is not None:
            seq_outputs = self.mbart(
                input_ids=None,
                attention_mask=attention_mask,
                encoder_outputs=enc_for_decoder_p2,
                decoder_input_ids=decoder_input_ids,
                decoder_attention_mask=decoder_attention_mask,
                labels=labels,
                use_cache=False,
                return_dict=True,
            )
        else:
            seq_outputs = self.mbart(
                input_ids=input_ids,
                attention_mask=attention_mask,
                decoder_input_ids=decoder_input_ids,
                decoder_attention_mask=decoder_attention_mask,
                labels=labels,
                use_cache=False,
                return_dict=True,
            )

        translation_loss = getattr(seq_outputs, "loss", None)
        if translation_loss is None or not torch.isfinite(translation_loss):
            translation_loss = torch.tensor(10.0, device=device)
        else:
            translation_loss = torch.clamp(translation_loss, 0.0, 100.0)

        label_smoothing_loss = torch.tensor(0.0, device=device)
        if self.label_smoothing_loss is not None:
            try:
                logits               = seq_outputs.logits
                label_smoothing_loss = self.label_smoothing_loss(logits, labels)
                if torch.isfinite(label_smoothing_loss):
                    translation_loss = label_smoothing_loss
            except Exception:
                pass

        rdrop_loss = torch.tensor(0.0, device=device)
        if use_rdrop and self.rdrop_loss is not None and self.training:
            try:
                if enc_for_decoder_p2 is not None:
                    seq_outputs2 = self.mbart(
                        input_ids=None,
                        attention_mask=attention_mask,
                        encoder_outputs=enc_for_decoder_p2,
                        decoder_input_ids=decoder_input_ids,
                        decoder_attention_mask=decoder_attention_mask,
                        labels=labels,
                        use_cache=False,
                        return_dict=True,
                    )
                else:
                    seq_outputs2 = self.mbart(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        decoder_input_ids=decoder_input_ids,
                        decoder_attention_mask=decoder_attention_mask,
                        labels=labels,
                        use_cache=False,
                        return_dict=True,
                    )
                logits1    = seq_outputs.logits
                logits2    = seq_outputs2.logits
                rdrop_loss = self.rdrop_loss(logits1, logits2, labels)
                if not torch.isfinite(rdrop_loss):
                    rdrop_loss = torch.tensor(0.0, device=device)
            except Exception:
                rdrop_loss = torch.tensor(0.0, device=device)

        total_loss = translation_loss + rdrop_loss

        if not torch.isfinite(total_loss):
            total_loss = translation_loss

        return {"loss": total_loss}

    # -------------------------------------------------------------------------
    # forward  (main router)
    # -------------------------------------------------------------------------
    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        src_texts: Optional[List[str]] = None,
        token_word_map: Optional[List[dict]] = None,
        labels: Optional[torch.Tensor] = None,
        use_dscd: bool = True,
        use_asbn: bool = False,
        return_dict: bool = True,
        domain_labels: Optional[torch.Tensor] = None,
        path: Optional[int] = None,
        use_rdrop: bool = False,
        **kwargs,
    ):
        if path == 1:
            return self.forward_path1(
                input_ids=input_ids,
                attention_mask=attention_mask,
                src_texts=src_texts,
                token_word_map=token_word_map,
                domain_labels=domain_labels,
            )

        elif path == 2:
            if labels is not None:
                cached_h_aug = self._last_path1_h_aug
                return self.forward_path2(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                    src_texts=src_texts,
                    token_word_map=token_word_map,
                    use_rdrop=use_rdrop,
                    path1_h_aug=cached_h_aug,
                )

        with self._step_lock:
            self.global_step += 1
            current_step = self.global_step

        if input_ids is None or attention_mask is None:
            raise ValueError("input_ids and attention_mask cannot be None")
        if input_ids.dim() != 2 or attention_mask.dim() != 2:
            raise ValueError(
                f"Expected 2D tensors, got {input_ids.shape}, {attention_mask.shape}"
            )

        batch_size, seq_len = int(input_ids.size(0)), int(input_ids.size(1))
        device = input_ids.device

        if torch.any(input_ids >= self.vocab_size) or torch.any(input_ids < 0):
            invalid_count = torch.sum(
                (input_ids >= self.vocab_size) | (input_ids < 0)
            ).item()
            print(f"[TATN] CRITICAL: {invalid_count} input_ids out of vocab bounds!")
            print(f"  Vocab size: {self.vocab_size}")
            print(f"  Max input_id: {torch.max(input_ids).item()}")
            print(f"  Min input_id: {torch.min(input_ids).item()}")
            input_ids = torch.clamp(input_ids, 0, self.vocab_size - 1)

        if (
            torch.cuda.is_available()
            and _MEMORY_CLEANUP_FREQUENCY > 0
            and current_step % _MEMORY_CLEANUP_FREQUENCY == 0
        ):
            for i in range(min(_NUM_GPUS, torch.cuda.device_count())):
                try:
                    with torch.cuda.device(i):
                        torch.cuda.empty_cache()
                except Exception:
                    pass
            if gc.isenabled():
                gc.collect()

        if self.training and _DSCD_ENABLE_TRAINING_CLUSTERING and use_dscd:
            if (
                current_step - self.last_discovery_step
                >= _PERIODIC_DISCOVERY_FREQUENCY
            ):
                try:
                    if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                        print("\n" + "=" * 80)
                        print(f"[TATN] PERIODIC DISCOVERY @ step {current_step}")
                        print("=" * 80)
                    start_time = time.time()
                    self.dscd.periodic_discovery_check(
                        current_step, _PERIODIC_DISCOVERY_FREQUENCY
                    )
                    elapsed = time.time() - start_time
                    self.last_discovery_step = current_step

                    if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                        summary = self.dscd.get_prototype_summary()
                        print(f"[TATN] Discovery completed in {elapsed:.2f}s")
                        print(
                            f"[TATN]   Homographs: {summary.get('num_homographs', 0)}"
                        )
                        print(
                            f"[TATN]   Total prototypes: {summary.get('total_prototypes', 0)}"
                        )
                        print("=" * 80 + "\n")

                except Exception as e:
                    if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                        print(f"[TATN] Discovery failed: {e}")
                        try:
                            traceback.print_exc()
                        except Exception:
                            pass

        if not self.training and _VALIDATION_CHECK_INTERVAL > 0:
            if (
                current_step - self.last_validation_step
                >= _VALIDATION_CHECK_INTERVAL
            ):
                try:
                    if _DEBUG_DISCOVERY:
                        print(f"\n[TATN-VALIDATION] Step {current_step}")
                        summary = self.dscd.get_prototype_summary()
                        print(f"  - Tokens: {summary.get('total_tokens', 0)}")
                        print(
                            f"  - Prototypes: {summary.get('total_prototypes', 0)}"
                        )
                        print(
                            f"  - Homographs: {summary.get('num_homographs', 0)}"
                        )
                    self.last_validation_step = current_step
                except Exception:
                    pass

        enc_outputs = None
        try:
            enc_outputs = self.mbart.model.encoder(
                input_ids=input_ids, attention_mask=attention_mask
            )
        except Exception:
            try:
                enc_outputs = self.mbart.get_encoder()(
                    input_ids=input_ids, attention_mask=attention_mask
                )
            except Exception as e:
                if _DEBUG_DISCOVERY:
                    print(f"[TATN] Encoder failed: {e}")
                enc_outputs = None

        h = _safe_get_last_hidden_state(enc_outputs)
        if h is None:
            try:
                embedding_layer = self.mbart.get_input_embeddings()
                if embedding_layer is None:
                    raise RuntimeError("No embedding layer available")
                h = embedding_layer(input_ids).to(device)
            except Exception as e:
                if _DEBUG_DISCOVERY:
                    print(f"[TATN] Embedding fallback failed: {e}")
                h = torch.zeros(
                    batch_size,
                    seq_len,
                    int(getattr(self.mbart.config, "d_model", 1024)),
                    device=device,
                )

        embed_dim     = int(h.size(-1))
        training_mode = labels is not None

        token_word_map = self._reconstruct_word_maps_before_dscd(
            input_ids, batch_size, seq_len, src_texts, token_word_map
        )

        if domain_labels is None:
            domain_labels = self._extract_domain_labels(
                batch_size=batch_size, device=device, src_texts=src_texts
            )

        if use_dscd:
            try:
                h_detached = h.detach()
                raw_dscd   = self.dscd.forward(
                    h_detached,
                    token_types=None,
                    train_mode=self.training,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_word_map=token_word_map,
                )
            except Exception as e:
                if _DEBUG_DISCOVERY:
                    print(f"[TATN] DSCD forward failed: {e}")
                raw_dscd = {
                    "h_augmented": h.detach().clone(),
                    "proto_probs": [
                        [
                            torch.tensor(
                                [1.0], dtype=torch.float32, device=device
                            )
                            for _ in range(seq_len)
                        ]
                        for _ in range(batch_size)
                    ],
                    "uncertainties": [
                        [
                            torch.tensor(0.0, device=device)
                            for _ in range(seq_len)
                        ]
                        for _ in range(batch_size)
                    ],
                    "gates": [
                        [
                            torch.tensor(0.0, device=device)
                            for _ in range(seq_len)
                        ]
                        for _ in range(batch_size)
                    ],
                    "span_preds": [
                        [
                            torch.tensor(0.0, device=device)
                            for _ in range(seq_len)
                        ]
                        for _ in range(batch_size)
                    ],
                    "proto_assignments": [
                        torch.zeros(seq_len, dtype=torch.long, device=device)
                        for _ in range(batch_size)
                    ],
                }
        else:
            raw_dscd = {
                "h_augmented": h.detach().clone(),
                "proto_probs": [
                    [
                        torch.tensor([1.0], dtype=torch.float32, device=device)
                        for _ in range(seq_len)
                    ]
                    for _ in range(batch_size)
                ],
                "uncertainties": [
                    [
                        torch.tensor(0.0, device=device)
                        for _ in range(seq_len)
                    ]
                    for _ in range(batch_size)
                ],
                "gates": [
                    [
                        torch.tensor(0.0, device=device)
                        for _ in range(seq_len)
                    ]
                    for _ in range(batch_size)
                ],
                "span_preds": [
                    [
                        torch.tensor(0.0, device=device)
                        for _ in range(seq_len)
                    ]
                    for _ in range(batch_size)
                ],
                "proto_assignments": [
                    torch.zeros(seq_len, dtype=torch.long, device=device)
                    for _ in range(batch_size)
                ],
            }

        dscd  = _normalize_dscd_outputs(
            raw_dscd, batch_size, seq_len, device, embed_dim, fallback_h=h
        )
        h_aug = dscd.get("h_augmented", h)

        if not isinstance(h_aug, torch.Tensor) or h_aug.shape != h.shape:
            if _DEBUG_DISCOVERY:
                print(
                    f"[TATN] h_augmented shape mismatch "
                    f"(expected {h.shape}, got {getattr(h_aug, 'shape', None)})"
                )
            h_aug = h

        sense_assignments, ambiguity_scores = self.get_sense_assignments_from_dscd(
            dscd, h_aug
        )

        try:
            sense_adapted_enc = self.scea(h_aug, sense_assignments, ambiguity_scores)
        except Exception:
            sense_adapted_enc = h_aug

        asbn_loss      = torch.tensor(0.0, device=device)
        domain_loss    = torch.tensor(0.0, device=device)
        domain_accuracy = torch.tensor(0.0, device=device)

        if training_mode and use_asbn and _ENABLE_ASBN_TRAINING and domain_labels is not None:
            try:
                if _DEBUG_DISCOVERY and current_step % 100 == 0:
                    print(f"\n[TATN-ASBN] Applying ASBN at step {current_step}")
                    print(f"  Domain labels: {domain_labels.tolist()}")

                h_asbn, asbn_losses = self.asbn.forward(
                    sense_adapted_enc,
                    proto_probs=dscd.get("proto_probs", None),
                    uncertainties=dscd.get("uncertainties", None),
                    gates=dscd.get("gates", None),
                    token_word_map=token_word_map,
                    domain_labels=domain_labels,
                    global_step=current_step,
                )

                if isinstance(asbn_losses, dict):
                    encoder_loss    = asbn_losses.get(
                        "encoder_loss", torch.tensor(0.0, device=device)
                    )
                    domain_loss     = asbn_losses.get(
                        "domain_loss", torch.tensor(0.0, device=device)
                    )
                    domain_accuracy = asbn_losses.get(
                        "domain_accuracy", torch.tensor(0.0, device=device)
                    )
                    if isinstance(encoder_loss, torch.Tensor) and torch.isfinite(encoder_loss):
                        asbn_loss = encoder_loss
                    else:
                        asbn_loss = torch.tensor(0.0, device=device)
                else:
                    asbn_loss = torch.tensor(0.0, device=device)

                if _DEBUG_DISCOVERY and current_step % 100 == 0:
                    print(f"  ASBN loss: {asbn_loss.item():.4f}")
                    print(f"  Domain loss: {domain_loss.item():.4f}")
                    print(f"  Domain accuracy: {domain_accuracy.item():.2%}")

            except Exception as e:
                if _DEBUG_DISCOVERY:
                    print(f"[TATN-ASBN] ASBN forward failed: {e}")
                    try:
                        traceback.print_exc()
                    except Exception:
                        pass
                asbn_loss       = torch.tensor(0.0, device=device)
                domain_loss     = torch.tensor(0.0, device=device)
                domain_accuracy = torch.tensor(0.0, device=device)

        try:
            enc_for_decoder = BaseModelOutput(
                last_hidden_state=sense_adapted_enc,
                hidden_states=(
                    getattr(enc_outputs, "hidden_states", None)
                    if enc_outputs
                    else None
                ),
                attentions=(
                    getattr(enc_outputs, "attentions", None)
                    if enc_outputs
                    else None
                ),
            )
        except Exception:
            enc_for_decoder = (sense_adapted_enc,)

        if training_mode:
            try:
                pad_id = getattr(self.tokenizer, "pad_token_id", 1)

                if labels is not None:
                    decoder_input_ids = labels.clone()
                    decoder_input_ids = torch.where(
                        decoder_input_ids == -100,
                        torch.full_like(decoder_input_ids, pad_id),
                        decoder_input_ids,
                    )
                    bos_column = torch.full(
                        (batch_size, 1),
                        int(self.mbart.config.decoder_start_token_id),
                        dtype=torch.long,
                        device=device,
                    )
                    decoder_input_ids = torch.cat(
                        [bos_column, decoder_input_ids[:, :-1]], dim=1
                    )
                    decoder_attention_mask = (decoder_input_ids != pad_id).long()
                else:
                    decoder_input_ids = None
                    decoder_attention_mask = None

                seq_outputs = self.mbart(
                    input_ids=None,
                    attention_mask=attention_mask,
                    encoder_outputs=enc_for_decoder,
                    decoder_input_ids=decoder_input_ids,
                    decoder_attention_mask=decoder_attention_mask,
                    labels=labels,
                    use_cache=False,
                    output_hidden_states=True,
                    return_dict=True,
                )

                dec_hidden = None
                try:
                    if (
                        hasattr(seq_outputs, "decoder_hidden_states")
                        and seq_outputs.decoder_hidden_states is not None
                    ):
                        dec_hidden = seq_outputs.decoder_hidden_states[-1]
                except Exception:
                    dec_hidden = None

                if dec_hidden is not None:
                    try:
                        refined_dec    = self.hfcaa(
                            dec_hidden, sense_adapted_enc, ambiguity_scores
                        )
                        refined_logits = self.mbart.lm_head(refined_dec)

                        if labels is not None:
                            shift_logits   = refined_logits[..., :-1, :].contiguous()
                            shift_labels   = labels[..., 1:].contiguous()
                            translation_loss = F.cross_entropy(
                                shift_logits.view(-1, shift_logits.size(-1)),
                                shift_labels.view(-1),
                                ignore_index=-100,
                            )
                            if not torch.isfinite(translation_loss):
                                translation_loss = getattr(seq_outputs, "loss", None)
                        else:
                            translation_loss = getattr(seq_outputs, "loss", None)
                    except Exception:
                        translation_loss = getattr(seq_outputs, "loss", None)
                        refined_logits   = getattr(seq_outputs, "logits", None)
                else:
                    translation_loss = getattr(seq_outputs, "loss", None)
                    refined_logits   = getattr(seq_outputs, "logits", None)

                if translation_loss is None or not torch.isfinite(translation_loss):
                    translation_loss = torch.tensor(10.0, device=device)
                else:
                    translation_loss = torch.clamp(translation_loss, 0.0, 100.0)

                token_penalty      = torch.tensor(0.0, device=device)
                confidence_penalty = torch.tensor(0.0, device=device)
                length_penalty     = torch.tensor(0.0, device=device)

                try:
                    logits        = refined_logits if refined_logits is not None \
                                    else seq_outputs.logits
                    predicted_ids = torch.argmax(logits, dim=-1)

                    reference_ids = labels
                    valid_mask    = (reference_ids != -100) & (reference_ids != pad_id)

                    if valid_mask.sum() > 0:
                        mismatches    = (predicted_ids != reference_ids) & valid_mask
                        mismatch_rate = mismatches.float().sum() / valid_mask.float().sum()
                        token_penalty = mismatch_rate * 2.0

                        probs     = torch.softmax(logits, dim=-1)
                        max_probs = torch.max(probs, dim=-1)[0]
                        wrong_positions = (predicted_ids != reference_ids) & valid_mask

                        if wrong_positions.sum() > 0:
                            wrong_confidences  = max_probs[wrong_positions]
                            confidence_penalty = wrong_confidences.mean() * 1.5

                        pred_lengths = valid_mask.sum(dim=1).float()
                        ref_lengths  = (reference_ids != -100).sum(dim=1).float()
                        length_diff  = torch.abs(pred_lengths - ref_lengths) / torch.clamp(
                            ref_lengths, min=1.0
                        )
                        length_penalty = length_diff.mean() * 0.5

                except Exception as e:
                    if _DEBUG_DISCOVERY:
                        print(f"Penalty computation failed: {e}")

            except Exception as e:
                if _DEBUG_DISCOVERY:
                    print(f"[TATN] Decoder forward failed: {e}")
                    try:
                        traceback.print_exc()
                    except Exception:
                        pass
                translation_loss   = torch.tensor(10.0, device=device)
                token_penalty      = torch.tensor(0.0, device=device)
                confidence_penalty = torch.tensor(0.0, device=device)
                length_penalty     = torch.tensor(0.0, device=device)

            dscd_reg = torch.tensor(0.0, device=device)
            try:
                dscd_reg = self._entropy_reg_from_proto_probs_static(
                    dscd.get("proto_probs", []),
                    gates_list=dscd.get("gates", []),
                    min_gate=0.01,
                )
                if not isinstance(dscd_reg, torch.Tensor):
                    dscd_reg = torch.tensor(float(dscd_reg), device=device)
                if not torch.isfinite(dscd_reg):
                    dscd_reg = torch.tensor(0.0, device=device)
                else:
                    dscd_reg = torch.clamp(dscd_reg.to(device), 0.0, 5.0)
            except Exception:
                dscd_reg = torch.tensor(0.0, device=device)

            self._last_translation_loss = (
                float(translation_loss.item())
                if isinstance(translation_loss, torch.Tensor)
                else 0.0
            )
            self._last_asbn_loss = (
                float(asbn_loss.item()) if isinstance(asbn_loss, torch.Tensor) else 0.0
            )
            self._last_domain_loss = (
                float(domain_loss.item()) if isinstance(domain_loss, torch.Tensor) else 0.0
            )
            self._last_domain_accuracy = (
                float(domain_accuracy.item())
                if isinstance(domain_accuracy, torch.Tensor)
                else 0.0
            )
            self._last_dscd_loss = (
                float(dscd_reg.item()) if isinstance(dscd_reg, torch.Tensor) else 0.0
            )
            self._last_token_penalty = (
                float(token_penalty.item())
                if isinstance(token_penalty, torch.Tensor)
                else 0.0
            )
            self._last_confidence_penalty = (
                float(confidence_penalty.item())
                if isinstance(confidence_penalty, torch.Tensor)
                else 0.0
            )
            self._last_length_penalty = (
                float(length_penalty.item())
                if isinstance(length_penalty, torch.Tensor)
                else 0.0
            )

            total_loss = (
                translation_loss
                + _LAMBDA_ASBN * asbn_loss
                + _LAMBDA_TOKEN * token_penalty
                + _LAMBDA_CONFIDENCE * confidence_penalty
                + _LAMBDA_LENGTH * length_penalty
                + _LAMBDA_DSCD * dscd_reg
            )

            if not isinstance(total_loss, torch.Tensor):
                total_loss = torch.tensor(float(total_loss), device=device)
            if total_loss.numel() != 1:
                total_loss = total_loss.mean()

            if not torch.isfinite(total_loss):
                if _DEBUG_DISCOVERY:
                    print(
                        f"[TATN] NaN/Inf in total_loss ({total_loss}), "
                        "using translation loss"
                    )
                total_loss = translation_loss

            if _DEBUG_DISCOVERY and current_step % 100 == 0:
                print(f"\n{'='*60}")
                print(f"LOSS BREAKDOWN (Step {current_step}):")
                print(f"  Translation:       {translation_loss.item():.4f}")
                print(f"  ASBN Loss:         {asbn_loss.item():.4f} (x{_LAMBDA_ASBN})")
                print(f"  Domain Loss:       {domain_loss.item():.4f}")
                print(f"  Domain Accuracy:   {domain_accuracy.item():.2%}")
                print(f"  Token Penalty:     {token_penalty.item():.4f} (x{_LAMBDA_TOKEN})")
                print(f"  Confidence Pen.:   {confidence_penalty.item():.4f} (x{_LAMBDA_CONFIDENCE})")
                print(f"  Length Penalty:    {length_penalty.item():.4f} (x{_LAMBDA_LENGTH})")
                print(f"  DSCD Reg:          {dscd_reg.item():.4f} (x{_LAMBDA_DSCD})")
                print(f"  TOTAL:             {total_loss.item():.4f}")
                print(f"{'='*60}\n")

            try:
                del enc_outputs, h, raw_dscd
            except Exception:
                pass
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            # training forward returns dict
            return {"loss": total_loss}

        # Inference path
        if not return_dict:
            return sense_adapted_enc

        pad_id = getattr(self.tokenizer, "pad_token_id", 1)

        # ---------------------------------------------------------------------
        # FIXED generate() usage must handle input_ids=None (see separate generate())
        # ---------------------------------------------------------------------

        try:
            inf_decoder_input_ids      = torch.full(
                (batch_size, 1),
                int(self.mbart.config.decoder_start_token_id),
                dtype=torch.long,
                device=device,
            )
            inf_decoder_attention_mask = torch.ones(
                (batch_size, 1), dtype=torch.long, device=device
            )

            seq_outputs_inf = self.mbart(
                input_ids=None,
                attention_mask=attention_mask,
                encoder_outputs=enc_for_decoder,
                decoder_input_ids=inf_decoder_input_ids,
                decoder_attention_mask=inf_decoder_attention_mask,
                use_cache=False,
                output_hidden_states=True,
                return_dict=True,
            )

            dec_hidden_inf = None
            try:
                if (
                    hasattr(seq_outputs_inf, "decoder_hidden_states")
                    and seq_outputs_inf.decoder_hidden_states is not None
                ):
                    dec_hidden_inf = seq_outputs_inf.decoder_hidden_states[-1]
            except Exception:
                dec_hidden_inf = None

            if dec_hidden_inf is not None:
                try:
                    refined_dec_inf = self.hfcaa(
                        dec_hidden_inf, sense_adapted_enc, ambiguity_scores
                    )
                    refined_logits_inf = self.mbart.lm_head(refined_dec_inf)
                except Exception:
                    refined_logits_inf = getattr(seq_outputs_inf, "logits", None)
            else:
                refined_logits_inf = getattr(seq_outputs_inf, "logits", None)

            decoder_attention = None
            try:
                if (
                    hasattr(seq_outputs_inf, "cross_attentions")
                    and seq_outputs_inf.cross_attentions is not None
                    and len(seq_outputs_inf.cross_attentions) > 0
                ):
                    decoder_attention = seq_outputs_inf.cross_attentions[-1]
            except Exception:
                decoder_attention = None

        except Exception:
            refined_logits_inf = None
            seq_outputs_inf    = None
            decoder_attention  = None

        explanations_list: List[List[Dict[str, Any]]] = []

        if _ENABLE_TRG_INFERENCE:
            if _DEBUG_DISCOVERY:
                print(
                    f"\n[TATN-INFERENCE] Starting TRG for {batch_size} samples"
                )

            tokens_batch: List[List[str]] = []

            for b in range(batch_size):
                try:
                    ids_b = input_ids[b].detach().cpu().tolist()
                    if hasattr(self.tokenizer, "convert_ids_to_tokens"):
                        toks = self.tokenizer.convert_ids_to_tokens(ids_b)
                    else:
                        toks = []
                    if not toks:
                        toks = ["UNK"] * seq_len
                    elif len(toks) < seq_len:
                        toks = toks + [""] * (seq_len - len(toks))
                    elif len(toks) > seq_len:
                        toks = toks[:seq_len]
                except Exception:
                    toks = ["UNK"] * seq_len
                tokens_batch.append(toks)

            try:
                total_explanations = 0
                for b in range(batch_size):
                    per_sent = {
                        "proto_probs": self._safe_take_key_static(
                            dscd, "proto_probs", b, seq_len, device
                        ),
                        "uncertainties": self._safe_take_key_static(
                            dscd, "uncertainties", b, seq_len, device
                        ),
                        "gates": self._safe_take_key_static(
                            dscd, "gates", b, seq_len, device
                        ),
                        "span_preds": self._safe_take_key_static(
                            dscd, "span_preds", b, seq_len, device
                        ),
                    }

                    try:
                        exps = self.trg.process_sentence_for_explanations(
                            tokens_batch[b],
                            per_sent,
                            token_word_map=(
                                token_word_map[b]
                                if token_word_map and b < len(token_word_map)
                                else None
                            ),
                            uncertainty_threshold=_TRG_UNCERTAINTY_THRESHOLD,
                            decoder_attention=decoder_attention,
                        )
                        batch_exps = exps if isinstance(exps, list) else []
                        explanations_list.append(batch_exps)
                        total_explanations += len(batch_exps)

                        if _DEBUG_DISCOVERY and b < 2:
                            print(
                                f"[TATN-INFERENCE] Sample {b}: "
                                f"{len(batch_exps)} explanations"
                            )

                    except Exception as e:
                        if _DEBUG_DISCOVERY:
                            print(
                                f"[TATN-INFERENCE] TRG failed for sample {b}: {e}"
                            )
                        explanations_list.append([])

                if _DEBUG_DISCOVERY:
                    print(
                        f"\n[TATN-INFERENCE] Total explanations: {total_explanations}"
                    )
                    if total_explanations == 0:
                        print("[TATN-INFERENCE] NO EXPLANATIONS GENERATED")

            except Exception as e:
                if _DEBUG_DISCOVERY:
                    print(f"[TATN-INFERENCE] TRG generation failed: {e}")
                    try:
                        traceback.print_exc()
                    except Exception:
                        pass
                explanations_list = [[] for _ in range(batch_size)]
        else:
            explanations_list = [[] for _ in range(batch_size)]

        outputs = {
            "encoder_outputs":          enc_for_decoder,
            "dscd_outputs":             dscd,
            "sense_augmented_embeddings": h_aug,
            "sense_adapted_enc":        sense_adapted_enc,
            "refined_logits":           refined_logits_inf,
            "explanations":             explanations_list,
            "asbn_loss":                asbn_loss,
            "domain_loss":              domain_loss,
            "domain_accuracy":          domain_accuracy,
            "ambiguity_signals": {
                "span":       dscd.get("span_preds", []),
                "uncertainty": dscd.get("uncertainties", []),
                "confidence": [
                    [
                        1.0
                        - (
                            float(u)
                            if isinstance(u, (float, int))
                            else (
                                float(u.item())
                                if isinstance(u, torch.Tensor)
                                else 1.0
                            )
                        )
                        for u in row
                    ]
                    for row in dscd.get("uncertainties", [])
                ],
                "proto_probs": dscd.get("proto_probs", []),
            },
        }

        try:
            del h, raw_dscd
        except Exception:
            pass
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return outputs

    def forward_with_explanations(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        src_texts: Optional[List[str]] = None,
        token_word_map: Optional[List[dict]] = None,
    ):
        return self.forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            src_texts=src_texts,
            token_word_map=token_word_map,
            labels=None,
        )

    def forward_with_dscd_for_inference(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        src_texts: Optional[List[str]] = None,
        token_word_map: Optional[List[dict]] = None,
    ) -> Tuple[Dict[str, Any], Dict[str, Any], torch.Tensor]:
        was_training = self.training
        try:
            self.eval()
            with torch.no_grad():
                out = self.forward(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    src_texts=src_texts,
                    token_word_map=token_word_map,
                    labels=None,
                    use_dscd=True,
                )
        finally:
            if was_training:
                self.train()

        dscd_out  = out.get("dscd_outputs", {})
        h_aug     = out.get("sense_augmented_embeddings", None)
        enc_out   = out.get("encoder_outputs", None)

        if h_aug is None:
            device    = input_ids.device
            embed_dim = int(getattr(self.mbart.config, "d_model", 1024))
            h_aug     = torch.zeros(
                int(input_ids.size(0)),
                int(input_ids.size(1)),
                embed_dim,
                device=device,
            )

        return dscd_out, out, h_aug

    # -------------------------------------------------------------------------
    # generate  (FIXED: robust to input_ids=None and encoder_outputs passed)
    # -------------------------------------------------------------------------
    def generate(
        self,
        input_ids: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        src_texts: Optional[List[str]] = None,
        token_word_map: Optional[List[dict]] = None,
        max_new_tokens: int = 128,
        num_beams: int = 4,
        length_penalty: float = 1.0,
        early_stopping: bool = True,
        **generate_kwargs,
    ) -> torch.Tensor:
        # Accept encoder_outputs via kwargs but pop it to avoid double-key errors.
        provided_enc = generate_kwargs.pop("encoder_outputs", None)
        embed_dim = int(getattr(self.mbart.config, "d_model", 1024))

        # Determine device, batch_size, seq_len from available inputs.
        if input_ids is not None:
            device = input_ids.device
            batch_size = int(input_ids.size(0))
            seq_len = int(input_ids.size(1))
        elif attention_mask is not None:
            device = attention_mask.device
            batch_size = int(attention_mask.size(0))
            seq_len = int(attention_mask.size(1))
        elif provided_enc is not None and hasattr(provided_enc, "last_hidden_state") and provided_enc.last_hidden_state is not None:
            device = provided_enc.last_hidden_state.device
            batch_size = int(provided_enc.last_hidden_state.size(0))
            seq_len = int(provided_enc.last_hidden_state.size(1))
        else:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            batch_size = 1
            seq_len = int(getattr(self.mbart.config, "max_position_embeddings", 1024))

        was_training = self.training
        self.eval()
        try:
            with torch.no_grad():
                enc_outputs = None
                if input_ids is not None:
                    try:
                        enc_outputs = self.mbart.model.encoder(
                            input_ids=input_ids,
                            attention_mask=attention_mask,
                        )
                    except Exception:
                        enc_outputs = None
                elif provided_enc is not None:
                    enc_outputs = provided_enc

                h = _safe_get_last_hidden_state(enc_outputs)
                if h is None:
                    if input_ids is not None:
                        try:
                            embedding_layer = self.mbart.get_input_embeddings()
                            h = embedding_layer(input_ids).to(device)
                        except Exception:
                            h = torch.zeros(batch_size, seq_len, embed_dim, device=device)
                    else:
                        h = torch.zeros(batch_size, seq_len, embed_dim, device=device)

                raw_dscd = None
                if provided_enc is None:
                    try:
                        if token_word_map is None:
                            token_word_map = self._reconstruct_word_maps_before_dscd(
                                input_ids if input_ids is not None else torch.zeros((batch_size, seq_len), dtype=torch.long, device=device),
                                batch_size,
                                seq_len,
                                src_texts,
                                None,
                            )
                        raw_dscd = self.dscd.forward(
                            h.detach(),
                            token_types=None,
                            train_mode=False,
                            input_ids=input_ids,
                            attention_mask=attention_mask,
                            token_word_map=token_word_map,
                        )
                    except Exception:
                        raw_dscd = None
                else:
                    # If encoder_outputs provided externally, attempt to use DSCD on its last_hidden_state if available.
                    try:
                        if hasattr(provided_enc, "last_hidden_state") and provided_enc.last_hidden_state is not None:
                            raw_dscd = self.dscd.forward(
                                provided_enc.last_hidden_state.detach(),
                                token_types=None,
                                train_mode=False,
                                input_ids=None,
                                attention_mask=None,
                                token_word_map=token_word_map,
                            )
                        else:
                            raw_dscd = None
                    except Exception:
                        raw_dscd = None

                if raw_dscd is not None:
                    try:
                        dscd = _normalize_dscd_outputs(
                            raw_dscd, batch_size, seq_len, device, embed_dim, fallback_h=h
                        )
                    except Exception:
                        dscd = None
                else:
                    dscd = None

                if dscd is not None:
                    h_aug = dscd.get("h_augmented", h)
                    if not isinstance(h_aug, torch.Tensor) or h_aug.shape != h.shape:
                        h_aug = h
                    sense_assignments, ambiguity_scores = self.get_sense_assignments_from_dscd(
                        dscd, h_aug
                    )
                    try:
                        sense_adapted_enc = self.scea(h_aug, sense_assignments, ambiguity_scores)
                    except Exception:
                        sense_adapted_enc = h_aug
                else:
                    sense_adapted_enc = h

                enc_for_generate = BaseModelOutput(
                    last_hidden_state=sense_adapted_enc,
                    hidden_states=(
                        getattr(enc_outputs, "hidden_states", None)
                        if enc_outputs is not None
                        else None
                    ),
                    attentions=(
                        getattr(enc_outputs, "attentions", None)
                        if enc_outputs is not None
                        else None
                    ),
                )

                generated_ids = self.mbart.generate(
                    input_ids=None,
                    attention_mask=attention_mask,
                    encoder_outputs=enc_for_generate,
                    decoder_start_token_id=int(self.mbart.config.decoder_start_token_id),
                    forced_bos_token_id=None,
                    max_new_tokens=max_new_tokens,
                    num_beams=num_beams,
                    length_penalty=length_penalty,
                    early_stopping=early_stopping,
                    use_cache=True,
                    **generate_kwargs,
                )

        finally:
            if was_training:
                self.train()

        return generated_ids

    # -------------------------------------------------------------------------
    # get_component_stats
    # -------------------------------------------------------------------------
    def get_component_stats(self) -> Dict[str, Any]:
        stats: Dict[str, Any] = {}

        stats["last_translation_loss"]   = self._last_translation_loss
        stats["last_asbn_loss"]          = self._last_asbn_loss
        stats["last_domain_loss"]        = self._last_domain_loss
        stats["last_domain_accuracy"]    = self._last_domain_accuracy
        stats["last_dscd_loss"]          = self._last_dscd_loss
        stats["last_token_penalty"]      = self._last_token_penalty
        stats["last_confidence_penalty"] = self._last_confidence_penalty
        stats["last_length_penalty"]     = self._last_length_penalty
        stats["last_path1_asbn_loss"]    = self._last_path1_asbn_loss
        stats["last_path1_dscd_loss"]    = self._last_path1_dscd_loss
        stats["global_step"]             = self.global_step

        try:
            dscd_summary = self.dscd.get_prototype_summary()
            stats["dscd"] = {
                "num_homographs":    dscd_summary.get("num_homographs", 0),
                "total_prototypes":  dscd_summary.get("total_prototypes", 0),
                "total_tokens":      dscd_summary.get("total_tokens", 0),
                "buffer_usage":      dscd_summary.get("buffer_usage", 0.0),
            }
        except Exception:
            stats["dscd"] = {
                "num_homographs":   0,
                "total_prototypes": 0,
                "total_tokens":     0,
                "buffer_usage":     0.0,
            }

        try:
            asbn_stats    = self.asbn.get_detailed_stats()
            stats["asbn"] = {
                "domain_loss":     asbn_stats.get("domain_loss", 0.0),
                "domain_accuracy": asbn_stats.get("domain_accuracy", 0.0),
                "source_accuracy": asbn_stats.get("source_accuracy", 0.0),
                "target_accuracy": asbn_stats.get("target_accuracy", 0.0),
                "asbn_loss":       asbn_stats.get("asbn_loss", 0.0),
                "num_updates":     asbn_stats.get("num_updates", 0),
            }
        except Exception:
            stats["asbn"] = {
                "domain_loss":     0.0,
                "domain_accuracy": 0.0,
                "source_accuracy": 0.0,
                "target_accuracy": 0.0,
                "asbn_loss":       0.0,
                "num_updates":     0,
            }

        try:
            trg_stats    = self.trg.get_statistics()
            stats["trg"] = trg_stats if isinstance(trg_stats, dict) else {}
        except Exception:
            stats["trg"] = {}

        try:
            scea_gates    = torch.sigmoid(self.scea.sense_gates).detach().cpu().tolist()
            stats["scea"] = {"sense_gates": scea_gates}
        except Exception:
            stats["scea"] = {"sense_gates": []}

        return stats

    def reset_component_stats(self):
        try:
            self.asbn.reset_stats()
        except Exception:
            pass
        try:
            self.trg.reset_statistics()
        except Exception:
            pass
        self._last_translation_loss   = 0.0
        self._last_asbn_loss          = 0.0
        self._last_domain_loss        = 0.0
        self._last_domain_accuracy    = 0.0
        self._last_dscd_loss          = 0.0
        self._last_token_penalty      = 0.0
        self._last_confidence_penalty = 0.0
        self._last_length_penalty     = 0.0
        self._last_path1_asbn_loss    = 0.0
        self._last_path1_dscd_loss    = 0.0

    def get_trainable_parameters(self) -> List[nn.Parameter]:
        params = []
        try:
            params += list(self.mbart.parameters())
        except Exception:
            pass
        try:
            params += list(self.dscd.parameters())
        except Exception:
            pass
        try:
            params += list(self.asbn.parameters())
        except Exception:
            pass
        try:
            params += list(self.scea.parameters())
        except Exception:
            pass
        try:
            params += list(self.hfcaa.parameters())
        except Exception:
            pass
        return params

    def get_non_critic_parameters(self) -> List[nn.Parameter]:
        params          = []
        critic_param_ids = set()

        try:
            critic_params = self.asbn.critic_parameters()
            critic_param_ids = {id(p) for p in critic_params}
        except Exception:
            critic_param_ids = set()

        for p in self.parameters():
            if id(p) not in critic_param_ids:
                params.append(p)
        return params

    def get_critic_parameters(self) -> List[nn.Parameter]:
        try:
            return list(self.asbn.critic_parameters())
        except Exception:
            return []

    def freeze_mbart(self):
        for param in self.mbart.parameters():
            param.requires_grad = False

    def unfreeze_mbart(self):
        for param in self.mbart.parameters():
            param.requires_grad = True

    def freeze_encoder(self):
        try:
            for param in self.mbart.model.encoder.parameters():
                param.requires_grad = False
        except Exception:
            pass

    def unfreeze_encoder(self):
        try:
            for param in self.mbart.model.encoder.parameters():
                param.requires_grad = True
        except Exception:
            pass

    def get_memory_usage(self) -> Dict[str, float]:
        stats = {}
        if torch.cuda.is_available():
            for i in range(min(_NUM_GPUS, torch.cuda.device_count())):
                try:
                    allocated = torch.cuda.memory_allocated(i) / (1024 ** 3)
                    reserved  = torch.cuda.memory_reserved(i) / (1024 ** 3)
                    stats[f"gpu_{i}_allocated_gb"] = round(allocated, 3)
                    stats[f"gpu_{i}_reserved_gb"]  = round(reserved, 3)
                except Exception:
                    pass
        return stats

    def cleanup_memory(self):
        if torch.cuda.is_available():
            for i in range(min(_NUM_GPUS, torch.cuda.device_count())):
                try:
                    with torch.cuda.device(i):
                        torch.cuda.empty_cache()
                except Exception:
                    pass
        if gc.isenabled():
            gc.collect()

    def save_dscd_state(self, path: str):
        try:
            state = {
                "dscd_state":    self.dscd.state_dict(),
                "global_step":   self.global_step,
                "vocab_size":    self.vocab_size,
                "en_token_id":   self.en_token_id,
                "bn_token_id":   self.bn_token_id,
            }
            torch.save(state, path)
            if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                print(f"[TATN] DSCD state saved to {path}")
        except Exception as e:
            print(f"[TATN] Failed to save DSCD state: {e}")

    def load_dscd_state(self, path: str):
        try:
            state = torch.load(path, map_location="cpu")
            if "dscd_state" in state:
                self.dscd.load_state_dict(state["dscd_state"])
            if "global_step" in state:
                self.global_step = int(state["global_step"])
            if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                print(f"[TATN] DSCD state loaded from {path}")
                print(f"  - global_step: {self.global_step}")
        except Exception as e:
            print(f"[TATN] Failed to load DSCD state: {e}")


# ==============================================================================
# Alias + Cell 6 completion banner
# ==============================================================================

TATNModelWithDSCDAndASBN = MemoryOptimizedTATNWithExplanations

print("=" * 80)
print("Cell 6: DUAL-PATH TATN MODEL | SCEA + HFCAA + SENSE-GATED FUSION ACTIVE")
print("=" * 80)
print("NEW MODULES REGISTERED:")
print("  self.scea  = SenseConditionedEncoderAdapter(1024, 5, 128)")
print("  self.hfcaa = HomographFocusedCrossAttentionAdapter(1024, 8)")
print("NEW HELPER:")
print("  get_sense_assignments_from_dscd -> sense_assignments[B,T], ambiguity_scores[B,T]")
print("FORWARD CHANGES:")
print("  FWD-1: DSCD -> get_sense_assignments_from_dscd -> sense_assignments, ambiguity_scores")
print("  FWD-2: SCEA(enc_out, sense_assignments, ambiguity_scores) -> sense_adapted_enc")
print("  FWD-3: h_aug fusion ALWAYS applied when shapes permit (HAUG gate removed)")
print("  FWD-4: fusion base = sense_adapted_enc (not raw enc_out)")
print("  FWD-5: decoder called with output_hidden_states=True")
print("  FWD-6: HFCAA(dec_hidden, sense_adapted_enc, ambiguity_scores) -> refined_dec")
print("  FWD-7: mbart.lm_head(refined_dec) applied manually -> refined_logits")
print("PATH2 FIX (C6-P2-1):")
print("  path1_h_aug=None -> SCEA still applied on h_subword (gate-closed path)")
print("  path1_h_aug != None -> SCEA + unconditional fusion (gate removed)")
print("FIX 6A/B: HAUG gating removed in forward and forward_path2 as requested")
print("=" * 80)

In [ ]:
# ==============================================================================
# CELL 6.5: BACK-TRANSLATION DATA AUGMENTATION
# Generates synthetic (bn_synthetic, en_real) pairs using mBART en→bn
# Reads: USE_BACK_TRANSLATION, BT_* params from Cell 0
# Produces: bt_train_pairs (List[Tuple[str,str]]) injected into training
# Run AFTER Cell 6 (model init) and BEFORE Cell 7 (training loop)
# ==============================================================================


import os
import gc
import sys
import time
import math
import random
import traceback
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any
from contextlib import nullcontext

import torch
import numpy as np
from tqdm import tqdm


try:
    USE_BACK_TRANSLATION = bool(USE_BACK_TRANSLATION)
except NameError:
    USE_BACK_TRANSLATION = False

try:
    BT_MONOLINGUAL_SAMPLE = int(BT_MONOLINGUAL_SAMPLE)
except NameError:
    BT_MONOLINGUAL_SAMPLE = 50000

try:
    BT_GEN_BATCH_SIZE = int(BT_GEN_BATCH_SIZE)
except NameError:
    BT_GEN_BATCH_SIZE = 16

try:
    BT_NUM_BEAMS = int(BT_NUM_BEAMS)
except NameError:
    BT_NUM_BEAMS = 4

try:
    BT_MIX_RATIO = float(BT_MIX_RATIO)
except NameError:
    BT_MIX_RATIO = 0.3

try:
    BT_NOISE_DROPOUT = float(BT_NOISE_DROPOUT)
except NameError:
    BT_NOISE_DROPOUT = 0.1

try:
    BT_MIN_LENGTH = int(BT_MIN_LENGTH)
except NameError:
    BT_MIN_LENGTH = 4

try:
    BT_MAX_NEW_TOKENS = int(BT_MAX_NEW_TOKENS)
except NameError:
    BT_MAX_NEW_TOKENS = 128

try:
    BT_CACHE_ENABLED = bool(BT_CACHE_ENABLED)
except NameError:
    BT_CACHE_ENABLED = True

try:
    BT_CACHE_PATH = str(BT_CACHE_PATH)
except NameError:
    BT_CACHE_PATH = "/kaggle/working/bt_cache.pt"

try:
    LAMBDA_BT = float(LAMBDA_BT)
except NameError:
    LAMBDA_BT = 1.0

try:
    SOURCE_LANGUAGE = str(SOURCE_LANGUAGE)
except NameError:
    SOURCE_LANGUAGE = "bn_IN"

try:
    TARGET_LANGUAGE = str(TARGET_LANGUAGE)
except NameError:
    TARGET_LANGUAGE = "en_XX"

try:
    MBART_MODEL_NAME = str(MBART_MODEL_NAME)
except NameError:
    MBART_MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

try:
    MBART50_BN_TOKEN_ID = int(MBART50_BN_TOKEN_ID)
except NameError:
    MBART50_BN_TOKEN_ID = 250028

try:
    MBART50_EN_TOKEN_ID = int(MBART50_EN_TOKEN_ID)
except NameError:
    MBART50_EN_TOKEN_ID = 250004

try:
    MAX_LENGTH = int(MAX_LENGTH)
except NameError:
    MAX_LENGTH = 256

# FIX-C6.5-1: Removed trailing slash to match Cell 0's CHECKPOINT_DIR="/kaggle/working"
try:
    CHECKPOINT_DIR = str(CHECKPOINT_DIR)
except NameError:
    CHECKPOINT_DIR = "/kaggle/working"

try:
    DATASET_CSV_PATH = str(DATASET_CSV_PATH)
except NameError:
    DATASET_CSV_PATH = ""

try:
    SRC_COL = str(SRC_COL)
except NameError:
    SRC_COL = "src"

try:
    TGT_COL = str(TGT_COL)
except NameError:
    TGT_COL = "tgt"

try:
    DEVICE = DEVICE
except NameError:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# FIX-C6.5-2: Read AMP_DTYPE from Cell 0 globals (T4→float16, L4/A100→bfloat16)
try:
    AMP_DTYPE = AMP_DTYPE
except NameError:
    if torch.cuda.is_available():
        _major, _ = torch.cuda.get_device_capability(0)
        AMP_DTYPE = torch.bfloat16 if _major >= 8 else torch.float16
    else:
        AMP_DTYPE = torch.float16

try:
    USE_BF16 = bool(USE_BF16)
except NameError:
    USE_BF16 = (AMP_DTYPE == torch.bfloat16)

try:
    USE_AMP = bool(USE_AMP)
except NameError:
    USE_AMP = True

try:
    SEED = int(SEED)
except NameError:
    SEED = 42

try:
    VERBOSE_LOGGING = bool(VERBOSE_LOGGING)
except NameError:
    VERBOSE_LOGGING = False

try:
    DEBUG_DISCOVERY = bool(DEBUG_DISCOVERY)
except NameError:
    DEBUG_DISCOVERY = False


# ==============================================================================
# BACK-TRANSLATION NOISE: token-level dropout on synthetic Bengali output
# ==============================================================================


def _apply_bt_noise(
    tokens: List[str],
    noise_dropout: float,
    rng: random.Random,
) -> List[str]:
    if noise_dropout <= 0.0:
        return tokens
    return [tok for tok in tokens if rng.random() >= noise_dropout]


# ==============================================================================
# ENGLISH MONOLINGUAL SENTENCE EXTRACTION
# ==============================================================================


def _load_english_monolingual(
    dataset_csv_path: str,
    tgt_col: str,
    num_samples: int,
    seed: int,
) -> List[str]:
    sentences: List[str] = []
    try:
        import pandas as pd
        if not os.path.exists(dataset_csv_path):
            print(f"[BT] WARNING: CSV not found at {dataset_csv_path}, using fallback sentences.")
            return _bt_fallback_sentences()

        df = pd.read_csv(dataset_csv_path, on_bad_lines="skip", encoding="utf-8")
        if tgt_col not in df.columns:
            print(f"[BT] WARNING: Column '{tgt_col}' not in CSV. Available: {list(df.columns)}")
            return _bt_fallback_sentences()

        col_data = df[tgt_col].dropna().astype(str).tolist()
        rng = random.Random(seed)
        rng.shuffle(col_data)
        col_data = col_data[:num_samples]

        for sent in col_data:
            sent = sent.strip()
            if sent and len(sent.split()) >= 3:
                sentences.append(sent)

        print(f"[BT] Loaded {len(sentences):,} English monolingual sentences from '{tgt_col}' column.")

    except Exception as e:
        print(f"[BT] ERROR loading monolingual data: {type(e).__name__}: {e}")
        sentences = _bt_fallback_sentences()

    return sentences[:num_samples]


def _bt_fallback_sentences() -> List[str]:
    return [
        "The water tap is broken.",
        "I will call you tomorrow.",
        "She turned the page of the book.",
        "He went to the bank to deposit money.",
        "The fruit fell from the tree.",
        "The stars are shining in the sky.",
        "I am learning a new language.",
        "The leaf of the tree is green.",
        "He scored a goal in the match.",
        "The time has come to make a decision.",
    ]


# ==============================================================================
# CORE BT GENERATION: en→bn using mBART en_XX→bn_IN direction
# ==============================================================================


def _generate_bt_batch(
    en_sentences: List[str],
    bt_tokenizer,
    bt_model_core,
    bt_num_beams: int,
    bt_min_length: int,
    bt_max_new_tokens: int,
    bn_token_id: int,
    device: torch.device,
    use_amp: bool,
    amp_dtype: torch.dtype,
) -> List[str]:
    try:
        bt_tokenizer.src_lang = TARGET_LANGUAGE
        bt_tokenizer.tgt_lang = SOURCE_LANGUAGE

        enc = bt_tokenizer(
            en_sentences,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )

        input_ids      = enc["input_ids"].to(device, non_blocking=True)
        attention_mask = enc["attention_mask"].to(device, non_blocking=True)

        # FIX-C6.5-3: Always wrap with no_grad; use nullcontext when AMP is off
        # so autocast and no_grad are never double-nested incorrectly
        amp_ctx = (
            torch.amp.autocast(device_type="cuda", dtype=amp_dtype)
            if use_amp and torch.cuda.is_available()
            else nullcontext()
        )

        # FIX-C6.5-5: Unwrap DataParallel/module wrapper if present
        core = bt_model_core.module if hasattr(bt_model_core, "module") else bt_model_core

        with torch.no_grad():
            with amp_ctx:
                generated = core.mbart.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    forced_bos_token_id=bn_token_id,
                    num_beams=bt_num_beams,
                    min_length=bt_min_length,
                    max_new_tokens=bt_max_new_tokens,
                    early_stopping=True,
                    no_repeat_ngram_size=3,
                )

        decoded = bt_tokenizer.batch_decode(generated, skip_special_tokens=True)

        try:
            del input_ids, attention_mask, generated
            torch.cuda.empty_cache()
        except Exception:
            pass

        return decoded

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"[BT] OOM on batch of {len(en_sentences)}, skipping batch.")
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass
            return [""] * len(en_sentences)
        raise

    except Exception as e:
        print(f"[BT] Batch generation failed: {type(e).__name__}: {str(e)[:120]}")
        return [""] * len(en_sentences)


# ==============================================================================
# BT CACHE LOAD / SAVE
# ==============================================================================


def _load_bt_cache(cache_path: str) -> Optional[List[Tuple[str, str]]]:
    if not BT_CACHE_ENABLED:
        return None
    if not os.path.exists(cache_path):
        return None
    try:
        cached = torch.load(cache_path, map_location="cpu", weights_only=False)
        if isinstance(cached, list) and len(cached) > 0:
            print(f"[BT] Loaded {len(cached):,} BT pairs from cache: {cache_path}")
            return cached
    except Exception as e:
        print(f"[BT] Cache load failed: {type(e).__name__}: {e}")
    return None


def _save_bt_cache(bt_pairs: List[Tuple[str, str]], cache_path: str) -> None:
    if not BT_CACHE_ENABLED:
        return
    try:
        # FIX-C6.5-4: Only call makedirs when dirname is non-empty
        _cache_dir = os.path.dirname(cache_path)
        if _cache_dir:
            os.makedirs(_cache_dir, exist_ok=True)
        torch.save(bt_pairs, cache_path)
        size_mb = Path(cache_path).stat().st_size / (1024 ** 2)
        print(f"[BT] Saved {len(bt_pairs):,} BT pairs to cache: {cache_path} ({size_mb:.1f} MB)")
    except Exception as e:
        print(f"[BT] Cache save failed: {type(e).__name__}: {e}")


# ==============================================================================
# BUILD BT DATASET: full pipeline
# ==============================================================================


def build_bt_pairs(
    bt_tokenizer,
    bt_model_core,
    device: torch.device,
) -> List[Tuple[str, str]]:
    print("\n" + "=" * 80)
    print("CELL 6.5: BACK-TRANSLATION — en→bn SYNTHETIC PAIR GENERATION")
    print("=" * 80)
    print(f"  USE_BACK_TRANSLATION:  {USE_BACK_TRANSLATION}")
    print(f"  BT_MONOLINGUAL_SAMPLE: {BT_MONOLINGUAL_SAMPLE:,}")
    print(f"  BT_GEN_BATCH_SIZE:     {BT_GEN_BATCH_SIZE}")
    print(f"  BT_NUM_BEAMS:          {BT_NUM_BEAMS}")
    print(f"  BT_MIX_RATIO:          {BT_MIX_RATIO}")
    print(f"  BT_NOISE_DROPOUT:      {BT_NOISE_DROPOUT}")
    print(f"  BT_MIN_LENGTH:         {BT_MIN_LENGTH}")
    print(f"  BT_MAX_NEW_TOKENS:     {BT_MAX_NEW_TOKENS}")
    print(f"  BT_CACHE_ENABLED:      {BT_CACHE_ENABLED}")
    print(f"  BT_CACHE_PATH:         {BT_CACHE_PATH}")
    print(f"  LAMBDA_BT:             {LAMBDA_BT}")
    print(f"  AMP_DTYPE:             {AMP_DTYPE}")
    print(f"  Direction:             {TARGET_LANGUAGE} → {SOURCE_LANGUAGE}")
    print("=" * 80)

    cached = _load_bt_cache(BT_CACHE_PATH)
    if cached is not None:
        return cached

    en_sentences = _load_english_monolingual(
        DATASET_CSV_PATH,
        TGT_COL,
        BT_MONOLINGUAL_SAMPLE,
        SEED,
    )

    if not en_sentences:
        print("[BT] ERROR: No English sentences available. Returning empty BT pairs.")
        return []

    print(f"[BT] Generating bn synthetic translations for {len(en_sentences):,} sentences...")

    bt_model_core.eval()
    bt_pairs: List[Tuple[str, str]] = []
    rng_noise = random.Random(SEED + 1)
    skipped   = 0
    start_t   = time.time()

    with tqdm(
        total=len(en_sentences),
        desc="BT en→bn",
        ncols=110,
        file=sys.stdout,
        leave=True,
    ) as pbar:
        for i in range(0, len(en_sentences), BT_GEN_BATCH_SIZE):
            batch_en = en_sentences[i : i + BT_GEN_BATCH_SIZE]

            bn_generated = _generate_bt_batch(
                en_sentences      = batch_en,
                bt_tokenizer      = bt_tokenizer,
                bt_model_core     = bt_model_core,
                bt_num_beams      = BT_NUM_BEAMS,
                bt_min_length     = BT_MIN_LENGTH,
                bt_max_new_tokens = BT_MAX_NEW_TOKENS,
                bn_token_id       = MBART50_BN_TOKEN_ID,
                device            = device,
                use_amp           = USE_AMP,
                amp_dtype         = AMP_DTYPE,
            )

            for en_sent, bn_sent in zip(batch_en, bn_generated):
                bn_sent = bn_sent.strip()
                en_sent = en_sent.strip()

                if not bn_sent or not en_sent:
                    skipped += 1
                    continue

                if len(bn_sent.split()) < 2:
                    skipped += 1
                    continue

                if BT_NOISE_DROPOUT > 0.0:
                    bn_tokens = bt_tokenizer.tokenize(bn_sent)
                    bn_tokens = _apply_bt_noise(bn_tokens, BT_NOISE_DROPOUT, rng_noise)
                    if not bn_tokens:
                        skipped += 1
                        continue
                    bn_sent = bt_tokenizer.convert_tokens_to_string(bn_tokens).strip()
                    if not bn_sent:
                        skipped += 1
                        continue

                bt_pairs.append((bn_sent, en_sent))

            pbar.update(len(batch_en))
            pbar.set_postfix(
                pairs=len(bt_pairs),
                skipped=skipped,
                refresh=False,
            )

    elapsed = time.time() - start_t
    speed   = len(en_sentences) / elapsed if elapsed > 0 else 0.0

    print(f"\n[BT] Generation complete.")
    print(f"  Pairs generated:  {len(bt_pairs):,}")
    print(f"  Skipped/empty:    {skipped:,}")
    print(f"  Time:             {elapsed:.1f}s ({elapsed / 60:.2f} min)")
    print(f"  Speed:            {speed:.1f} sents/s")

    if len(bt_pairs) > 0:
        print(f"\n[BT] Sample BT pairs (first 3):")
        for idx, (bn_s, en_s) in enumerate(bt_pairs[:3], 1):
            print(f"  [{idx}] BN: {bn_s[:80]}")
            print(f"       EN: {en_s[:80]}")

    _save_bt_cache(bt_pairs, BT_CACHE_PATH)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return bt_pairs


# ==============================================================================
# BT DATASET WRAPPER
# ==============================================================================


def build_bt_dataset(
    bt_pairs: List[Tuple[str, str]],
    tokenizer,
    word_tokenizer=None,
) -> Optional[Any]:
    if not bt_pairs:
        print("[BT] No BT pairs — bt_dataset will be None.")
        return None

    try:
        DualPathTranslationDataset = globals().get("DualPathTranslationDataset", None)
        if DualPathTranslationDataset is None:
            print("[BT] ERROR: DualPathTranslationDataset not found in globals. Run Cell 2 first.")
            return None

        bt_dataset = DualPathTranslationDataset(
            pairs          = bt_pairs,
            tokenizer      = tokenizer,
            word_tokenizer = word_tokenizer,
            max_length     = MAX_LENGTH,
            split          = "train",
            domain         = globals().get("TRAIN_DOMAIN", 0),
        )
        print(f"[BT] bt_dataset created: {len(bt_dataset):,} samples.")
        return bt_dataset

    except Exception as e:
        print(f"[BT] ERROR building bt_dataset: {type(e).__name__}: {e}")
        if VERBOSE_LOGGING or DEBUG_DISCOVERY:
            traceback.print_exc()
        return None


# ==============================================================================
# MIX BT PAIRS INTO TRAINING DATA
# ==============================================================================


def mix_bt_into_train(
    train_pairs: List[Tuple[str, str]],
    bt_pairs:    List[Tuple[str, str]],
    bt_mix_ratio: float,
    seed: int,
) -> List[Tuple[str, str]]:
    if not bt_pairs or bt_mix_ratio <= 0.0:
        print(f"[BT] mix_bt_into_train: no BT pairs or BT_MIX_RATIO=0. Returning original train set.")
        return train_pairs

    n_bt_target = int(len(train_pairs) * bt_mix_ratio)
    n_bt_target = min(n_bt_target, len(bt_pairs))

    if n_bt_target == 0:
        print(f"[BT] mix_bt_into_train: computed 0 BT samples to mix in. Returning original.")
        return train_pairs

    rng = random.Random(seed + 7)
    sampled_bt = rng.sample(bt_pairs, n_bt_target)
    mixed      = train_pairs + sampled_bt
    rng.shuffle(mixed)

    print(f"[BT] Mixing: {len(train_pairs):,} real + {n_bt_target:,} BT = {len(mixed):,} total train pairs.")
    return mixed


# ==============================================================================
# MAIN ENTRY POINT
# ==============================================================================


bt_train_pairs: List[Tuple[str, str]] = []
bt_dataset = None

if not USE_BACK_TRANSLATION:
    print("[BT] USE_BACK_TRANSLATION=False — Cell 6.5 skipped.")
    print("[BT] bt_train_pairs = [] | bt_dataset = None")
else:
    try:
        _bt_tokenizer  = globals().get("tokenizer", None)

        # FIX-C6.5-6: Try multiple model variable names used across Cell 6 variants
        _bt_model_core = (
            globals().get("tatn_model", None)
            or globals().get("model", None)
            or globals().get("tatn_model_wrapped", None)
        )

        if _bt_tokenizer is None:
            print("[BT] ERROR: 'tokenizer' not found in globals. Run Cell 5 (tokenizer init) first.")
        elif _bt_model_core is None:
            print("[BT] ERROR: 'tatn_model' / 'model' not found in globals. Run Cell 6 (model init) first.")
        else:
            _core = _bt_model_core.module if hasattr(_bt_model_core, "module") else _bt_model_core

            bt_train_pairs = build_bt_pairs(
                bt_tokenizer  = _bt_tokenizer,
                bt_model_core = _core,
                device        = DEVICE,
            )

            _bt_wt = globals().get("word_tokenizer", None)

            bt_dataset = build_bt_dataset(
                bt_pairs       = bt_train_pairs,
                tokenizer      = _bt_tokenizer,
                word_tokenizer = _bt_wt,
            )

    except Exception as _bt_exc:
        print(f"[BT] FATAL ERROR in Cell 6.5: {type(_bt_exc).__name__}: {_bt_exc}")
        traceback.print_exc()
        bt_train_pairs = []
        bt_dataset     = None

    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# FIX-C6.5-7: Safe length display for bt_dataset (avoids crash if not Sized)
try:
    _bt_dataset_info = f"Ready — {len(bt_dataset):,} samples" if bt_dataset is not None else "None"
except Exception:
    _bt_dataset_info = "Ready (size unknown)" if bt_dataset is not None else "None"

print("\n" + "=" * 80)
print("CELL 6.5 SUMMARY")
print("=" * 80)
print(f"  USE_BACK_TRANSLATION: {USE_BACK_TRANSLATION}")
print(f"  bt_train_pairs:       {len(bt_train_pairs):,} synthetic (bn_synthetic, en_real) pairs")
print(f"  bt_dataset:           {_bt_dataset_info}")
print(f"  LAMBDA_BT:            {LAMBDA_BT}")
print(f"  BT_MIX_RATIO:         {BT_MIX_RATIO}")
print(f"  AMP_DTYPE:            {AMP_DTYPE}")
print()
print("  HOW TO USE IN CELL 7:")
print("  Pass bt_train_pairs to mix_bt_into_train(train_pairs, bt_train_pairs, BT_MIX_RATIO, SEED)")
print("  OR pass bt_dataset as a secondary DataLoader and weight its loss by LAMBDA_BT")
print("=" * 80)


In [ ]:
# ==============================================================================
# CELL 7: DUAL-PATH TRAINING LOOP - PATH 1 (DSCD+Contrastive) + PATH 2 (TRANSLATION)
# ==============================================================================

import os
import time
import math
import gc
import traceback
import sys
from datetime import datetime
from pathlib import Path
from collections import defaultdict, deque
from typing import Optional, Dict, Any, List

import numpy as np
import torch
from torch.cuda.amp import GradScaler
from torch.amp import autocast as torch_amp_autocast
from tqdm import tqdm
from contextlib import nullcontext
import threading
from torch.utils.data import Dataset
import torch.nn as nn

_missing_upload_fns = [
    fn for fn in ("upload_epoch_files_to_gdrive", "upload_checkpoint_to_gdrive")
    if fn not in globals() or not callable(globals()[fn])
]
if _missing_upload_fns:
    raise RuntimeError(
        f"[CELL7] Missing Drive upload functions: {_missing_upload_fns}\n"
        f"  → Run the Authentication Cell first, then re-run Cell 7."
    )
print("[CELL7] ✅ Pre-flight: Drive upload functions verified.")

try:
    _VERBOSE_LOGGING = bool(VERBOSE_LOGGING)
except (NameError, TypeError):
    _VERBOSE_LOGGING = False

try:
    _DEBUG_DISCOVERY = bool(DEBUG_DISCOVERY)
except (NameError, TypeError):
    _DEBUG_DISCOVERY = False

DEBUG_PRINT_INTERVAL = 400
_cell7_dbg_counts = defaultdict(int)


def cell7_dbg(key: str, msg: str, limit: int = 10):
    if not (_VERBOSE_LOGGING or _DEBUG_DISCOVERY):
        return
    _cell7_dbg_counts[key] += 1
    if _cell7_dbg_counts[key] <= limit:
        print(f"[CELL7-DBG] {msg}")


try:
    _DEVICE = DEVICE
except (NameError, TypeError):
    _DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    _EPOCHS = int(EPOCHS)
except (NameError, ValueError, TypeError):
    _EPOCHS = 1

try:
    _BATCH_SIZE = int(BATCH_SIZE)
except (NameError, ValueError, TypeError):
    _BATCH_SIZE = 4

try:
    _ACCUMULATION_STEPS = int(ACCUMULATION_STEPS)
except (NameError, ValueError, TypeError):
    _ACCUMULATION_STEPS = 8

try:
    _GRAD_CLIP_NORM = float(GRAD_CLIP_NORM)
except (NameError, ValueError, TypeError):
    _GRAD_CLIP_NORM = 1.0

try:
    _MEMORY_CLEANUP_FREQUENCY = int(MEMORY_CLEANUP_FREQUENCY)
except (NameError, ValueError, TypeError):
    _MEMORY_CLEANUP_FREQUENCY = 100

try:
    _USE_MULTI_GPU = bool(USE_MULTI_GPU)
    _NUM_GPUS = int(NUM_GPUS)
except (NameError, ValueError, TypeError):
    _NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
    _USE_MULTI_GPU = _NUM_GPUS > 1

try:
    _USE_AMP = bool(USE_AMP)
except (NameError, TypeError):
    _USE_AMP = True

try:
    _AMP_DTYPE = AMP_DTYPE
except (NameError, TypeError):
    if torch.cuda.is_available():
        _major, _ = torch.cuda.get_device_capability(0)
        _AMP_DTYPE = torch.bfloat16 if _major >= 8 else torch.float16
    else:
        _AMP_DTYPE = torch.float16

try:
    _USE_BF16 = bool(USE_BF16)
except (NameError, TypeError):
    _USE_BF16 = (_AMP_DTYPE == torch.bfloat16)

try:
    _SOURCE_LANGUAGE = str(SOURCE_LANGUAGE)
    _TARGET_LANGUAGE = str(TARGET_LANGUAGE)
except (NameError, TypeError):
    _SOURCE_LANGUAGE = "bn_IN"
    _TARGET_LANGUAGE = "en_XX"

try:
    _MAX_LENGTH = int(MAX_LENGTH)
except (NameError, ValueError, TypeError):
    _MAX_LENGTH = 256

try:
    _VALIDATION_CHECK_INTERVAL = int(VALIDATION_CHECK_INTERVAL)
except (NameError, ValueError, TypeError):
    _VALIDATION_CHECK_INTERVAL = 500

try:
    _PERIODIC_DISCOVERY_FREQUENCY = int(PERIODIC_DISCOVERY_FREQUENCY)
except (NameError, ValueError, TypeError):
    _PERIODIC_DISCOVERY_FREQUENCY = 300

try:
    _TRAIN_DOMAIN = int(TRAIN_DOMAIN)
    _TEST_DOMAIN = int(TEST_DOMAIN)
except (NameError, ValueError, TypeError):
    _TRAIN_DOMAIN = 0
    _TEST_DOMAIN = 1

try:
    _LAMBDA_TRG = float(LAMBDA_TRG)
except (NameError, ValueError, TypeError):
    _LAMBDA_TRG = 0.001

try:
    _LAMBDA_ASBN = float(LAMBDA_ASBN)
except (NameError, ValueError, TypeError):
    _LAMBDA_ASBN = 0.0

try:
    _WARMUP_STEPS = int(WARMUP_STEPS)
except (NameError, ValueError, TypeError):
    _WARMUP_STEPS = 800
    print(f"[CELL7] WARNING: WARMUP_STEPS not defined, using default {_WARMUP_STEPS}")

try:
    _LABEL_SMOOTHING = float(LABEL_SMOOTHING)
except (NameError, ValueError, TypeError):
    _LABEL_SMOOTHING = 0.1

try:
    _USE_LR_SCHEDULER = bool(USE_LR_SCHEDULER)
except (NameError, TypeError):
    _USE_LR_SCHEDULER = True
    print(f"[CELL7] WARNING: USE_LR_SCHEDULER not defined, defaulting to True")

try:
    _USE_DUAL_PATH_TRAINING = bool(USE_DUAL_PATH_TRAINING)
except (NameError, TypeError):
    _USE_DUAL_PATH_TRAINING = True

try:
    _HOMOGRAPH_REFERENCE_LIST = set(str(w).lower() for w in HOMOGRAPH_REFERENCE_LIST_BN)
except (NameError, TypeError):
    _HOMOGRAPH_REFERENCE_LIST = {
        "কল", "কাল", "পাতা", "ব্যাংক", "ফল", "মাথা", "বার", "হার", "তারা",
        "পানি", "দল", "বাজার", "নাম", "কথা", "বই", "ঘর", "মন", "হাত"
    }
    _HOMOGRAPH_REFERENCE_LIST = set(str(w).lower() for w in _HOMOGRAPH_REFERENCE_LIST)

try:
    _EVAL_LENGTH_PENALTY = float(EVAL_LENGTH_PENALTY)
except (NameError, ValueError, TypeError):
    _EVAL_LENGTH_PENALTY = 1.0

try:
    _EVAL_REPETITION_PENALTY = float(EVAL_REPETITION_PENALTY)
except (NameError, ValueError, TypeError):
    _EVAL_REPETITION_PENALTY = 1.1

try:
    _EVAL_MIN_LENGTH = int(EVAL_MIN_LENGTH)
except (NameError, ValueError, TypeError):
    _EVAL_MIN_LENGTH = 5

try:
    _EVAL_NUM_BEAMS = int(EVAL_NUM_BEAMS)
except (NameError, ValueError, TypeError):
    _EVAL_NUM_BEAMS = 8

try:
    _EVAL_NO_REPEAT_NGRAM_SIZE = int(EVAL_NO_REPEAT_NGRAM_SIZE)
except (NameError, ValueError, TypeError):
    _EVAL_NO_REPEAT_NGRAM_SIZE = 3

try:
    _BLEU_EVAL_SAMPLES = int(BLEU_EVAL_SAMPLES)
except (NameError, ValueError, TypeError):
    _BLEU_EVAL_SAMPLES = 500

try:
    _BLEU_SEED = int(BLEU_SEED)
except (NameError, ValueError, TypeError):
    _BLEU_SEED = 42

try:
    _USE_RDROP = bool(USE_RDROP)
except (NameError, TypeError):
    _USE_RDROP = False

assert _USE_RDROP == False, (
    f"[CELL7] _USE_RDROP={_USE_RDROP} but Cell 0 sets USE_RDROP=False. "
    f"Restart the kernel and re-run cells in order (Cell 0 → Cell 7)."
)

try:
    _USE_BACK_TRANSLATION = bool(USE_BACK_TRANSLATION)
except (NameError, TypeError):
    _USE_BACK_TRANSLATION = False

try:
    _LAMBDA_BT = float(LAMBDA_BT)
except (NameError, ValueError, TypeError):
    _LAMBDA_BT = 1.0

try:
    _CHECKPOINT_DIR = str(CHECKPOINT_DIR)
except (NameError, TypeError):
    _CHECKPOINT_DIR = "/kaggle/working"

_BENGALI_PUNCT_SET = set(['।', '॥'])
_COMMON_PUNCT_SET = set(['.', ',', ';', ':', '!', '?', '"', "'", '-', '(', ')', '[', ']', '{', '}', '/', '\\'])
_PUNCT_SET = _BENGALI_PUNCT_SET | _COMMON_PUNCT_SET


def _is_punctuation_only(token: str) -> bool:
    if not token or not isinstance(token, str):
        return False
    clean = (
        token.replace("▁", "")
        .replace("Ġ", "")
        .replace("##", "")
        .replace("</w>", "")
        .strip()
    )
    if not clean:
        return False
    if clean in _BENGALI_PUNCT_SET:
        return True
    if clean in _COMMON_PUNCT_SET:
        return True
    if len(clean) == 1 and not clean.isalnum():
        return True
    return all(c in _PUNCT_SET for c in clean)


def clear_all_gpu_caches():
    gc.collect()
    if not torch.cuda.is_available():
        return
    try:
        for i in range(torch.cuda.device_count()):
            with torch.cuda.device(i):
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass
    except Exception:
        pass


def get_amp_ctx():
    if not _USE_AMP or not torch.cuda.is_available():
        return nullcontext()
    try:
        return torch_amp_autocast(device_type="cuda", dtype=_AMP_DTYPE)
    except Exception:
        return nullcontext()


_PROTOBUF_COMPAT_ERROR_SHOWN = globals().get("PROTOBUF_COMPAT_ERROR_SHOWN", False)


def get_dscd_homographs(model: torch.nn.Module) -> set:
    try:
        core = model.module if hasattr(model, "module") else model
        dscd = getattr(core, "dscd", None)
        if dscd is None:
            return set()
        if hasattr(dscd, "get_discovered_homographs"):
            discovered = dscd.get_discovered_homographs()
            return set(w for w in discovered if not _is_punctuation_only(w))
        homographs = set()
        lock = None
        if hasattr(dscd, "buffer_lock"):
            lock = dscd.buffer_lock
        elif hasattr(dscd, "clustering_lock"):
            lock = dscd.clustering_lock
        stores_snapshot = {}
        if lock:
            with lock:
                stores_snapshot = dict(dscd.prototype_stores.items())
        else:
            stores_snapshot = dict(dscd.prototype_stores.items())
        for token, store in stores_snapshot.items():
            try:
                if store.size > 1:
                    clean_token = str(token).replace("▁", "").replace("Ġ", "").replace("##", "").strip().lower()
                    if clean_token and not _is_punctuation_only(clean_token):
                        homographs.add(clean_token)
            except Exception:
                continue
        return homographs
    except Exception:
        return set()


def log_haug_gate_status(model: torch.nn.Module, global_step: int):
    try:
        core = model.module if hasattr(model, "module") else model
        dscd = getattr(core, "dscd", None)
        if dscd is None or not hasattr(dscd, "get_prototype_summary"):
            print(f"[HAUG] Gate status: UNKNOWN (DSCD unavailable) at step {global_step}")
            return
        summary = dscd.get_prototype_summary()
        n_hom = int(summary.get("num_homographs", 0))
        n_tok = max(int(summary.get("total_tokens", 1)), 1)
        quality = float(summary.get("quality_score", 0.0))
        h_rate = n_hom / n_tok
        print(
            f"[HAUG] Gate: OPEN ✅ | "
            f"quality={quality:.4f} | "
            f"homographs={n_hom} | "
            f"homograph_rate={h_rate * 100:.2f}% | "
            f"step={global_step}"
        )
    except Exception as e:
        print(f"[HAUG] Gate status: ERROR {type(e).__name__} at step {global_step}")


def get_gated_h_aug(model: torch.nn.Module, last_path1_h_aug) -> Optional[torch.Tensor]:
    if last_path1_h_aug is None:
        return None
    try:
        return last_path1_h_aug
    except Exception:
        return None


def compute_test_loss(
    model: torch.nn.Module,
    test_loader: torch.utils.data.DataLoader,
    device: torch.device,
    max_batches: int = 50,
) -> float:
    core_model = model.module if hasattr(model, "module") else model
    was_training = core_model.training
    try:
        core_model.train()
    except Exception:
        pass

    total_loss = 0.0
    count = 0
    with torch.no_grad():
        for batch_idx, batch in enumerate(test_loader):
            if batch_idx >= max_batches:
                break
            if batch is None:
                continue
            try:
                input_ids = batch["input_ids"].to(device, non_blocking=True)
                attention_mask = batch["attention_mask"].to(device, non_blocking=True)
                labels = batch["labels"].to(device, non_blocking=True)
                if input_ids.size(0) == 0:
                    continue

                if hasattr(core_model, "forward_path2"):
                    fwd_out = core_model.forward_path2(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels,
                        src_texts=batch.get("src_text", None),
                        token_word_map=batch.get("token_word_map", None),
                        use_rdrop=False,
                        path1_h_aug=None,
                    )
                else:
                    try:
                        fwd_out = model(
                            input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels,
                            path=2,
                        )
                    except Exception:
                        fwd_out = None

                loss_tensor = None
                if isinstance(fwd_out, torch.Tensor):
                    loss_tensor = fwd_out
                elif isinstance(fwd_out, dict):
                    loss_tensor = fwd_out.get("translation_loss", None)
                    if loss_tensor is None:
                        loss_tensor = fwd_out.get("loss", None)
                    if loss_tensor is None:
                        logits_raw = fwd_out.get("refined_logits", fwd_out.get("logits", None))
                        if logits_raw is not None:
                            import torch.nn.functional as F
                            try:
                                shift_logits = logits_raw[..., :-1, :].contiguous()
                                shift_labels = labels[..., 1:].contiguous()
                                loss_tensor = F.cross_entropy(
                                    shift_logits.view(-1, shift_logits.size(-1)),
                                    shift_labels.view(-1),
                                    ignore_index=-100,
                                )
                            except Exception:
                                loss_tensor = torch.tensor(0.0, device=device)
                        else:
                            loss_tensor = torch.tensor(0.0, device=device)
                elif isinstance(fwd_out, (list, tuple)) and len(fwd_out) > 0:
                    loss_tensor = fwd_out[0]
                else:
                    loss_tensor = torch.tensor(0.0, device=device)

                if isinstance(loss_tensor, torch.Tensor) and torch.isfinite(loss_tensor):
                    total_loss += float(loss_tensor.mean().item())
                    count += 1
            except Exception:
                continue

    try:
        if was_training:
            core_model.train()
        else:
            core_model.eval()
    except Exception:
        pass

    return total_loss / count if count > 0 else 0.0


def compute_bleu_on_val(
    model: torch.nn.Module,
    tokenizer,
    val_dataset: Optional[Dataset],
    source_lang: str,
    target_lang: str,
    max_length: int,
    device: torch.device,
    num_samples: int = 500,
) -> float:
    try:
        import sacrebleu
    except ImportError:
        print("BLEU: sacrebleu not installed, skipping BLEU computation")
        return 0.0
    if val_dataset is None:
        return 0.0
    if not isinstance(val_dataset, Dataset):
        print(
            f"BLEU ERROR: val_dataset is type {type(val_dataset).__name__}, "
            f"expected Dataset. Pass val_dataset=val_dataset (Dataset object), "
            f"NOT the DataLoader. Returning 0.0."
        )
        return 0.0
    total = len(val_dataset)
    if total == 0:
        return 0.0
    indices = list(range(total))
    np.random.seed(_BLEU_SEED)
    np.random.shuffle(indices)
    indices = indices[:min(num_samples, total)]
    hypotheses: List[str] = []
    references: List[str] = []
    core_model = model.module if hasattr(model, "module") else model
    was_training = core_model.training
    core_model.eval()
    try:
        tokenizer.src_lang = source_lang
        tokenizer.tgt_lang = target_lang
    except Exception:
        pass
    en_token_id = (
        getattr(core_model, "en_token_id", None)
        or (tokenizer.lang_code_to_id.get("en_XX") if hasattr(tokenizer, "lang_code_to_id") else None)
        or 250004
    )
    with tqdm(total=len(indices), desc="BLEU Translating", ncols=90, leave=False, position=0, file=sys.stdout) as bleu_bar:
        for idx in indices:
            bleu_bar.update(1)
            try:
                sample = val_dataset[idx]
                if isinstance(sample, dict):
                    src_text = sample.get("src_text", None)
                    ref_text = sample.get("tgt_text", sample.get("target_text", sample.get("translation", None)))
                elif isinstance(sample, (list, tuple)) and len(sample) >= 2:
                    src_text = sample[0]
                    ref_text = sample[1]
                else:
                    continue
                if not src_text or not ref_text:
                    continue
                translation = ""
                try:
                    enc = tokenizer(
                        src_text,
                        return_tensors="pt",
                        padding=True,
                        truncation=True,
                        max_length=max_length,
                    )
                    input_ids_b = enc["input_ids"].to(device)
                    attention_mask_b = enc["attention_mask"].to(device)
                    with torch.no_grad():
                        generated = core_model.mbart.generate(
                            input_ids=input_ids_b,
                            attention_mask=attention_mask_b,
                            forced_bos_token_id=en_token_id,
                            max_new_tokens=128,
                            min_length=_EVAL_MIN_LENGTH,
                            num_beams=_EVAL_NUM_BEAMS,
                            no_repeat_ngram_size=_EVAL_NO_REPEAT_NGRAM_SIZE,
                            early_stopping=True,
                            length_penalty=_EVAL_LENGTH_PENALTY,
                            repetition_penalty=_EVAL_REPETITION_PENALTY,
                        )
                    translation = tokenizer.decode(generated[0], skip_special_tokens=True)
                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        torch.cuda.empty_cache()
                        translation = ""
                except Exception:
                    translation = ""
                if translation and translation.strip() and ref_text and ref_text.strip():
                    hypotheses.append(translation.strip())
                    references.append(ref_text.strip())
                bleu_bar.set_postfix(ok=len(hypotheses), refresh=False)
            except Exception:
                continue
    if was_training:
        core_model.train()
    if not hypotheses:
        print(f"BLEU WARNING: hypotheses list is empty after {len(indices)} samples — check {source_lang}→{target_lang}")
        return 0.0
    try:
        bleu = sacrebleu.corpus_bleu(hypotheses, [references])
        return float(bleu.score)
    except Exception:
        return 0.0


def comprehensive_epoch_validation(
    model: torch.nn.Module,
    tokenizer,
    epoch: int,
    global_step: int,
    source_lang: str,
    target_lang: str,
    max_length: int,
    device: torch.device,
    val_dataset: Optional[Dataset] = None,
    test_loader: Optional[torch.utils.data.DataLoader] = None,
) -> Dict[str, Any]:
    global _PROTOBUF_COMPAT_ERROR_SHOWN
    print("=" * 80)
    print(f"EPOCH {epoch} COMPREHENSIVE VALIDATION (Step {global_step})")
    print("=" * 80)
    core_model = model.module if hasattr(model, "module") else model
    was_training = core_model.training
    if not isinstance(device, torch.device):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log_haug_gate_status(model, global_step)
    dscd_homographs = get_dscd_homographs(model)
    if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
        print(f"[VALIDATION] DSCD discovered homographs: {len(dscd_homographs)}")
        if dscd_homographs:
            print(f"[VALIDATION] Sample: {list(dscd_homographs)[:10]}")
    validation_results: Dict[str, Any] = {
        "epoch": epoch,
        "step": global_step,
        "translations_success": 0,
        "translations_failed": 0,
        "explanations_generated": 0,
        "dscd_homographs_explained": 0,
        "reference_homographs_explained": 0,
        "avg_explanation_confidence": 0.0,
        "dscd_quality_score": 0.0,
        "dscd_multi_sense_tokens": 0,
        "dscd_total_prototypes": 0,
        "asbn_domain_loss": 0.0,
        "asbn_domain_accuracy": 0.0,
        "asbn_source_accuracy": 0.0,
        "asbn_target_accuracy": 0.0,
        "trg_total_explanations": 0,
        "test_loss": 0.0,
        "bleu_200": 0.0,
        "validation_completed": False,
    }
    try:
        if test_loader is not None:
            print("[VALIDATION] Computing test loss...")
            try:
                test_loss_avg = compute_test_loss(
                    model=model,
                    test_loader=test_loader,
                    device=device,
                    max_batches=50,
                )
                validation_results["test_loss"] = test_loss_avg
                print(f"[VALIDATION] Test Loss (avg over ≤50 batches): {test_loss_avg:.4f}")
            except Exception as e:
                print(f"[VALIDATION] Test loss computation failed: {type(e).__name__}: {str(e)[:100]}")
            finally:
                clear_all_gpu_caches()
        else:
            print("[VALIDATION] No test_loader provided, skipping test loss.")

        core_model.eval()

        if val_dataset is not None:
            print(f"[VALIDATION] Computing BLEU on {_BLEU_EVAL_SAMPLES} val samples...")
            try:
                bleu_score = compute_bleu_on_val(
                    model=model,
                    tokenizer=tokenizer,
                    val_dataset=val_dataset,
                    source_lang=source_lang,
                    target_lang=target_lang,
                    max_length=max_length,
                    device=device,
                    num_samples=_BLEU_EVAL_SAMPLES,
                )
                validation_results["bleu_200"] = bleu_score
                print(f"[VALIDATION] BLEU-{_BLEU_EVAL_SAMPLES}: {bleu_score:.2f}")
            except Exception as e:
                print(f"[VALIDATION] BLEU computation failed: {type(e).__name__}: {str(e)[:100]}")
            finally:
                try:
                    dscd_cv = getattr(core_model, "dscd", None)
                    if dscd_cv and hasattr(dscd_cv, "cleanup_memory"):
                        dscd_cv.cleanup_memory()
                except Exception:
                    pass
                clear_all_gpu_caches()
                core_model.eval()
        else:
            print("[VALIDATION] No val_dataset provided, skipping BLEU.")

        val_sentences = []
        if val_dataset is not None and len(val_dataset) > 0:
            total_val = len(val_dataset)
            sample_indices = list(range(total_val))
            np.random.shuffle(sample_indices)
            sample_indices = sample_indices[:min(10, total_val)]
            for si in sample_indices:
                try:
                    sample = val_dataset[si]
                    if isinstance(sample, dict):
                        src = sample.get("src_text", "")
                        ref = sample.get("tgt_text", sample.get("target_text", sample.get("translation", "")))
                    elif isinstance(sample, (list, tuple)) and len(sample) >= 2:
                        src = sample[0]
                        ref = sample[1]
                    else:
                        continue
                    if src and ref:
                        val_sentences.append((str(src).strip(), str(ref).strip(), f"sample_{si}"))
                except Exception:
                    continue

        if not val_sentences:
            val_sentences = [
                ("আমি কলটি বন্ধ করে দিলাম।", "I turned off the tap.", "tap/call"),
                ("আগামীকাল আমি একটি বই কিনব।", "Tomorrow I will buy a book.", "tomorrow/yesterday"),
                ("গতকাল আমি একটি বই কিনেছিলাম।", "Yesterday I bought a book.", "tomorrow/yesterday"),
                ("পাতাটি পড়ে গেছে।", "The leaf has fallen.", "leaf/page"),
                ("সে ব্যাংকে গেল।", "He went to the bank.", "bank/embankment"),
                ("আমি ভালো আছি।", "I am fine.", "No ambiguity"),
                ("সে মিষ্টি করে কথা বলে।", "She speaks sweetly.", "No ambiguity"),
                ("এটি আমার বই।", "This is my book.", "No ambiguity"),
                ("আজ আবহাওয়া ভালো।", "Weather is good today.", "No ambiguity"),
                ("ফলটি সুস্বাদু।", "The fruit is delicious.", "fruit/result"),
            ]

        print(f"[VALIDATION] Testing {len(val_sentences)} samples:")
        print("-" * 80)
        confidences = []
        dscd_homograph_words_detected = set()
        reference_homograph_words_detected = set()
        try:
            try:
                tokenizer.src_lang = source_lang
                tokenizer.tgt_lang = target_lang
                if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                    actual_src = getattr(tokenizer, "src_lang", "NOT_SET")
                    actual_tgt = getattr(tokenizer, "tgt_lang", "NOT_SET")
                    print(f"[VALIDATION] Tokenizer languages set: src={actual_src}, tgt={actual_tgt}")
            except Exception as e:
                print(f"[VALIDATION] Warning: Could not set tokenizer languages: {type(e).__name__}")
            for idx, (src, expected, note) in enumerate(val_sentences, 1):
                translation = ""
                explanation_status = ""
                error_detail = ""
                try:
                    if "translate_with_explanations" in globals():
                        try:
                            res = translate_with_explanations(
                                model, tokenizer, src,
                                source_lang=source_lang,
                                target_lang=target_lang,
                                device=device,
                                max_length=max_length,
                            )
                            translation = str(res.get("translation", ""))
                            if translation == "ERROR_DURING_TRANSLATION":
                                translation = ""
                            exps = res.get("explanations", [])
                            error_info = res.get("error", "")
                            if translation and translation.strip():
                                validation_results["translations_success"] += 1
                            else:
                                validation_results["translations_failed"] += 1
                                if error_info:
                                    error_detail = f" {error_info}"
                                else:
                                    error_detail = " (empty result)"
                            validation_results["explanations_generated"] += len(exps)
                            if exps:
                                explanation_status = f"[{len(exps)} expl]"
                                for exp in exps:
                                    try:
                                        conf = exp.get("confidence", 0.5)
                                        confidences.append(float(conf))
                                        word = exp.get("ambiguous_word", "").strip()
                                        clean_word = word.replace("▁", "").replace("Ġ", "").strip()
                                        if clean_word and not _is_punctuation_only(clean_word):
                                            if clean_word.lower() in dscd_homographs:
                                                validation_results["dscd_homographs_explained"] += 1
                                                dscd_homograph_words_detected.add(clean_word)
                                            if clean_word.lower() in _HOMOGRAPH_REFERENCE_LIST:
                                                validation_results["reference_homographs_explained"] += 1
                                                reference_homograph_words_detected.add(clean_word)
                                    except Exception:
                                        pass
                            else:
                                explanation_status = "[no expl]"
                        except Exception as e:
                            explanation_status = f"[error: {type(e).__name__}]"
                            error_detail = f" {str(e)[:50]}"
                            translation = ""
                            validation_results["translations_failed"] += 1
                    else:
                        explanation_status = "[unavailable]"
                        error_detail = " (function not found)"
                        validation_results["translations_failed"] += 1
                    if translation and translation.strip():
                        print(f"   {idx:2d}. {explanation_status}")
                        print(f"       SRC : {src[:120]}")
                        print(f"       HYP : {translation[:120]}")
                        print(f"       REF : {expected[:120]}")
                    else:
                        print(f"   {idx:2d}. Translation FAILED ({note}){error_detail}")
                        print(f"       SRC : {src[:120]}")
                        print(f"       REF : {expected[:120]}")
                    if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                        print(f"   DEBUG: Bengali input was: {src[:50]}")
                except Exception as e:
                    validation_results["translations_failed"] += 1
                    print(f"   {idx:2d}. ERROR ({note}) - {type(e).__name__}")
                    print(f"       SRC : {src[:120]}")
                    if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                        try:
                            traceback.print_exc()
                        except Exception:
                            pass
                finally:
                    if torch.cuda.is_available():
                        try:
                            torch.cuda.synchronize()
                        except Exception:
                            pass
                    clear_all_gpu_caches()
        except Exception:
            pass

        print("-" * 80)
        print("[VALIDATION] DSCD Prototype Quality Check:")
        try:
            dscd = core_model.dscd if hasattr(core_model, "dscd") else None
            if dscd and hasattr(dscd, "validate_prototypes"):
                lock = None
                if hasattr(dscd, "buffer_lock"):
                    lock = dscd.buffer_lock
                elif hasattr(dscd, "clustering_lock"):
                    lock = dscd.clustering_lock
                if lock:
                    with lock:
                        quality_results = dscd.validate_prototypes(cluster_missing=False)
                else:
                    quality_results = dscd.validate_prototypes(cluster_missing=False)
                validation_results["dscd_quality_score"] = quality_results.get("quality_score", 0.0)
                validation_results["dscd_multi_sense_tokens"] = quality_results.get("multi_sense_tokens", 0)
                validation_results["dscd_total_prototypes"] = quality_results.get("total_prototypes", 0)
                print(f"  - Quality Score: {validation_results['dscd_quality_score']:.1%}")
                print(f"  - Multi-sense tokens: {validation_results['dscd_multi_sense_tokens']}")
                print(f"  - Total prototypes: {validation_results['dscd_total_prototypes']}")
            else:
                print("  - Validation not available")
        except Exception as e:
            print(f"  - Validation failed: {type(e).__name__}")

        print("-" * 80)
        print("[VALIDATION] DSCD Sense Registry (Top Multi-Sense Tokens):")
        try:
            dscd_sr = core_model.dscd if hasattr(core_model, "dscd") else None
            if dscd_sr is not None:
                sense_registry_data = {}
                if hasattr(dscd_sr, "sense_registry"):
                    sense_registry_data = dict(dscd_sr.sense_registry)
                elif hasattr(dscd_sr, "prototype_stores"):
                    lock_sr = None
                    if hasattr(dscd_sr, "buffer_lock"):
                        lock_sr = dscd_sr.buffer_lock
                    elif hasattr(dscd_sr, "clustering_lock"):
                        lock_sr = dscd_sr.clustering_lock
                    if lock_sr:
                        with lock_sr:
                            stores_sr = dict(dscd_sr.prototype_stores.items())
                    else:
                        stores_sr = dict(dscd_sr.prototype_stores.items())
                    for tok, store in stores_sr.items():
                        try:
                            n_senses = store.size if hasattr(store, "size") else len(getattr(store, "centroids", []))
                            if n_senses > 1:
                                clean_tok = str(tok).replace("▁", "").replace("Ġ", "").replace("##", "").strip()
                                if clean_tok and not _is_punctuation_only(clean_tok):
                                    sense_registry_data[clean_tok] = n_senses
                        except Exception:
                            continue
                if sense_registry_data:
                    sorted_senses = sorted(sense_registry_data.items(), key=lambda x: x[1], reverse=True)
                    print(f"  - Total multi-sense tokens in registry: {len(sorted_senses)}")
                    print(f"  {'Token':<20} {'Senses':<8}")
                    print(f"  {'-'*20} {'-'*8}")
                    for tok_display, n_s in sorted_senses[:15]:
                        in_ref = "✓" if tok_display.lower() in _HOMOGRAPH_REFERENCE_LIST else ""
                        print(f"  {tok_display:<20} {n_s:<8} {in_ref}")
                else:
                    print("  - No multi-sense tokens found in registry yet")
            else:
                print("  - DSCD not available, sense registry display skipped")
        except Exception as e:
            print(f"  - Sense registry display failed: {type(e).__name__}")

        print("-" * 80)
        print("[VALIDATION] ASBN Training Statistics:")
        try:
            asbn = core_model.asbn if hasattr(core_model, "asbn") else None
            if asbn and hasattr(asbn, "get_detailed_stats"):
                asbn_stats = asbn.get_detailed_stats()
                validation_results["asbn_domain_loss"] = asbn_stats.get("domain_loss", 0.0)
                validation_results["asbn_domain_accuracy"] = asbn_stats.get("domain_accuracy", 0.0)
                validation_results["asbn_source_accuracy"] = asbn_stats.get("source_accuracy", 0.0)
                validation_results["asbn_target_accuracy"] = asbn_stats.get("target_accuracy", 0.0)
                print(f"  - Domain Loss: {validation_results['asbn_domain_loss']:.4f}")
                print(f"  - Domain Accuracy: {validation_results['asbn_domain_accuracy']:.2%}")
                print(f"  - Source Accuracy: {validation_results['asbn_source_accuracy']:.2%}")
                print(f"  - Target Accuracy: {validation_results['asbn_target_accuracy']:.2%}")
            elif asbn and hasattr(asbn, "get_asbn_stats"):
                asbn_stats = asbn.get_asbn_stats()
                validation_results["asbn_domain_loss"] = asbn_stats.get("domain_loss", 0.0)
                validation_results["asbn_domain_accuracy"] = asbn_stats.get("domain_accuracy", 0.0)
                print(f"  - Domain Loss: {validation_results['asbn_domain_loss']:.4f}")
                print(f"  - Domain Accuracy: {validation_results['asbn_domain_accuracy']:.2%}")
            else:
                print("  - ASBN statistics not available")
        except Exception as e:
            print(f"  - ASBN stats retrieval failed: {type(e).__name__}")

        print("-" * 80)
        print("[VALIDATION] TRG Explanation Statistics:")
        try:
            trg = core_model.trg if hasattr(core_model, "trg") else None
            if trg and hasattr(trg, "get_statistics"):
                trg_stats = trg.get_statistics()
                validation_results["trg_total_explanations"] = trg_stats.get("explanations_generated", 0)
                print(f"  - Total explanations: {validation_results['trg_total_explanations']}")
                print(f"  - High confidence rate: {trg_stats.get('high_confidence_rate', 0):.1%}")
                print(f"  - DSCD homograph rate: {trg_stats.get('dscd_homograph_rate', 0):.1%}")
            else:
                print("  - TRG statistics not available")
        except Exception as e:
            print(f"  - TRG stats retrieval failed: {type(e).__name__}")

        if confidences:
            validation_results["avg_explanation_confidence"] = sum(confidences) / len(confidences)

        print("-" * 80)
        print("[VALIDATION] Summary:")
        print(f"  - Translations: {validation_results['translations_success']}/{len(val_sentences)} successful")
        print(f"  - Explanations generated: {validation_results['explanations_generated']}")
        print(f"  - Avg explanation confidence: {validation_results['avg_explanation_confidence']:.3f}")
        print(f"  - DSCD homographs explained: {validation_results['dscd_homographs_explained']}")
        print(f"  - Reference homographs explained: {validation_results['reference_homographs_explained']}")
        if dscd_homograph_words_detected:
            print(f"  - DSCD homographs detected: {', '.join(sorted(dscd_homograph_words_detected))}")
        print(f"  - DSCD Quality Score: {validation_results['dscd_quality_score']:.1%}")
        print(f"  - Multi-sense tokens: {validation_results['dscd_multi_sense_tokens']}")
        print(f"  - ASBN Domain Accuracy: {validation_results['asbn_domain_accuracy']:.2%}")
        print(f"  - Test Loss: {validation_results['test_loss']:.4f}")
        print(f"  - BLEU-{_BLEU_EVAL_SAMPLES}: {validation_results['bleu_200']:.2f}")

        warnings_list = []
        if validation_results["translations_failed"] > len(val_sentences) // 2:
            warnings_list.append("High translation failure rate")
        if validation_results["explanations_generated"] == 0:
            warnings_list.append("No explanations generated")
        if validation_results["dscd_quality_score"] < 0.3:
            warnings_list.append("Low DSCD quality score")
        if validation_results["dscd_multi_sense_tokens"] < 10:
            warnings_list.append("Very few multi-sense tokens")
        if warnings_list:
            print("[VALIDATION] Health Warnings:")
            for w in warnings_list:
                print(f"  - {w}")
        else:
            print("[VALIDATION] All systems healthy")

        validation_results["validation_completed"] = True
    except Exception as e:
        print(f"[VALIDATION] Critical error: {type(e).__name__}: {str(e)[:200]}")
        if _VERBOSE_LOGGING or _DEBUG_DISCOVERY:
            try:
                traceback.print_exc()
            except Exception:
                pass
        validation_results["validation_completed"] = False
    finally:
        if was_training:
            core_model.train()
        clear_all_gpu_caches()
    print("=" * 80)
    return validation_results


def print_gpu_mem(prefix: str):
    if not torch.cuda.is_available():
        return
    try:
        lines = [f"[{prefix}] GPU mem (GB):"]
        for i in range(torch.cuda.device_count()):
            try:
                alloc = torch.cuda.memory_allocated(i) / 1024**3
                resv = torch.cuda.memory_reserved(i) / 1024**3
                lines.append(f"  GPU {i}: alloc={alloc:.2f} resv={resv:.2f}")
            except Exception:
                lines.append(f"  GPU {i}: mem query failed")
        print("\n".join(lines))
    except Exception:
        pass


def get_cluster_count(model: torch.nn.Module) -> int:
    try:
        core = model
        while hasattr(core, "module"):
            core = core.module
        dscd = getattr(core, "dscd", None)
        if dscd is None:
            return 0
        stores = getattr(dscd, "prototype_stores", None)
        if stores is None:
            return 0
        lock = None
        if hasattr(dscd, "buffer_lock"):
            lock = dscd.buffer_lock
        elif hasattr(dscd, "clustering_lock"):
            lock = dscd.clustering_lock
        if lock:
            with lock:
                return len(stores)
        else:
            return len(stores)
    except Exception:
        return 0


def get_dscd_safe(model: torch.nn.Module):
    try:
        core = model
        while hasattr(core, "module"):
            core = core.module
        return getattr(core, "dscd", None)
    except Exception:
        return None


def print_top_clusters(model: torch.nn.Module, topn: int = 5):
    dscd = get_dscd_safe(model)
    if dscd is None:
        return
    try:
        print("[CLUSTER] Top 5 clusters")
        print("-" * 90)
        print(f"{'Rank':<6}{'Token':<15}{'Count':<12}{'Protos':<10}{'Mu':<15}{'Tau':<12}")
        print("-" * 90)
        items = []
        lock = None
        if hasattr(dscd, "buffer_lock"):
            lock = dscd.buffer_lock
        elif hasattr(dscd, "clustering_lock"):
            lock = dscd.clustering_lock
        if lock:
            with lock:
                stores_snapshot = list(dscd.prototype_stores.items())
        else:
            stores_snapshot = list(dscd.prototype_stores.items())
        for token, store in stores_snapshot:
            try:
                total_count = sum(getattr(store, "counts", []) or [])
                protos = store.size if hasattr(store, "size") else len(getattr(store, "centroids", []))
                clean_token = str(token).replace("▁", "").replace("Ġ", "").strip().lower()
                if _is_punctuation_only(clean_token):
                    continue
                mu = getattr(store, "mu", 0.0)
                tau = getattr(store, "tau", 0.0)
                items.append((token, total_count, protos, mu, tau))
            except Exception:
                continue
        items.sort(key=lambda x: x[1], reverse=True)
        for i, (tok, cnt, prot, mu, tau) in enumerate(items[:topn], 1):
            token_display = str(tok)[:12]
            print(f"{i:<6}{token_display:<15}{cnt:<12}{prot:<10}{mu:<15.6f}{tau:<12.6f}")
        print("-" * 90)
    except Exception as e:
        if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
            print(f"[CLUSTER-DBG] print_top_clusters error: {type(e).__name__}")


def check_discovery_status(model: torch.nn.Module, global_step: int):
    try:
        core = model
        while hasattr(core, "module"):
            core = core.module
        dscd = getattr(core, "dscd", None)
        if dscd is None:
            return
        if hasattr(dscd, "get_prototype_summary"):
            summary = dscd.get_prototype_summary()
            if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                print(f"[DISCOVERY-STATUS] Step {global_step}")
                print(f"  - Total tokens: {summary.get('total_tokens', 0)}")
                print(f"  - Homographs: {summary.get('num_homographs', 0)}")
                print(f"  - Total prototypes: {summary.get('total_prototypes', 0)}")
                print(f"  - Quality score: {summary.get('quality_score', 0.0):.1%}")
        if hasattr(dscd, "discovered_log") and dscd.discovered_log:
            total_discovered = len(dscd.discovered_log)
            if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                print(f"[DISCOVERY-STATUS] Discovery events: {total_discovered}")
                recent = dscd.discovered_log[-3:] if len(dscd.discovered_log) > 3 else dscd.discovered_log
                for entry in recent:
                    discovered = entry.get("discovered", 0)
                    candidates = entry.get("candidates", 0)
                    print(f"  - {discovered}/{candidates} homographs discovered")
        else:
            if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                print(f"[DISCOVERY-STATUS] No discoveries yet at step {global_step}")
    except Exception as e:
        if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
            print(f"[DISCOVERY-STATUS] Error: {e}")


def print_path_loss_summary(training_stats: Dict[str, Any], validate_every: int, global_step: int, use_dual_path: bool):
    print("=" * 80)
    print(f"LOSS SUMMARY AT STEP {global_step}")
    print("=" * 80)
    lookback_window = min(validate_every, len(training_stats["path1_losses"]), len(training_stats["path2_losses"]))
    if use_dual_path and lookback_window > 0:
        recent_p1_fwd = training_stats["path1_losses"][-lookback_window:] if training_stats["path1_losses"] else []
        recent_p2_fwd = training_stats["path2_losses"][-lookback_window:] if training_stats["path2_losses"] else []
        recent_bwd = training_stats["backward_losses"][-lookback_window:] if training_stats["backward_losses"] else []
        p1_fwd_avg = float(np.mean(recent_p1_fwd)) if recent_p1_fwd else 0.0
        p2_fwd_avg = float(np.mean(recent_p2_fwd)) if recent_p2_fwd else 0.0
        bwd_avg = float(np.mean(recent_bwd)) if recent_bwd else 0.0
        p1_count = training_stats["path1_batches"]
        p2_count = training_stats["path2_batches"]
        print(f"\nPATH 1 (DSCD + Contrastive, weight=0.1):")
        print(f"  - Forward Loss:  {p1_fwd_avg:.4f}")
        print(f"  - Backward Loss: {bwd_avg:.4f}")
        print(f"  - Total Batches: {p1_count}")
        print(f"\nPATH 2 (Translation Subword):")
        print(f"  - Forward Loss:  {p2_fwd_avg:.4f}")
        print(f"  - Backward Loss: {bwd_avg:.4f}")
        print(f"  - Total Batches: {p2_count}")
        print(f"\nCOMBINED:")
        print(f"  - Total Batches: {p1_count + p2_count}")
        print(f"  - Optimizer Updates: {training_stats['optimizer_updates']}")
    else:
        recent_fwd = training_stats["total_loss"][-lookback_window:] if training_stats["total_loss"] else []
        recent_bwd = training_stats["backward_losses"][-lookback_window:] if training_stats["backward_losses"] else []
        fwd_avg = float(np.mean(recent_fwd)) if recent_fwd else 0.0
        bwd_avg = float(np.mean(recent_bwd)) if recent_bwd else 0.0
        print(f"\nPATH MODE: Path 2 Only")
        print(f"  - Forward Loss:  {fwd_avg:.4f}")
        print(f"  - Backward Loss: {bwd_avg:.4f}")
        print(f"  - Total Batches: {training_stats['batches_processed']}")
        print(f"  - Optimizer Updates: {training_stats['optimizer_updates']}")
    print("=" * 80)


def _build_epoch_checkpoint_dscd_state(core_model: torch.nn.Module) -> dict:
    proto_state = {}
    try:
        if hasattr(core_model, "dscd"):
            dscd_final = core_model.dscd
            stores_final = getattr(dscd_final, "prototype_stores", {})
            lock_final = None
            if hasattr(dscd_final, "buffer_lock"):
                lock_final = dscd_final.buffer_lock
            elif hasattr(dscd_final, "clustering_lock"):
                lock_final = dscd_final.clustering_lock
            if lock_final:
                with lock_final:
                    stores_snapshot_final = dict(stores_final.items())
            else:
                stores_snapshot_final = dict(stores_final.items())
            for word, store in stores_snapshot_final.items():
                try:
                    if store.size > 0:
                        proto_state[word] = store.state_dict()
                except Exception:
                    continue
    except Exception as e:
        print(f"[CHECKPOINT] Prototype state build failed: {e}")
    return proto_state


def train_memory_efficient_tatn(
    model: torch.nn.Module,
    tokenizer,
    train_loader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    phi_optimizer: Optional[torch.optim.Optimizer] = None,
    epochs: Optional[int] = None,
    accumulation_steps: Optional[int] = None,
    validate_every: Optional[int] = None,
    enable_validation: bool = True,
    use_dual_path: bool = None,
    test_loader: Optional[torch.utils.data.DataLoader] = None,
    val_dataset: Optional[Dataset] = None,
) -> torch.nn.Module:

    if epochs is None:
        epochs = _EPOCHS
    if accumulation_steps is None:
        accumulation_steps = _ACCUMULATION_STEPS
    if validate_every is None:
        validate_every = _VALIDATION_CHECK_INTERVAL
    if use_dual_path is None:
        use_dual_path = _USE_DUAL_PATH_TRAINING

    if val_dataset is not None and not isinstance(val_dataset, Dataset):
        print(
            f"[TRAIN] ERROR: val_dataset is type {type(val_dataset).__name__}, "
            f"expected Dataset. Pass val_dataset=val_dataset (Dataset object), "
            f"NOT val_loader (DataLoader). BLEU computation will be skipped."
        )
        val_dataset = None

    print(f"[TRAIN] Starting training: epochs={epochs}, batch={_BATCH_SIZE}, accum_steps={accumulation_steps}")
    print(f"[TRAIN] Validation: {'enabled' if enable_validation and validate_every > 0 else 'disabled'}")
    print(f"[TRAIN] DP enabled: {_USE_MULTI_GPU}, GPUs: {_NUM_GPUS}, Device: {_DEVICE}")
    print(f"[TRAIN] Discovery frequency: {_PERIODIC_DISCOVERY_FREQUENCY} steps")
    print(f"[TRAIN] Dual-path training: {'ENABLED' if use_dual_path else 'DISABLED'}")
    print(f"[TRAIN] test_loader: {'provided' if test_loader is not None else 'not provided'}")
    print(f"[TRAIN] val_dataset: {'provided (size=' + str(len(val_dataset)) + ')' if val_dataset is not None else 'not provided'}")
    print(f"[TRAIN] Checkpoint dir: {_CHECKPOINT_DIR}")
    print(f"[TRAIN] Back-translation: {'ENABLED' if _USE_BACK_TRANSLATION else 'DISABLED'}, LAMBDA_BT={_LAMBDA_BT}")
    print(f"[TRAIN] AMP dtype: {_AMP_DTYPE}, BF16: {_USE_BF16}, GradScaler: {'DISABLED (BF16)' if _USE_BF16 else 'ENABLED'}")

    # FIX-C7-BT-1: Check bt_train_pairs (correct variable name from Cell 6.5)
    if _USE_BACK_TRANSLATION:
        _bt_check = globals().get("bt_train_pairs", None)
        if _bt_check and len(_bt_check) > 0:
            print(f"[TRAIN] ✅ bt_train_pairs found: {len(_bt_check):,} BT pairs ready")
        else:
            print("[TRAIN] ⚠️ USE_BACK_TRANSLATION=True but bt_train_pairs not found or empty")
            print("[TRAIN]    → Run Cell 6.5 first to generate bt_train_pairs")

    if "translate_with_explanations" not in globals():
        print("[TRAIN] WARNING: translate_with_explanations not found in globals!")
        print("[TRAIN] Validation will fail. Please ensure Cell 8 is executed before training.")

    model.train()
    clear_all_gpu_caches()

    scaler = GradScaler(enabled=_USE_AMP and torch.cuda.is_available() and not _USE_BF16)

    scheduler = None
    if _USE_LR_SCHEDULER:
        try:
            from transformers import get_cosine_schedule_with_warmup
            total_optimizer_steps = len(train_loader) * epochs // max(1, accumulation_steps)
            warmup_steps = _WARMUP_STEPS
            scheduler = get_cosine_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_optimizer_steps,
            )
            print(f"[TRAIN] ✅ Learning rate scheduler created:")
            print(f"[TRAIN]    - Type: Cosine with warmup")
            print(f"[TRAIN]    - Total optimizer steps: {total_optimizer_steps}")
            print(f"[TRAIN]    - Warmup steps (optimizer): {warmup_steps}")
            print(f"[TRAIN]    - Initial LR: {optimizer.param_groups[0]['lr']:.2e}")
            print(f"[TRAIN]    - Expected: Train loss will converge to <1.5 (from current ~2.4)")
            print(f"[TRAIN]    - Expected BLEU gain: +10-15 points")
        except ImportError:
            print("[TRAIN] WARNING: transformers not available, cannot create scheduler")
            print("[TRAIN] Training will proceed without LR scheduling (lower BLEU expected)")
            scheduler = None
        except Exception as e:
            print(f"[TRAIN] WARNING: Scheduler creation failed: {type(e).__name__}")
            print("[TRAIN] Training will proceed without LR scheduling")
            scheduler = None
    else:
        print("[TRAIN] Learning rate scheduler disabled (USE_LR_SCHEDULER=False)")

    global_step = 0
    accumulated_steps = 0
    pending_validation = False
    last_path1_h_aug = None
    _warmup_discovery_done = False

    training_stats: Dict[str, Any] = {
        "total_loss": [],
        "epoch_losses": [],
        "backward_losses": [],
        "batches_processed": 0,
        "optimizer_updates": 0,
        "skipped_batches": 0,
        "oom_errors": 0,
        "runtime_errors": 0,
        "exceptions": 0,
        "epoch_validations": [],
        "dscd_quality_history": [],
        "multi_sense_ratio_history": [],
        "asbn_domain_accuracy_history": [],
        "asbn_domain_loss_history": [],
        "trg_explanation_history": [],
        "test_loss_history": [],
        "bleu_200_history": [],
        "discovery_runs": 0,
        "discovery_homographs_found": 0,
        "learning_rates": [],
        "path1_batches": 0,
        "path2_batches": 0,
        "path1_losses": [],
        "path2_losses": [],
        "last_forward_loss": 0.0,
        "bt_batches": 0,
        "bt_losses": [],
        "last_backward_loss": 0.0,
    }

    last_backward_loss = 0.0
    cached_cluster_count = 0
    last_test_loss = 0.0
    last_bleu_score = 0.0

    _bt_pairs = globals().get("bt_train_pairs", None)
    _bt_iter = iter(_bt_pairs) if (_USE_BACK_TRANSLATION and _bt_pairs and len(_bt_pairs) > 0) else None

    for epoch in range(1, epochs + 1):
        epoch_start = time.time()
        epoch_losses: List[float] = []
        skip_reasons = defaultdict(int)

        print("=" * 80)
        print(f"EPOCH {epoch}/{epochs} STARTED")
        print("=" * 80)

        try:
            core = model.module if hasattr(model, "module") else model
            trg = getattr(core, "trg", None)
            if trg and hasattr(trg, "reset_statistics"):
                try:
                    trg.reset_statistics()
                    print(f"[TRAIN] TRG statistics reset for epoch {epoch}")
                except Exception:
                    pass
        except Exception as e:
            if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                print(f"[TRAIN] TRG stats reset failed: {e}")

        try:
            core = model.module if hasattr(model, "module") else model
            asbn = getattr(core, "asbn", None)
            if asbn and hasattr(asbn, "reset_stats"):
                try:
                    asbn.reset_stats()
                    print(f"[TRAIN] ASBN statistics reset for epoch {epoch}")
                except Exception:
                    pass
        except Exception as e:
            if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                print(f"[TRAIN] ASBN stats reset failed: {e}")

        try:
            optimizer.zero_grad(set_to_none=True)
        except Exception:
            pass

        progress = None
        batch_idx = 0
        try:
            progress = tqdm(
                total=len(train_loader),
                desc=f"Epoch {epoch}/{epochs}",
                ncols=120,
                leave=False,
                position=0,
                file=sys.stdout,
            )
            for batch in train_loader:
                batch_idx += 1
                global_step += 1
                training_stats["batches_processed"] += 1

                if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                    if global_step % DEBUG_PRINT_INTERVAL == 0:
                        print(f"[TRAIN-DEBUG] Epoch {epoch} Batch {batch_idx} GlobalStep {global_step}")
                        check_discovery_status(model, global_step)

                if _PERIODIC_DISCOVERY_FREQUENCY and _PERIODIC_DISCOVERY_FREQUENCY > 0:
                    if global_step % _PERIODIC_DISCOVERY_FREQUENCY == 0:
                        try:
                            core = model.module if hasattr(model, "module") else model
                            dscd = getattr(core, "dscd", None)
                            if dscd and hasattr(dscd, "periodic_discovery_check"):
                                print(f"\n[DISCOVERY] Running periodic check at step {global_step}...")
                                num_discovered = dscd.periodic_discovery_check(
                                    global_step=global_step,
                                    frequency=_PERIODIC_DISCOVERY_FREQUENCY,
                                    cluster_missing=False,
                                )
                                training_stats["discovery_runs"] += 1
                                training_stats["discovery_homographs_found"] = num_discovered
                                if num_discovered > 0:
                                    print(f"[DISCOVERY] Found {num_discovered} new homographs!")
                                else:
                                    print("[DISCOVERY] No new homographs found this check")
                                cached_cluster_count = get_cluster_count(model)
                        except Exception as e:
                            if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                                print(f"[DISCOVERY] Periodic check failed: {type(e).__name__}: {str(e)[:200]}")

                if (
                    _WARMUP_STEPS > 0
                    and global_step == _WARMUP_STEPS
                    and not _warmup_discovery_done
                ):
                    try:
                        core = model.module if hasattr(model, "module") else model
                        dscd = getattr(core, "dscd", None)
                        if dscd and hasattr(dscd, "periodic_discovery_check"):
                            print(f"[DISCOVERY] Post-warmup forced discovery at step {global_step}...")
                            num_discovered = dscd.periodic_discovery_check(
                                global_step=global_step,
                                frequency=_PERIODIC_DISCOVERY_FREQUENCY,
                                cluster_missing=False,
                            )
                            training_stats["discovery_runs"] += 1
                            training_stats["discovery_homographs_found"] = num_discovered
                            if num_discovered > 0:
                                print(f"[DISCOVERY] Post-warmup: Found {num_discovered} new homographs!")
                            else:
                                print("[DISCOVERY] Post-warmup: No new homographs at warmup boundary")
                            cached_cluster_count = get_cluster_count(model)
                        _warmup_discovery_done = True
                    except Exception as e:
                        if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                            print(f"[DISCOVERY] Post-warmup forced discovery failed: {type(e).__name__}: {str(e)[:200]}")
                        _warmup_discovery_done = True

                if enable_validation and validate_every and validate_every > 0 and global_step % validate_every == 0:
                    if accumulated_steps > 0:
                        try:
                            optimizer.zero_grad(set_to_none=True)
                        except Exception:
                            pass
                    print_path_loss_summary(training_stats, validate_every, global_step, use_dual_path)
                    val_result = comprehensive_epoch_validation(
                        model=model,
                        tokenizer=tokenizer,
                        epoch=epoch,
                        global_step=global_step,
                        source_lang=_SOURCE_LANGUAGE,
                        target_lang=_TARGET_LANGUAGE,
                        max_length=_MAX_LENGTH,
                        device=_DEVICE,
                        val_dataset=val_dataset,
                        test_loader=test_loader,
                    )
                    try:
                        core = model.module if hasattr(model, "module") else model
                        dscd = getattr(core, "dscd", None)
                        if dscd and hasattr(dscd, "cleanup_memory"):
                            print("[VALIDATION] Running DSCD cleanup after validation...")
                            dscd.cleanup_memory()
                    except Exception as e:
                        if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                            print(f"[VALIDATION] DSCD cleanup failed: {type(e).__name__}")
                    if val_result:
                        training_stats["epoch_validations"].append(val_result)
                        training_stats["test_loss_history"].append(val_result.get("test_loss", 0.0))
                        training_stats["bleu_200_history"].append(val_result.get("bleu_200", 0.0))
                        last_test_loss = val_result.get("test_loss", last_test_loss)
                        last_bleu_score = val_result.get("bleu_200", last_bleu_score)
                        cached_cluster_count = get_cluster_count(model)
                    else:
                        pending_validation = True

                if batch is None:
                    training_stats["skipped_batches"] += 1
                    skip_reasons["batch_none"] += 1
                    progress.update(1)
                    continue

                try:
                    input_ids = batch["input_ids"]
                    attention_mask = batch["attention_mask"]
                    labels = batch["labels"]
                    batch_size = int(input_ids.size(0))
                    domain_labels = batch.get("domain_labels", None)
                    if domain_labels is not None:
                        if not isinstance(domain_labels, torch.Tensor):
                            domain_labels = None
                        elif domain_labels.dim() == 0:
                            domain_labels = domain_labels.unsqueeze(0)
                    if domain_labels is None:
                        domain_labels = torch.full(
                            (batch_size,), _TRAIN_DOMAIN, dtype=torch.long, device=torch.device("cpu")
                        )
                    if _USE_MULTI_GPU and _NUM_GPUS > 0:
                        keep = (batch_size // _NUM_GPUS) * _NUM_GPUS
                        if keep == 0:
                            training_stats["skipped_batches"] += 1
                            skip_reasons["dp_keep_zero"] += 1
                            progress.update(1)
                            continue
                        if keep != batch_size:
                            input_ids = input_ids[:keep]
                            attention_mask = attention_mask[:keep]
                            labels = labels[:keep]
                            domain_labels = domain_labels[:keep]
                            batch_size = keep

                    input_ids = input_ids.to(_DEVICE, non_blocking=True)
                    attention_mask = attention_mask.to(_DEVICE, non_blocking=True)
                    labels = labels.to(_DEVICE, non_blocking=True)
                    domain_labels = domain_labels.to(_DEVICE, non_blocking=True)

                    if input_ids.size(0) == 0:
                        training_stats["skipped_batches"] += 1
                        skip_reasons["empty_batch"] += 1
                        progress.update(1)
                        continue

                    if use_dual_path:
                        selected_path = 1 if (batch_idx % 2 == 1) else 2
                    else:
                        selected_path = 2

                    if selected_path == 1:
                        training_stats["path1_batches"] += 1
                        forward_kwargs = {
                            "input_ids": input_ids,
                            "attention_mask": attention_mask,
                            "src_texts": batch.get("src_text", None),
                            "token_word_map": batch.get("token_word_map", None),
                            "domain_labels": domain_labels,
                        }
                        amp_ctx = get_amp_ctx()
                        with amp_ctx:
                            try:
                                core = model.module if hasattr(model, "module") else model
                                if hasattr(core, "forward_path1"):
                                    forward_out = core.forward_path1(**forward_kwargs)
                                else:
                                    forward_kwargs["labels"] = None
                                    forward_kwargs["path"] = 1
                                    forward_out = model(**forward_kwargs)
                            except Exception:
                                forward_kwargs["labels"] = None
                                forward_kwargs["path"] = 1
                                forward_out = model(**forward_kwargs)

                        if isinstance(forward_out, torch.Tensor):
                            loss_tensor = forward_out
                        elif isinstance(forward_out, dict):
                            if "loss" in forward_out:
                                loss_tensor = forward_out["loss"]
                            elif "asbn_loss" in forward_out:
                                loss_tensor = forward_out["asbn_loss"]
                            else:
                                raise RuntimeError("Path 1 forward returned dict without 'loss' or 'asbn_loss'")
                        elif isinstance(forward_out, (list, tuple)) and len(forward_out) > 0 and isinstance(forward_out[0], torch.Tensor):
                            loss_tensor = forward_out[0]
                        else:
                            raise RuntimeError("Path 1 forward did not return a recognizable loss tensor")

                        if not isinstance(loss_tensor, torch.Tensor):
                            loss_tensor = torch.tensor(float(loss_tensor), device=_DEVICE)
                        else:
                            loss_tensor = loss_tensor.to(_DEVICE)
                        if loss_tensor.numel() > 1:
                            loss_tensor = loss_tensor.mean()

                        contrastive_loss = torch.tensor(0.0, device=_DEVICE)
                        try:
                            core = model.module if hasattr(model, "module") else model
                            dscd = getattr(core, "dscd", None)
                            if dscd is not None and hasattr(dscd, "compute_dscd_contrastive_loss"):
                                cl = dscd.compute_dscd_contrastive_loss()
                                if isinstance(cl, torch.Tensor) and torch.isfinite(cl):
                                    contrastive_loss = cl.to(_DEVICE)
                        except Exception as e:
                            if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                                print(f"[PATH1] contrastive_loss computation failed: {type(e).__name__}, skipping")
                            contrastive_loss = torch.tensor(0.0, device=_DEVICE)

                        p1_total = loss_tensor + 0.1 * contrastive_loss
                        if p1_total.numel() > 1:
                            loss_val = float(p1_total.mean().item())
                            p1_total = p1_total.mean()
                        else:
                            loss_val = float(p1_total.item())
                        training_stats["last_forward_loss"] = loss_val
                        epoch_losses.append(loss_val)
                        training_stats["total_loss"].append(loss_val)
                        training_stats["path1_losses"].append(loss_val)
                        loss_tensor = p1_total

                        try:
                            core = model.module if hasattr(model, "module") else model
                            cached_h_aug = getattr(core, "_last_path1_h_aug", None)
                            if cached_h_aug is not None and isinstance(cached_h_aug, torch.Tensor):
                                last_path1_h_aug = cached_h_aug.detach()
                        except Exception:
                            pass

                    else:
                        training_stats["path2_batches"] += 1
                        forward_kwargs = {
                            "input_ids": input_ids,
                            "attention_mask": attention_mask,
                            "labels": labels,
                            "src_texts": batch.get("src_text", None),
                            "token_word_map": batch.get("token_word_map", None),
                        }
                        amp_ctx = get_amp_ctx()
                        with amp_ctx:
                            try:
                                core = model.module if hasattr(model, "module") else model
                                gated_h_aug = get_gated_h_aug(model, last_path1_h_aug)
                                forward_out = core.forward_path2(
                                    **forward_kwargs,
                                    use_rdrop=_USE_RDROP,
                                    path1_h_aug=gated_h_aug,
                                )
                            except Exception:
                                forward_kwargs["path"] = 2
                                forward_out = model(**forward_kwargs)

                        if isinstance(forward_out, torch.Tensor):
                            loss_tensor = forward_out
                        elif isinstance(forward_out, dict):
                            translation_loss = None
                            asbn_loss = None
                            domain_accuracy = None
                            if "translation_loss" in forward_out:
                                translation_loss = forward_out["translation_loss"]
                            if "asbn_loss" in forward_out:
                                asbn_loss = forward_out["asbn_loss"]
                            if "domain_accuracy" in forward_out:
                                domain_accuracy = forward_out["domain_accuracy"]
                            if translation_loss is None and "loss" in forward_out:
                                translation_loss = forward_out["loss"]
                            if "logits" in forward_out and "labels" in forward_out:
                                logits_raw = forward_out["logits"]
                                labels_raw = forward_out["labels"]
                            elif "logits_path2" in forward_out and "labels" in forward_out:
                                logits_raw = forward_out["logits_path2"]
                                labels_raw = forward_out["labels"]
                            else:
                                logits_raw = None
                                labels_raw = None
                            if logits_raw is not None and labels_raw is not None:
                                logits_use = logits_raw
                                labels_use = labels_raw
                                if logits_use.dim() == 3:
                                    vocab_size = logits_use.size(-1)
                                    logits_use = logits_use.view(-1, vocab_size)
                                if labels_use.dim() > 1:
                                    labels_use = labels_use.view(-1)
                                try:
                                    pad_id = getattr(tokenizer, "pad_token_id", -100)
                                except Exception:
                                    pad_id = -100
                                ce_fct = nn.CrossEntropyLoss(ignore_index=pad_id, label_smoothing=_LABEL_SMOOTHING)
                                translation_loss = ce_fct(logits_use, labels_use)
                            if translation_loss is None:
                                raise RuntimeError("No translation loss available in forward_out for path 2")
                            loss_tensor = translation_loss
                            if asbn_loss is not None and _LAMBDA_ASBN > 0.0:
                                try:
                                    if isinstance(domain_accuracy, torch.Tensor):
                                        da_val = float(domain_accuracy.item())
                                    else:
                                        da_val = float(domain_accuracy) if domain_accuracy is not None else 0.0
                                except Exception:
                                    da_val = 0.0
                                if da_val > 0.10:
                                    loss_tensor = loss_tensor + _LAMBDA_ASBN * asbn_loss
                        elif isinstance(forward_out, (list, tuple)) and len(forward_out) > 0 and isinstance(forward_out[0], torch.Tensor):
                            loss_tensor = forward_out[0]
                        else:
                            raise RuntimeError("Model forward did not return a recognizable loss tensor")

                        if not isinstance(loss_tensor, torch.Tensor):
                            loss_tensor = torch.tensor(float(loss_tensor), device=_DEVICE)
                        else:
                            loss_tensor = loss_tensor.to(_DEVICE)
                        if loss_tensor.numel() > 1:
                            loss_tensor = loss_tensor.mean()

                        # FIX-C7-BT-3: bt_pair[0]=bn_synthetic, bt_pair[1]=en_real
                        # src_lang must be _SOURCE_LANGUAGE (bn_IN), tgt_lang must be _TARGET_LANGUAGE (en_XX)
                        if _USE_BACK_TRANSLATION and _bt_iter is not None:
                            try:
                                try:
                                    bt_pair = next(_bt_iter)
                                except StopIteration:
                                    _bt_pairs_refresh = globals().get("bt_train_pairs", None)
                                    if _bt_pairs_refresh and len(_bt_pairs_refresh) > 0:
                                        _bt_iter = iter(_bt_pairs_refresh)
                                        bt_pair = next(_bt_iter)
                                    else:
                                        bt_pair = None

                                if bt_pair is not None:
                                    bt_src_text, bt_tgt_text = bt_pair[0], bt_pair[1]
                                    try:
                                        tokenizer.src_lang = _SOURCE_LANGUAGE
                                        tokenizer.tgt_lang = _TARGET_LANGUAGE
                                    except Exception:
                                        pass
                                    bt_enc = tokenizer(
                                        bt_src_text,
                                        return_tensors="pt",
                                        padding=True,
                                        truncation=True,
                                        max_length=_MAX_LENGTH,
                                    )
                                    bt_tgt_enc = tokenizer(
                                        bt_tgt_text,
                                        return_tensors="pt",
                                        padding=True,
                                        truncation=True,
                                        max_length=_MAX_LENGTH,
                                    )
                                    bt_input_ids = bt_enc["input_ids"].to(_DEVICE, non_blocking=True)
                                    bt_attention_mask = bt_enc["attention_mask"].to(_DEVICE, non_blocking=True)
                                    bt_labels = bt_tgt_enc["input_ids"].to(_DEVICE, non_blocking=True)
                                    bt_labels[bt_labels == tokenizer.pad_token_id] = -100

                                    amp_ctx_bt = get_amp_ctx()
                                    with amp_ctx_bt:
                                        core = model.module if hasattr(model, "module") else model
                                        bt_fwd = core.forward_path2(
                                            input_ids=bt_input_ids,
                                            attention_mask=bt_attention_mask,
                                            labels=bt_labels,
                                            src_texts=[bt_src_text],
                                            token_word_map=None,
                                            use_rdrop=False,
                                            path1_h_aug=None,
                                        )

                                    if isinstance(bt_fwd, torch.Tensor):
                                        bt_loss = bt_fwd
                                    elif isinstance(bt_fwd, dict):
                                        bt_loss = bt_fwd.get("translation_loss", bt_fwd.get("loss", None))
                                    elif isinstance(bt_fwd, (list, tuple)) and len(bt_fwd) > 0:
                                        bt_loss = bt_fwd[0]
                                    else:
                                        bt_loss = None

                                    if bt_loss is not None and isinstance(bt_loss, torch.Tensor):
                                        if bt_loss.numel() > 1:
                                            bt_loss = bt_loss.mean()
                                        bt_loss = bt_loss.to(_DEVICE)
                                        if torch.isfinite(bt_loss):
                                            loss_tensor = loss_tensor + _LAMBDA_BT * bt_loss
                                            training_stats["bt_batches"] += 1
                                            training_stats["bt_losses"].append(float(bt_loss.item()))
                            except Exception as e:
                                if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                                    print(f"[BT] Back-translation loss failed at step {global_step}: {type(e).__name__}: {str(e)[:100]}")

                        if loss_tensor.numel() > 1:
                            loss_val = float(loss_tensor.mean().item())
                            loss_tensor = loss_tensor.mean()
                        else:
                            loss_val = float(loss_tensor.item())
                        training_stats["last_forward_loss"] = loss_val
                        epoch_losses.append(loss_val)
                        training_stats["total_loss"].append(loss_val)
                        training_stats["path2_losses"].append(loss_val)

                    loss_scaled = loss_tensor / max(1, accumulation_steps)
                    last_backward_loss = float(loss_scaled.item())
                    training_stats["backward_losses"].append(last_backward_loss)

                    try:
                        if scaler.is_enabled():
                            scaler.scale(loss_scaled).backward()
                        else:
                            loss_scaled.backward()
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                    except RuntimeError as e:
                        if "out of memory" in str(e).lower():
                            training_stats["oom_errors"] += 1
                            training_stats["skipped_batches"] += 1
                            skip_reasons["oom_backward"] += 1
                            print(f"[OOM] Step {global_step} - Emergency cleanup")
                            try:
                                optimizer.zero_grad(set_to_none=True)
                            except Exception:
                                pass
                            for p in model.parameters():
                                p.grad = None
                            if torch.cuda.is_available():
                                torch.cuda.empty_cache()
                            gc.collect()
                            accumulated_steps = 0
                            progress.update(1)
                            continue
                        else:
                            raise

                    accumulated_steps += 1
                    if accumulated_steps >= accumulation_steps:
                        try:
                            if scaler.is_enabled():
                                scaler.unscale_(optimizer)
                                torch.nn.utils.clip_grad_norm_(model.parameters(), _GRAD_CLIP_NORM)
                                scaler.step(optimizer)
                                scaler.update()
                            else:
                                torch.nn.utils.clip_grad_norm_(model.parameters(), _GRAD_CLIP_NORM)
                                optimizer.step()
                            if scheduler is not None:
                                scheduler.step()
                            current_lr = optimizer.param_groups[0]["lr"]
                            training_stats["learning_rates"].append(current_lr)
                            if global_step % DEBUG_PRINT_INTERVAL == 0 and (_DEBUG_DISCOVERY or _VERBOSE_LOGGING):
                                print(f"[TRAIN-DEBUG] Current learning rate: {current_lr:.2e}")
                            optimizer.zero_grad(set_to_none=True)
                            training_stats["optimizer_updates"] += 1
                            if torch.cuda.is_available():
                                torch.cuda.empty_cache()
                        except RuntimeError as e:
                            if "out of memory" in str(e).lower():
                                training_stats["oom_errors"] += 1
                                training_stats["skipped_batches"] += 1
                                skip_reasons["oom"] += 1
                                print(f"[OOM] Step {global_step} - Emergency cleanup")
                                try:
                                    optimizer.zero_grad(set_to_none=True)
                                except Exception:
                                    pass
                                for p in model.parameters():
                                    p.grad = None
                                if torch.cuda.is_available():
                                    torch.cuda.empty_cache()
                                gc.collect()
                                accumulated_steps = 0
                                progress.update(1)
                                continue
                            else:
                                training_stats["runtime_errors"] += 1
                                skip_reasons["opt_runtime"] += 1
                                print(f"[ERROR] Runtime error during optimizer step: {type(e).__name__}")
                        except Exception as e:
                            training_stats["exceptions"] += 1
                            skip_reasons["opt_exception"] += 1
                            print(f"[ERROR] Exception during optimizer step: {type(e).__name__}")
                        finally:
                            accumulated_steps = 0

                    if pending_validation:
                        try:
                            optimizer.zero_grad(set_to_none=True)
                        except Exception:
                            pass
                        print_path_loss_summary(training_stats, validate_every, global_step, use_dual_path)
                        val_result = comprehensive_epoch_validation(
                            model=model,
                            tokenizer=tokenizer,
                            epoch=epoch,
                            global_step=global_step,
                            source_lang=_SOURCE_LANGUAGE,
                            target_lang=_TARGET_LANGUAGE,
                            max_length=_MAX_LENGTH,
                            device=_DEVICE,
                            val_dataset=val_dataset,
                            test_loader=test_loader,
                        )
                        try:
                            core = model.module if hasattr(model, "module") else model
                            dscd = getattr(core, "dscd", None)
                            if dscd and hasattr(dscd, "cleanup_memory"):
                                print("[VALIDATION] Running DSCD cleanup after validation...")
                                dscd.cleanup_memory()
                        except Exception as e:
                            if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                                print(f"[VALIDATION] DSCD cleanup failed: {type(e).__name__}")
                        if val_result:
                            training_stats["epoch_validations"].append(val_result)
                            training_stats["test_loss_history"].append(val_result.get("test_loss", 0.0))
                            training_stats["bleu_200_history"].append(val_result.get("bleu_200", 0.0))
                            last_test_loss = val_result.get("test_loss", last_test_loss)
                            last_bleu_score = val_result.get("bleu_200", last_bleu_score)
                        pending_validation = False
                        cached_cluster_count = get_cluster_count(model)

                    if global_step % DEBUG_PRINT_INTERVAL == 0:
                        print_gpu_mem("TRAIN-DEBUG")
                        cached_cluster_count = get_cluster_count(model)
                        path_str = f"p{selected_path}" if use_dual_path else "p2"
                        print(f"[TRAIN-DEBUG] step={global_step} {path_str} loss={training_stats['last_forward_loss']:.4f} clusters={cached_cluster_count}")
                        print_top_clusters(model, topn=5)

                    if global_step % _MEMORY_CLEANUP_FREQUENCY == 0:
                        clear_all_gpu_caches()

                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        training_stats["oom_errors"] += 1
                        training_stats["skipped_batches"] += 1
                        skip_reasons["oom"] += 1
                        print(f"[OOM] Step {global_step} - Emergency cleanup")
                        try:
                            optimizer.zero_grad(set_to_none=True)
                        except Exception:
                            pass
                        for p in model.parameters():
                            p.grad = None
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                        gc.collect()
                        accumulated_steps = 0
                        progress.update(1)
                        continue
                    else:
                        training_stats["runtime_errors"] += 1
                        training_stats["skipped_batches"] += 1
                        skip_reasons["runtime"] += 1
                        print("=" * 80)
                        print(f"[RUNTIME ERROR] - Step {global_step}")
                        print("=" * 80)
                        print(f"  Error: {str(e)}")
                        print(f"  Path: {selected_path}")
                        print(f"  Batch size: {batch_size}")
                        traceback.print_exc()
                        print("=" * 80)
                        try:
                            optimizer.zero_grad(set_to_none=True)
                        except Exception:
                            pass
                        accumulated_steps = 0
                        progress.update(1)
                        continue
                except Exception as e:
                    training_stats["exceptions"] += 1
                    training_stats["skipped_batches"] += 1
                    skip_reasons["exceptions"] += 1
                    print(f"[EXCEPTION] Exception at step {global_step}: {type(e).__name__}")
                    if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                        try:
                            traceback.print_exc()
                        except Exception:
                            pass
                    try:
                        optimizer.zero_grad(set_to_none=True)
                    except Exception:
                        pass
                    accumulated_steps = 0
                    progress.update(1)
                    continue

                processed_batches = training_stats["batches_processed"] - training_stats["skipped_batches"]
                recent_losses = training_stats["total_loss"][-50:] if training_stats["total_loss"] else [0.0]
                running_train_loss = float(np.mean(recent_losses))
                current_lr_display = optimizer.param_groups[0]["lr"]

                postfix = {
                    "loss": f"{running_train_loss:.3f}",
                    "test": f"{last_test_loss:.3f}",
                    "bleu": f"{last_bleu_score:.2f}",
                    "lr": f"{current_lr_display:.1e}",
                    "disc": training_stats.get("discovery_homographs_found", 0),
                }
                progress.set_postfix(postfix, refresh=False)
                progress.update(1)

        finally:
            if progress is not None:
                try:
                    progress.close()
                except Exception:
                    pass

        if accumulated_steps > 0:
            try:
                if scaler.is_enabled():
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), _GRAD_CLIP_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), _GRAD_CLIP_NORM)
                    optimizer.step()
                if scheduler is not None:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                training_stats["optimizer_updates"] += 1
            except Exception as e:
                print(f"[EPOCH-FLUSH] Exception on epoch flush: {type(e).__name__}")
            finally:
                accumulated_steps = 0

        epoch_duration_min = (time.time() - epoch_start) / 60.0
        processed_batches = training_stats["batches_processed"] - training_stats["skipped_batches"]
        expected_updates = max(1, math.floor(processed_batches / max(1, accumulation_steps)))
        success_rate = 100.0 * training_stats["optimizer_updates"] / expected_updates if expected_updates > 0 else 0.0
        cached_cluster_count = get_cluster_count(model)
        avg_epoch_loss = float(np.mean(epoch_losses)) if epoch_losses else 0.0
        training_stats["epoch_losses"].append(avg_epoch_loss)

        print("=" * 80)
        print(f"EPOCH {epoch}/{epochs} SUMMARY")
        print("=" * 80)
        print(f"  Duration (min): {epoch_duration_min:.2f}")
        print(f"  Optimizer updates: {training_stats['optimizer_updates']}")
        print(f"  Batches: processed={processed_batches}, skipped={training_stats['skipped_batches']}")
        print(f"  Success rate: {success_rate:.1f}%")
        print(f"  Clustered tokens: {cached_cluster_count}")
        print(f"  Avg epoch loss: {avg_epoch_loss:.6f}")
        print(f"  Discovery runs: {training_stats['discovery_runs']}")
        print(f"  Homographs discovered: {training_stats['discovery_homographs_found']}")
        if use_dual_path:
            print(f"  Path 1 batches: {training_stats['path1_batches']}")
            print(f"  Path 2 batches: {training_stats['path2_batches']}")
        if training_stats["path1_losses"]:
            print(f"  Path 1 avg loss (DSCD+Contrastive): {np.mean(training_stats['path1_losses']):.4f}")
        if training_stats["path2_losses"]:
            print(f"  Path 2 avg loss: {np.mean(training_stats['path2_losses']):.4f}")
        if _USE_BACK_TRANSLATION and training_stats["bt_batches"] > 0:
            print(f"  BT batches: {training_stats['bt_batches']}")
            print(f"  BT avg loss: {np.mean(training_stats['bt_losses']):.4f}")
        if scheduler is not None and training_stats["learning_rates"]:
            current_lr = training_stats["learning_rates"][-1]
            print(f"  Current learning rate: {current_lr:.2e}")
        if skip_reasons:
            print("  Skip reasons:")
            for k, v in sorted(skip_reasons.items(), key=lambda x: -x[1]):
                print(f"    - {k}: {v}")
        print("=" * 80)

        if enable_validation:
            try:
                print(f"[TRAIN] Running comprehensive validation after epoch {epoch}...")
                try:
                    optimizer.zero_grad(set_to_none=True)
                except Exception:
                    pass
                validation_results = comprehensive_epoch_validation(
                    model=model,
                    tokenizer=tokenizer,
                    epoch=epoch,
                    global_step=global_step,
                    source_lang=_SOURCE_LANGUAGE,
                    target_lang=_TARGET_LANGUAGE,
                    max_length=_MAX_LENGTH,
                    device=_DEVICE,
                    val_dataset=val_dataset,
                    test_loader=test_loader,
                )
                try:
                    core = model.module if hasattr(model, "module") else model
                    dscd = getattr(core, "dscd", None)
                    if dscd and hasattr(dscd, "cleanup_memory"):
                        print("[VALIDATION] Running DSCD cleanup after validation...")
                        dscd.cleanup_memory()
                except Exception as e:
                    if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                        print(f"[VALIDATION] DSCD cleanup failed: {type(e).__name__}")

                if validation_results and validation_results.get("validation_completed", False):
                    training_stats["epoch_validations"].append(validation_results)
                    training_stats["dscd_quality_history"].append(validation_results.get("dscd_quality_score", 0.0))
                    training_stats["asbn_domain_accuracy_history"].append(validation_results.get("asbn_domain_accuracy", 0.0))
                    training_stats["asbn_domain_loss_history"].append(validation_results.get("asbn_domain_loss", 0.0))
                    training_stats["trg_explanation_history"].append(validation_results.get("trg_total_explanations", 0))
                    training_stats["test_loss_history"].append(validation_results.get("test_loss", 0.0))
                    training_stats["bleu_200_history"].append(validation_results.get("bleu_200", 0.0))
                    last_test_loss = validation_results.get("test_loss", last_test_loss)
                    last_bleu_score = validation_results.get("bleu_200", last_bleu_score)

                    try:
                        core_model = model.module if hasattr(model, "module") else model
                        dscd = getattr(core_model, "dscd", None)
                        lock = None
                        if dscd is not None:
                            if hasattr(dscd, "buffer_lock"):
                                lock = dscd.buffer_lock
                            elif hasattr(dscd, "clustering_lock"):
                                lock = dscd.clustering_lock
                        if dscd is not None:
                            if lock:
                                with lock:
                                    total_tokens = len(dscd.prototype_stores)
                            else:
                                total_tokens = len(dscd.prototype_stores)
                            multi_sense = validation_results.get("dscd_multi_sense_tokens", 0)
                            ratio = multi_sense / total_tokens if total_tokens > 0 else 0.0
                            training_stats["multi_sense_ratio_history"].append(ratio)
                        else:
                            training_stats["multi_sense_ratio_history"].append(0.0)
                    except Exception:
                        training_stats["multi_sense_ratio_history"].append(0.0)

                    proto_epoch_path = get_epoch_prototype_path(epoch)
                    # FIX-C7-6: Wrapped save_dscd_prototypes in try/except with torch.save fallback
                    try:
                        save_dscd_prototypes(dscd, proto_epoch_path)
                    except Exception as e:
                        try:
                            core_model_ps = model.module if hasattr(model, "module") else model
                            dscd_ps = getattr(core_model_ps, "dscd", None)
                            if dscd_ps is not None:
                                torch.save(dscd_ps.state_dict(), proto_epoch_path)
                                print(f"[TRAIN] Prototype epoch saved (fallback): {proto_epoch_path}")
                            else:
                                print(f"[TRAIN] Prototype epoch save skipped (DSCD unavailable): {e}")
                        except Exception as e2:
                            print(f"[TRAIN] Prototype epoch save failed (both methods): {e2}")

                    try:
                        epoch_ckpt_path = get_epoch_checkpoint_path(epoch)
                        _epoch_ckpt_dir = os.path.dirname(str(epoch_ckpt_path))
                        if _epoch_ckpt_dir:
                            os.makedirs(_epoch_ckpt_dir, exist_ok=True)
                        core_model_ckpt = model.module if hasattr(model, "module") else model
                        proto_state_epoch = _build_epoch_checkpoint_dscd_state(core_model_ckpt)
                        epoch_checkpoint_data = {
                            "epoch": epoch,
                            "global_step": global_step,
                            "avg_epoch_loss": avg_epoch_loss,
                            "model_state_dict": core_model_ckpt.state_dict(),
                            "optimizer_state_dict": optimizer.state_dict(),
                            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
                            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
                            "dscd_state": proto_state_epoch,
                            "training_stats": training_stats,
                            "config": {
                                "SPAN_THRESHOLD": globals().get("SPAN_THRESHOLD", 0.08),
                                "TAU_LOW": globals().get("TAU_LOW", 0.15),
                                "LAMBDA_ASBN": globals().get("LAMBDA_ASBN", 0.0),
                                "LAMBDA_DSCD": globals().get("LAMBDA_DSCD", 0.01),
                                "LAMBDA_TRG": globals().get("LAMBDA_TRG", 0.001),
                                "PERIODIC_DISCOVERY_FREQUENCY": _PERIODIC_DISCOVERY_FREQUENCY,
                                "NUM_EPOCHS": epochs,
                                "BATCH_SIZE": _BATCH_SIZE,
                                "LEARNING_RATE": optimizer.param_groups[0]["lr"] if optimizer and optimizer.param_groups else 0.0,
                                "WARMUP_STEPS": _WARMUP_STEPS,
                                "USE_LR_SCHEDULER": _USE_LR_SCHEDULER,
                                "USE_DUAL_PATH_TRAINING": use_dual_path,
                                "USE_BACK_TRANSLATION": _USE_BACK_TRANSLATION,
                                "LAMBDA_BT": _LAMBDA_BT,
                            },
                        }
                        torch.save(epoch_checkpoint_data, epoch_ckpt_path)
                        file_size_mb = Path(epoch_ckpt_path).stat().st_size / 1024**2
                        print(f"[TRAIN] ✅ Epoch {epoch} checkpoint saved: {epoch_ckpt_path} ({file_size_mb:.1f} MB)")

                        try:
                            upload_epoch_files_to_gdrive(epoch, checkpoint_dir=_CHECKPOINT_DIR)
                        except Exception as _upload_exc:
                            print(f"[TRAIN] ⚠️ Drive upload for epoch {epoch} failed (non-fatal): {type(_upload_exc).__name__}: {_upload_exc}")

                    except Exception as e:
                        print(f"[TRAIN] ❌ Epoch {epoch} checkpoint save FAILED: {type(e).__name__}: {e}")
                        if _VERBOSE_LOGGING or _DEBUG_DISCOVERY:
                            try:
                                traceback.print_exc()
                            except Exception:
                                pass

                else:
                    training_stats["multi_sense_ratio_history"].append(0.0)
            except Exception as e:
                print(f"[TRAIN] Epoch validation failed: {type(e).__name__}")
                if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                    try:
                        traceback.print_exc()
                    except Exception:
                        pass

    print("=" * 80)
    print("TRAINING COMPLETE - SAVING FINAL CHECKPOINT")
    print("=" * 80)
    try:
        checkpoint_path = get_checkpoint_path()
        _final_ckpt_dir = os.path.dirname(str(checkpoint_path))
        if _final_ckpt_dir:
            os.makedirs(_final_ckpt_dir, exist_ok=True)
        core_model = model.module if hasattr(model, "module") else model

        proto_final_path = get_epoch_prototype_path("final")
        # FIX-C7-6: Wrapped final prototype save with torch.save fallback
        try:
            save_dscd_prototypes(core_model.dscd, proto_final_path)
        except Exception as e:
            try:
                dscd_final_fb = getattr(core_model, "dscd", None)
                if dscd_final_fb is not None:
                    torch.save(dscd_final_fb.state_dict(), proto_final_path)
                    print(f"[CHECKPOINT] Prototype final saved (fallback): {proto_final_path}")
                else:
                    print(f"[CHECKPOINT] Prototype final save skipped: {e}")
            except Exception as e2:
                print(f"[CHECKPOINT] Prototype final save failed (both methods): {e2}")

        checkpoint_data = {
            "epochs_trained": epochs,
            "global_steps": global_step,
            "final_train_loss": training_stats["epoch_losses"][-1] if training_stats["epoch_losses"] else 0.0,
            "last_epoch_checkpoint": str(get_epoch_checkpoint_path(epochs)),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "training_stats": training_stats,
            "config": {
                "SPAN_THRESHOLD": globals().get("SPAN_THRESHOLD", 0.08),
                "TAU_LOW": globals().get("TAU_LOW", 0.15),
                "LAMBDA_ASBN": globals().get("LAMBDA_ASBN", 0.0),
                "LAMBDA_DSCD": globals().get("LAMBDA_DSCD", 0.01),
                "LAMBDA_TRG": globals().get("LAMBDA_TRG", 0.001),
                "PERIODIC_DISCOVERY_FREQUENCY": _PERIODIC_DISCOVERY_FREQUENCY,
                "NUM_EPOCHS": epochs,
                "BATCH_SIZE": _BATCH_SIZE,
                "LEARNING_RATE": optimizer.param_groups[0]["lr"] if optimizer and optimizer.param_groups else 0.0,
                "WARMUP_STEPS": _WARMUP_STEPS,
                "USE_LR_SCHEDULER": _USE_LR_SCHEDULER,
                "USE_DUAL_PATH_TRAINING": use_dual_path,
                "USE_BACK_TRANSLATION": _USE_BACK_TRANSLATION,
                "LAMBDA_BT": _LAMBDA_BT,
            },
        }
        torch.save(checkpoint_data, checkpoint_path)
        file_size_mb = Path(checkpoint_path).stat().st_size / 1024**2
        print("CHECKPOINT SAVED")
        print(f"  Path: {checkpoint_path}")
        print(f"  Size: {file_size_mb:.2f} MB  ← lightweight metadata only")
        print(f"  Full model weights → {get_epoch_checkpoint_path(epochs)}")
        print(f"  Epochs trained: {epochs}")
        print(f"  Global steps: {global_step}")
        print(f"  Final train loss: {training_stats['epoch_losses'][-1] if training_stats['epoch_losses'] else 0.0:.4f}")
        print(f"  Prototype file: {proto_final_path}")
        print("=" * 80)

        try:
            upload_checkpoint_to_gdrive(str(get_checkpoint_path()), epoch=None)
        except Exception as _upload_final_exc:
            print(f"[TRAIN] ⚠️ Drive upload of final checkpoint failed (non-fatal): {type(_upload_final_exc).__name__}: {_upload_final_exc}")

    except Exception as e:
        print(f"[FINAL CHECKPOINT SAVE FAILED] {type(e).__name__}: {e}")
        if _VERBOSE_LOGGING or _DEBUG_DISCOVERY:
            try:
                traceback.print_exc()
            except Exception:
                pass

    print("=" * 80)
    print("TRAINING COMPLETED - FINAL SUMMARY")
    print("=" * 80)
    processed_batches = training_stats["batches_processed"] - training_stats["skipped_batches"]
    expected_updates = max(1, math.floor(processed_batches / max(1, accumulation_steps)))
    success_rate = 100.0 * training_stats["optimizer_updates"] / expected_updates if expected_updates > 0 else 0.0
    print(f"[TRAIN] Batches processed: {processed_batches}")
    print(f"[TRAIN] Batches skipped: {training_stats['skipped_batches']}")
    print(f"[TRAIN] Optimizer updates: {training_stats['optimizer_updates']}")
    print(f"[TRAIN] Expected optimizer updates: {expected_updates}")
    print(f"[TRAIN] Success Rate: {success_rate:.1f}%")
    print(f"[TRAIN] Total Steps: {global_step}")
    print(f"[TRAIN] Clustered Token Types: {cached_cluster_count}")
    print(f"[TRAIN] Discovery Runs: {training_stats['discovery_runs']}")
    print(f"[TRAIN] Total Homographs Found: {training_stats['discovery_homographs_found']}")
    print(f"[TRAIN] LR Scheduler: {'Enabled' if scheduler is not None else 'Disabled'}")
    print(f"[TRAIN] Dual-Path Mode: {'Enabled' if use_dual_path else 'Disabled'}")
    if use_dual_path:
        print(f"[TRAIN] Path 1 Total Batches: {training_stats['path1_batches']}")
        print(f"[TRAIN] Path 2 Total Batches: {training_stats['path2_batches']}")
    if _USE_BACK_TRANSLATION:
        print(f"[TRAIN] BT Total Batches: {training_stats['bt_batches']}")
        if training_stats["bt_losses"]:
            print(f"[TRAIN] BT Avg Loss: {np.mean(training_stats['bt_losses']):.4f}")
    if training_stats["dscd_quality_history"]:
        print("[TRAIN] DSCD Quality Score Trend:")
        for i, score in enumerate(training_stats["dscd_quality_history"], 1):
            print(f"  Epoch {i}: {score:.1%}")
    if training_stats["asbn_domain_accuracy_history"]:
        print("[TRAIN] ASBN Domain Accuracy Trend:")
        for i, acc in enumerate(training_stats["asbn_domain_accuracy_history"], 1):
            print(f"  Epoch {i}: {acc:.1%}")
    if training_stats["asbn_domain_loss_history"]:
        print("[TRAIN] ASBN Domain Loss Trend:")
        for i, loss_val in enumerate(training_stats["asbn_domain_loss_history"], 1):
            print(f"  Epoch {i}: {loss_val:.4f}")
    if training_stats["trg_explanation_history"]:
        print("[TRAIN] TRG Explanation Count Trend:")
        for i, count in enumerate(training_stats["trg_explanation_history"], 1):
            print(f"  Epoch {i}: {count} explanations")
    if training_stats["test_loss_history"]:
        print("[TRAIN] Test Loss Trend at validation checkpoints:")
        for i, tl in enumerate(training_stats["test_loss_history"], 1):
            print(f"  Checkpoint {i}: {tl:.4f}")
    if training_stats["bleu_200_history"]:
        print(f"[TRAIN] BLEU-{_BLEU_EVAL_SAMPLES} Trend at validation checkpoints:")
        for i, bl in enumerate(training_stats["bleu_200_history"], 1):
            print(f"  Checkpoint {i}: {bl:.2f}")
    if training_stats["learning_rates"]:
        print("[TRAIN] Learning Rate Progression:")
        print(f"  Initial LR: {training_stats['learning_rates'][0]:.2e}")
        print(f"  Final LR: {training_stats['learning_rates'][-1]:.2e}")
        print(f"  Total LR updates: {len(training_stats['learning_rates'])}")
    print("=" * 80)
    return model


# ==============================================================================
# PHASE 5: TRAINING INVOCATION
# ==============================================================================

print("=" * 80)
print("Cell 7: DUAL-PATH Training Loop - PATH 1 (DSCD+Contrastive) + PATH 2 (TRANSLATION)")
print("=" * 80)
print("Configuration:")
print(f"  - Discovery frequency: {_PERIODIC_DISCOVERY_FREQUENCY} steps")
print(f"  - Validation interval: {_VALIDATION_CHECK_INTERVAL} steps")
print(f"  - Train domain: {_TRAIN_DOMAIN}")
print(f"  - Test domain: {_TEST_DOMAIN}")
print(f"  - Gradient clip: {_GRAD_CLIP_NORM}")
print(f"  - Memory cleanup: Every {_MEMORY_CLEANUP_FREQUENCY} steps")
print(f"  - Lambda TRG: {_LAMBDA_TRG}")
print(f"  - Lambda ASBN: {_LAMBDA_ASBN}")
print(f"  - Warmup steps (optimizer): {_WARMUP_STEPS}")
print(f"  - Label smoothing: {_LABEL_SMOOTHING}")
print(f"  - LR scheduler: {'Enabled' if _USE_LR_SCHEDULER else 'Disabled'}")
print(f"  - Dual-path training: {'Enabled' if _USE_DUAL_PATH_TRAINING else 'Disabled (default)'}")
print(f"  - Eval length penalty: {_EVAL_LENGTH_PENALTY}")
print(f"  - Eval repetition penalty: {_EVAL_REPETITION_PENALTY}")
print(f"  - Eval num beams: {_EVAL_NUM_BEAMS}")
print(f"  - Eval min length: {_EVAL_MIN_LENGTH}")
print(f"  - Eval no_repeat_ngram_size: {_EVAL_NO_REPEAT_NGRAM_SIZE}")
print(f"  - BLEU eval samples: {_BLEU_EVAL_SAMPLES}")
print(f"  - Use R-Drop: {_USE_RDROP}")
print(f"  - Back-translation: {'Enabled' if _USE_BACK_TRANSLATION else 'Disabled'}, LAMBDA_BT={_LAMBDA_BT}")
print(f"  - AMP dtype: {_AMP_DTYPE}")
print(f"  - GradScaler: {'DISABLED (BF16 path)' if _USE_BF16 else 'ENABLED'}")
print(f"  - Checkpoint dir: {_CHECKPOINT_DIR}")
print()

# FIX-C7-BT-1: Corrected variable name — original checked bt_augmented_train_loader (does not exist)
# Cell 6.5 produces bt_train_pairs (List[Tuple[str,str]]), not a DataLoader
if _USE_BACK_TRANSLATION:
    _bt_check = globals().get("bt_train_pairs", None)
    if _bt_check and len(_bt_check) > 0:
        print(f"[PHASE 5] ✅ bt_train_pairs found: {len(_bt_check):,} BT pairs ready for injection")
    else:
        print("[PHASE 5] ⚠️ USE_BACK_TRANSLATION=True but bt_train_pairs not found or empty")
        print("[PHASE 5]    → Run Cell 6.5 first to generate bt_train_pairs before Cell 7")
else:
    print("[PHASE 5] Back-translation disabled (USE_BACK_TRANSLATION=False)")

_tatn_model   = globals().get("tatn_model",   None)
_optimizer    = globals().get("optimizer",    None)
_train_loader = globals().get("train_loader", None)
_tokenizer    = globals().get("tokenizer",    None)
_val_dataset  = globals().get("val_dataset",  None)
_test_loader  = globals().get("test_loader",  None)

_missing_vars = []
if _tatn_model   is None: _missing_vars.append("tatn_model")
if _optimizer    is None: _missing_vars.append("optimizer")
if _train_loader is None: _missing_vars.append("train_loader")
if _tokenizer    is None: _missing_vars.append("tokenizer")

if _missing_vars:
    raise RuntimeError(
        f"[PHASE 5] Cannot start training — missing required variables: {_missing_vars}\n"
        f"  → Ensure Cell 5 (tokenizer), Cell 6 (model+optimizer+loaders) have run successfully."
    )

if _val_dataset is None:
    print("[PHASE 5] WARNING: val_dataset not found — BLEU computation will be skipped")
if _test_loader is None:
    print("[PHASE 5] WARNING: test_loader not found — test loss computation will be skipped")

print(f"[PHASE 5] tatn_model:   {type(_tatn_model).__name__}")
print(f"[PHASE 5] optimizer:    {type(_optimizer).__name__}")
print(f"[PHASE 5] train_loader: {len(_train_loader)} batches")
print(f"[PHASE 5] tokenizer:    {type(_tokenizer).__name__}")
print(f"[PHASE 5] val_dataset:  {type(_val_dataset).__name__ + ' (' + str(len(_val_dataset)) + ' samples)' if _val_dataset is not None else 'None'}")
print(f"[PHASE 5] test_loader:  {type(_test_loader).__name__ if _test_loader is not None else 'None'}")

tatn_model = train_memory_efficient_tatn(
    model              = _tatn_model,
    tokenizer          = _tokenizer,
    train_loader       = _train_loader,
    optimizer          = _optimizer,
    phi_optimizer      = globals().get("phi_optimizer", None),
    epochs             = _EPOCHS,
    accumulation_steps = _ACCUMULATION_STEPS,
    validate_every     = _VALIDATION_CHECK_INTERVAL,
    enable_validation  = True,
    use_dual_path      = _USE_DUAL_PATH_TRAINING,
    test_loader        = _test_loader,
    val_dataset        = _val_dataset,
)

print("\n[CELL 7] ✅ Training complete. tatn_model updated in globals.")


In [ ]:
# =======================================================================================
# CELL 8: INFERENCE & EVALUATION PIPELINE (DUAL-PATH COMPATIBLE)
# =======================================================================================
import os
import glob
import time
import math
import torch
import traceback
from typing import List, Dict, Any, Tuple, Optional
from collections import defaultdict
from transformers.modeling_outputs import BaseModelOutput
import threading
import gc

try:
    SOURCE_LANG = str(SOURCE_LANGUAGE)
    TARGET_LANG = str(TARGET_LANGUAGE)
except (NameError, TypeError):
    SOURCE_LANG = "bn_IN"
    TARGET_LANG = "en_XX"

try:
    MAXLEN = int(MAX_LENGTH)
except (NameError, ValueError, TypeError):
    MAXLEN = 256

_DEVICE = globals().get("DEVICE", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
if not isinstance(_DEVICE, torch.device):
    try:
        _DEVICE = torch.device(str(_DEVICE))
    except Exception:
        _DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_VERBOSE_LOGGING = bool(globals().get("VERBOSE_LOGGING",  False))
_DEBUG_DISCOVERY = bool(globals().get("DEBUG_DISCOVERY",  False))
_DEBUG_TIMING    = bool(globals().get("DEBUG_TIMING",     False))
_USE_MULTI_GPU   = bool(globals().get("USE_MULTI_GPU", torch.cuda.is_available() and (torch.cuda.device_count() > 1)))

try:
    SPAN_THRESHOLD = float(SPAN_THRESHOLD)
except (NameError, ValueError, TypeError):
    SPAN_THRESHOLD = 0.20

try:
    UNCERTAINTY_THRESHOLD = float(UNCERTAINTY_THRESHOLD)
except (NameError, ValueError, TypeError):
    UNCERTAINTY_THRESHOLD = 0.10

try:
    HOMOGRAPH_REFERENCE_LIST = set(str(w).lower() for w in HOMOGRAPH_REFERENCE_LIST_BN)
except (NameError, TypeError):
    HOMOGRAPH_REFERENCE_LIST = {
        "কল", "কাল", "পাতা", "ব্যাংক", "ফল", "মাথা", "বার", "হার", "তারা",
        "পানি", "দল", "বাজার", "নাম", "কথা", "বই", "ঘর", "মন", "হাত"
    }
    HOMOGRAPH_REFERENCE_LIST = set(str(w).lower() for w in HOMOGRAPH_REFERENCE_LIST)

_MBART50_EN_TOKEN_ID = int(globals().get("MBART50_EN_TOKEN_ID", 250004))
_MBART50_BN_TOKEN_ID = int(globals().get("MBART50_BN_TOKEN_ID", 250028))

try:
    TEST_DOMAIN = int(TEST_DOMAIN)
except (NameError, ValueError, TypeError):
    TEST_DOMAIN = 1

try:
    EVAL_NUM_BEAMS = int(EVAL_NUM_BEAMS)
except (NameError, ValueError, TypeError):
    EVAL_NUM_BEAMS = 4

try:
    EVAL_LENGTH_PENALTY = float(EVAL_LENGTH_PENALTY)
except (NameError, ValueError, TypeError):
    EVAL_LENGTH_PENALTY = 1.0

try:
    EVAL_NO_REPEAT_NGRAM_SIZE = int(EVAL_NO_REPEAT_NGRAM_SIZE)
except (NameError, ValueError, TypeError):
    EVAL_NO_REPEAT_NGRAM_SIZE = 2

try:
    EVAL_MIN_LENGTH = int(EVAL_MIN_LENGTH)
except (NameError, ValueError, TypeError):
    EVAL_MIN_LENGTH = 1

try:
    HAUG_GATE_QUALITY_THRESHOLD = float(HAUG_GATE_QUALITY_THRESHOLD)
except (NameError, ValueError, TypeError):
    HAUG_GATE_QUALITY_THRESHOLD = 0.08

try:
    HAUG_GATE_HOMOGRAPH_THRESHOLD = float(HAUG_GATE_HOMOGRAPH_THRESHOLD)
except (NameError, ValueError, TypeError):
    HAUG_GATE_HOMOGRAPH_THRESHOLD = 0.03

BENGALI_PUNCT_SET = set(['।', '॥'])
COMMON_PUNCT_SET  = set(['.', ',', ';', ':', '!', '?', '"', "'", '-', '(', ')', '[', ']', '{', '}', '/', '\\'])
PUNCT_SET         = BENGALI_PUNCT_SET | COMMON_PUNCT_SET


def is_punctuation_only(token: str) -> bool:
    if not token or not isinstance(token, str):
        return False
    clean = (
        token.replace("▁", "")
        .replace("Ġ", "")
        .replace("##", "")
        .replace("</w>", "")
        .strip()
    )
    if not clean:
        return False
    if clean in BENGALI_PUNCT_SET:
        return True
    if clean in COMMON_PUNCT_SET:
        return True
    if len(clean) == 1 and not clean.isalnum():
        return True
    return all(c in PUNCT_SET for c in clean)


def clean_token(token: str) -> str:
    if not isinstance(token, str):
        return ""
    cleaned = token.replace("▁", "").replace("Ġ", "").replace("##", "").strip()
    for punct in [".", ",", "!", "?", ";", ":", "-"]:
        cleaned = cleaned.replace(punct, "")
    return cleaned


def get_dscd_homographs(model: torch.nn.Module) -> set:
    try:
        core = model.module if hasattr(model, 'module') else model
        dscd = getattr(core, 'dscd', None)
        if dscd is None:
            return set()

        if hasattr(dscd, 'get_discovered_homographs'):
            try:
                discovered = dscd.get_discovered_homographs()
                return set(w for w in discovered if not is_punctuation_only(w))
            except Exception:
                pass

        homographs = set()
        lock = None
        if hasattr(dscd, 'buffer_lock'):
            lock = dscd.buffer_lock
        elif hasattr(dscd, 'clustering_lock'):
            lock = dscd.clustering_lock

        if lock:
            with lock:
                for token, store in dscd.prototype_stores.items():
                    try:
                        if store.size() >= 2:
                            clean_tok = clean_token(str(token))
                            if clean_tok and not is_punctuation_only(str(token)):
                                homographs.add(clean_tok)
                    except Exception:
                        continue
        else:
            for token, store in dscd.prototype_stores.items():
                try:
                    if store.size() >= 2:
                        clean_tok = clean_token(str(token))
                        if clean_tok and not is_punctuation_only(str(token)):
                            homographs.add(clean_tok)
                except Exception:
                    continue

        return homographs
    except Exception:
        return set()


def _log_scea_gates(dscd_out_dict: Dict[str, Any], prefix: str = "[SCEA]"):
    try:
        gates = dscd_out_dict.get("sense_gates", None)
        if gates is None:
            gates = dscd_out_dict.get("scea_gates", dscd_out_dict.get("slot_gates", None))
        if gates is None:
            print(f"{prefix} sense_gates: NOT FOUND in DSCD outputs")
            return

        if isinstance(gates, torch.Tensor):
            g = gates.detach().float()
            if g.min().item() < 0.0 or g.max().item() > 1.0:
                g = torch.sigmoid(g)
            if g.dim() == 3:
                g = g.mean(dim=1)
            if g.dim() == 2:
                mean_per_sense = g.mean(dim=0)
                max_per_sense  = g.max(dim=0).values
                sense_strs = []
                for sense_idx, (mv, mx) in enumerate(zip(mean_per_sense.tolist(), max_per_sense.tolist())):
                    sense_strs.append(f"s{sense_idx}=mean:{mv:.4f}/max:{mx:.4f}")
                print(f"{prefix} Gate values ({len(sense_strs)} senses): {', '.join(sense_strs)}")
            elif g.dim() == 1:
                vals = [f"s{i}={v:.4f}" for i, v in enumerate(g.tolist())]
                print(f"{prefix} Gate values: {', '.join(vals)}")
            else:
                print(f"{prefix} sense_gates shape={tuple(g.shape)}, mean={g.mean().item():.4f}")
        else:
            print(f"{prefix} sense_gates type={type(gates).__name__} (expected Tensor)")
    except Exception as e:
        print(f"{prefix} sense_gates log error: {type(e).__name__}: {str(e)[:80]}")


def _log_hfcaa_gate(fwd_out: Any, prefix: str = "[HFCAA]"):
    try:
        if not isinstance(fwd_out, dict):
            print(f"{prefix} gate mean: NOT AVAILABLE (forward output is not dict)")
            return

        gate = (
            fwd_out.get("hfcaa_gate_mean", None)
            or fwd_out.get("hfcaa_gate", None)
            or fwd_out.get("decoder_gate_mean", None)
        )

        if gate is None:
            print(f"{prefix} gate mean: NOT FOUND in forward outputs")
            return

        if isinstance(gate, torch.Tensor):
            gate_val = gate.detach().float()
            if gate_val.numel() > 1:
                gate_mean = gate_val.mean().item()
                gate_max  = gate_val.max().item()
                gate_min  = gate_val.min().item()
                print(f"{prefix} Gate mean={gate_mean:.4f} | max={gate_max:.4f} | min={gate_min:.4f}")
            else:
                print(f"{prefix} Gate mean={float(gate_val.item()):.4f}")
        elif isinstance(gate, (int, float)):
            print(f"{prefix} Gate mean={float(gate):.4f}")
        else:
            print(f"{prefix} gate type={type(gate).__name__}, value={str(gate)[:60]}")
    except Exception as e:
        print(f"{prefix} gate log error: {type(e).__name__}: {str(e)[:80]}")


def _log_ambiguity_scores(dscd_out_dict: Dict[str, Any], prefix: str = "[AMB]"):
    try:
        scores = (
            dscd_out_dict.get("ambiguity_scores", None)
            or dscd_out_dict.get("span_scores", None)
            or dscd_out_dict.get("uncertainty_scores", None)
            or dscd_out_dict.get("proto_probs", None)
        )

        if scores is None:
            print(f"{prefix} Ambiguity scores: NOT FOUND in DSCD outputs")
            return

        if isinstance(scores, torch.Tensor):
            s = scores.detach().float()
            if s.numel() == 0:
                print(f"{prefix} Ambiguity scores: EMPTY tensor")
                return
            s_flat   = s.view(-1)
            s_mean   = float(s_flat.mean().item())
            s_max    = float(s_flat.max().item())
            s_min    = float(s_flat.min().item())
            high_amb = float((s_flat > SPAN_THRESHOLD).float().mean().item())
            print(
                f"{prefix} Ambiguity distribution: mean={s_mean:.4f} | "
                f"max={s_max:.4f} | min={s_min:.4f} | "
                f"frac>{SPAN_THRESHOLD:.2f}={high_amb:.2%} | shape={tuple(s.shape)}"
            )
        elif isinstance(scores, (list, tuple)):
            if len(scores) == 0:
                print(f"{prefix} Ambiguity scores: EMPTY list")
                return
            flat = []
            for item in scores:
                if isinstance(item, (int, float)):
                    flat.append(float(item))
                elif isinstance(item, torch.Tensor):
                    flat.extend(item.detach().float().view(-1).tolist())
            if flat:
                import statistics
                print(
                    f"{prefix} Ambiguity distribution: mean={statistics.mean(flat):.4f} | "
                    f"max={max(flat):.4f} | min={min(flat):.4f} | n={len(flat)}"
                )
        else:
            print(f"{prefix} Ambiguity scores type={type(scores).__name__}")
    except Exception as e:
        print(f"{prefix} Ambiguity scores log error: {type(e).__name__}: {str(e)[:80]}")


def _log_prototype_separation(model: torch.nn.Module, prefix: str = "[PROTO-SEP]", top_n: int = 10):
    try:
        core = model.module if hasattr(model, 'module') else model
        dscd = getattr(core, 'dscd', None)
        if dscd is None:
            print(f"{prefix} DSCD not available")
            return

        lock = None
        if hasattr(dscd, 'buffer_lock'):
            lock = dscd.buffer_lock
        elif hasattr(dscd, 'clustering_lock'):
            lock = dscd.clustering_lock

        if lock:
            with lock:
                stores_snapshot = list(dscd.prototype_stores.items())
        else:
            stores_snapshot = list(dscd.prototype_stores.items())

        if not stores_snapshot:
            print(f"{prefix} No prototype stores found")
            return

        sep_items = []
        for token, store in stores_snapshot:
            try:
                n_protos = store.size() if hasattr(store, 'size') else len(getattr(store, 'centroids', []))
                if n_protos < 1:
                    continue
                mu        = float(getattr(store, 'mu',  0.0))
                tau       = float(getattr(store, 'tau', 0.0))
                sep_score = mu / (tau + 1e-8) if tau >= 0 else 0.0
                clean_tok = clean_token(str(token))
                if not clean_tok or is_punctuation_only(str(token)):
                    continue
                sep_items.append((clean_tok, n_protos, mu, tau, sep_score))
            except Exception:
                continue

        if not sep_items:
            print(f"{prefix} No valid prototype stores with mu/tau found")
            return

        sep_items.sort(key=lambda x: x[4], reverse=True)

        print(f"\n{prefix} Prototype Separation (top {min(top_n, len(sep_items))} / {len(sep_items)} tokens):")
        print(f"{'Token':<16}{'Protos':<8}{'Mu':<14}{'Tau':<14}{'Sep(mu/tau)':<14}")
        print("-" * 66)
        for tok, n_p, mu, tau, sep in sep_items[:top_n]:
            flag = "✅" if mu > 0.1 and sep > 1.0 else ("⚠️ " if mu > 0.01 else "❌")
            print(f"{tok[:15]:<16}{n_p:<8}{mu:<14.6f}{tau:<14.6f}{sep:<14.4f} {flag}")
        print("-" * 66)

        all_mu   = [x[2] for x in sep_items]
        all_sep  = [x[4] for x in sep_items]
        well_sep = sum(1 for x in sep_items if x[2] > 0.1 and x[4] > 1.0)
        print(
            f"{prefix} Stats: avg_mu={sum(all_mu)/len(all_mu):.6f} | "
            f"avg_sep={sum(all_sep)/len(all_sep):.4f} | "
            f"well_separated={well_sep}/{len(sep_items)} "
            f"({well_sep/len(sep_items)*100:.1f}%)"
        )
    except Exception as e:
        print(f"{prefix} separation log error: {type(e).__name__}: {str(e)[:100]}")


def _log_haug_gate_status_c8(model: torch.nn.Module, prefix: str = "[HAUG]") -> str:
    try:
        core = model.module if hasattr(model, 'module') else model
        dscd = getattr(core, 'dscd', None)
        if dscd is None or not hasattr(dscd, 'get_prototype_summary'):
            status = "UNKNOWN (DSCD unavailable)"
            print(f"{prefix} Gate status: {status}")
            return status

        summary      = dscd.get_prototype_summary()
        n_hom        = int(summary.get("num_homographs", 0))
        n_tok        = max(int(summary.get("total_tokens", 1)), 1)
        quality      = float(summary.get("quality_score", 0.0))
        hrate        = n_hom / n_tok
        _q_thresh    = float(globals().get("HAUG_GATE_QUALITY_THRESHOLD",   HAUG_GATE_QUALITY_THRESHOLD))
        _h_thresh    = float(globals().get("HAUG_GATE_HOMOGRAPH_THRESHOLD", HAUG_GATE_HOMOGRAPH_THRESHOLD))
        gate_open    = quality > _q_thresh and hrate > _h_thresh
        status       = "OPEN  ✅" if gate_open else "CLOSED ⛔"
        print(
            f"{prefix} Gate status: {status} | "
            f"quality={quality:.4f} (need >{_q_thresh:.2f}) | "
            f"homograph_rate={hrate*100:.2f}% (need >{_h_thresh*100:.2f}%)"
        )
        return "OPEN" if gate_open else "CLOSED"
    except Exception as e:
        status = f"ERROR ({type(e).__name__})"
        print(f"{prefix} Gate status: {status}")
        return status


class InferenceStatistics:
    def __init__(self):
        self._lock = threading.Lock()
        self.reset()

    def reset(self):
        with self._lock:
            self.total_inferences             = 0
            self.successful_translations      = 0
            self.failed_translations          = 0
            self.total_explanations           = 0
            self.high_confidence_explanations = 0
            self.low_confidence_explanations  = 0
            self.total_confidence             = 0.0
            self.dscd_homographs_explained    = set()
            self.reference_homographs_explained = set()
            self.avg_span                     = 0.0
            self.avg_uncertainty              = 0.0
            self.dscd_empty_warnings          = 0
            self.token_counts                 = defaultdict(int)
            self.token_confidences            = defaultdict(list)

    def record_inference(self, result: Dict[str, Any], dscd_homographs: Optional[set] = None):
        with self._lock:
            self.total_inferences += 1

            if result.get('translation') and result['translation'] != "ERROR DURING TRANSLATION":
                self.successful_translations += 1
            else:
                self.failed_translations += 1

            explanations = result.get('explanations', [])
            self.total_explanations += len(explanations)

            for exp in explanations:
                try:
                    conf = exp.get('confidence', 0.5)
                    self.total_confidence += float(conf)

                    if conf >= 0.65:
                        self.high_confidence_explanations += 1
                    elif conf < 0.4:
                        self.low_confidence_explanations += 1

                    word = str(exp.get('ambiguous_word', '')).strip()

                    if is_punctuation_only(word):
                        continue

                    clean_word = clean_token(word)

                    if not clean_word:
                        continue

                    self.token_counts[clean_word] += 1
                    self.token_confidences[clean_word].append(float(conf))

                    if dscd_homographs and clean_word in dscd_homographs:
                        self.dscd_homographs_explained.add(clean_word)

                    if clean_word.lower() in HOMOGRAPH_REFERENCE_LIST:
                        self.reference_homographs_explained.add(clean_word)

                    self.avg_span        += float(exp.get('span', 0.0))
                    self.avg_uncertainty += float(exp.get('uncertainty', 0.0))

                except Exception:
                    pass

    def get_summary(self) -> Dict[str, Any]:
        with self._lock:
            total_exp = max(self.total_explanations, 1)

            unique_tokens   = len(self.token_counts)
            diversity_ratio = unique_tokens / total_exp if total_exp > 0 else 0.0

            return {
                'total_inferences':               self.total_inferences,
                'successful_translations':        self.successful_translations,
                'failed_translations':            self.failed_translations,
                'success_rate':                   self.successful_translations / max(self.total_inferences, 1),
                'total_explanations':             self.total_explanations,
                'explanations_per_inference':     self.total_explanations / max(self.total_inferences, 1),
                'high_confidence_rate':           self.high_confidence_explanations / total_exp,
                'low_confidence_rate':            self.low_confidence_explanations / total_exp,
                'avg_confidence':                 self.total_confidence / total_exp,
                'avg_span':                       self.avg_span / total_exp,
                'avg_uncertainty':                self.avg_uncertainty / total_exp,
                'dscd_homographs_explained':      list(self.dscd_homographs_explained),
                'reference_homographs_explained': list(self.reference_homographs_explained),
                'dscd_empty_warnings':            self.dscd_empty_warnings,
                'unique_tokens_explained':        unique_tokens,
                'diversity_ratio':                diversity_ratio,
            }

    def print_summary(self):
        summary = self.get_summary()
        print("\n" + "=" * 80)
        print("INFERENCE STATISTICS SUMMARY")
        print("=" * 80)
        print(f"Total inferences: {summary['total_inferences']}")
        print(f"Success rate: {summary['success_rate']:.1%}")
        print(f"Total explanations: {summary['total_explanations']}")
        print(f"Explanations per inference: {summary['explanations_per_inference']:.2f}")
        print(f"Unique tokens explained: {summary['unique_tokens_explained']}")
        print(f"Diversity ratio: {summary['diversity_ratio']:.2%}")
        print(f"Avg confidence: {summary['avg_confidence']:.3f}")
        print(f"High confidence rate: {summary['high_confidence_rate']:.1%}")
        print(f"Avg span: {summary['avg_span']:.3f}")
        print(f"Avg uncertainty: {summary['avg_uncertainty']:.3f}")

        if summary['dscd_homographs_explained']:
            print(f"\nDSCD homographs explained ({len(summary['dscd_homographs_explained'])})")
            print(f"  {', '.join(summary['dscd_homographs_explained'])}")

        if summary['reference_homographs_explained']:
            print(f"\nReference homographs explained ({len(summary['reference_homographs_explained'])})")
            print(f"  {', '.join(summary['reference_homographs_explained'])}")

        if summary['dscd_empty_warnings'] > 0:
            print(f"\nDSCD empty warnings: {summary['dscd_empty_warnings']}")
        print("=" * 80 + "\n")


INFERENCE_STATS = InferenceStatistics()


def to_device_batch(enc: Any, device: torch.device):
    try:
        if hasattr(enc, "to") and callable(getattr(enc, "to")):
            return enc.to(device)
    except Exception:
        pass

    if isinstance(enc, dict):
        out = {}
        try:
            for k, v in enc.items():
                try:
                    if isinstance(v, torch.Tensor):
                        out[k] = v.to(device)
                    elif isinstance(v, dict):
                        out[k] = to_device_batch(v, device)
                    elif isinstance(v, (list, tuple)):
                        out[k] = [
                            t.to(device) if isinstance(t, torch.Tensor) else t
                            for t in v
                        ]
                    else:
                        out[k] = v
                except Exception:
                    out[k] = v
            return out
        except Exception:
            return enc

    return enc


def extract_dscd_outputs(raw_out: Any) -> Dict[str, Any]:
    if raw_out is None:
        return {}

    if isinstance(raw_out, dict):
        if "dscd_outputs" in raw_out and isinstance(raw_out["dscd_outputs"], dict):
            return raw_out["dscd_outputs"]
        if "dscd" in raw_out and isinstance(raw_out["dscd"], dict):
            return raw_out["dscd"]
        if "proto_probs" in raw_out or "uncertainties" in raw_out:
            return raw_out
        for key in ("dscd_outputs", "dscd", "dscd_out"):
            if key in raw_out and isinstance(raw_out[key], dict):
                return raw_out[key]
        return raw_out

    if isinstance(raw_out, (list, tuple)):
        for item in raw_out:
            if isinstance(item, dict):
                return extract_dscd_outputs(item)

    return {}


def is_subword_token(token: str) -> bool:
    if not token or len(token.strip()) == 0:
        return True

    token = token.strip()

    if is_punctuation_only(token):
        return True

    if (
        token.startswith("##")
        or token.startswith("▁▁")
        or token.startswith("@@")
    ):
        return True

    if len(token) < 2:
        return True

    if (len(token) == 1 and token in PUNCT_SET) or token.isdigit():
        return True

    return False


def should_filter_explanation(expl: Dict[str, Any], span_th: float, u_th: float) -> bool:
    try:
        token = expl.get('ambiguous_word', expl.get('token', ''))

        if is_punctuation_only(str(token)):
            return True

        span        = float(expl.get('span', 0.0))
        uncertainty = float(expl.get('uncertainty', 0.0))

        if is_subword_token(str(token)):
            return True

        if span < span_th and uncertainty < u_th:
            return True

        return False
    except Exception:
        return True


def has_bengali_chars(text: str) -> bool:
    if not text or not isinstance(text, str):
        return False
    return any('\u0980' <= c <= '\u09FF' for c in text)


def _get_lang_token_id_from_tokenizer(tokenizer, candidates: List[str]) -> Optional[int]:
    try:
        if hasattr(tokenizer, "lang_code_to_id") and isinstance(tokenizer.lang_code_to_id, dict):
            for c in candidates:
                if c in tokenizer.lang_code_to_id:
                    return int(tokenizer.lang_code_to_id[c])
    except Exception:
        pass

    try:
        if hasattr(tokenizer, "get_lang_id"):
            for c in candidates:
                try:
                    lid = tokenizer.get_lang_id(c)
                    if lid is not None and int(lid) >= 0:
                        return int(lid)
                except Exception:
                    continue
    except Exception:
        pass

    return None


def force_english_bos(tokenizer, mbart_model):
    try:
        lang_candidates = ["en_XX", "en"]
        lang_id = _get_lang_token_id_from_tokenizer(tokenizer, lang_candidates)
        if lang_id is not None and lang_id >= 0:
            return int(lang_id)

        if hasattr(mbart_model, 'config'):
            try:
                bos_id = getattr(mbart_model.config, 'forced_bos_token_id', None)
                if bos_id is not None and int(bos_id) >= 0:
                    return int(bos_id)
            except Exception:
                pass

        return _MBART50_EN_TOKEN_ID
    except Exception:
        return _MBART50_EN_TOKEN_ID


def force_bengali_bos(tokenizer, mbart_model):
    try:
        lang_candidates = ["bn_IN", "bn"]
        lang_id = _get_lang_token_id_from_tokenizer(tokenizer, lang_candidates)
        if lang_id is not None and lang_id >= 0:
            return int(lang_id)
        return _MBART50_BN_TOKEN_ID
    except Exception:
        return _MBART50_BN_TOKEN_ID


@torch.inference_mode()
def translate_with_explanations(
    model,
    tokenizer,
    input_sentence: str,
    source_lang: str = "bn_IN",
    target_lang: str  = "en_XX",
    device: Optional[torch.device] = None,
    max_length: Optional[int]       = None,
    span_threshold: Optional[float]       = None,
    uncertainty_threshold: Optional[float] = None,
    track_stats: bool = True,
) -> Dict[str, Any]:
    device  = _DEVICE if device is None else device
    max_len = MAXLEN if max_length is None else int(max_length)
    span_th = SPAN_THRESHOLD if span_threshold is None else float(span_threshold)
    u_th    = UNCERTAINTY_THRESHOLD if uncertainty_threshold is None else float(uncertainty_threshold)

    span_th = min(span_th, 0.20)
    u_th    = min(u_th, 0.10)

    if not input_sentence or not input_sentence.strip():
        return {
            "input_sentence":           input_sentence,
            "translation":              "",
            "ambiguous_words_detected": 0,
            "explanations":             [],
            "quality_metrics":          {},
            "dscd_validated":           False,
            "haug_gate_status":         "UNKNOWN",
            "error":                    "Empty input"
        }

    if not has_bengali_chars(input_sentence):
        if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
            print(f"[INF] WARNING: Input does not contain Bengali characters: {input_sentence[:50]}")

    try:
        tokenizer.src_lang = source_lang
    except Exception:
        if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
            print("[INF] Warning: Could not set tokenizer.src_lang")

    try:
        tokenizer.tgt_lang = target_lang
    except Exception:
        pass

    if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
        print(f"\n[INF] Starting inference:")
        print(f"[INF]   Input: {input_sentence[:60]}")
        print(f"[INF]   Languages: {source_lang} -> {target_lang}")
        print(f"[INF]   Thresholds: span={span_th:.2f}, uncertainty={u_th:.2f}")

    try:
        core = model.module if (_USE_MULTI_GPU and hasattr(model, "module")) else model
        dscd = getattr(core, 'dscd', None)
        if dscd and hasattr(dscd, 'cleanup_memory'):
            if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                print("[INF] Running DSCD cleanup before inference...")
            dscd.cleanup_memory()
    except Exception as e:
        if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
            print(f"[INF] DSCD cleanup failed: {type(e).__name__}")

    dscd_homographs  = get_dscd_homographs(model)
    haug_gate_status = "UNKNOWN"

    try:
        enc = tokenizer(
            input_sentence,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_len,
        )
        enc = to_device_batch(enc, device)

        model.eval()
        core = model.module if (_USE_MULTI_GPU and hasattr(model, "module")) else model

        src_texts = [input_sentence]

        dscd_validated = False
        try:
            dscd = core.dscd if hasattr(core, 'dscd') else None
            if dscd:
                lock = getattr(dscd, 'buffer_lock', None) or getattr(dscd, 'clustering_lock', None)

                num_stores  = 0
                multi_sense = 0

                if lock:
                    try:
                        with lock:
                            num_stores  = len(dscd.prototype_stores)
                            multi_sense = sum(
                                1
                                for store in dscd.prototype_stores.values()
                                if hasattr(store, 'centroids') and len(store.centroids) >= 2
                            )
                    except Exception:
                        pass
                else:
                    try:
                        num_stores  = len(dscd.prototype_stores)
                        multi_sense = sum(
                            1
                            for store in dscd.prototype_stores.values()
                            if hasattr(store, 'centroids') and len(store.centroids) >= 2
                        )
                    except Exception:
                        pass

                if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                    print(
                        f"[INF] DSCD state: {num_stores} tokens, "
                        f"{multi_sense} multi-sense, {len(dscd_homographs)} discovered"
                    )

                if num_stores == 0:
                    if track_stats:
                        INFERENCE_STATS.dscd_empty_warnings += 1
                else:
                    dscd_validated = True
        except Exception as e:
            if _DEBUG_DISCOVERY:
                print(f"[INF] DSCD validation failed: {e}")

        with torch.inference_mode():
            dscd_out_dict: Dict[str, Any] = {}
            fwd_outputs_raw: Any = None

            try:
                if not hasattr(core, "mbart"):
                    raise RuntimeError("Model backend missing .mbart")

                if _DEBUG_DISCOVERY:
                    print("[INF] Step 1: Running DSCD-augmented forward pass for explanations...")

                fwd_outputs_raw = core(
                    input_ids=enc.get("input_ids"),
                    attention_mask=enc.get("attention_mask"),
                    src_texts=src_texts,
                    token_word_map=None,
                    labels=None,
                    return_dict=True,
                    path=2
                )

                dscd_out_dict = extract_dscd_outputs(fwd_outputs_raw)

                if _DEBUG_DISCOVERY:
                    print(f"[INF] DSCD outputs extracted: {list(dscd_out_dict.keys()) if isinstance(dscd_out_dict, dict) else 'NOT_DICT'}")

                if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                    _log_scea_gates(dscd_out_dict, prefix="[SCEA]")

                if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                    if isinstance(fwd_outputs_raw, dict):
                        _log_hfcaa_gate(fwd_outputs_raw, prefix="[HFCAA]")
                    else:
                        _log_hfcaa_gate({}, prefix="[HFCAA]")

                if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                    _log_ambiguity_scores(dscd_out_dict, prefix="[AMB]")

            except Exception as e:
                if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                    print(f"[INF] DSCD forward error: {e}")
                    try:
                        traceback.print_exc()
                    except Exception:
                        pass
                dscd_out_dict   = {}
                fwd_outputs_raw = None

            forced_id        = force_english_bos(tokenizer, core.mbart)
            model_vocab_size = getattr(core, 'vocab_size', None) or getattr(getattr(core, 'mbart', None), 'vocab_size', None) or 250054
            try:
                model_vocab_size = int(model_vocab_size)
            except Exception:
                model_vocab_size = 250054

            if forced_id is None:
                forced_id = _MBART50_EN_TOKEN_ID

            if forced_id >= model_vocab_size:
                if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                    print(f"[INF] ⚠️  forced_id ({forced_id}) >= vocab_size ({model_vocab_size}) -- clamping")
                forced_id = min(forced_id, model_vocab_size - 1)

            if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                print(f"[INF] Target language: {target_lang} -> Token ID: {forced_id}")
                print(f"[INF] Vocab size: {model_vocab_size}, Token ID valid: {forced_id < model_vocab_size}")

            try:
                if _DEBUG_DISCOVERY:
                    print("[INF] Step 2: Running DSCD-augmented forward pass for translation...")

                with torch.no_grad():
                    fwd_out = core(
                        input_ids=enc.get("input_ids"),
                        attention_mask=enc.get("attention_mask"),
                        src_texts=src_texts,
                        token_word_map=None,
                        labels=None,
                        return_dict=True,
                        path=2
                    )

                if not dscd_out_dict and isinstance(fwd_out, dict):
                    dscd_out_dict = extract_dscd_outputs(fwd_out)
                    if dscd_out_dict and (_DEBUG_DISCOVERY or _VERBOSE_LOGGING):
                        print("[INF] dscd_out_dict re-populated from 2nd forward pass")
                        _log_scea_gates(dscd_out_dict,       prefix="[SCEA-2]")
                        _log_hfcaa_gate(fwd_out,             prefix="[HFCAA-2]")
                        _log_ambiguity_scores(dscd_out_dict, prefix="[AMB-2]")

                if not isinstance(fwd_out, dict):
                    raise ValueError(
                        f"[INF] Step 2 forward returned {type(fwd_out).__name__}, expected dict"
                    )

                h_sense = fwd_out.get('sense_augmented_embeddings', None)
                if h_sense is None:
                    h_sense = fwd_out.get('encoder_last_hidden_state', None)
                if h_sense is None:
                    _enc_out = fwd_out.get('encoder_outputs', None)
                    if _enc_out is not None and hasattr(_enc_out, 'last_hidden_state'):
                        h_sense = _enc_out.last_hidden_state
                    elif isinstance(_enc_out, torch.Tensor):
                        h_sense = _enc_out
                if h_sense is None:
                    h_sense = fwd_out.get('encoder_hidden_states', None)
                if h_sense is None:
                    h_sense = fwd_out.get('last_hidden_state', None)
                if h_sense is None:
                    raise ValueError("No encoder outputs found in forward pass")

                encoder_outputs_wrapped = BaseModelOutput(
                    last_hidden_state=h_sense,
                    hidden_states=None,
                    attentions=None
                )

                old_forced_bos = getattr(core.mbart.config, 'forced_bos_token_id', None)
                try:
                    core.mbart.config.forced_bos_token_id = forced_id

                    try:
                        generated = core.mbart.generate(
                            input_ids=None,
                            encoder_outputs=encoder_outputs_wrapped,
                            attention_mask=enc.get("attention_mask"),
                            max_length=max_len,
                            min_length=EVAL_MIN_LENGTH,
                            num_beams=EVAL_NUM_BEAMS,
                            early_stopping=True,
                            length_penalty=EVAL_LENGTH_PENALTY,
                            no_repeat_ngram_size=EVAL_NO_REPEAT_NGRAM_SIZE,
                            repetition_penalty=1.2,
                            forced_bos_token_id=forced_id,
                        )

                    except RuntimeError as oom_err:
                        if "out of memory" in str(oom_err).lower():
                            if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                                print("[INF] OOM during generation, retrying with reduced settings")
                            torch.cuda.empty_cache() if torch.cuda.is_available() else None

                            with torch.no_grad():
                                fwd_out = core(
                                    input_ids=enc.get("input_ids"),
                                    attention_mask=enc.get("attention_mask"),
                                    src_texts=src_texts,
                                    token_word_map=None,
                                    labels=None,
                                    return_dict=True,
                                    path=2
                                )

                            if not isinstance(fwd_out, dict):
                                raise ValueError(
                                    f"[INF] OOM retry forward returned {type(fwd_out).__name__}, expected dict"
                                )

                            h_sense = fwd_out.get('sense_augmented_embeddings', None)
                            if h_sense is None:
                                h_sense = fwd_out.get('encoder_last_hidden_state', None)
                            if h_sense is None:
                                _enc_out_oom = fwd_out.get('encoder_outputs', None)
                                if _enc_out_oom is not None and hasattr(_enc_out_oom, 'last_hidden_state'):
                                    h_sense = _enc_out_oom.last_hidden_state
                                elif isinstance(_enc_out_oom, torch.Tensor):
                                    h_sense = _enc_out_oom
                            if h_sense is None:
                                h_sense = fwd_out.get('encoder_hidden_states', None)
                            if h_sense is None:
                                h_sense = fwd_out.get('last_hidden_state', None)

                            encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=h_sense)

                            generated = core.mbart.generate(
                                input_ids=None,
                                encoder_outputs=encoder_outputs_wrapped,
                                attention_mask=enc.get("attention_mask"),
                                max_length=min(max_len, 128),
                                min_length=EVAL_MIN_LENGTH,
                                num_beams=max(1, min(4, EVAL_NUM_BEAMS)),
                                early_stopping=True,
                                length_penalty=EVAL_LENGTH_PENALTY,
                                no_repeat_ngram_size=EVAL_NO_REPEAT_NGRAM_SIZE,
                                repetition_penalty=1.2,
                                forced_bos_token_id=forced_id,
                            )
                        else:
                            raise

                finally:
                    if old_forced_bos is not None:
                        core.mbart.config.forced_bos_token_id = old_forced_bos

                if generated is None:
                    translation = ""
                else:
                    try:
                        gen_ids     = generated[0] if isinstance(generated, (list, tuple)) else generated[0]
                        gen_ids     = gen_ids.detach().cpu().numpy().tolist() if hasattr(gen_ids, "detach") else gen_ids
                        translation = tokenizer.decode(gen_ids, skip_special_tokens=True)
                    except Exception:
                        try:
                            translation = tokenizer.decode(generated[0], skip_special_tokens=True)
                        except Exception:
                            translation = ""

                if _DEBUG_DISCOVERY:
                    print(f"[INF] Translation: {translation[:60] if translation else 'EMPTY'}")

                if not translation or not translation.strip():
                    error_msg = "Empty generation result"
                    if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                        print(f"[INF] ERROR: {error_msg}")
                    return {
                        "input_sentence":           input_sentence,
                        "translation":              "",
                        "ambiguous_words_detected": 0,
                        "explanations":             [],
                        "quality_metrics":          {},
                        "dscd_validated":           dscd_validated,
                        "haug_gate_status":         "UNKNOWN",
                        "error":                    error_msg
                    }

            except Exception as e:
                if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                    print(f"[INF] Generation error: {type(e).__name__}: {str(e)}")
                    try:
                        traceback.print_exc()
                    except Exception:
                        pass

                return {
                    "input_sentence":           input_sentence,
                    "translation":              "",
                    "ambiguous_words_detected": 0,
                    "explanations":             [],
                    "quality_metrics":          {},
                    "dscd_validated":           dscd_validated,
                    "haug_gate_status":         "UNKNOWN",
                    "error":                    f"Generation failed: {type(e).__name__}"
                }

            if _DEBUG_DISCOVERY:
                print("[INF] Step 5: Calling TRG to generate explanations...")

            out_explanations: List[Dict[str, Any]] = []

            try:
                trg = core.trg if hasattr(core, 'trg') else None

                if trg and hasattr(trg, 'process_sentence_for_explanations'):
                    try:
                        tokens_list = tokenizer.convert_ids_to_tokens(enc['input_ids'][0].tolist())

                        if _DEBUG_DISCOVERY:
                            print(f"[INF] Calling TRG with {len(tokens_list)} tokens")

                        trg_explanations = trg.process_sentence_for_explanations(
                            tokens=tokens_list,
                            dscd_outputs=dscd_out_dict,
                            token_word_map=None,
                            uncertainty_threshold=u_th,
                            decoder_attention=None
                        )

                        if _DEBUG_DISCOVERY:
                            print(f"[INF] TRG returned {len(trg_explanations) if isinstance(trg_explanations, list) else 0} explanations")

                        if isinstance(trg_explanations, list):
                            for exp in trg_explanations:
                                try:
                                    raw_word = exp.get('token', '')

                                    if is_punctuation_only(str(raw_word)):
                                        continue

                                    clean_word = clean_token(str(raw_word)) if raw_word else ''

                                    if not clean_word:
                                        continue

                                    if should_filter_explanation(exp, span_th, u_th):
                                        continue

                                    s = float(exp.get('span', 0.0))
                                    u = float(exp.get('uncertainty', 0.0))
                                    confidence = max(s, u)

                                    expl_text = exp.get('explanation', '')
                                    if not expl_text:
                                        continue

                                    out_explanations.append({
                                        "ambiguous_word": clean_word,
                                        "position":       exp.get("token_idx", "N/A"),
                                        "explanation":    expl_text,
                                        "uncertainty":    u,
                                        "span":           s,
                                        "confidence":     confidence,
                                        "is_real_amb":    bool((s > span_th) or (u > u_th)),
                                    })
                                except Exception as e:
                                    if _DEBUG_DISCOVERY:
                                        print(f"[INF] Error processing TRG explanation: {e}")
                                    continue

                    except Exception as e:
                        if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                            print(f"[INF] TRG processing failed: {e}")
                            try:
                                traceback.print_exc()
                            except Exception:
                                pass
                else:
                    if _DEBUG_DISCOVERY:
                        print("[INF] TRG not available or missing process_sentence_for_explanations()")

            except Exception as e:
                if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                    print(f"[INF] TRG invocation error: {e}")

            real_amb_count = sum(1 for e in out_explanations if e.get('is_real_amb', False))

            quality_metrics = {
                'total_raw_explanations': len(out_explanations),
                'filtered_explanations':  0,
                'high_confidence_count':  sum(1 for e in out_explanations if e.get('confidence', 0) >= 0.65),
                'low_confidence_count':   sum(1 for e in out_explanations if e.get('confidence', 0) < 0.4),
                'avg_confidence':         sum(e.get('confidence', 0) for e in out_explanations) / max(len(out_explanations), 1),
                'avg_span':               sum(e.get('span', 0) for e in out_explanations) / max(len(out_explanations), 1),
                'avg_uncertainty':        sum(e.get('uncertainty', 0) for e in out_explanations) / max(len(out_explanations), 1),
            }

            if _DEBUG_DISCOVERY:
                print(
                    f"[INF] Final: {len(out_explanations)} explanations "
                    f"(real ambiguous: {real_amb_count})"
                )

            try:
                haug_gate_status = _log_haug_gate_status_c8(model, prefix="[HAUG]")
            except Exception:
                haug_gate_status = "UNKNOWN"

            result = {
                "input_sentence":           input_sentence,
                "translation":              translation,
                "ambiguous_words_detected": int(real_amb_count),
                "explanations":             out_explanations,
                "quality_metrics":          quality_metrics,
                "dscd_validated":           dscd_validated,
                "haug_gate_status":         haug_gate_status,
            }

            if track_stats:
                INFERENCE_STATS.record_inference(result, dscd_homographs=dscd_homographs)

            return result

    except Exception as e:
        if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
            print(f"[INF] ERROR: {type(e).__name__}: {str(e)[:200]}")
            try:
                traceback.print_exc()
            except Exception:
                pass

        error_result = {
            "input_sentence":           input_sentence,
            "translation":              "ERROR DURING TRANSLATION",
            "ambiguous_words_detected": 0,
            "explanations":             [],
            "quality_metrics":          {},
            "dscd_validated":           False,
            "haug_gate_status":         "UNKNOWN",
            "error":                    f"{type(e).__name__}: {str(e)[:150]}",
        }

        if track_stats:
            INFERENCE_STATS.record_inference(error_result, dscd_homographs=dscd_homographs)

        return error_result

    finally:
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass

        try:
            if gc.isenabled():
                gc.collect()
        except Exception:
            pass


def demonstrate_system(model, tokenizer, sentences: Optional[List[str]] = None):
    if sentences is None:
        sentences = [
            "আমি কল বন্ধ করেছি।",
            "কাল আমি বই কিনব।",
            "পাতা ঝরে পড়েছে।",
            "তিনি ব্যাংক গেছেন।",
            "আজ ভাল আবহাওয়া।",
        ]

    print("=" * 80)
    print("TATN DEMO: Translation + Explanations")
    print("=" * 80)

    print("\n[DEMO] Gate Diagnostics:")
    _log_haug_gate_status_c8(model, prefix="[HAUG]")

    _log_prototype_separation(model, prefix="[PROTO-SEP]", top_n=5)

    INFERENCE_STATS.reset()

    for s in sentences:
        print(f"\nInput: {s}")
        res = translate_with_explanations(
            model,
            tokenizer,
            s,
            source_lang=SOURCE_LANG,
            target_lang=TARGET_LANG,
        )
        print("Translation:", res.get("translation", ""))
        print("Ambiguous words detected:", res.get("ambiguous_words_detected", 0))

        haug_status = res.get("haug_gate_status", "UNKNOWN")
        print(f"HAUG Gate: {haug_status}")

        quality = res.get("quality_metrics", {})
        if quality:
            print(
                f"Quality: conf={quality.get('avg_confidence', 0):.3f}, "
                f"high={quality.get('high_confidence_count', 0)}, "
                f"low={quality.get('low_confidence_count', 0)}"
            )

        if res.get("explanations"):
            for idx, ex in enumerate(res["explanations"], 1):
                print(
                    f"  {idx}. '{ex['ambiguous_word']}' "
                    f"pos={ex['position']} conf={ex.get('confidence', 0):.3f}"
                )
                print("     ", ex.get("explanation", "")[:200])
        else:
            print("  No explanations")

    print("=" * 80)
    INFERENCE_STATS.print_summary()


def dscd_discovery_warmup(
    model,
    tokenizer,
    num_sents: int = 8000,
    batch_size: int = 64,
    max_len: Optional[int] = None,
):
    if max_len is None:
        max_len = MAXLEN

    core = model.module if (_USE_MULTI_GPU and hasattr(model, "module")) else model

    try:
        dscd = getattr(core, "dscd", None)
        if dscd is None:
            print("[WARMUP] Model has no dscd component")
            return

        print("\n" + "=" * 80)
        print("[WARMUP] Starting DSCD discovery warmup")
        print("=" * 80)

        orig_enable = getattr(dscd, "enable_training_clustering", False)
        orig_n_min  = getattr(dscd, "n_min", None)
        orig_buffer = getattr(dscd, "buffer_size", None)

        try:
            if hasattr(dscd, "enable_training_clustering"):
                dscd.enable_training_clustering = True
            if hasattr(dscd, "n_min"):
                dscd.n_min = max(3, int(getattr(dscd, "n_min", 5)))
            if hasattr(dscd, "buffer_size"):
                dscd.buffer_size = max(200, int(getattr(dscd, "buffer_size", 300)))
        except Exception:
            pass

        texts: List[str] = []
        try:
            if "load_and_preprocess_optimized" in globals():
                pairs = load_and_preprocess_optimized(num_sents)
                texts = [bn for (bn, _) in pairs][:num_sents]
            else:
                base = [
                    "আমি কল বন্ধ করেছি।",
                    "কাল আমি বই কিনব।",
                    "পাতা ঝরে পড়েছে।",
                    "তিনি ব্যাংক গেছেন।",
                ]
                while len(texts) < num_sents:
                    texts.extend(base)
                texts = texts[:num_sents]
        except Exception:
            texts = ["আমি কল বন্ধ করেছি।"] * num_sents

        processed = 0
        core.eval()

        print(f"\n[WARMUP] Processing {len(texts)} sentences (batch={batch_size})...")

        start_time = time.time()
        last_print  = start_time

        with torch.inference_mode():
            for i in range(0, len(texts), batch_size):
                batch = texts[i : i + batch_size]
                try:
                    enc = tokenizer(
                        batch,
                        return_tensors="pt",
                        padding=True,
                        truncation=True,
                        max_length=max_len,
                    )
                    enc = to_device_batch(enc, _DEVICE)

                    _fwd_called = False
                    if hasattr(core, 'forward_path1'):
                        try:
                            core.forward_path1(
                                input_ids=enc.get("input_ids"),
                                attention_mask=enc.get("attention_mask"),
                                src_texts=batch,
                                token_word_map=None,
                            )
                            _fwd_called = True
                        except Exception:
                            _fwd_called = False

                    if not _fwd_called:
                        try:
                            core(
                                input_ids=enc.get("input_ids"),
                                attention_mask=enc.get("attention_mask"),
                                src_texts=batch,
                                token_word_map=None,
                                labels=None,
                                return_dict=True,
                                path=1,
                            )
                            _fwd_called = True
                        except Exception:
                            _fwd_called = False

                    if not _fwd_called:
                        core(
                            input_ids=enc.get("input_ids"),
                            attention_mask=enc.get("attention_mask"),
                            src_texts=batch,
                            token_word_map=None,
                            labels=None,
                            return_dict=True,
                            path=2,
                        )

                    processed += len(batch)

                    current_time = time.time()
                    if (i // batch_size) % 10 == 0 or (current_time - last_print) > 5:
                        elapsed = current_time - start_time
                        rate    = processed / elapsed if elapsed > 0 else 0
                        eta     = (len(texts) - processed) / rate if rate > 0 else 0
                        print(
                            f"[WARMUP] {processed}/{len(texts)} "
                            f"({processed/len(texts)*100:.1f}%) | "
                            f"{rate:.1f} sent/s | ETA {eta:.0f}s"
                        )
                        last_print = current_time

                    del enc

                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        print(f"[WARMUP] OOM at batch {i//batch_size}, cleaning up...")
                        torch.cuda.empty_cache() if torch.cuda.is_available() else None
                        gc.collect()
                        continue
                    else:
                        print(f"[WARMUP] Batch {i//batch_size} failed: {str(e)[:100]}")
                        continue
                except Exception as e:
                    print(f"[WARMUP] Batch {i//batch_size} failed: {str(e)[:100]}")
                    continue

        total_time = time.time() - start_time
        print(
            f"\n[WARMUP] Completed in {total_time:.1f}s "
            f"({processed/total_time:.1f} sent/s)"
        )
        print("-" * 80)

        try:
            if dscd and hasattr(dscd, 'cleanup_memory'):
                print("[WARMUP] Running DSCD cleanup after warmup...")
                dscd.cleanup_memory()
        except Exception as e:
            if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                print(f"[WARMUP] DSCD cleanup failed: {type(e).__name__}")

        try:
            lock = None
            if hasattr(dscd, 'buffer_lock'):
                lock = dscd.buffer_lock
            elif hasattr(dscd, 'clustering_lock'):
                lock = dscd.clustering_lock

            if lock:
                with lock:
                    stores = dict(dscd.prototype_stores)
            else:
                stores = dict(dscd.prototype_stores)

            num_types    = len(stores)
            total_protos = (
                sum(store.size() for store in stores.values()) if stores else 0
            )
            multi = (
                sum(1 for store in stores.values() if store.size() >= 2)
                if stores
                else 0
            )

            print("[WARMUP] Summary:")
            print(f"  - Token types: {num_types}")
            print(f"  - Total prototypes: {total_protos}")
            print(f"  - Multi-sense tokens: {multi}")

            if num_types > 0:
                print(f"  - Multi-sense ratio: {multi/num_types:.1%}")

            dscd_homographs_w = get_dscd_homographs(model)

            print(f"\n[WARMUP] Discovered Homographs: {len(dscd_homographs_w)}")
            if dscd_homographs_w:
                print(f"  Sample: {list(dscd_homographs_w)[:10]}")

            reference_found = dscd_homographs_w.intersection(
                set(w.lower() for w in HOMOGRAPH_REFERENCE_LIST)
            )

            print(f"\n[WARMUP] Reference List Comparison:")
            print(f"  - Reference list: {len(HOMOGRAPH_REFERENCE_LIST)} words")
            print(f"  - Found in DSCD: {len(reference_found)}")
            print(
                f"  - Coverage: {len(reference_found)/len(HOMOGRAPH_REFERENCE_LIST):.1%}"
            )

            print("\n[WARMUP] Prototype Separation Scores for Found Reference Tokens:")
            sep_found = []
            for token_key, store in stores.items():
                try:
                    clean_tok = clean_token(str(token_key))
                    if clean_tok.lower() in reference_found:
                        mu  = float(getattr(store, 'mu', 0.0))
                        tau = float(getattr(store, 'tau', 0.0))
                        sep = mu / (tau + 1e-8)
                        n_p = store.size() if hasattr(store, 'size') else 0
                        sep_found.append((clean_tok, n_p, mu, tau, sep))
                except Exception:
                    continue

            if sep_found:
                sep_found.sort(key=lambda x: x[4], reverse=True)
                print(f"  {'Token':<16}{'Protos':<8}{'Mu':<14}{'Tau':<14}{'Sep':<12}{'Status'}")
                print("  " + "-" * 64)
                for tok, n_p, mu, tau, sep in sep_found:
                    flag = "✅ GOOD" if mu > 0.1 and sep > 1.0 else ("⚠️  WEAK" if mu > 0.01 else "❌ NONE")
                    print(f"  {tok[:15]:<16}{n_p:<8}{mu:<14.6f}{tau:<14.6f}{sep:<12.4f}{flag}")
            else:
                print("  No reference tokens found in prototype stores yet")

            if num_types == 0:
                print("\n[WARMUP] CRITICAL: NO PROTOTYPES CREATED")
            elif len(reference_found) < len(HOMOGRAPH_REFERENCE_LIST) // 2:
                print("\n[WARMUP] WARNING: < 50% reference coverage")
            else:
                print("\n[WARMUP] SUCCESS")

        except Exception as e:
            print(f"[WARMUP] Validation failed: {e}")

    finally:
        try:
            if dscd is not None:
                if hasattr(dscd, "enable_training_clustering"):
                    dscd.enable_training_clustering = orig_enable
                if hasattr(dscd, "n_min") and orig_n_min is not None:
                    dscd.n_min = orig_n_min
                if hasattr(dscd, "buffer_size") and orig_buffer is not None:
                    dscd.buffer_size = orig_buffer
        except Exception:
            pass

        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass

        try:
            if gc.isenabled():
                gc.collect()
        except Exception:
            pass

        print("=" * 80 + "\n")


def final_evaluation_with_bleu(
    model,
    tokenizer,
    test_data: List[Tuple[str, str]],
    device: Optional[torch.device] = None,
    max_length: Optional[int] = None,
    batch_size: int = 16,
) -> Dict[str, Any]:
    device  = _DEVICE if device is None else device
    max_len = MAXLEN if max_length is None else int(max_length)

    print("\n" + "=" * 80)
    print("FINAL EVALUATION WITH BLEU/CHRF++")
    print("=" * 80)
    print(f"Test samples: {len(test_data)}")
    print(f"Batch size: {batch_size}")
    print(f"Max length: {max_len}")
    print("=" * 80 + "\n")

    try:
        core = model.module if (_USE_MULTI_GPU and hasattr(model, "module")) else model
        dscd = getattr(core, 'dscd', None)
        if dscd and hasattr(dscd, 'cleanup_memory'):
            print("[EVAL] Running DSCD cleanup before evaluation...")
            dscd.cleanup_memory()
    except Exception as e:
        if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
            print(f"[EVAL] DSCD cleanup failed: {type(e).__name__}")

    print("\n[EVAL] Prototype Separation Diagnostics:")
    _log_prototype_separation(model, prefix="[PROTO-SEP]", top_n=10)

    _log_haug_gate_status_c8(model, prefix="[HAUG]")

    INFERENCE_STATS.reset()

    predictions                    = []
    references                     = []
    translations_with_explanations = []

    model.eval()

    try:
        from sacrebleu.metrics import BLEU, CHRF
        bleu_metric       = BLEU()
        chrf_metric       = CHRF()
        metrics_available = True
    except ImportError:
        print("[EVAL] WARNING: sacrebleu not available, BLEU/CHRF scores will not be computed")
        metrics_available = False

    start_time = time.time()

    with torch.inference_mode():
        for i in range(0, len(test_data), batch_size):
            batch = test_data[i:i+batch_size]

            for src, ref in batch:
                try:
                    result = translate_with_explanations(
                        model,
                        tokenizer,
                        src,
                        source_lang=SOURCE_LANG,
                        target_lang=TARGET_LANG,
                        device=device,
                        max_length=max_len,
                        track_stats=True,
                    )

                    translation = result.get('translation', '')

                    predictions.append(translation)
                    references.append(ref)
                    translations_with_explanations.append({
                        'source':          src,
                        'reference':       ref,
                        'translation':     translation,
                        'explanations':    result.get('explanations', []),
                        'ambiguous_words': result.get('ambiguous_words_detected', 0),
                        'haug_gate':       result.get('haug_gate_status', 'UNKNOWN'),
                    })

                except Exception as e:
                    if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                        print(f"[EVAL] Translation failed for: {src[:50]} - {type(e).__name__}")
                    predictions.append("")
                    references.append(ref)
                    translations_with_explanations.append({
                        'source':          src,
                        'reference':       ref,
                        'translation':     "ERROR",
                        'explanations':    [],
                        'ambiguous_words': 0,
                        'haug_gate':       'UNKNOWN',
                    })

            if (i // batch_size) % 10 == 0:
                elapsed   = time.time() - start_time
                processed = min(i + batch_size, len(test_data))
                rate      = processed / elapsed if elapsed > 0 else 0
                eta       = (len(test_data) - processed) / rate if rate > 0 else 0
                print(f"[EVAL] {processed}/{len(test_data)} ({processed/len(test_data)*100:.1f}%) | {rate:.1f} sent/s | ETA {eta:.0f}s")

    total_time = time.time() - start_time
    print(f"\n[EVAL] Translation completed in {total_time:.1f}s ({len(test_data)/total_time:.1f} sent/s)")

    results = {
        'total_samples':                  len(test_data),
        'successful_translations':        sum(1 for p in predictions if p and p != "ERROR"),
        'failed_translations':            sum(1 for p in predictions if not p or p == "ERROR"),
        'total_time':                     total_time,
        'throughput':                     len(test_data) / total_time,
        'predictions':                    predictions,
        'references':                     references,
        'translations_with_explanations': translations_with_explanations,
    }

    if metrics_available and predictions and references:
        try:
            valid_preds = []
            valid_refs  = []
            for p, r in zip(predictions, references):
                if p and p != "ERROR" and r:
                    valid_preds.append(p)
                    valid_refs.append(r)

            if valid_preds:
                bleu_score = bleu_metric.corpus_score(valid_preds, [valid_refs])
                chrf_score = chrf_metric.corpus_score(valid_preds, [valid_refs])

                results['bleu'] = float(bleu_score.score)
                results['chrf'] = float(chrf_score.score)

                print("\n" + "=" * 80)
                print("METRIC SCORES")
                print("=" * 80)
                print(f"BLEU:    {results['bleu']:.2f}")
                print(f"CHRF++:  {results['chrf']:.2f}")
                print(f"Valid samples: {len(valid_preds)}/{len(predictions)}")
                print("=" * 80)
            else:
                print("[EVAL] WARNING: No valid translations for BLEU/CHRF computation")
                results['bleu'] = 0.0
                results['chrf'] = 0.0
        except Exception as e:
            print(f"[EVAL] Metric computation failed: {type(e).__name__}: {str(e)[:100]}")
            results['bleu'] = 0.0
            results['chrf'] = 0.0
    else:
        results['bleu'] = 0.0
        results['chrf'] = 0.0

    print("\n" + "=" * 80)
    print("EVALUATION SUMMARY")
    print("=" * 80)
    print(f"Total samples: {results['total_samples']}")
    print(f"Successful: {results['successful_translations']}")
    print(f"Failed: {results['failed_translations']}")
    print(f"Success rate: {results['successful_translations']/results['total_samples']:.1%}")
    print(f"Throughput: {results['throughput']:.1f} sent/s")
    print("=" * 80 + "\n")

    INFERENCE_STATS.print_summary()

    return results


# =======================================================================================
# CRITICAL FIX — load_checkpoint_for_resume(): DSCD per-store restore + epoch fallback
# =======================================================================================
def load_checkpoint_for_resume(
    model: torch.nn.Module, optimizer, checkpoint_path: Optional[str] = None
) -> Tuple[bool, int, int, float]:
    if checkpoint_path is None:
        try:
            checkpoint_path = get_checkpoint_path()
        except (NameError, Exception):
            checkpoint_path = os.path.join(
                globals().get("CHECKPOINT_DIR", "/kaggle/working"),
                globals().get("CHECKPOINT_FILENAME", "tatn_final.pt")
            )

    if not os.path.exists(checkpoint_path):
        print(f"[CHECKPOINT] Not found: {checkpoint_path}")
        return False, 0, 0, 0.0

    try:
        ckpt = torch.load(checkpoint_path, map_location=_DEVICE, weights_only=False)
    except Exception as e:
        print(f"[CHECKPOINT] Load failed: {e}")
        return False, 0, 0, 0.0

    core = model.module if (_USE_MULTI_GPU and hasattr(model, "module")) else model

    state = ckpt.get("model_state_dict", ckpt)
    try:
        core.load_state_dict(state, strict=False)
    except Exception as e:
        print(f"[CHECKPOINT] model.load_state_dict failed: {e}")
        try:
            if isinstance(state, dict):
                new_state = {}
                for k, v in state.items():
                    new_key = k.replace("module.", "") if k.startswith("module.") else k
                    new_state[new_key] = v
                core.load_state_dict(new_state, strict=False)
        except Exception:
            pass

    try:
        if optimizer is not None and "optimizer_state_dict" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    except Exception as e:
        print(f"[CHECKPOINT] optimizer.load_state_dict failed: {e}")

    restored      = 0
    skipped       = 0
    dscd_restored = False

    try:
        if "dscd_state" in ckpt and ckpt["dscd_state"]:
            dscd_state = ckpt["dscd_state"]
            print("[CHECKPOINT] Restoring DSCD prototype stores...")

            dscd = core.dscd if hasattr(core, 'dscd') else None

            if dscd is not None and isinstance(dscd_state, dict):
                lock = None
                if hasattr(dscd, 'buffer_lock'):
                    lock = dscd.buffer_lock
                elif hasattr(dscd, 'clustering_lock'):
                    lock = dscd.clustering_lock

                def _do_restore():
                    nonlocal restored, skipped
                    for token, store_sd in dscd_state.items():
                        try:
                            if not isinstance(store_sd, dict):
                                skipped += 1
                                continue
                            if token not in dscd.prototype_stores:
                                if hasattr(dscd, '_create_store'):
                                    dscd.prototype_stores[token] = dscd._create_store()
                                elif hasattr(dscd, 'MemoryEfficientPrototypeStore'):
                                    dscd.prototype_stores[token] = dscd.MemoryEfficientPrototypeStore()
                                else:
                                    skipped += 1
                                    continue
                            dscd.prototype_stores[token].load_state_dict(store_sd)
                            restored += 1
                        except Exception as per_store_err:
                            if _DEBUG_DISCOVERY:
                                print(f"[CHECKPOINT] Store restore failed for '{token}': {per_store_err}")
                            skipped += 1
                            continue

                if lock:
                    with lock:
                        _do_restore()
                else:
                    _do_restore()

                if lock:
                    with lock:
                        num_tokens   = len(dscd.prototype_stores)
                        total_protos = sum(store.size() for store in dscd.prototype_stores.values())
                        multi_sense  = sum(1 for store in dscd.prototype_stores.values() if store.size() >= 2)
                else:
                    num_tokens   = len(dscd.prototype_stores)
                    total_protos = sum(store.size() for store in dscd.prototype_stores.values())
                    multi_sense  = sum(1 for store in dscd.prototype_stores.values() if store.size() >= 2)

                print(f"[CHECKPOINT] DSCD per-store restore: restored={restored}, skipped={skipped}")
                print(f"[CHECKPOINT] DSCD state after restore:")
                print(f"  - Tokens:      {num_tokens}")
                print(f"  - Prototypes:  {total_protos}")
                print(f"  - Multi-sense: {multi_sense}")

                _log_prototype_separation(model, prefix="[PROTO-SEP]", top_n=8)

                if restored > 0:
                    dscd_restored = True

                if num_tokens == 0:
                    print("[CHECKPOINT] WARNING: DSCD state empty after restore")
            else:
                if dscd is None:
                    print("[CHECKPOINT] Model has no .dscd attribute")
                else:
                    print(f"[CHECKPOINT] dscd_state is not a dict (type={type(dscd_state).__name__}), skipping per-store restore")
        else:
            print("[CHECKPOINT] No dscd_state key in checkpoint")

    except Exception as e:
        print(f"[CHECKPOINT] DSCD restore block failed: {e}")
        if _DEBUG_DISCOVERY:
            traceback.print_exc()

    if not dscd_restored:
        print("[CHECKPOINT] Inline DSCD restore yielded 0 stores — scanning for epoch prototype files...")
        try:
            _ckpt_dir   = globals().get("CHECKPOINT_DIR", os.path.dirname(checkpoint_path))
            proto_files = sorted(
                glob.glob(os.path.join(_ckpt_dir, "dscd_prototypes_epoch*.pt")),
                key=os.path.getmtime,
                reverse=True
            )

            if proto_files:
                latest_proto_file = proto_files[0]
                print(f"[CHECKPOINT] Found {len(proto_files)} epoch prototype file(s). Loading latest: {latest_proto_file}")

                load_dscd_prototypes = globals().get("load_dscd_prototypes", None)
                if load_dscd_prototypes is not None and callable(load_dscd_prototypes):
                    try:
                        load_dscd_prototypes(core, latest_proto_file)
                        print(f"[CHECKPOINT] Fallback prototype load succeeded from: {latest_proto_file}")

                        dscd = core.dscd if hasattr(core, 'dscd') else None
                        if dscd is not None:
                            lock = getattr(dscd, 'buffer_lock', None) or getattr(dscd, 'clustering_lock', None)
                            if lock:
                                with lock:
                                    fb_tokens   = len(dscd.prototype_stores)
                                    fb_protos   = sum(store.size() for store in dscd.prototype_stores.values())
                                    fb_multi    = sum(1 for store in dscd.prototype_stores.values() if store.size() >= 2)
                            else:
                                fb_tokens   = len(dscd.prototype_stores)
                                fb_protos   = sum(store.size() for store in dscd.prototype_stores.values())
                                fb_multi    = sum(1 for store in dscd.prototype_stores.values() if store.size() >= 2)

                            print(f"[CHECKPOINT] Fallback DSCD state: tokens={fb_tokens}, protos={fb_protos}, multi-sense={fb_multi}")
                            _log_prototype_separation(model, prefix="[PROTO-SEP-FB]", top_n=8)
                    except Exception as fb_err:
                        print(f"[CHECKPOINT] Fallback prototype load failed: {fb_err}")
                else:
                    print("[CHECKPOINT] load_dscd_prototypes() not found in globals — skipping fallback load")
            else:
                print(f"[CHECKPOINT] No dscd_prototypes_epoch*.pt files found in {_ckpt_dir}")

        except Exception as scan_err:
            print(f"[CHECKPOINT] Epoch prototype file scan failed: {scan_err}")

    epoch    = int(ckpt.get("epochs_trained", ckpt.get("epoch", 0)))
    step     = int(
        ckpt.get(
            "global_step",
            ckpt.get("global_steps", ckpt.get("step", 0))
        )
    )
    avg_loss = float(
        ckpt.get(
            "final_train_loss",
            ckpt.get("avg_epoch_loss", ckpt.get("avg_loss", 0.0)),
        )
    )

    print(f"[CHECKPOINT] Loaded: epoch={epoch} step={step} loss={avg_loss:.6f}")
    return True, epoch, step, avg_loss


# =======================================================================================
# Cell 8 execution block
# =======================================================================================

print("\n" + "=" * 80)
print("Cell 8: Inference & Evaluation Pipeline (DUAL-PATH COMPATIBLE) - LOADED")
print("=" * 80)
print("Configuration:")
print(f"  - Source language: {SOURCE_LANG}")
print(f"  - Target language: {TARGET_LANG}")
print(f"  - Span threshold: {SPAN_THRESHOLD}")
print(f"  - Uncertainty threshold: {UNCERTAINTY_THRESHOLD}")
print(f"  - Max length: {MAXLEN}")
print(f"  - Device: {_DEVICE}")
print(f"  - mBART-50 Bengali token ID (fallback): {_MBART50_BN_TOKEN_ID}")
print(f"  - mBART-50 English token ID (fallback): {_MBART50_EN_TOKEN_ID}")
print(f"  - Eval num beams: {EVAL_NUM_BEAMS}")
print(f"  - Eval length penalty: {EVAL_LENGTH_PENALTY}")
print(f"  - Eval no repeat ngram size: {EVAL_NO_REPEAT_NGRAM_SIZE}")
print(f"  - Eval min length: {EVAL_MIN_LENGTH}")
print(f"  - HAUG gate quality threshold: {HAUG_GATE_QUALITY_THRESHOLD}")
print(f"  - HAUG gate homograph threshold: {HAUG_GATE_HOMOGRAPH_THRESHOLD}")
print("\n✅ ALL FIXES APPLIED:")
print("  ✅ [FIX-CRITICAL-1] _log_haug_gate_status_c8(): Removed / 100.0 — gate can now open")
print("  ✅ [FIX-BUG-2]      is_subword_token(): ▁ no longer classified as subword marker")
print("  ✅ [FIX-BUG-3]      translate_with_explanations(): src_lang set before tokenize, tgt_lang in separate guard")
print("  ✅ [FIX-BUG-4]      load_checkpoint_for_resume(): global_step (singular) checked first")
print("  ✅ [FIX-GEN-1]      translate_with_explanations() Step 2: h_sense lookup order fixed")
print("  ✅ [FIX-GEN-2]      translate_with_explanations() OOM retry: same robust h_sense chain")
print("  ✅ [FIX-C8-6]       HAUG_GATE_QUALITY_THRESHOLD / HAUG_GATE_HOMOGRAPH_THRESHOLD guarded with try/except")
print("  ✅ [FIX-C8-7]       dscd_discovery_warmup(): forward_path1 → path=1 → path=2 safe fallback chain")
print("  ✅ [FIX-C8-8]       All INFERENCE_STATS error calls use record_inference() (record_inference_error removed)")
print("  ✅ [FIX-C8-9]       fwd_out isinstance(dict) guard added before .get() in translate_with_explanations Step 2 and OOM retry")
print("  ✅ [VAL-1]          _log_scea_gates(): SCEA sense_gates logged per sense slot")
print("  ✅ [VAL-2]          _log_hfcaa_gate(): HFCAA gate mean logged")
print("  ✅ [VAL-3]          _log_ambiguity_scores(): Ambiguity score distribution logged")
print("  ✅ [VAL-4]          _log_prototype_separation(): mu/tau per token")
print("  ✅ [Critical]       load_checkpoint_for_resume(): per-store load_state_dict() loop")
print("  ✅ [Critical]       load_checkpoint_for_resume(): checkpoint_path defaults to get_checkpoint_path()")
print("  ✅ [Important]      load_checkpoint_for_resume(): fallback glob uses CHECKPOINT_DIR from Cell 0")
print("=" * 80 + "\n")


In [ ]:
# ===========================================================================================
# CELL 9: COMPREHENSIVE TESTING & EVALUATION (DUAL-PATH COMPATIBLE) - FIXED
# ===========================================================================================
from typing import Dict, List, Tuple, Optional, Any
import os
import torch
import traceback
import time
import functools
from collections import defaultdict

# Configuration guards (match other cells if they exist)
try:
    _USE_MULTI_GPU = bool(USE_MULTI_GPU)
except (NameError, TypeError):
    _USE_MULTI_GPU = torch.cuda.is_available() and torch.cuda.device_count() > 1

try:
    _SOURCE_LANGUAGE = str(SOURCE_LANGUAGE)
except (NameError, TypeError):
    _SOURCE_LANGUAGE = "bn"

try:
    _TARGET_LANGUAGE = str(TARGET_LANGUAGE)
except (NameError, TypeError):
    _TARGET_LANGUAGE = "en"

try:
    _VERBOSE_LOGGING = bool(VERBOSE_LOGGING)
except (NameError, TypeError):
    _VERBOSE_LOGGING = False

try:
    _DEBUG_DISCOVERY = bool(DEBUG_DISCOVERY)
except (NameError, TypeError):
    _DEBUG_DISCOVERY = False

try:
    _DEBUG_TIMING = bool(DEBUG_TIMING)
except (NameError, TypeError):
    _DEBUG_TIMING = False

try:
    _SPAN_THRESHOLD = float(SPAN_THRESHOLD)
except (NameError, ValueError, TypeError):
    _SPAN_THRESHOLD = 0.10

try:
    _UNCERTAINTY_THRESHOLD = float(UNCERTAINTY_THRESHOLD)
except (NameError, ValueError, TypeError):
    _UNCERTAINTY_THRESHOLD = 0.10

try:
    _MAX_LENGTH = int(MAX_LENGTH)
except (NameError, ValueError, TypeError):
    _MAX_LENGTH = 64

try:
    _DEVICE = DEVICE
except (NameError, TypeError):
    _DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    _HOMOGRAPH_REFERENCE_LIST = set(str(w).lower() for w in HOMOGRAPH_REFERENCE_LIST_BN)
except (NameError, TypeError):
    _HOMOGRAPH_REFERENCE_LIST = {
        "কল", "কাল", "পাতা", "ব্যাংক", "ফল", "মাথা", "বার", "হার", "তারা",
        "পানি", "দল", "বাজার", "নাম", "কথা", "বই", "ঘর", "মন", "হাত"
    }
    _HOMOGRAPH_REFERENCE_LIST = set(str(w).lower() for w in _HOMOGRAPH_REFERENCE_LIST)


def _get_cluster_count(model: torch.nn.Module) -> int:
    try:
        core = model.module if hasattr(model, "module") else model
        dscd = getattr(core, "dscd", None)
        if dscd is None:
            return 0

        lock = None
        if hasattr(dscd, 'buffer_lock'):
            lock = dscd.buffer_lock
        elif hasattr(dscd, 'clustering_lock'):
            lock = dscd.clustering_lock

        if lock:
            with lock:
                stores = getattr(dscd, "prototype_stores", {}) or {}
                return len(stores)
        else:
            stores = getattr(dscd, "prototype_stores", {}) or {}
            return len(stores)
    except Exception:
        return 0


def _get_dscd_homographs(model: torch.nn.Module) -> set:
    try:
        core = model.module if hasattr(model, "module") else model
        dscd = getattr(core, "dscd", None)
        if dscd is None:
            return set()

        if hasattr(dscd, 'get_discovered_homographs'):
            try:
                return dscd.get_discovered_homographs()
            except Exception:
                pass

        homographs = set()

        lock = None
        if hasattr(dscd, 'buffer_lock'):
            lock = dscd.buffer_lock
        elif hasattr(dscd, 'clustering_lock'):
            lock = dscd.clustering_lock

        if lock:
            with lock:
                prototype_stores = getattr(dscd, "prototype_stores", {}) or {}
                for token, store in prototype_stores.items():
                    try:
                        if hasattr(store, 'size') and store.size() >= 1:
                            clean_token = (
                                str(token)
                                .replace('▁', '')
                                .replace('Ġ', '')
                                .replace('##', '')
                                .strip()
                                .lower()
                            )
                            homographs.add(clean_token)
                    except Exception:
                        continue
        else:
            prototype_stores = getattr(dscd, "prototype_stores", {}) or {}
            for token, store in prototype_stores.items():
                try:
                    if hasattr(store, 'size') and store.size() >= 1:
                        clean_token = (
                            str(token)
                            .replace('▁', '')
                            .replace('Ġ', '')
                            .replace('##', '')
                            .strip()
                            .lower()
                        )
                        homographs.add(clean_token)
                except Exception:
                    continue

        return homographs
    except Exception:
        return set()


def _print_top_clusters(model: torch.nn.Module, top_n: int = 5):
    try:
        core = model.module if hasattr(model, "module") else model
        dscd = getattr(core, "dscd", None)
        if dscd is None:
            return

        lock = None
        if hasattr(dscd, 'buffer_lock'):
            lock = dscd.buffer_lock
        elif hasattr(dscd, 'clustering_lock'):
            lock = dscd.clustering_lock

        if lock:
            with lock:
                prototype_stores = dict(getattr(dscd, "prototype_stores", {}) or {})
        else:
            prototype_stores = dict(getattr(dscd, "prototype_stores", {}) or {})

        if not prototype_stores:
            print("[CLUSTER] No clusters found yet")
            return

        cluster_info = []
        for token, store in prototype_stores.items():
            try:
                total_count = sum(getattr(store, "counts", []))
            except Exception:
                total_count = 0
            try:
                n_protos = len(getattr(store, "centroids", []))
            except Exception:
                n_protos = 0
            cluster_info.append({
                'token':  token,
                'count':  total_count,
                'protos': n_protos,
                'mu':     getattr(store, "mu", 0.0),
                'tau':    getattr(store, "tau", 0.0)
            })

        cluster_info.sort(key=lambda x: x['count'], reverse=True)

        print(f"\n[CLUSTER] Top {min(top_n, len(cluster_info))} clusters:")
        print("-" * 90)
        print(f"{'Rank':<6}{'Token':<15}{'Count':<12}{'Protos':<10}{'Mu':<15}{'Tau':<12}")
        print("-" * 90)

        for rank, info in enumerate(cluster_info[:top_n], 1):
            token_str     = str(info['token'])
            token_display = token_str[:12] if len(token_str) > 12 else token_str
            print(
                f"{rank:<6}{token_display:<15}{info['count']:<12}{info['protos']:<10}"
                f"{info['mu']:<15.6f}{info['tau']:<12.6f}"
            )

        print("-" * 90)

    except Exception as e:
        if _DEBUG_DISCOVERY:
            print(f"[CLUSTER] Error: {str(e)[:100]}")


def _timed(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        if _DEBUG_TIMING:
            start   = time.time()
            result  = func(*args, **kwargs)
            elapsed = time.time() - start
            print(f"[TIMING] {func.__name__}: {elapsed:.2f}s")
            return result
        else:
            return func(*args, **kwargs)
    return wrapper


@torch.inference_mode()
@_timed
def comprehensive_post_training_testing(
    model: torch.nn.Module,
    tokenizer,
    run_warmup:        bool                       = True,
    compare_baseline:  bool                       = False,
    baseline_metrics:  Optional[Dict[str, Any]]   = None
) -> Dict[str, Any]:
    print("\n" + "=" * 80)
    print("COMPREHENSIVE POST-TRAINING EVALUATION")
    print("=" * 80)

    if 'translate_with_explanations' not in globals():
        print("[EVAL] ERROR: translate_with_explanations not found!")
        print("[EVAL] Cell 8 must be executed before Cell 9.")
        return {
            "error":                   "translate_with_explanations not found",
            "total_tests":             0,
            "successful_translations": 0,
        }

    test_sentences: List[Tuple[str, str, str, List[str]]] = [
        ("আমি কল বন্ধ করেছি।",
         "I turned off the tap",
         "কল = tap/call", ["কল"]),
        ("কাল আমি বই কিনব।",
         "Tomorrow I will buy a book",
         "কাল = tomorrow/yesterday", ["কাল"]),
        ("কাল আমি বই কিনেছিলাম।",
         "Yesterday I bought a book",
         "কাল = tomorrow/yesterday", ["কাল"]),
        ("পাতা ঝরে পড়েছে।",
         "The leaf has fallen",
         "পাতা = leaf/page", ["পাতা"]),
        ("তিনি ব্যাংক গেছেন।",
         "He went to the bank",
         "ব্যাংক = bank/embankment", ["ব্যাংক"]),
        ("ফল খুব সুস্বাদু।",
         "The fruit is delicious",
         "ফল = fruit/result", ["ফল"]),
        ("মাথা ব্যথা করছে।",
         "Head is aching",
         "মাথা = head/top", ["মাথা"]),
        ("কল থেকে কল এসেছে।",
         "A call came from the tap",
         "Multiple কল", ["কল"]),
        ("কালকে কাল মেঘ দেখা গেছে।",
         "Yesterday black clouds were seen",
         "Multiple কাল", ["কাল"]),
        ("আজ ভাল আবহাওয়া।",
         "Weather is good today",
         "Simple", []),
        ("আমি ভালো আছি।",
         "I am fine",
         "Simple", []),
        ("সে খুব মিষ্টি কথা বলে।",
         "She speaks sweetly",
         "Simple", []),
        ("এটা আমার বই।",
         "This is my book",
         "Simple", []),
        ("তিনি ব্যাংকে কাজ করেন এবং ব্য��ংকে বসে থাকেন।",
         "He works at the bank and sits on the embankment",
         "Long with multiple", ["ব্যাংক"]),
    ]

    core_model = model.module if (_USE_MULTI_GPU and hasattr(model, "module")) else model
    core_model.eval()

    quality_metrics = {
        'total_confidence':       0.0,
        'confidence_samples':     0,
        'high_confidence_count':  0,
        'medium_confidence_count':0,
        'low_confidence_count':   0,
        'confidences':            [],
        'spans':                  [],
        'uncertainties':          [],
    }

    homograph_tracking = {
        'test_expected_homographs':  set(),
        'dscd_discovered_homographs':set(),
        'explained_homographs':      set(),
        'homograph_explanations':    defaultdict(list),
    }

    error_tracking = {
        'translation_failures': 0,
        'dscd_failures':        0,
        'trg_failures':         0,
        'timeout_errors':       0,
        'oom_errors':           0,
        'other_errors':         0,
        'error_details':        [],
        'per_test_status':      [],
    }

    timing_metrics = {
        'total_time':       0.0,
        'per_test_times':   [],
        'avg_test_time':    0.0,
    }

    discovery_validated = False
    try:
        dscd = getattr(core_model, "dscd", None)
        if dscd and hasattr(dscd, 'discovered_log'):
            lock = None
            if hasattr(dscd, 'buffer_lock'):
                lock = dscd.buffer_lock
            elif hasattr(dscd, 'clustering_lock'):
                lock = dscd.clustering_lock

            if lock:
                with lock:
                    discovered_log = getattr(dscd, 'discovered_log', [])
                    if discovered_log:
                        discovery_validated = True
                        last_discovery = discovered_log[-1]
                        discovered  = last_discovery.get('discovered', 0)
                        candidates  = last_discovery.get('candidates', 0)
                        if _DEBUG_DISCOVERY:
                            print(f"[EVAL] Discovery log: {discovered}/{candidates} homographs")
            else:
                discovered_log = getattr(dscd, 'discovered_log', [])
                if discovered_log:
                    discovery_validated = True
                    last_discovery = discovered_log[-1]
                    discovered  = last_discovery.get('discovered', 0)
                    candidates  = last_discovery.get('candidates', 0)
                    if _DEBUG_DISCOVERY:
                        print(f"[EVAL] Discovery log: {discovered}/{candidates} homographs")
        else:
            if _DEBUG_DISCOVERY:
                print(f"[EVAL] No discovery log found")
    except Exception as e:
        if _DEBUG_DISCOVERY:
            print(f"[EVAL] Discovery validation failed: {e}")

    asbn_stats: Dict[str, Any] = {}
    try:
        asbn = getattr(core_model, "asbn", None)
        if asbn:
            if hasattr(asbn, 'get_detailed_stats'):
                asbn_stats = asbn.get_detailed_stats()
            elif hasattr(asbn, 'get_asbn_stats'):
                asbn_stats = asbn.get_asbn_stats()

            if asbn_stats and _DEBUG_DISCOVERY:
                print(f"[EVAL] ASBN: domain_acc={asbn_stats.get('domain_accuracy', 0):.2%}")
    except Exception as e:
        if _DEBUG_DISCOVERY:
            print(f"[EVAL] ASBN stats failed: {e}")

    trg_stats: Dict[str, Any] = {}
    try:
        trg = getattr(core_model, "trg", None)
        if trg and hasattr(trg, 'get_statistics'):
            trg_stats = trg.get_statistics()
            if _DEBUG_DISCOVERY:
                print(f"[EVAL] TRG: {trg_stats.get('explanations_generated', 0)} total")
    except Exception as e:
        if _DEBUG_DISCOVERY:
            print(f"[EVAL] TRG stats failed: {e}")

    homograph_tracking['dscd_discovered_homographs'] = _get_dscd_homographs(core_model)
    print(f"[EVAL] DSCD discovered: {len(homograph_tracking['dscd_discovered_homographs'])} homographs")
    if homograph_tracking['dscd_discovered_homographs'] and _DEBUG_DISCOVERY:
        print(f"[EVAL] Sample: {list(homograph_tracking['dscd_discovered_homographs'])[:10]}")

    # Warmup / discovery run (guarded)
    if run_warmup:
        try:
            dscd = getattr(core_model, "dscd", None)
            if dscd is not None:
                lock = None
                if hasattr(dscd, 'buffer_lock'):
                    lock = dscd.buffer_lock
                elif hasattr(dscd, 'clustering_lock'):
                    lock = dscd.clustering_lock

                if lock:
                    with lock:
                        stores      = getattr(dscd, "prototype_stores", None)
                        store_count = len(stores) if stores else 0
                else:
                    stores      = getattr(dscd, "prototype_stores", None)
                    store_count = len(stores) if stores else 0

                if store_count == 0 and 'dscd_discovery_warmup' in globals():
                    print("[EVAL] Running warmup (num_sents=4000)...")
                    try:
                        # Call warmup function safely with correct args
                        if callable(globals().get('dscd_discovery_warmup')):
                            globals()['dscd_discovery_warmup'](core_model, tokenizer, num_sents=4000, batch_size=64)
                        else:
                            dscd_discovery_warmup(core_model, tokenizer, num_sents=4000, batch_size=64)
                        homograph_tracking['dscd_discovered_homographs'] = _get_dscd_homographs(core_model)
                    except Exception as e:
                        print(f"[EVAL] Warmup failed: {e}")
        except Exception:
            if _DEBUG_DISCOVERY:
                try:
                    traceback.print_exc()
                except Exception:
                    pass

    # Populate expected homographs set
    for _, _, _, expected_homos in test_sentences:
        homograph_tracking['test_expected_homographs'].update([h.lower() for h in expected_homos])

    total_tests             = len(test_sentences)
    successful_translations = 0
    total_explanations      = 0
    total_high_span         = 0
    total_real_ambiguous    = 0

    print(f"\n[EVAL] Running {total_tests} tests...")
    print("-" * 80)

    try:
        tokenizer.src_lang = _SOURCE_LANGUAGE
        tokenizer.tgt_lang = _TARGET_LANGUAGE
    except Exception:
        pass

    def _is_real_amb(expl: Dict[str, Any]) -> bool:
        try:
            s = float(expl.get("span", 0.0))
            u = float(expl.get("uncertainty", 0.0))
            return (s > _SPAN_THRESHOLD) or (u > _UNCERTAINTY_THRESHOLD)
        except Exception:
            return False

    def _compute_similarity(pred: str, expected: str) -> float:
        try:
            pred_words = set(pred.lower().split())
            exp_words  = set(expected.lower().split())
            if not exp_words:
                return 0.0
            overlap = len(pred_words & exp_words)
            return overlap / len(exp_words)
        except Exception:
            return 0.0

    eval_start = time.time()

    for idx, (src_text, expected_translation, desc, expected_homos) in enumerate(test_sentences, 1):
        test_start = time.time()

        print(f"\nTest {idx}/{total_tests}: {desc}")
        print("=" * 60)

        test_status = {
            'test_id':            idx,
            'success':            False,
            'translation_ok':     False,
            'explanations_count': 0,
            'error':              None,
        }

        try:
            # Always call the canonical pipeline (translate_with_explanations)
            result = translate_with_explanations(
                core_model if core_model is not None else model,
                tokenizer,
                src_text,
                source_lang=_SOURCE_LANGUAGE,
                target_lang=_TARGET_LANGUAGE,
                device=_DEVICE,
                max_length=_MAX_LENGTH,
                span_threshold=_SPAN_THRESHOLD,
                uncertainty_threshold=_UNCERTAINTY_THRESHOLD,
                track_stats=False
            )

            if result is None or not isinstance(result, dict):
                print(f"[EVAL] Invalid result type: {type(result)}")
                error_tracking['translation_failures'] += 1
                test_status['error'] = 'invalid_result'
                error_tracking['per_test_status'].append(test_status)
                continue

            if 'error' in result and result['error']:
                print(f"[EVAL] Translation error: {result['error']}")
                error_tracking['translation_failures'] += 1
                test_status['error'] = 'translation_error'
                error_tracking['per_test_status'].append(test_status)
                continue

            translation  = str(result.get("translation", "") or "")
            amb_count    = int(result.get("ambiguous_words_detected", 0))
            explanations = result.get("explanations", []) or []

            similarity = _compute_similarity(translation, expected_translation)

            print(f"Input:       {src_text}")
            print(f"Expected:    {expected_translation}")
            print(f"Translation: {translation}")
            print(f"Similarity:  {similarity:.1%}")
            print(f"Ambiguous:   {amb_count}")

            if explanations:
                print("\nExplanations:")
                high_span_local = 0
                real_amb_local  = 0

                for j, expl in enumerate(explanations, 1):
                    span_val = float(expl.get("span", 0.0))
                    u_val    = float(expl.get("uncertainty", 0.0))
                    conf_val = float(expl.get("confidence", max(span_val, u_val)))

                    marker = f"[S>{_SPAN_THRESHOLD:.2f}]" if span_val > _SPAN_THRESHOLD else "          "

                    word = expl.get("ambiguous_word", expl.get("token", "N/A"))
                    pos  = expl.get("position", expl.get("token_idx", "N/A"))

                    print(f"  {j}. {marker} '{word}' @ {pos}")
                    print(f"       conf={conf_val:.3f} | U={u_val:.3f} | S={span_val:.3f}")
                    text = str(expl.get("explanation", ""))
                    if len(text) > 120:
                        text = text[:120] + "..."
                    print(f"       {text}")

                    quality_metrics['confidences'].append(conf_val)
                    quality_metrics['spans'].append(span_val)
                    quality_metrics['uncertainties'].append(u_val)
                    quality_metrics['total_confidence'] = (
                        quality_metrics.get('total_confidence', 0.0) + conf_val
                    )
                    quality_metrics['confidence_samples'] += 1

                    if conf_val >= 0.65:
                        quality_metrics['high_confidence_count'] += 1
                    elif conf_val >= 0.4:
                        quality_metrics['medium_confidence_count'] += 1
                    else:
                        quality_metrics['low_confidence_count'] += 1

                    if span_val > _SPAN_THRESHOLD:
                        high_span_local += 1
                    if _is_real_amb(expl):
                        real_amb_local += 1

                    clean_word = str(word).replace('▁', '').replace('Ġ', '').strip().lower()
                    homograph_tracking['explained_homographs'].add(clean_word)
                    homograph_tracking['homograph_explanations'][clean_word].append({
                        'sentence':    src_text,
                        'confidence':  conf_val,
                        'span':        span_val,
                        'uncertainty': u_val,
                    })

                total_explanations   += len(explanations)
                total_high_span      += high_span_local
                total_real_ambiguous += real_amb_local
                test_status['explanations_count'] = len(explanations)
            else:
                print("No explanations")

            if translation and translation.strip() and translation not in (
                "Error occurred",
                "Translation generation failed",
                "ERROR DURING TRANSLATION",
            ):
                successful_translations  += 1
                test_status['translation_ok'] = True
                test_status['success']        = True
                print("✅ Success")
            else:
                print("❌ Translation failed")
                error_tracking['translation_failures'] += 1
                test_status['error'] = 'translation_failed'

            del result
            if explanations:
                del explanations

        except RuntimeError as e:
            error_str = str(e).lower()
            if "out of memory" in error_str:
                print(f"[EVAL] OOM: {str(e)[:100]}")
                error_tracking['oom_errors'] += 1
                test_status['error'] = 'oom'
            elif "timeout" in error_str:
                print(f"[EVAL] Timeout: {str(e)[:100]}")
                error_tracking['timeout_errors'] += 1
                test_status['error'] = 'timeout'
            else:
                print(f"[EVAL] Runtime: {type(e).__name__}")
                error_tracking['other_errors'] += 1
                test_status['error'] = 'runtime'
            error_tracking['error_details'].append(f"Test {idx}: {type(e).__name__}")
        except Exception as e:
            print(f"[EVAL] Error: {type(e).__name__}")
            error_tracking['other_errors'] += 1
            test_status['error'] = type(e).__name__
            error_tracking['error_details'].append(f"Test {idx}: {type(e).__name__}")
            if _DEBUG_DISCOVERY:
                try:
                    traceback.print_exc()
                except Exception:
                    pass

        error_tracking['per_test_status'].append(test_status)

        test_time = time.time() - test_start
        timing_metrics['per_test_times'].append(test_time)

        print("-" * 60)

        if idx % 3 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()

    timing_metrics['total_time'] = time.time() - eval_start
    if timing_metrics['per_test_times']:
        timing_metrics['avg_test_time'] = (
            sum(timing_metrics['per_test_times']) / len(timing_metrics['per_test_times'])
        )

    if quality_metrics['confidence_samples'] > 0:
        quality_metrics['avg_confidence'] = (
            quality_metrics['total_confidence'] / quality_metrics['confidence_samples']
        )
        quality_metrics['avg_span'] = (
            sum(quality_metrics['spans']) / len(quality_metrics['spans'])
            if quality_metrics['spans'] else 0.0
        )
        quality_metrics['avg_uncertainty'] = (
            sum(quality_metrics['uncertainties']) / len(quality_metrics['uncertainties'])
            if quality_metrics['uncertainties'] else 0.0
        )

        if quality_metrics['confidences']:
            sorted_conf = sorted(quality_metrics['confidences'])
            quality_metrics['confidence_p25'] = sorted_conf[len(sorted_conf) // 4]
            quality_metrics['confidence_p50'] = sorted_conf[len(sorted_conf) // 2]
            quality_metrics['confidence_p75'] = sorted_conf[3 * len(sorted_conf) // 4]
    else:
        quality_metrics['avg_confidence']  = 0.0
        quality_metrics['avg_span']        = 0.0
        quality_metrics['avg_uncertainty'] = 0.0

    explained_from_dscd = homograph_tracking['explained_homographs'].intersection(
        homograph_tracking['dscd_discovered_homographs']
    )

    test_expected_discovered = homograph_tracking['test_expected_homographs'].intersection(
        homograph_tracking['dscd_discovered_homographs']
    )

    reference_discovered = _HOMOGRAPH_REFERENCE_LIST.intersection(
        homograph_tracking['dscd_discovered_homographs']
    )

    homograph_tracking['explained_from_dscd_rate'] = (
        len(explained_from_dscd) / len(homograph_tracking['dscd_discovered_homographs'])
        if homograph_tracking['dscd_discovered_homographs'] else 0.0
    )
    homograph_tracking['test_expected_discovery_rate'] = (
        len(test_expected_discovered) / len(homograph_tracking['test_expected_homographs'])
        if homograph_tracking['test_expected_homographs'] else 0.0
    )
    homograph_tracking['reference_discovery_rate'] = (
        len(reference_discovered) / len(_HOMOGRAPH_REFERENCE_LIST)
        if _HOMOGRAPH_REFERENCE_LIST else 0.0
    )

    try:
        dscd_stats = {"total_words": 0, "multi_sense_words": 0, "total_prototypes": 0}
        dscd = getattr(core_model, "dscd", None)
        if dscd is not None and hasattr(dscd, "prototype_stores"):
            lock = None
            if hasattr(dscd, 'buffer_lock'):
                lock = dscd.buffer_lock
            elif hasattr(dscd, 'clustering_lock'):
                lock = dscd.clustering_lock

            if lock:
                with lock:
                    stores = dict(getattr(dscd, "prototype_stores") or {})
            else:
                stores = dict(getattr(dscd, "prototype_stores") or {})

            total_words  = 0
            multi        = 0
            total_protos = 0
            for key, store in stores.items():
                try:
                    sz = int(store.size()) if hasattr(store, "size") else 0
                except Exception:
                    sz = 0
                total_words  += 1
                total_protos += sz
                if sz >= 2:
                    multi += 1
            dscd_stats = {
                "total_words":       total_words,
                "multi_sense_words": multi,
                "total_prototypes":  total_protos,
            }
    except Exception as e:
        if _DEBUG_DISCOVERY:
            print(f"[EVAL] DSCD stats failed: {e}")
        dscd_stats = {"total_words": 0, "multi_sense_words": 0, "total_prototypes": 0}

    print("\n" + "=" * 80)
    print("COMPREHENSIVE EVALUATION SUMMARY")
    print("=" * 80)

    print(f"\n[TRANSLATION QUALITY]")
    print(f"  Total tests:  {total_tests}")
    print(f"  Successful:   {successful_translations}")
    print(f"  Success rate: {successful_translations / total_tests * 100:.1f}%")

    print(f"\n[AMBIGUITY DETECTION]")
    print(f"  Total explanations:    {total_explanations}")
    print(f"  High-span (S>{_SPAN_THRESHOLD}): {total_high_span}")
    print(f"  Real ambiguous:        {total_real_ambiguous}")
    if total_tests > 0:
        print(f"  Avg explanations/test: {total_explanations / total_tests:.2f}")

    print(f"\n[EXPLANATION QUALITY]")
    print(f"  Avg confidence:   {quality_metrics['avg_confidence']:.3f}")
    print(f"  Avg span:         {quality_metrics['avg_span']:.3f}")
    print(f"  Avg uncertainty:  {quality_metrics['avg_uncertainty']:.3f}")

    if 'confidence_p50' in quality_metrics:
        print(
            f"  Confidence P25/P50/P75: "
            f"{quality_metrics.get('confidence_p25', 0):.3f} / "
            f"{quality_metrics.get('confidence_p50', 0):.3f} / "
            f"{quality_metrics.get('confidence_p75', 0):.3f}"
        )

    print(f"  High  (>=0.65):    {quality_metrics['high_confidence_count']}")
    print(f"  Medium (0.4-0.65): {quality_metrics['medium_confidence_count']}")
    print(f"  Low   (<0.4):      {quality_metrics['low_confidence_count']}")

    print(f"\n[HOMOGRAPH DISCOVERY]")
    print(f"  DSCD discovered:     {len(homograph_tracking['dscd_discovered_homographs'])}")
    print(f"  Explained:           {len(homograph_tracking['explained_homographs'])}")
    print(f"  Explanation rate:    {homograph_tracking['explained_from_dscd_rate']:.1%}")
    print(f"  Test discovery rate: {homograph_tracking['test_expected_discovery_rate']:.1%}")

    if homograph_tracking['explained_homographs']:
        print(f"\n  Explained homographs (top 10):")
        for homo in sorted(homograph_tracking['explained_homographs'])[:10]:
            exps     = homograph_tracking['homograph_explanations'].get(homo, [])
            count    = len(exps)
            avg_conf = sum(e['confidence'] for e in exps) / len(exps) if exps else 0.0
            in_dscd  = "[D]" if homo in homograph_tracking['dscd_discovered_homographs'] else "   "
            in_ref   = "[R]" if homo in _HOMOGRAPH_REFERENCE_LIST else "   "
            print(f"    {in_dscd} {in_ref} '{homo}': {count} x conf={avg_conf:.3f}")

    print(f"\n[REFERENCE COMPARISON]")
    print(f"  Reference:  {len(_HOMOGRAPH_REFERENCE_LIST)} words")
    print(f"  Discovered: {len(reference_discovered)}/{len(_HOMOGRAPH_REFERENCE_LIST)}")
    print(f"  Coverage:   {homograph_tracking['reference_discovery_rate']:.1%}")

    print(f"\n[DSCD PROTOTYPES]")
    print(f"  Word types:       {dscd_stats['total_words']}")
    print(f"  Multi-sense:      {dscd_stats['multi_sense_words']}")
    print(f"  Total prototypes: {dscd_stats['total_prototypes']}")
    if dscd_stats['total_words'] > 0:
        print(
            f"  Multi-sense ratio: "
            f"{dscd_stats['multi_sense_words'] / dscd_stats['total_words']:.1%}"
        )

    if asbn_stats:
        print(f"\n[ASBN]")
        print(f"  Domain accuracy: {asbn_stats.get('domain_accuracy', 0):.2%}")
        if 'source_accuracy' in asbn_stats:
            print(f"  Source accuracy: {asbn_stats['source_accuracy']:.2%}")
            print(f"  Target accuracy: {asbn_stats['target_accuracy']:.2%}")

    if trg_stats:
        print(f"\n[TRG]")
        print(f"  Total explanations: {trg_stats.get('explanations_generated', 0)}")
        print(f"  High confidence:    {trg_stats.get('high_confidence_rate', 0):.1%}")

    print(f"\n[PERFORMANCE]")
    print(f"  Total time:    {timing_metrics['total_time']:.2f}s")
    print(f"  Avg time/test: {timing_metrics['avg_test_time']:.2f}s")

    total_errors = sum([
        error_tracking['translation_failures'],
        error_tracking['dscd_failures'],
        error_tracking['trg_failures'],
        error_tracking['timeout_errors'],
        error_tracking['oom_errors'],
        error_tracking['other_errors'],
    ])

    if total_errors > 0:
        print(f"\n[ERRORS]")
        print(f"  Total:       {total_errors}")
        print(f"  Translation: {error_tracking['translation_failures']}")
        print(f"  OOM:         {error_tracking['oom_errors']}")
        print(f"  Other:       {error_tracking['other_errors']}")

    if compare_baseline and baseline_metrics and isinstance(baseline_metrics, dict):
        print(f"\n[BASELINE COMPARISON]")
        try:
            baseline_success  = float(baseline_metrics.get('success_rate_pct', 0))
            current_success   = (
                successful_translations / total_tests * 100.0
            ) if total_tests > 0 else 0.0
            success_delta     = current_success - baseline_success

            baseline_expl     = int(baseline_metrics.get('total_explanations', 0))
            expl_delta        = total_explanations - baseline_expl

            baseline_quality_dict = baseline_metrics.get('quality_metrics', {})
            if isinstance(baseline_quality_dict, dict):
                baseline_quality = float(baseline_quality_dict.get('avg_confidence', 0))
            else:
                baseline_quality = 0.0
            quality_delta = quality_metrics['avg_confidence'] - baseline_quality

            print(f"  Translation:  {current_success:.1f}% ({success_delta:+.1f}%)")
            print(f"  Explanations: {total_explanations} ({expl_delta:+d})")
            print(
                f"  Confidence:   {quality_metrics['avg_confidence']:.3f} "
                f"({quality_delta:+.3f})"
            )

            baseline_homo_dict = baseline_metrics.get('homograph_tracking', {})
            if isinstance(baseline_homo_dict, dict):
                baseline_homo_rate = float(baseline_homo_dict.get('explained_from_dscd_rate', 0))
                homo_delta = (
                    homograph_tracking['explained_from_dscd_rate'] - baseline_homo_rate
                )
                print(
                    f"  Explanation rate: "
                    f"{homograph_tracking['explained_from_dscd_rate']:.1%} "
                    f"({homo_delta:+.1%})"
                )
        except Exception as e:
            print(f"  Comparison failed: {type(e).__name__}")

    warnings = []
    if successful_translations < total_tests * 0.5:
        warnings.append("High translation failure (>50%)")
    if total_explanations == 0:
        warnings.append("No explanations generated")
    if dscd_stats['total_words'] < 100:
        warnings.append("Very few prototypes (<100)")
    if quality_metrics['low_confidence_count'] > quality_metrics['high_confidence_count']:
        warnings.append("More low than high confidence")
    if homograph_tracking['explained_from_dscd_rate'] < 0.3:
        warnings.append("Low explanation rate (<30%)")
    if not discovery_validated:
        warnings.append("Discovery log missing")
    if asbn_stats and asbn_stats.get('domain_accuracy', 0) < 0.5:
        warnings.append("ASBN domain accuracy <50%")

    if warnings:
        print(f"\n[WARNINGS]")
        for w in warnings:
            print(f"  - {w}")
    else:
        print(f"\n[HEALTH] All systems nominal")

    print("=" * 80)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # If INFERENCE_STATS exists, print summary (guarded)
    if 'INFERENCE_STATS' in globals() and hasattr(globals().get('INFERENCE_STATS'), 'print_summary'):
        try:
            globals()['INFERENCE_STATS'].print_summary()
        except Exception:
            pass

    return {
        "total_tests":             total_tests,
        "successful_translations": successful_translations,
        "success_rate_pct":        (successful_translations / total_tests * 100.0) if total_tests > 0 else 0.0,
        "total_explanations":      total_explanations,
        "total_high_span":         total_high_span,
        "total_real_ambiguous":    total_real_ambiguous,
        "dscd_stats":              dscd_stats,
        "quality_metrics":         quality_metrics,
        "homograph_tracking":      homograph_tracking,
        "error_tracking":          error_tracking,
        "asbn_stats":              asbn_stats,
        "trg_stats":               trg_stats,
        "discovery_validated":     discovery_validated,
        "timing_metrics":          timing_metrics,
    }


def test_evaluation_pipeline(model, tokenizer) -> bool:
    print("\n" + "=" * 60)
    print("[TEST] Testing evaluation pipeline")
    print("=" * 60)

    try:
        result = comprehensive_post_training_testing(
            model,
            tokenizer,
            run_warmup=False,
            compare_baseline=False
        )

        assert 'total_tests' in result
        assert 'quality_metrics' in result
        assert 'homograph_tracking' in result

        print("✅ Evaluation pipeline test passed")
        print("=" * 60 + "\n")
        return True

    except Exception as e:
        print(f"❌ Evaluation pipeline test failed: {e}")
        try:
            traceback.print_exc()
        except Exception:
            pass
        print("=" * 60 + "\n")
        return False


print("\n" + "=" * 80)
print("Cell 9: Testing & Evaluation (DUAL-PATH COMPATIBLE)")
print("=" * 80)
print("Evaluation metrics:")
print("  - Translation quality (success rate)")
print("  - Ambiguity detection (high-span, real ambiguous)")
print("  - Explanation quality (confidence distribution)")
print("  - Homograph discovery (DSCD vs reference)")
print("  - DSCD prototype statistics")
print("  - ASBN domain accuracy")
print("  - TRG explanation statistics")
print("  - Performance timing (total, avg per test)")
print("  - Error tracking (OOM, timeout, other)")
print()
print(f"Configuration:")
print(f"  - Source language: {_SOURCE_LANGUAGE}")
print(f"  - Target language: {_TARGET_LANGUAGE}")
print(f"  - Span threshold: {_SPAN_THRESHOLD}")
print(f"  - Uncertainty threshold: {_UNCERTAINTY_THRESHOLD}")
print(f"  - Max length: {_MAX_LENGTH}")
print(f"  - Device: {_DEVICE}")
print(f"  - Reference list: {len(_HOMOGRAPH_REFERENCE_LIST)} words")
print("\nDUAL-PATH COMPATIBILITY:")
print("  Uses Cell 8 functions (already fixed)")
print("  No direct model.forward() calls")
print("  Only inspection functions (no API changes)")
print("  Fully compatible with dual-path system")
print("=" * 80 + "\n")

In [ ]:
# ==============================================================================
# CELL 10: TATN MAIN PIPELINE (DUAL-PATH COMPATIBLE)
# ==============================================================================

import os
import sys
import time
import random
import traceback
import inspect
from typing import Tuple, Optional, Dict, Any
import gc
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers.modeling_outputs import BaseModelOutput
import collections

try:
    if hasattr(torch.serialization, 'add_safe_globals'):
        torch.serialization.add_safe_globals([
            collections.defaultdict,
            collections.OrderedDict,
            collections.deque
        ])
        print("✓ Registered safe globals for PyTorch 2.6+")
except (AttributeError, Exception):
    pass


def _g(name, default):
    return globals().get(name, default)


try:
    USE_MULTI_GPU                = bool(_g("USE_MULTI_GPU", False))
    NUM_GPUS                     = int(_g("NUM_GPUS", torch.cuda.device_count() if torch.cuda.is_available() else 0))
    DEVICE                       = _g("DEVICE", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    SOURCE_LANGUAGE              = str(_g("SOURCE_LANGUAGE", "bn_IN"))
    TARGET_LANGUAGE              = str(_g("TARGET_LANGUAGE", "en_XX"))
    NUM_SAMPLES                  = int(_g("NUM_SAMPLES", 70000))
    MAX_LENGTH                   = int(_g("MAX_LENGTH", 256))
    BATCH_SIZE                   = int(_g("BATCH_SIZE", 4))
    EPOCHS                       = int(_g("EPOCHS", 1))
    ACCUMULATION_STEPS           = int(_g("ACCUMULATION_STEPS", 16))
    LR_NMT                       = float(_g("LR_NMT",     3e-5))
    LR_PHI                       = float(_g("LR_PHI",     1e-4))
    LR_ADAPTER                   = float(_g("LR_ADAPTER", 5e-5))
    WEIGHT_DECAY                 = float(_g("WEIGHT_DECAY", 0.01))
    ENABLE_ASBN_TRAINING         = bool(_g("ENABLE_ASBN_TRAINING", True))
    VALIDATION_CHECK_INTERVAL    = int(_g("VALIDATION_CHECK_INTERVAL", 500))
    PERIODIC_DISCOVERY_FREQUENCY = int(_g("PERIODIC_DISCOVERY_FREQUENCY", 300))
    DSCD_WARMUP_SAMPLES          = int(_g("DSCD_WARMUP_SAMPLES", 9000))
    SPAN_THRESHOLD               = float(_g("SPAN_THRESHOLD", 0.20))
    UNCERTAINTY_THRESHOLD        = float(_g("UNCERTAINTY_THRESHOLD", 0.15))
    HOMOGRAPH_REFERENCE_LIST_BN  = set(_g("HOMOGRAPH_REFERENCE_LIST_BN",
        ["কল", "কাল", "পাতা", "ব্যাংক", "ফল", "মাথা", "বার", "হার", "তারা"]))
    TRAIN_RATIO                  = float(_g("TRAIN_RATIO", 0.80))
    VAL_RATIO                    = float(_g("VAL_RATIO",   0.10))
    TEST_RATIO                   = float(_g("TEST_RATIO",  0.10))
    TRAIN_DOMAIN                 = int(_g("TRAIN_DOMAIN", 0))
    TEST_DOMAIN                  = int(_g("TEST_DOMAIN",  1))
    SEED                         = int(_g("SEED", 42))
    EVAL_BATCH_SIZE              = int(_g("EVAL_BATCH_SIZE", _g("BATCH_SIZE", 4)))
    FREEZE_ENCODER               = bool(_g("FREEZE_ENCODER", False))
    DEBUG_TIMING                 = bool(_g("DEBUG_TIMING",    False))
    DEBUG_DISCOVERY              = bool(_g("DEBUG_DISCOVERY", False))
    MBART_MODEL_NAME             = str(_g("MBART_MODEL_NAME", "facebook/mbart-large-50-many-to-many-mmt"))
    USE_BACK_TRANSLATION         = bool(_g("USE_BACK_TRANSLATION", False))
    LAMBDA_BT                    = float(_g("LAMBDA_BT", 1.0))
except (ValueError, TypeError):
    NUM_GPUS                     = torch.cuda.device_count() if torch.cuda.is_available() else 0
    USE_MULTI_GPU                = NUM_GPUS > 1
    DEVICE                       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    SOURCE_LANGUAGE              = "bn_IN"
    TARGET_LANGUAGE              = "en_XX"
    NUM_SAMPLES                  = 70000
    MAX_LENGTH                   = 256
    BATCH_SIZE                   = 4
    EPOCHS                       = 1
    ACCUMULATION_STEPS           = 16
    LR_NMT                       = 3e-5
    LR_PHI                       = 1e-4
    LR_ADAPTER                   = 5e-5
    WEIGHT_DECAY                 = 0.01
    ENABLE_ASBN_TRAINING         = True
    VALIDATION_CHECK_INTERVAL    = 500
    PERIODIC_DISCOVERY_FREQUENCY = 300
    DSCD_WARMUP_SAMPLES          = 9000
    SPAN_THRESHOLD               = 0.20
    UNCERTAINTY_THRESHOLD        = 0.15
    HOMOGRAPH_REFERENCE_LIST_BN  = {"কল", "কাল", "পাতা", "ব্যাংক"}
    TRAIN_RATIO                  = 0.80
    VAL_RATIO                    = 0.10
    TEST_RATIO                   = 0.10
    TRAIN_DOMAIN                 = 0
    TEST_DOMAIN                  = 1
    SEED                         = 42
    EVAL_BATCH_SIZE              = 4
    FREEZE_ENCODER               = False
    DEBUG_TIMING                 = False
    DEBUG_DISCOVERY              = False
    MBART_MODEL_NAME             = "facebook/mbart-large-50-many-to-many-mmt"
    USE_BACK_TRANSLATION         = False
    LAMBDA_BT                    = 1.0

MIN_TRAIN_SAMPLES = 10

try:
    CHECKPOINT_DIR = str(globals()["CHECKPOINT_DIR"])
except (KeyError, TypeError):
    CHECKPOINT_DIR = "/kaggle/working"

try:
    CHECKPOINT_PATH = get_checkpoint_path()
except (NameError, Exception):
    CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "tatn_final.pt")

try:
    CHECKPOINT_EPOCH_FILENAME_TEMPLATE = str(globals()["CHECKPOINT_EPOCH_FILENAME_TEMPLATE"])
except (KeyError, TypeError):
    CHECKPOINT_EPOCH_FILENAME_TEMPLATE = "tatn_epoch_{epoch}.pt"


def _safe_clear_gpu_caches():
    try:
        if "clear_all_gpu_caches" in globals():
            globals()["clear_all_gpu_caches"]()
            return
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                try:
                    with torch.cuda.device(i):
                        torch.cuda.empty_cache()
                except Exception:
                    pass
        if gc.isenabled():
            gc.collect()
    except Exception:
        pass


def _safe_get(d: dict, *keys, default=None):
    if not isinstance(d, dict):
        return default
    result = d
    for key in keys:
        if not isinstance(result, dict):
            return default
        result = result.get(key, None)
        if result is None:
            return default
    return result


def _safe_tokenizer_from_pretrained(model_name: str, local_files_only: bool = False):
    try:
        from transformers import MBart50TokenizerFast
        tok = MBart50TokenizerFast.from_pretrained(
            model_name,
            src_lang="bn_IN",
            tgt_lang="en_XX",
            local_files_only=local_files_only
        )
        required = ['encode', 'decode', 'convert_ids_to_tokens', '__call__']
        for method in required:
            if not hasattr(tok, method):
                raise RuntimeError(f"Tokenizer missing: {method}")
        return tok
    except Exception as e:
        print(f"[TOKENIZER] Load failed: {e}")
        raise


def _get_dscd_stores_safe(dscd):
    try:
        prototype_stores = getattr(dscd, 'prototype_stores', None)
        if prototype_stores is None:
            return {}
        lock = None
        if hasattr(dscd, 'buffer_lock'):
            lock = dscd.buffer_lock
        elif hasattr(dscd, 'clustering_lock'):
            lock = dscd.clustering_lock
        try:
            if lock:
                try:
                    with lock:
                        return dict(prototype_stores)
                except Exception:
                    return dict(prototype_stores)
            else:
                return dict(prototype_stores)
        except Exception:
            return {}
    except Exception:
        return {}


def _get_core_model(model):
    return model.module if hasattr(model, "module") else model


def _get_robust_fallback_pairs():
    if "get_fallback_dataset" in globals():
        try:
            pairs = globals()["get_fallback_dataset"]()
            if pairs and len(pairs) >= MIN_TRAIN_SAMPLES:
                print(f"[PIPELINE] Using Cell 2 fallback dataset: {len(pairs)} pairs")
                return pairs
        except Exception as e:
            print(f"[PIPELINE] Cell 2 fallback failed: {e}")

    fallback_pairs = [
        ("আমি কল বন্ধ করেছি",              "i turned off the tap"),
        ("সে আমাকে পরে কল করবে",            "he will call me later"),
        ("আমরা প্রতিদিন তাজা ফল খাই",        "we eat fresh fruits every day"),
        ("তার কঠোর পরিশ্রমের ভালো ফল হয়েছে", "his hard work has brought good results"),
        ("গাছে নতুন পাতাগুলো গজিয়েছে",       "new leaves have sprouted on the tree"),
        ("আমি বইয়ের পাতা উল্টাচ্ছি",         "i am turning the pages of the book"),
        ("কাল আমি বাজারে গিয়েছিলাম",         "yesterday i went to the market"),
        ("কাল আমি তোমার সাথে দেখা করব",       "tomorrow i will meet you"),
        ("তারা আকাশে উজ্জ্বল",               "the stars are bright in the sky"),
        ("তারা বাড়িতে নেই",                  "they are not at home"),
        ("ব্যাংক নদীর ধারে ভেঙে গেছে",        "the bank by the river has collapsed"),
        ("আমি ব্যাংকে টাকা জমা দিয়েছি",       "i deposited money in the bank"),
        ("বার বার চেষ্টা করতে হবে",           "you have to try again and again"),
        ("আমি বার খুলে ভিতরে ঢুকলাম",         "i opened the bar and entered"),
        ("তার মাথা ব্যথা করছে",               "his head is hurting"),
        ("আমি মাথা নেড়ে সম্মতি দিলাম",        "i nodded my head in agreement"),
        ("সে হার মেনে নিয়েছে",               "he accepted defeat"),
        ("আমি গলায় সোনার হার পরেছি",          "i am wearing a gold necklace"),
        ("পানি খুব ঠান্ডা",                   "the water is very cold"),
        ("আমি পানি খাচ্ছি",                   "i am drinking water"),
        ("দল খেলায় জিতেছে",                  "the team won the game"),
        ("বাজার থেকে সবজি কিনলাম",            "i bought vegetables from the market"),
        ("তার নাম আহমেদ",                    "his name is ahmed"),
        ("কথা বলা বন্ধ করো",                 "stop talking"),
        ("বই পড়তে ভালো লাগে",                "i like reading books"),
        ("আমি একটি নতুন বই কিনেছি",           "i bought a new book"),
        ("ঘর পরিষ্কার করা হয়েছে",             "the house has been cleaned"),
        ("আমি ঘরে বসে আছি",                  "i am sitting at home"),
        ("মন ভালো নেই",                      "my mind is not good"),
        ("হাত ধুয়ে নাও",                     "wash your hands"),
        ("আমি তার হাত ধরলাম",                "i held his hand"),
        ("দিন কেটে যাচ্ছে",                   "the day is passing by"),
        ("রাত হয়ে এসেছে",                    "night has come"),
        ("আমি রাত জেগে পড়েছি",               "i studied staying up at night"),
        ("জল খুব গরম",                       "the water is very hot"),
        ("বাড়ি যাচ্ছি",                       "i am going home"),
        ("আমার বাড়ি ঢাকায়",                  "my house is in dhaka"),
        ("পার্কে অনেক মানুষ",                 "there are many people in the park"),
        ("আমি প্রতিদিন পার্কে হাঁটি",          "i walk in the park every day"),
        ("নদী বইছে",                          "the river is flowing"),
        ("আমি নদীর ধারে দাঁড়িয়ে আছি",         "i am standing by the river"),
        ("বন খুব সুন্দর",                     "the forest is very beautiful"),
        ("আমি বন দেখতে গিয়েছিলাম",            "i went to see the forest"),
        ("সে ভালো ছেলে",                      "he is a good boy"),
        ("আমরা একসাথে খেলি",                  "we play together"),
        ("আকাশ নীল রঙের",                    "the sky is blue"),
        ("ফুল খুব সুন্দর",                    "the flower is very beautiful"),
        ("সূর্য পূর্বে ওঠে",                  "the sun rises in the east"),
        ("চাঁদ রাতে ওঠে",                     "the moon rises at night"),
        ("আমি স্কুলে যাই",                    "i go to school"),
    ]
    print(f"[PIPELINE] Using inline fallback dataset: {len(fallback_pairs)} pairs")
    return fallback_pairs


def initialize_environment():
    print("[PIPELINE] Initializing environment...")
    if torch.cuda.is_available():
        gcnt = torch.cuda.device_count()
        print(f"[PIPELINE] GPUs: {gcnt}")
        for i in range(gcnt):
            try:
                name = torch.cuda.get_device_name(i)
                mem  = torch.cuda.get_device_properties(i).total_memory / 1024**3
                print(f"  GPU {i}: {name} ({mem:.1f} GB)")
            except Exception:
                print(f"  GPU {i}: Unknown")
        _safe_clear_gpu_caches()
    else:
        print("[PIPELINE] CPU only")
    return True


def validate_component_compatibility(model_core, tokenizer):
    print("\n[VALIDATION] Checking component compatibility...")
    issues = []

    try:
        model_vocab     = model_core.vocab_size
        tokenizer_vocab = len(tokenizer) if hasattr(tokenizer, "__len__") else getattr(tokenizer, "vocab_size", 0)
        if model_vocab < tokenizer_vocab:
            issues.append(f"CRITICAL: model vocab ({model_vocab}) < tokenizer vocab ({tokenizer_vocab})")
        elif model_vocab > tokenizer_vocab:
            print(f"  ✅ Vocabulary: model={model_vocab}, tokenizer={tokenizer_vocab}")
            print(f"     Note: Model has {model_vocab - tokenizer_vocab} extra tokens (preserves pretrained weights)")
        else:
            print(f"  ✅ Vocabulary: {model_vocab}")
    except Exception as e:
        issues.append(f"Vocab check failed: {e}")

    try:
        model_embed_dim = int(getattr(model_core.mbart.config, "d_model", 1024))
        print(f"  ✅ Model embed_dim: {model_embed_dim}")
        if hasattr(model_core, 'dscd'):
            dscd_embed_dim = getattr(model_core.dscd, 'embed_dim', None)
            if dscd_embed_dim is not None and dscd_embed_dim != model_embed_dim:
                issues.append(f"DSCD embed_dim mismatch: {dscd_embed_dim} != {model_embed_dim}")
            else:
                print(f"  ✅ DSCD embed_dim: {dscd_embed_dim}")
        if hasattr(model_core, 'asbn'):
            asbn_embed_dim = getattr(model_core.asbn, 'embed_dim', None)
            if asbn_embed_dim is not None and asbn_embed_dim != model_embed_dim:
                issues.append(f"ASBN embed_dim mismatch: {asbn_embed_dim} != {model_embed_dim}")
            else:
                print(f"  ✅ ASBN embed_dim: {asbn_embed_dim}")
    except Exception as e:
        issues.append(f"Embed_dim check failed: {e}")

    try:
        embedding_layer = model_core.mbart.get_input_embeddings()
        if embedding_layer is None:
            issues.append("Model has no input embeddings")
        else:
            actual_embed_dim  = embedding_layer.embedding_dim
            actual_vocab_size = embedding_layer.num_embeddings
            print(f"  ✅ Embedding layer: dim={actual_embed_dim}, vocab={actual_vocab_size}")
    except Exception as e:
        issues.append(f"Embedding layer check failed: {e}")

    if issues:
        print("\n[VALIDATION] ❌ FAILED - Issues found:")
        for issue in issues:
            print(f"  - {issue}")
        raise RuntimeError("Component compatibility validation failed")
    else:
        print("[VALIDATION] ✅ All components compatible")
    return True


def validate_dataset_compatibility(dataset, tokenizer, model_vocab_size):
    print("\n[VALIDATION] Checking dataset compatibility...")
    try:
        sample_batch = []
        for i in range(min(5, len(dataset))):
            try:
                sample_batch.append(dataset[i])
            except Exception:
                continue

        if not sample_batch:
            print("[VALIDATION] ⚠️  Could not load samples")
            return True

        max_input_id = 0
        min_input_id = float('inf')

        for item in sample_batch:
            input_ids = item.get('input_ids', None)
            if input_ids is not None:
                if isinstance(input_ids, torch.Tensor):
                    max_input_id = max(max_input_id, input_ids.max().item())
                    min_input_id = min(min_input_id, input_ids.min().item())
                elif isinstance(input_ids, list):
                    max_input_id = max(max_input_id, max(input_ids))
                    min_input_id = min(min_input_id, min(input_ids))

        print(f"  Input IDs range: [{min_input_id}, {max_input_id}]")
        print(f"  Model vocab size: {model_vocab_size}")

        if max_input_id >= model_vocab_size:
            raise RuntimeError(
                f"Dataset contains out-of-bounds token IDs!\n"
                f"  Max ID: {max_input_id}\n"
                f"  Vocab size: {model_vocab_size}\n"
                f"  → Cell 2 tokenization error or vocab mismatch"
            )
        if min_input_id < 0:
            raise RuntimeError(f"Dataset contains negative token IDs: {min_input_id}")

        print("[VALIDATION] ✅ Dataset token IDs valid")
        return True
    except Exception as e:
        print(f"[VALIDATION] Dataset check failed: {e}")
        raise


def test_model_forward_pass(model, tokenizer, device):
    print("\n[VALIDATION] Testing model forward pass...")
    try:
        core_model   = model.module if hasattr(model, 'module') else model
        was_training = core_model.training
        core_model.eval()

        test_text = "আমি কল বন্ধ করেছি।"
        try:
            tokenizer.src_lang = SOURCE_LANGUAGE
            tokenizer.tgt_lang = TARGET_LANGUAGE
        except Exception:
            pass

        inputs = tokenizer(
            test_text,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH
        )

        test_device    = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        input_ids      = inputs['input_ids'].to(test_device)
        attention_mask = inputs['attention_mask'].to(test_device)

        original_device = next(core_model.parameters()).device
        if test_device != original_device:
            core_model = core_model.to(test_device)

        try:
            with torch.no_grad():
                print("  [TEST 1] Testing forward pass (inference mode)...")
                outputs = core_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    src_texts=[test_text],
                    token_word_map=None,
                    labels=None,
                    return_dict=True,
                    path=2
                )

            if isinstance(outputs, torch.Tensor):
                print("  ✅ Forward pass successful (tensor output)")
                print(f"  Output shape: {outputs.shape}")
            elif isinstance(outputs, dict):
                print("  ✅ Forward pass successful (dict output)")
                print(f"  Keys: {list(outputs.keys())}")

                print("\n  [TEST 2] Validating DSCD-augmented generation capability...")
                h_augmented = outputs.get('sense_augmented_embeddings', None)
                if h_augmented is not None and isinstance(h_augmented, torch.Tensor):
                    print(f"    ✅ DSCD embeddings: shape={h_augmented.shape}")
                    try:
                        enc_wrapped = BaseModelOutput(
                            last_hidden_state=h_augmented,
                            hidden_states=None,
                            attentions=None,
                        )
                        print(f"    ✅ BaseModelOutput created successfully")

                        print("\n  [TEST 3] Testing generate() with DSCD-augmented embeddings...")
                        generated = core_model.generate(
                            input_ids=None,
                            attention_mask=attention_mask,
                            encoder_outputs=enc_wrapped,
                            max_length=32,
                            num_beams=3,
                            early_stopping=True
                        )
                        if generated is not None and generated.numel() > 0:
                            translation = tokenizer.decode(generated[0], skip_special_tokens=True)
                            print(f"    ✅ Generation successful: '{translation}'")
                            print(f"    ✅ DSCD-AUGMENTED GENERATION WORKING!")
                        else:
                            print(f"    ⚠️  Generation returned empty output")
                    except Exception as e:
                        print(f"    ❌ DSCD-augmented generation failed: {e}")
                        raise RuntimeError(f"DSCD-augmented generation validation failed: {e}")
                else:
                    print(f"    ⚠️  No DSCD embeddings in output")
            else:
                print(f"  ⚠️  Unexpected output type: {type(outputs)}")

            print("\n[VALIDATION] ✅ Model forward pass and generation validated")
        finally:
            if test_device != original_device:
                core_model = core_model.to(original_device)
            if was_training:
                core_model.train()

        return True
    except Exception as e:
        print(f"[VALIDATION] ❌ Forward pass failed: {e}")
        try:
            traceback.print_exc()
        except Exception:
            pass
        raise RuntimeError(f"Model forward pass validation failed: {e}")


def main_pipeline() -> Tuple[object, object]:
    print("\n" + "=" * 80)
    print("TATN MAIN PIPELINE (DUAL-PATH COMPATIBLE)")
    print("=" * 80)
    print(f"Configuration:")
    print(f"  - Span threshold:        {SPAN_THRESHOLD}")
    print(f"  - Uncertainty threshold: {UNCERTAINTY_THRESHOLD}")
    print(f"  - Discovery frequency:   {PERIODIC_DISCOVERY_FREQUENCY}")
    print(f"  - ASBN Training:         {'DISABLED' if not ENABLE_ASBN_TRAINING else 'ENABLED'}")
    print(f"  - Epochs:                {EPOCHS}")
    print(f"  - Batch size:            {BATCH_SIZE}")
    print(f"  - LR (mBART base):       {LR_NMT:.2e}")
    print(f"  - LR (SCEA/HFCAA):       {LR_ADAPTER:.2e}")
    print(f"  - LR (ASBN critic):      {LR_PHI:.2e}")
    print(f"  - Back-translation:      {'ENABLED' if USE_BACK_TRANSLATION else 'DISABLED'}, LAMBDA_BT={LAMBDA_BT}")
    print("=" * 80)

    pipeline_start = time.time()
    if DEBUG_TIMING:
        phase_start = time.time()

    initialize_environment()
    if DEBUG_TIMING:
        print(f"[TIMING] Initialization: {time.time() - phase_start:.2f}s")
        phase_start = time.time()

    # ── PHASE 1: Tokenizer ──────────────────────────────────────────────────
    print("\n[PHASE 1] Loading tokenizer...")
    tokenizer = _safe_tokenizer_from_pretrained(MBART_MODEL_NAME)
    try:
        tokenizer.src_lang = SOURCE_LANGUAGE
        tokenizer.tgt_lang = TARGET_LANGUAGE
    except Exception:
        pass

    try:
        if not hasattr(tokenizer, 'pad_token_id') or tokenizer.pad_token_id is None:
            if hasattr(tokenizer, 'add_special_tokens'):
                tokenizer.add_special_tokens({"pad_token": "<pad>"})
    except Exception:
        pass

    vocab_size = getattr(tokenizer, 'vocab_size', None)
    if vocab_size is None:
        try:
            vocab_size = len(tokenizer)
        except Exception:
            vocab_size = 250054

    print(f"[PHASE 1] Tokenizer loaded (vocab: {vocab_size})")

    if "validate_tokenizer_vocab" in globals():
        try:
            print("[PHASE 1] Validating tokenizer vocabulary...")
            globals()["validate_tokenizer_vocab"](tokenizer, expected_vocab_size=None)
        except Exception as e:
            print(f"[PHASE 1] Tokenizer validation warning: {e}")

    if DEBUG_TIMING:
        print(f"[TIMING] Tokenizer: {time.time() - phase_start:.2f}s")
        phase_start = time.time()

    # ── PHASE 2: Data loading & val/test split ──────────────────────────────
    print(f"\n[PHASE 2] Loading data ({NUM_SAMPLES} samples)...")
    pairs = None

    if "load_and_preprocess_optimized" in globals():
        try:
            pairs = globals()["load_and_preprocess_optimized"](NUM_SAMPLES)
            print(f"[PHASE 2] Loaded {len(pairs)} pairs from load_and_preprocess_optimized")
        except Exception as e:
            print(f"[PHASE 2] Data loading failed: {e}")
            pairs = None
    else:
        print("[PHASE 2] load_and_preprocess_optimized not found")

    if pairs is None or len(pairs) < MIN_TRAIN_SAMPLES:
        print(f"[PHASE 2] Insufficient pairs ({len(pairs) if pairs else 0}), using robust fallback")
        pairs = _get_robust_fallback_pairs()

    if len(pairs) < MIN_TRAIN_SAMPLES:
        raise RuntimeError(
            f"[PHASE 2] FATAL: Could not load enough training pairs. "
            f"Got {len(pairs)}, need at least {MIN_TRAIN_SAMPLES}. "
            f"Check DATASET_CSV_PATH in Cell 0."
        )

    print(f"[PHASE 2] Total pairs available: {len(pairs)}")

    n           = len(pairs)
    n_train     = int(n * TRAIN_RATIO)
    n_val       = int(n * VAL_RATIO)
    rng_split   = random.Random(SEED)
    pairs_shuf  = pairs[:]
    rng_split.shuffle(pairs_shuf)
    train_pairs = pairs_shuf[:n_train]
    val_pairs   = pairs_shuf[n_train : n_train + n_val]
    test_pairs  = pairs_shuf[n_train + n_val :]

    if "DualPathTranslationDataset" not in globals():
        raise RuntimeError("DualPathTranslationDataset not found - run Cell 2")

    word_tokenizer = globals().get("word_tokenizer", None)

    dataset = globals()["DualPathTranslationDataset"](
        train_pairs,
        tokenizer,
        word_tokenizer=word_tokenizer,
        max_length=MAX_LENGTH,
        split="train",
        domain=TRAIN_DOMAIN
    )

    if len(dataset) < MIN_TRAIN_SAMPLES:
        raise RuntimeError(
            f"[PHASE 2] FATAL: Dataset has {len(dataset)} valid samples after init, "
            f"need at least {MIN_TRAIN_SAMPLES}. "
            f"Check that CSV contains valid Bengali-English pairs."
        )

    val_dataset = globals()["DualPathTranslationDataset"](
        val_pairs,
        tokenizer,
        word_tokenizer=word_tokenizer,
        max_length=MAX_LENGTH,
        split="val",
        domain=TEST_DOMAIN
    )

    test_dataset = globals()["DualPathTranslationDataset"](
        test_pairs,
        tokenizer,
        word_tokenizer=word_tokenizer,
        max_length=MAX_LENGTH,
        split="test",
        domain=TEST_DOMAIN
    )

    collate_fn = globals().get("safe_collate", None)

    if "create_optimized_dataloader" in globals():
        try:
            train_loader = globals()["create_optimized_dataloader"](
                dataset, batch_size=BATCH_SIZE, shuffle=True
            )
        except Exception as e:
            print(f"[PHASE 2] create_optimized_dataloader failed: {e}, falling back to DataLoader")
            dataloader_kwargs = {
                'batch_size':  BATCH_SIZE,
                'shuffle':     True,
                'num_workers': 0,
                'pin_memory':  torch.cuda.is_available(),
                'drop_last':   False,
            }
            if collate_fn is not None:
                dataloader_kwargs['collate_fn'] = collate_fn
            train_loader = DataLoader(dataset, **dataloader_kwargs)
    else:
        dataloader_kwargs = {
            'batch_size':  BATCH_SIZE,
            'shuffle':     True,
            'num_workers': 0,
            'pin_memory':  torch.cuda.is_available(),
            'drop_last':   False,
        }
        if collate_fn is not None:
            dataloader_kwargs['collate_fn'] = collate_fn
        train_loader = DataLoader(dataset, **dataloader_kwargs)

    test_loader_kwargs = {
        'batch_size':  EVAL_BATCH_SIZE,
        'shuffle':     False,
        'num_workers': 0,
        'pin_memory':  torch.cuda.is_available(),
        'drop_last':   False,
    }
    if collate_fn is not None:
        test_loader_kwargs['collate_fn'] = collate_fn
    test_loader = DataLoader(test_dataset, **test_loader_kwargs)

    print(f"[PHASE 2] Train: {len(dataset)} samples, {len(train_loader)} batches")
    print(f"[PHASE 2] Val: {len(val_dataset)} samples (Dataset object) | Test: {len(test_dataset)} samples")

    # BT-augmented loader construction
    bt_dataset              = None
    bt_augmented_train_loader = None

    if USE_BACK_TRANSLATION:
        bt_train_pairs = globals().get("bt_train_pairs", None)
        if bt_train_pairs is not None and len(bt_train_pairs) > 0:
            try:
                bt_combined_pairs = train_pairs + list(bt_train_pairs)
                bt_dataset = globals()["DualPathTranslationDataset"](
                    bt_combined_pairs,
                    tokenizer,
                    word_tokenizer=word_tokenizer,
                    max_length=MAX_LENGTH,
                    split="train",
                    domain=TRAIN_DOMAIN
                )
                bt_loader_kwargs = {
                    'batch_size':  BATCH_SIZE,
                    'shuffle':     True,
                    'num_workers': 0,
                    'pin_memory':  torch.cuda.is_available(),
                    'drop_last':   False,
                }
                if collate_fn is not None:
                    bt_loader_kwargs['collate_fn'] = collate_fn
                bt_augmented_train_loader = DataLoader(bt_dataset, **bt_loader_kwargs)
                globals()["bt_augmented_train_loader"] = bt_augmented_train_loader
                print(
                    f"[PHASE 2] BT-augmented loader built: "
                    f"{len(bt_combined_pairs)} pairs "
                    f"({len(train_pairs)} original + {len(bt_train_pairs)} BT), "
                    f"{len(bt_augmented_train_loader)} batches"
                )
            except Exception as e:
                print(f"[PHASE 2] WARNING: BT loader construction failed: {type(e).__name__}: {e}")
                bt_augmented_train_loader = None
                globals()["bt_augmented_train_loader"] = None
        else:
            print("[PHASE 2] WARNING: USE_BACK_TRANSLATION=True but bt_train_pairs not found in globals — no BT loader built")
            globals()["bt_augmented_train_loader"] = None
    else:
        globals()["bt_augmented_train_loader"] = None

    del pairs, pairs_shuf, val_pairs, test_pairs
    _safe_clear_gpu_caches()

    if DEBUG_TIMING:
        print(f"[TIMING] Data loading: {time.time() - phase_start:.2f}s")
        phase_start = time.time()

    # ── PHASE 3: Model ──────────────────────────────────────────────────────
    print("\n[PHASE 3] Initializing model...")
    if "MemoryOptimizedTATNWithExplanations" not in globals():
        raise RuntimeError("Model class not found - run Cell 6")

    model_core = globals()["MemoryOptimizedTATNWithExplanations"](tokenizer)

    try:
        validate_component_compatibility(model_core, tokenizer)
    except Exception as e:
        print(f"[PHASE 3] ❌ Component validation failed: {e}")
        raise

    try:
        validate_dataset_compatibility(dataset, tokenizer, model_core.vocab_size)
    except Exception as e:
        print(f"[PHASE 3] ❌ Dataset validation failed: {e}")
        raise

    if USE_MULTI_GPU and NUM_GPUS > 1:
        device_ids = list(range(NUM_GPUS))
        print(f"[PHASE 3] Using DataParallel on {device_ids}")
        model = nn.DataParallel(model_core, device_ids=device_ids)
    else:
        model = model_core

    model      = model.to(DEVICE)
    core_model = _get_core_model(model)

    try:
        test_model_forward_pass(model, tokenizer, DEVICE)
    except Exception as e:
        print(f"[PHASE 3] ❌ Forward pass test failed: {e}")
        raise

    if FREEZE_ENCODER:
        try:
            for p in core_model.mbart.model.encoder.parameters():
                p.requires_grad = False
            print("[PHASE 3] Encoder frozen")
        except Exception:
            pass

    print(f"[PHASE 3] Model initialized and validated")
    if DEBUG_TIMING:
        print(f"[TIMING] Model init: {time.time() - phase_start:.2f}s")
        phase_start = time.time()

    # ── PHASE 4: Optimizers ─────────────────────────────────────────────────
    print("\n[PHASE 4] Setting up optimizers...")

    try:
        critic_params = list(
            core_model.asbn.critic_parameters()
        ) if hasattr(core_model, "asbn") and hasattr(core_model.asbn, "critic_parameters") else []
    except Exception:
        critic_params = []

    scea_params = []
    try:
        if hasattr(core_model, 'scea') and core_model.scea is not None:
            scea_params = [p for p in core_model.scea.parameters() if p.requires_grad]
            if not scea_params:
                for p in core_model.scea.parameters():
                    p.requires_grad = True
                scea_params = list(core_model.scea.parameters())
                print(f"[PHASE 4] ⚠️  SCEA params had requires_grad=False — force-enabled")
        else:
            print("[PHASE 4] ⚠️  SCEA module NOT found on core_model — OPT-1 skipped")
    except Exception as e:
        print(f"[PHASE 4] ⚠️  SCEA param extraction failed: {type(e).__name__}: {e}")
        scea_params = []

    hfcaa_params = []
    try:
        if hasattr(core_model, 'hfcaa') and core_model.hfcaa is not None:
            hfcaa_params = [p for p in core_model.hfcaa.parameters() if p.requires_grad]
            if not hfcaa_params:
                for p in core_model.hfcaa.parameters():
                    p.requires_grad = True
                hfcaa_params = list(core_model.hfcaa.parameters())
                print(f"[PHASE 4] ⚠️  HFCAA params had requires_grad=False — force-enabled")
        else:
            print("[PHASE 4] ⚠️  HFCAA module NOT found on core_model — OPT-2 skipped")
    except Exception as e:
        print(f"[PHASE 4] ⚠️  HFCAA param extraction failed: {type(e).__name__}: {e}")
        hfcaa_params = []

    critic_ids   = {id(p) for p in critic_params}
    adapter_ids  = {id(p) for p in scea_params} | {id(p) for p in hfcaa_params}
    excluded_ids = critic_ids | adapter_ids

    base_params = [
        p for p in core_model.parameters()
        if p.requires_grad and id(p) not in excluded_ids
    ]

    optimizer_param_groups = [
        {
            'params':       base_params,
            'lr':           LR_NMT,
            'weight_decay': WEIGHT_DECAY,
            'name':         'mbart_backbone'
        },
    ]

    if scea_params:
        optimizer_param_groups.append({
            'params':       scea_params,
            'lr':           LR_ADAPTER,
            'weight_decay': WEIGHT_DECAY,
            'name':         'scea_adapter'
        })

    if hfcaa_params:
        optimizer_param_groups.append({
            'params':       hfcaa_params,
            'lr':           LR_ADAPTER,
            'weight_decay': WEIGHT_DECAY,
            'name':         'hfcaa_adapter'
        })

    optimizer = torch.optim.AdamW(optimizer_param_groups)

    print(f"[PHASE 4] Optimizer param groups:")
    for i, pg in enumerate(optimizer.param_groups):
        n_params    = len(pg['params'])
        n_trainable = sum(1 for p in pg['params'] if p.requires_grad)
        print(f"  Group {i} [{pg.get('name', f'group_{i}')}]: "
              f"{n_params} tensors ({n_trainable} trainable), "
              f"lr={pg['lr']:.2e}, wd={pg['weight_decay']}")

    phi_optimizer     = None
    adapter_optimizer = None

    if critic_params and ENABLE_ASBN_TRAINING:
        phi_optimizer = torch.optim.AdamW(
            [p for p in critic_params if p.requires_grad],
            lr=LR_PHI,
            weight_decay=WEIGHT_DECAY
        )
        print(f"[PHASE 4] ASBN phi_optimizer: "
              f"{len([p for p in critic_params if p.requires_grad])} params, "
              f"lr={LR_PHI:.2e}")
    elif not ENABLE_ASBN_TRAINING:
        print(f"[PHASE 4] ASBN optimizer DISABLED (training disabled)")

    print(f"[PHASE 4] Optimizers ready | "
          f"base={len(base_params)} | "
          f"scea={len(scea_params)} | "
          f"hfcaa={len(hfcaa_params)} | "
          f"critic={len(critic_params)}")

    if DEBUG_TIMING:
        phase_start = time.time()

    # ── PHASE 5: Training ───────────────────────────────────────────────────
    print("\n[PHASE 5] Training...")
    print(f"  - ASBN Training: {'DISABLED' if not ENABLE_ASBN_TRAINING else 'ENABLED'}")
    print(f"  - ASBN Optimizer: {'None (ASBN disabled)' if phi_optimizer is None else 'Active'}")

    trained_model  = model
    training_stats = None

    if "train_memory_efficient_tatn" in globals():
        try:
            try:
                trg = getattr(core_model, 'trg', None)
                if trg and hasattr(trg, 'reset_statistics'):
                    trg.reset_statistics()
            except Exception:
                pass

            train_fn  = globals()["train_memory_efficient_tatn"]
            train_sig = inspect.signature(train_fn)

            _bt_loader = globals().get("bt_augmented_train_loader", None)
            effective_train_loader = (
                _bt_loader
                if (_bt_loader is not None and USE_BACK_TRANSLATION)
                else train_loader
            )

            if USE_BACK_TRANSLATION and _bt_loader is not None:
                print(f"[PHASE 5] Using BT-augmented train loader (len={len(effective_train_loader)})")
            elif USE_BACK_TRANSLATION and _bt_loader is None:
                print("[PHASE 5] WARNING: USE_BACK_TRANSLATION=True but bt_augmented_train_loader not found — using plain train_loader")
            else:
                print(f"[PHASE 5] Using plain train_loader (len={len(effective_train_loader)})")

            train_kwargs = dict(
                phi_optimizer=phi_optimizer,
                epochs=EPOCHS,
                accumulation_steps=ACCUMULATION_STEPS,
                validate_every=VALIDATION_CHECK_INTERVAL,
                enable_validation=(VALIDATION_CHECK_INTERVAL > 0),
                val_dataset=val_dataset,
                test_loader=test_loader,
            )
            if 'adapter_optimizer' in train_sig.parameters:
                train_kwargs['adapter_optimizer'] = adapter_optimizer

            trained_model = train_fn(
                model,
                tokenizer,
                effective_train_loader,
                optimizer,
                **train_kwargs
            )
            print("[PHASE 5] Training complete")
        except Exception as e:
            print(f"[PHASE 5] Training failed: {e}")
            trained_model = model
    else:
        print("[PHASE 5] Skipping training (function not found)")

    if DEBUG_TIMING:
        print(f"[TIMING] Training: {time.time() - phase_start:.2f}s")
        phase_start = time.time()

    del train_loader, dataset, val_dataset, test_loader, test_dataset
    if bt_dataset is not None:
        del bt_dataset
    if bt_augmented_train_loader is not None:
        del bt_augmented_train_loader
    _safe_clear_gpu_caches()

    core_model = _get_core_model(trained_model)

    # ── PHASE 6: Discovery check ────────────────────────────────────────────
    if DEBUG_TIMING:
        phase_start = time.time()

    print("\n[PHASE 6] Discovery check...")
    discovery_success = False

    try:
        dscd = getattr(core_model, 'dscd', None)
        if dscd is None:
            print("[PHASE 6] No DSCD module")
        else:
            print("[PHASE 6] Running periodic discovery check...")
            if hasattr(dscd, 'periodic_discovery_check'):
                try:
                    sig    = inspect.signature(dscd.periodic_discovery_check)
                    params = list(sig.parameters.keys())
                    print(f"[PHASE 6] periodic_discovery_check params: {params}")

                    total_steps    = int(EPOCHS * NUM_SAMPLES // BATCH_SIZE)
                    num_discovered = 0

                    if 'cluster_missing' in params:
                        if len(params) >= 3:
                            num_discovered = dscd.periodic_discovery_check(total_steps, PERIODIC_DISCOVERY_FREQUENCY, cluster_missing=False)
                        elif len(params) >= 2:
                            num_discovered = dscd.periodic_discovery_check(total_steps, cluster_missing=False)
                        else:
                            num_discovered = dscd.periodic_discovery_check(cluster_missing=False)
                    else:
                        if len(params) >= 2:
                            num_discovered = dscd.periodic_discovery_check(total_steps, PERIODIC_DISCOVERY_FREQUENCY)
                        elif len(params) >= 1:
                            num_discovered = dscd.periodic_discovery_check(total_steps)
                        else:
                            num_discovered = dscd.periodic_discovery_check()

                    num_discovered    = num_discovered if num_discovered is not None else 0
                    discovery_success = True
                    print(f"[PHASE 6] Discovery complete: {num_discovered} homographs found")
                except Exception as e:
                    print(f"[PHASE 6] periodic_discovery_check failed: {e}")
                    try:
                        if hasattr(dscd, 'discover_homographs'):
                            num_discovered    = dscd.discover_homographs()
                            discovery_success = True
                            print(f"[PHASE 6] Fallback discovery: {num_discovered} homographs")
                        else:
                            print("[PHASE 6] discover_homographs not available")
                    except Exception as e2:
                        print(f"[PHASE 6] Fallback discovery failed: {e2}")
            else:
                print("[PHASE 6] periodic_discovery_check not available")
                if hasattr(dscd, 'discover_homographs'):
                    try:
                        num_discovered    = dscd.discover_homographs()
                        discovery_success = True
                        print(f"[PHASE 6] discover_homographs: {num_discovered} homographs")
                    except Exception as e:
                        print(f"[PHASE 6] discover_homographs failed: {e}")

            stores = _get_dscd_stores_safe(dscd)

            def _store_size(s):
                try:
                    if callable(getattr(s, "size", None)):
                        return int(s.size())
                    return int(getattr(s, "size", 0))
                except Exception:
                    return 0

            total_protos = sum(_store_size(store) for store in stores.values())
            multi_sense  = sum(1 for store in stores.values() if _store_size(store) >= 2)

            print("[PHASE 6] Discovery state:")
            print(f"  - Tokens:      {len(stores)}")
            print(f"  - Prototypes:  {total_protos}")
            print(f"  - Multi-sense: {multi_sense}")

            if len(stores) == 0:
                print("[PHASE 6] WARNING: No prototypes created")
            else:
                discovery_success = True
    except Exception as e:
        print(f"[PHASE 6] Discovery failed: {e}")
        if DEBUG_TIMING:
            try:
                traceback.print_exc()
            except Exception:
                pass

    if DEBUG_TIMING:
        print(f"[TIMING] Discovery: {time.time() - phase_start:.2f}s")
    _safe_clear_gpu_caches()
    if DEBUG_TIMING:
        phase_start = time.time()

    # ── PHASE 7: DSCD warmup ────────────────────────────────────────────────
    print("\n[PHASE 7] DSCD warmup...")
    if "dscd_discovery_warmup" in globals():
        try:
            warmup_samples = min(9000, DSCD_WARMUP_SAMPLES)
            print(f"[PHASE 7] Processing {warmup_samples} warmup samples...")
            warmup_start = time.time()
            globals()["dscd_discovery_warmup"](
                trained_model, tokenizer,
                num_sents=warmup_samples, batch_size=64, max_len=MAX_LENGTH
            )
            warmup_duration = time.time() - warmup_start
            print(f"[PHASE 7] Warmup complete ({warmup_samples} samples in {warmup_duration:.1f}s)")
        except Exception as e:
            print(f"[PHASE 7] Warmup failed: {e}")
    else:
        print("[PHASE 7] Skipping warmup (function not found)")

    if DEBUG_TIMING:
        print(f"[TIMING] Warmup: {time.time() - phase_start:.2f}s")
    _safe_clear_gpu_caches()
    if DEBUG_TIMING:
        phase_start = time.time()

    # ── PHASE 8: Baseline evaluation ────────────────────────────────────────
    print("\n[PHASE 8] Baseline evaluation...")
    baseline_metrics = None

    try:
        dscd_baseline  = getattr(core_model, 'dscd', None)
        has_prototypes = False

        if dscd_baseline:
            stores         = _get_dscd_stores_safe(dscd_baseline)
            has_prototypes = len(stores) > 0

        if not has_prototypes:
            print("[PHASE 8] Skipping baseline (no prototypes)")
        elif "comprehensive_post_training_testing" in globals():
            print("[PHASE 8] ⚡ Running baseline with DSCD-augmented generation...")
            try:
                trg = getattr(core_model, 'trg', None)
                if trg and hasattr(trg, 'reset_statistics'):
                    trg.reset_statistics()
            except Exception:
                pass
            baseline_metrics = globals()["comprehensive_post_training_testing"](
                trained_model, tokenizer, run_warmup=False
            )
            baseline_success = baseline_metrics.get('success_rate_pct', 0)
            baseline_expl    = baseline_metrics.get('total_explanations', 0)
            print(f"[PHASE 8] Baseline: {baseline_success:.1f}% success, {baseline_expl} explanations")
        else:
            print("[PHASE 8] Skipping baseline (function not found)")
    except Exception as e:
        print(f"[PHASE 8] Baseline evaluation failed: {e}")
        if DEBUG_TIMING:
            try:
                traceback.print_exc()
            except Exception:
                pass

    if DEBUG_TIMING:
        print(f"[TIMING] Baseline: {time.time() - phase_start:.2f}s")
    _safe_clear_gpu_caches()
    if DEBUG_TIMING:
        phase_start = time.time()

    # ── PHASE 9: Post-training evaluation ───────────────────────────────────
    print("\n[PHASE 9] Post-training evaluation...")
    eval_metrics = None

    if "comprehensive_post_training_testing" in globals():
        try:
            print("[PHASE 9] Running comprehensive post-training tests...")
            try:
                trg = getattr(core_model, 'trg', None)
                if trg and hasattr(trg, 'reset_statistics'):
                    trg.reset_statistics()
            except Exception:
                pass
            eval_metrics = globals()["comprehensive_post_training_testing"](
                trained_model, tokenizer, run_warmup=False
            )
            eval_success = eval_metrics.get('success_rate_pct', 0)
            eval_expl    = eval_metrics.get('total_explanations', 0)
            print(f"[PHASE 9] Evaluation: {eval_success:.1f}% success, {eval_expl} explanations")
        except Exception as e:
            print(f"[PHASE 9] Evaluation failed: {e}")
    else:
        print("[PHASE 9] Skipping evaluation (function not found)")

    if DEBUG_TIMING:
        print(f"[TIMING] Evaluation: {time.time() - phase_start:.2f}s")
    _safe_clear_gpu_caches()
    if DEBUG_TIMING:
        phase_start = time.time()

    # ── PHASE 10: Save checkpoint ────────────────────────────────────────────
    print("\n[PHASE 10] Saving checkpoint...")
    try:
        final_checkpoint_path = get_checkpoint_path()
        os.makedirs(os.path.dirname(str(final_checkpoint_path)), exist_ok=True)
        save_core = _get_core_model(trained_model)

        dscd_state = None
        try:
            dscd_to_save = getattr(save_core, 'dscd', None)
            if dscd_to_save is not None:
                lock = None
                if hasattr(dscd_to_save, 'buffer_lock'):
                    lock = dscd_to_save.buffer_lock
                elif hasattr(dscd_to_save, 'clustering_lock'):
                    lock = dscd_to_save.clustering_lock

                def _build_dscd_state():
                    state = {}
                    for token, store in dscd_to_save.prototype_stores.items():
                        try:
                            store_sd = store.state_dict()
                            if store_sd:
                                state[token] = store_sd
                        except Exception as per_store_err:
                            if DEBUG_DISCOVERY:
                                print(f"[PHASE 10] store.state_dict() failed for '{token}': {per_store_err}")
                            continue
                    return state

                if lock:
                    with lock:
                        dscd_state = _build_dscd_state()
                else:
                    dscd_state = _build_dscd_state()

                num_saved          = len(dscd_state) if dscd_state else 0
                total_protos_saved = 0
                multi_sense_saved  = 0
                if dscd_state:
                    for token, store_sd in dscd_state.items():
                        try:
                            centroids = store_sd.get('centroids', store_sd.get('data', None))
                            if centroids is not None:
                                if isinstance(centroids, torch.Tensor):
                                    n = int(centroids.shape[0])
                                elif isinstance(centroids, list):
                                    n = len(centroids)
                                else:
                                    n = 0
                                total_protos_saved += n
                                if n >= 2:
                                    multi_sense_saved += 1
                        except Exception:
                            continue

                print(f"[PHASE 10] DSCD per-store state captured: "
                      f"tokens={num_saved}, protos={total_protos_saved}, multi-sense={multi_sense_saved}")

                if num_saved == 0:
                    print("[PHASE 10] WARNING: dscd_state is empty — no prototype stores to save")
                    dscd_state = None
            else:
                print("[PHASE 10] No DSCD module found on model — dscd_state=None")
        except Exception as e:
            print(f"[PHASE 10] WARNING: dscd_state capture failed: {e}")
            if DEBUG_DISCOVERY:
                traceback.print_exc()
            dscd_state = None

        optimizer_state = None
        try:
            optimizer_state = optimizer.state_dict()
        except Exception as e:
            print(f"[PHASE 10] WARNING: optimizer state capture failed: {e}")

        phi_optimizer_state = None
        if phi_optimizer is not None:
            try:
                phi_optimizer_state = phi_optimizer.state_dict()
            except Exception as e:
                print(f"[PHASE 10] WARNING: phi_optimizer state capture failed: {e}")

        checkpoint = {
            'model_state_dict':           save_core.state_dict(),
            'optimizer_state_dict':       optimizer_state,
            'phi_optimizer_state_dict':   phi_optimizer_state,
            'training_stats':             training_stats,
            'eval_metrics':               eval_metrics,
            'baseline_metrics':           baseline_metrics,
            'config': {
                'num_samples':                  NUM_SAMPLES,
                'max_length':                   MAX_LENGTH,
                'batch_size':                   BATCH_SIZE,
                'epochs':                       EPOCHS,
                'accumulation_steps':           ACCUMULATION_STEPS,
                'lr_nmt':                       LR_NMT,
                'lr_phi':                       LR_PHI,
                'lr_adapter':                   LR_ADAPTER,
                'weight_decay':                 WEIGHT_DECAY,
                'seed':                         SEED,
                'source_language':              SOURCE_LANGUAGE,
                'target_language':              TARGET_LANGUAGE,
                'enable_asbn_training':         ENABLE_ASBN_TRAINING,
                'validation_check_interval':    VALIDATION_CHECK_INTERVAL,
                'periodic_discovery_frequency': PERIODIC_DISCOVERY_FREQUENCY,
                'dscd_warmup_samples':          DSCD_WARMUP_SAMPLES,
                'span_threshold':               SPAN_THRESHOLD,
                'uncertainty_threshold':        UNCERTAINTY_THRESHOLD,
                'scea_trainable_params':        len(scea_params),
                'hfcaa_trainable_params':       len(hfcaa_params),
                'checkpoint_path':              str(final_checkpoint_path),
                'checkpoint_epoch_template':    CHECKPOINT_EPOCH_FILENAME_TEMPLATE,
                'use_back_translation':         USE_BACK_TRANSLATION,
                'lambda_bt':                    LAMBDA_BT,
            }
        }
        torch.save(checkpoint, final_checkpoint_path, _use_new_zipfile_serialization=False)
        print(f"[PHASE 10] Checkpoint saved: {final_checkpoint_path}")

        if dscd_state is not None:
            dscd_checkpoint_path = str(final_checkpoint_path).replace('.pt', '_dscd.pt')
            torch.save({'dscd_state': dscd_state}, dscd_checkpoint_path, _use_new_zipfile_serialization=False)
            print(f"[PHASE 10] DSCD checkpoint saved: {dscd_checkpoint_path}")
        else:
            print(f"[PHASE 10] DSCD checkpoint skipped (no prototype stores)")

        print(f"[PHASE 10]   - model_state_dict:         ✅")
        print(f"[PHASE 10]   - dscd_state (prototypes):  "
              f"{'✅ separate file: ' + str(final_checkpoint_path).replace('.pt', '_dscd.pt') + ' — ' + str(len(dscd_state)) + ' token stores' if dscd_state is not None else '⚠️  None (no DSCD module or empty stores)'}")
        print(f"[PHASE 10]   - optimizer_state_dict:     {'✅' if optimizer_state is not None else '⚠️  None'}")
        print(f"[PHASE 10]   - phi_optimizer_state_dict: {'✅' if phi_optimizer_state is not None else 'N/A (ASBN disabled)'}")
        print(f"[PHASE 10]   - eval_metrics:             {'✅' if eval_metrics is not None else '⚠️  None'}")
        print(f"[PHASE 10]   - baseline_metrics:         {'✅' if baseline_metrics is not None else '⚠️  None'}")
        print(f"[PHASE 10]   - SCEA params trained:      {len(scea_params)}")
        print(f"[PHASE 10]   - HFCAA params trained:     {len(hfcaa_params)}")
    except Exception as e:
        print(f"[PHASE 10] Checkpoint save failed: {e}")

    if DEBUG_TIMING:
        print(f"[TIMING] Checkpoint: {time.time() - phase_start:.2f}s")

    # ── PHASE 11: Final validation ───────────────────────────────────────────
    print("\n[PHASE 11] Final validation...")
    try:
        final_dscd = getattr(_get_core_model(trained_model), 'dscd', None)
        if final_dscd:
            final_stores = _get_dscd_stores_safe(final_dscd)

            def _store_size_final(s):
                try:
                    if callable(getattr(s, "size", None)):
                        return int(s.size())
                    return int(getattr(s, "size", 0))
                except Exception:
                    return 0

            final_protos = sum(_store_size_final(store) for store in final_stores.values())
            final_multi  = sum(1 for store in final_stores.values() if _store_size_final(store) >= 2)
            print(
                f"[PHASE 11] Final DSCD state: {len(final_stores)} tokens, "
                f"{final_protos} prototypes, {final_multi} multi-sense"
            )
        else:
            print("[PHASE 11] No DSCD module found for final validation")
    except Exception as e:
        print(f"[PHASE 11] Final validation failed: {e}")

    pipeline_duration = time.time() - pipeline_start
    print(f"\n[PIPELINE] Total time: {pipeline_duration / 60:.1f} minutes")
    print("[PIPELINE] ✅ Pipeline complete")

    return trained_model, tokenizer


print("=" * 80)
print("Cell 10: TATN MAIN PIPELINE (DUAL-PATH COMPATIBLE) — READY")
print(f"  LR_NMT (backbone):        {LR_NMT:.2e}")
print(f"  LR_ADAPTER (SCEA/HFCAA):  {LR_ADAPTER:.2e}")
print(f"  LR_PHI (ASBN critic):     {LR_PHI:.2e}")
print(f"  CHECKPOINT_PATH:          {CHECKPOINT_PATH}")
print(f"  CHECKPOINT_EPOCH_TEMPLATE:{CHECKPOINT_EPOCH_FILENAME_TEMPLATE}")
print(f"  USE_BACK_TRANSLATION:     {USE_BACK_TRANSLATION}")
print(f"  LAMBDA_BT:                {LAMBDA_BT}")
print("  main_pipeline() defined and available for Cell 11")
print("=" * 80)


In [ ]:
# ==============================================================================
# CELL 11: MAIN EXECUTION WRAPPER (DUAL-PATH COMPATIBLE)
# ==============================================================================

from datetime import datetime, timezone
import os
import traceback
import math
import sys
import time
import torch
import gc

try:
    NUM_SAMPLES                  = int(globals().get('NUM_SAMPLES', 70000))
    EPOCHS                       = int(globals().get('EPOCHS', 1))
    BATCH_SIZE                   = int(globals().get('BATCH_SIZE', 4))
    ACCUMULATION_STEPS           = int(globals().get('ACCUMULATION_STEPS', 16))

    raw_device = globals().get('DEVICE', "cuda" if torch.cuda.is_available() else "cpu")
    if isinstance(raw_device, torch.device):
        DEVICE = raw_device
    else:
        DEVICE = torch.device(str(raw_device))

    LR_ADAPTER                   = float(globals().get('LR_ADAPTER', 5e-5))
    ENABLE_ASBN_TRAINING         = bool(globals().get('ENABLE_ASBN_TRAINING', True))
    ENABLE_TRG_INFERENCE         = bool(globals().get('ENABLE_TRG_INFERENCE', True))
    PERIODIC_DISCOVERY_FREQUENCY = int(globals().get('PERIODIC_DISCOVERY_FREQUENCY', 300))
    VERBOSE_LOGGING              = bool(globals().get('VERBOSE_LOGGING', False))
    DEBUG_DISCOVERY              = bool(globals().get('DEBUG_DISCOVERY', False))
    DEBUG_TIMING                 = bool(globals().get('DEBUG_TIMING', False))
    NUM_GPUS                     = int(globals().get('NUM_GPUS', torch.cuda.device_count() if torch.cuda.is_available() else 0))
    USE_MULTI_GPU                = bool(globals().get('USE_MULTI_GPU', NUM_GPUS > 1))
    SPAN_THRESHOLD               = float(globals().get('SPAN_THRESHOLD', 0.20))
    UNCERTAINTY_THRESHOLD        = float(globals().get('UNCERTAINTY_THRESHOLD', 0.15))
    MAX_LENGTH                   = int(globals().get('MAX_LENGTH', 256))
    SOURCE_LANGUAGE              = str(globals().get('SOURCE_LANGUAGE', 'bn_IN'))
    TARGET_LANGUAGE              = str(globals().get('TARGET_LANGUAGE', 'en_XX'))
    USE_BACK_TRANSLATION         = bool(globals().get('USE_BACK_TRANSLATION', False))
    LAMBDA_BT                    = float(globals().get('LAMBDA_BT', 1.0))

    raw_list = globals().get('HOMOGRAPH_REFERENCE_LIST_BN', ["কল", "কাল", "পাতা", "ব্যাংক", "ফল", "মাথা"])
    HOMOGRAPH_REFERENCE_LIST_BN  = set(str(w) for w in raw_list)
    cell0_loaded                 = 'NUM_SAMPLES' in globals()

except (NameError, TypeError, ValueError) as e:
    print(f"[EXEC] Config load error: {e}")
    NUM_SAMPLES                  = 70000
    EPOCHS                       = 1
    BATCH_SIZE                   = 4
    ACCUMULATION_STEPS           = 16
    DEVICE                       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    LR_ADAPTER                   = 5e-5
    ENABLE_ASBN_TRAINING         = True
    ENABLE_TRG_INFERENCE         = True
    PERIODIC_DISCOVERY_FREQUENCY = 300
    VERBOSE_LOGGING              = False
    DEBUG_DISCOVERY              = False
    DEBUG_TIMING                 = False
    NUM_GPUS                     = torch.cuda.device_count() if torch.cuda.is_available() else 0
    USE_MULTI_GPU                = (NUM_GPUS > 1)
    SPAN_THRESHOLD               = 0.20
    UNCERTAINTY_THRESHOLD        = 0.15
    MAX_LENGTH                   = 256
    SOURCE_LANGUAGE              = 'bn_IN'
    TARGET_LANGUAGE              = 'en_XX'
    USE_BACK_TRANSLATION         = False
    LAMBDA_BT                    = 1.0
    HOMOGRAPH_REFERENCE_LIST_BN  = {"কল", "কাল", "পাতা", "ব্যাংক", "ফল", "মাথা"}
    cell0_loaded                 = False
    print("[EXEC] Using fallback configuration (Cell 0 not executed)")

try:
    CHECKPOINT_PATH = get_checkpoint_path()
except (NameError, Exception):
    CHECKPOINT_PATH = os.path.join(
        globals().get("CHECKPOINT_DIR", "/kaggle/working"),
        globals().get("CHECKPOINT_FILENAME", "tatn_final.pt")
    )


def _safe_div_ceil(a: int, b: int) -> int:
    try:
        if isinstance(a, int) and isinstance(b, int) and b > 0:
            return math.ceil(a / b)
    except Exception:
        pass
    return 0


def _format_duration(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    elif seconds < 3600:
        return f"{seconds/60:.1f}min"
    else:
        return f"{seconds/3600:.2f}hr"


def _safe_get(d: dict, *keys, default=None):
    if not isinstance(d, dict):
        return default
    result = d
    for key in keys:
        if not isinstance(result, dict):
            return default
        if key not in result:
            return default
        result = result[key]
    return result if result is not None else default


def _get_dscd_homographs(model):
    try:
        core = model.module if hasattr(model, 'module') else model
        dscd = getattr(core, 'dscd', None)

        if dscd and hasattr(dscd, 'get_discovered_homographs'):
            try:
                return dscd.get_discovered_homographs()
            except Exception:
                pass

        if dscd and hasattr(dscd, 'prototype_stores'):
            homographs = set()

            lock = None
            if hasattr(dscd, 'buffer_lock'):
                lock = dscd.buffer_lock
            elif hasattr(dscd, 'clustering_lock'):
                lock = dscd.clustering_lock

            try:
                if lock:
                    try:
                        with lock:
                            stores = dict(dscd.prototype_stores)
                    except Exception:
                        stores = dict(dscd.prototype_stores)
                else:
                    stores = dict(dscd.prototype_stores)
            except Exception:
                return set()

            for token, store in stores.items():
                try:
                    size_ok = False
                    if hasattr(store, 'size'):
                        size_attr = getattr(store, 'size')
                        if callable(size_attr):
                            try:
                                size_val = size_attr()
                                size_ok = int(size_val) >= 1
                            except Exception:
                                size_ok = False
                        elif isinstance(size_attr, int):
                            size_ok = size_attr >= 1

                    if size_ok:
                        clean = str(token).replace('▁', '').replace('Ġ', '').replace('##', '').strip().lower()
                        if clean:
                            homographs.add(clean)
                except Exception:
                    continue
            return homographs
    except Exception:
        pass
    return set()


def _safe_cleanup():
    try:
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                try:
                    with torch.cuda.device(i):
                        torch.cuda.empty_cache()
                except Exception:
                    pass
        if gc.isenabled():
            gc.collect()
    except Exception:
        pass


print("=" * 80)
print("MEMORY-OPTIMIZED TATN (DUAL-PATH COMPATIBLE)")
print("=" * 80)

user_login  = os.getenv("KAGGLE_USERNAME") or os.getenv("USER") or "manas0003"
start_time  = time.time()
now_utc     = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

print(f"User: {user_login}")
print(f"Started: {now_utc}")

print("\n[CONFIGURATION]")
print(f"  Cell 0 status: {'Loaded' if cell0_loaded else 'Using fallbacks'}")
print(f"  Samples: {NUM_SAMPLES}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Accumulation: {ACCUMULATION_STEPS}")
print(f"  Device: {DEVICE}")
print(f"  Multi-GPU: {'ENABLED' if USE_MULTI_GPU else 'DISABLED'} ({NUM_GPUS} GPUs)")
print(f"  Source language: {SOURCE_LANGUAGE}")
print(f"  Target language: {TARGET_LANGUAGE}")
print(f"  Span threshold: {SPAN_THRESHOLD}")
print(f"  Uncertainty threshold: {UNCERTAINTY_THRESHOLD}")
print(f"  Max length: {MAX_LENGTH}")
print(f"  LR Adapter: {LR_ADAPTER}")
print(f"  Discovery frequency: {PERIODIC_DISCOVERY_FREQUENCY}")
print(f"  Back-translation: {'ENABLED' if USE_BACK_TRANSLATION else 'DISABLED'}, LAMBDA_BT={LAMBDA_BT}")

if USE_MULTI_GPU and NUM_GPUS > 0:
    per_gpu = _safe_div_ceil(BATCH_SIZE, NUM_GPUS)
    print(f"  Batch per GPU: {per_gpu}")

print(f"  ASBN: {'Enabled' if ENABLE_ASBN_TRAINING else 'Disabled'}")
print(f"  TRG: {'Enabled' if ENABLE_TRG_INFERENCE else 'Disabled'}")
print(f"  Debug: {'Enabled' if DEBUG_DISCOVERY else 'Disabled'}")
print("=" * 80)

trained_model    = None
tokenizer        = None
pipeline_success = False
checkpoint_valid = False
failure_category = None
failure_details  = ""

if 'main_pipeline' not in globals():
    print("\nERROR: main_pipeline not found")
    print("   -> Run Cell 10 before executing Cell 11")
    failure_category = "MISSING_DEPENDENCY"
    failure_details  = "Cell 10 not executed"
else:
    try:
        print("\nStarting pipeline...")

        if DEBUG_TIMING:
            print("   Expected: ~15-45 min (config dependent)")

        pipeline_start           = time.time()
        trained_model, tokenizer = main_pipeline()
        pipeline_duration        = time.time() - pipeline_start

        print(f"\nPipeline completed: {_format_duration(pipeline_duration)}")
        pipeline_success = True

    except KeyboardInterrupt:
        print("\nInterrupted by user")
        failure_category = "USER_INTERRUPT"
        failure_details  = "Manual stop"

    except RuntimeError as e:
        msg = str(e).lower()

        if "tokenizer" in msg or "sentencepiece" in msg:
            print("\nTokenizer error")
            failure_category = "TOKENIZER_ERROR"
            failure_details  = str(e)[:200]
            print("\nFix:")
            print("   ! pip install transformers==4.30.2 sentencepiece tokenizers")
            print("   Then RESTART kernel and re-run Cells 0-11")

        elif "out of memory" in msg:
            print("\nOut of Memory")
            failure_category = "OOM_ERROR"
            failure_details  = "GPU OOM"
            print("\nFixes:")
            print("   1. Reduce BATCH_SIZE (try 2-4)")
            print("   2. Reduce NUM_SAMPLES (try 10k-20k)")
            print("   3. Increase ACCUMULATION_STEPS (32-64)")

        else:
            print(f"\nRuntime error: {type(e).__name__}")
            print(f"   {str(e)[:400]}")
            failure_category = "RUNTIME_ERROR"
            failure_details  = str(e)[:200]

        if VERBOSE_LOGGING or DEBUG_DISCOVERY:
            print("\n[TRACEBACK]")
            try:
                traceback.print_exc()
            except Exception:
                pass

    except Exception as e:
        print(f"\nUnexpected error: {type(e).__name__}")
        print(f"   {str(e)[:400]}")
        failure_category = "UNKNOWN_ERROR"
        failure_details  = str(e)[:200]

        if VERBOSE_LOGGING or DEBUG_DISCOVERY:
            print("\n[TRACEBACK]")
            try:
                traceback.print_exc()
            except Exception:
                pass

checkpoint_dict = None

if pipeline_success and trained_model is not None and tokenizer is not None:
    print("\n" + "=" * 80)
    print("PIPELINE SUCCEEDED")
    print("=" * 80)

    print("\n[CHECKPOINT]")

    try:
        if os.path.exists(CHECKPOINT_PATH):
            size_mb = os.path.getsize(CHECKPOINT_PATH) / (1024 ** 2)
            print(f"  File: {CHECKPOINT_PATH}")
            print(f"  Size: {size_mb:.1f} MB")

            checkpoint_dict = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)

            has_model      = 'model_state_dict' in checkpoint_dict and len(checkpoint_dict['model_state_dict']) > 0
            has_dscd       = 'dscd_state' in checkpoint_dict and len(checkpoint_dict.get('dscd_state', {})) > 0
            has_optimizer  = 'optimizer_state_dict' in checkpoint_dict and checkpoint_dict.get('optimizer_state_dict') is not None
            has_phi_opt    = 'phi_optimizer_state_dict' in checkpoint_dict and checkpoint_dict.get('phi_optimizer_state_dict') is not None
            has_adapter_lr = (
                has_optimizer and
                isinstance(checkpoint_dict.get('optimizer_state_dict'), dict) and
                'param_groups' in checkpoint_dict['optimizer_state_dict'] and
                any(
                    pg.get('lr') == checkpoint_dict.get('config', {}).get('lr_adapter', LR_ADAPTER)
                    for pg in checkpoint_dict['optimizer_state_dict'].get('param_groups', [])
                )
            )

            ckpt_cfg = checkpoint_dict.get('config', {})
            ckpt_bt  = ckpt_cfg.get('use_back_translation', None)
            ckpt_lbt = ckpt_cfg.get('lambda_bt', None)

            print(f"  Model:              {'Present' if has_model else 'MISSING'}")
            print(f"  DSCD:               {'Present' if has_dscd else 'MISSING'}")
            print(f"  Optimizer:          {'Present' if has_optimizer else 'MISSING'}")
            print(f"  Adapter LR group:   {'Present' if has_adapter_lr else 'Not detected (single-group optimizer)'}")
            print(f"  Phi Optimizer:      {'Present' if has_phi_opt else 'N/A (ASBN disabled or no critic params)'}")
            if ckpt_bt is not None:
                print(f"  Back-translation:   {ckpt_bt}, LAMBDA_BT={ckpt_lbt}")
            else:
                print(f"  Back-translation:   Not recorded in checkpoint config")

            if has_dscd:
                try:
                    dscd_state = checkpoint_dict['dscd_state']
                    num_tokens = len(dscd_state)

                    print(f"  Tokens: {num_tokens}")

                    if num_tokens > 0 and has_model and (has_optimizer or has_adapter_lr):
                        checkpoint_valid = True
                        print("  Status: VALID")
                    elif num_tokens > 0 and has_model:
                        checkpoint_valid = True
                        print("  Status: VALID (optimizer state absent)")
                    else:
                        print("  Status: EMPTY DSCD")
                except Exception as e:
                    print(f"  Status: VALIDATION ERROR ({str(e)[:50]})")
            else:
                print("  Status: MISSING DSCD")
        else:
            print(f"  NOT FOUND: {CHECKPOINT_PATH}")

    except Exception as e:
        print(f"  Validation failed: {e}")
        checkpoint_dict = None

    print("\n[COMPONENTS]")

    try:
        core = trained_model.module if hasattr(trained_model, 'module') else trained_model

        dscd = getattr(core, 'dscd', None)
        if dscd and hasattr(dscd, 'get_prototype_summary'):
            try:
                dscd_stats = dscd.get_prototype_summary()
                print("  DSCD:")
                print(f"    - Tokens: {dscd_stats.get('total_tokens', 0)}")
                print(f"    - Prototypes: {dscd_stats.get('total_prototypes', 0)}")
                print(f"    - Homographs: {dscd_stats.get('num_homographs', 0)}")
            except Exception:
                pass

        asbn = getattr(core, 'asbn', None)
        if asbn and hasattr(asbn, 'get_detailed_stats'):
            try:
                asbn_stats = asbn.get_detailed_stats()
                print("  ASBN:")
                print(f"    - Domain accuracy: {asbn_stats.get('domain_accuracy', 0):.2%} {'(DISABLED)' if not ENABLE_ASBN_TRAINING else ''}")
                if 'source_accuracy' in asbn_stats:
                    print(f"    - Source: {asbn_stats['source_accuracy']:.2%}")
                    print(f"    - Target: {asbn_stats['target_accuracy']:.2%}")
            except Exception:
                pass

        trg = getattr(core, 'trg', None)
        if trg and hasattr(trg, 'get_statistics'):
            try:
                trg_stats = trg.get_statistics()
                print("  TRG:")
                print(f"    - Explanations: {trg_stats.get('explanations_generated', 0)}")
                print(f"    - High confidence: {trg_stats.get('high_confidence_rate', 0):.1%}")
                print(f"    - DSCD homograph rate: {trg_stats.get('dscd_homograph_rate', 0):.1%}")
            except Exception:
                pass

    except Exception as e:
        print(f"  Stats failed: {e}")

    print("\n[METRICS]")

    try:
        if checkpoint_dict is not None:
            training_stats = checkpoint_dict.get('training_stats', None)
            if training_stats is not None:
                total_loss = training_stats.get('total_loss', [])
                updates    = training_stats.get('optimizer_updates', 0)
                print("  Training:")
                print(f"    - Updates: {updates}")
                if total_loss:
                    if len(total_loss) >= 100:
                        final = sum(total_loss[-100:]) / len(total_loss[-100:])
                    else:
                        final = sum(total_loss) / len(total_loss)
                    print(f"    - Final loss: {final:.6f}")
            else:
                print("  Training: stats not available (Cell 7 does not return stats)")

            eval_metrics = checkpoint_dict.get('eval_metrics', None)
            baseline     = checkpoint_dict.get('baseline_metrics', None)

            if eval_metrics is not None:
                final_success = eval_metrics.get('success_rate_pct', 0)
                total_expl    = eval_metrics.get('total_explanations', 0)

                print("  Evaluation:")
                if baseline is not None:
                    baseline_success = baseline.get('success_rate_pct', 0)
                    improvement      = final_success - baseline_success
                    print(f"    - Baseline -> Final: {baseline_success:.1f}% -> {final_success:.1f}%")
                    print(f"    - Improvement: {improvement:+.1f}%")
                else:
                    print(f"    - Success: {final_success:.1f}%")

                print(f"    - Explanations: {total_expl}")

                quality = eval_metrics.get('quality_metrics', {})
                if quality:
                    print(f"    - Avg confidence: {quality.get('avg_confidence', 0):.3f}")
            else:
                print("  Evaluation: metrics not available (Cell 9 may have been skipped)")

        elif os.path.exists(CHECKPOINT_PATH):
            print("  Checkpoint loaded but invalid format")
        else:
            print("  No checkpoint available")

    except Exception as e:
        print(f"  Metrics failed: {e}")

    if checkpoint_dict is not None:
        try:
            del checkpoint_dict
        except Exception:
            pass
    _safe_cleanup()

    print("\n[INFERENCE VALIDATION]")
    print("Testing disambiguation on ambiguous sentences...")
    print("-" * 80)

    inference_success        = 0
    inference_failed         = 0
    dscd_homographs_detected = set()
    inference_times          = []

    dscd_homographs = _get_dscd_homographs(trained_model)
    print(f"DSCD discovered: {len(dscd_homographs)} homographs")
    if dscd_homographs and DEBUG_DISCOVERY:
        print(f"  Sample: {list(dscd_homographs)[:10]}")

    test_sentences = [
        ("আমি কল বন্ধ করেছি।",   "কল (tap/call)"),
        ("কাল আমি বই কিনব।",      "কাল (tomorrow/yesterday)"),
        ("পাতা ঝরে পড়েছে।",      "পাতা (leaf/page)"),
    ]

    try:
        if 'translate_with_explanations' not in globals():
            print("translate_with_explanations not available")
            print("   -> Run Cell 8 before Cell 11")
        else:
            for idx, (sentence, desc) in enumerate(test_sentences, 1):
                try:
                    print(f"\n{idx}.  {desc}")
                    print(f"   Input: {sentence}")

                    inf_start = time.time()

                    res = translate_with_explanations(
                        trained_model,
                        tokenizer,
                        sentence,
                        source_lang=SOURCE_LANGUAGE,
                        target_lang=TARGET_LANGUAGE,
                        device=DEVICE,
                        max_length=MAX_LENGTH,
                    )

                    inf_time = time.time() - inf_start
                    inference_times.append(inf_time)

                    if isinstance(res, dict):
                        translation = res.get('translation', 'N/A')
                        amb_count   = res.get('ambiguous_words_detected', 0)
                        exs         = res.get('explanations', []) or []

                        print(f"   Translation: {translation}")
                        print(f"   Ambiguous: {amb_count}")
                        print(f"   Time: {inf_time:.3f}s")

                        if exs:
                            for exp in exs:
                                word  = exp.get('ambiguous_word', exp.get('token', 'N/A'))
                                clean = str(word).replace('▁', '').replace('Ġ', '').strip().lower()

                                if clean in dscd_homographs:
                                    dscd_homographs_detected.add(clean)

                                try:
                                    conf = float(exp.get('confidence', 0.5))
                                    span = float(exp.get('span', 0.0))
                                    u    = float(exp.get('uncertainty', 0.0))
                                    print(f"   -> '{word}': conf={conf:.3f}, s={span:.3f}, u={u:.3f}")
                                except Exception:
                                    print(f"   -> '{word}': (no metrics)")

                            inference_success += 1
                        else:
                            print("   No explanations")
                            inference_success += 1
                    else:
                        print("   Unexpected format")
                        inference_failed += 1

                    _safe_cleanup()

                except Exception as e:
                    print(f"   Failed: {type(e).__name__} - {str(e)[:100]}")
                    inference_failed += 1
                    if DEBUG_DISCOVERY:
                        try:
                            traceback.print_exc()
                        except Exception:
                            pass

            print("\n" + "-" * 80)
            print(f"Results: {inference_success}/{len(test_sentences)} successful")

            if inference_times:
                avg_time = sum(inference_times) / len(inference_times)
                print(f"Performance: {avg_time:.3f}s avg per sentence")

            if dscd_homographs_detected:
                print(f"DSCD homographs detected: {', '.join(sorted(dscd_homographs_detected))}")
            else:
                print("No DSCD homographs detected")
                if len(dscd_homographs) == 0:
                    print("   -> DSCD has no discoveries (run warmup)")
                else:
                    print(f"   -> Check TRG thresholds (span={SPAN_THRESHOLD}, u={UNCERTAINTY_THRESHOLD})")

            if 'INFERENCE_STATS' in globals():
                try:
                    print("\n" + "-" * 80)
                    print("AGGREGATED STATISTICS (from Cell 8):")
                    print("-" * 80)
                    INFERENCE_STATS.print_summary()
                except Exception as e:
                    if DEBUG_DISCOVERY:
                        print(f"Failed to print INFERENCE_STATS: {e}")
            else:
                if DEBUG_DISCOVERY:
                    print("\nINFERENCE_STATS not available (Cell 8 not loaded)")

    except Exception as e:
        print(f"Validation failed: {e}")
        if DEBUG_DISCOVERY:
            try:
                traceback.print_exc()
            except Exception:
                pass

    print("\n[SYSTEM TEST]")

    try:
        core = trained_model.module if hasattr(trained_model, 'module') else trained_model

        dscd_ok  = hasattr(core, 'dscd') and hasattr(core.dscd, 'forward')
        asbn_ok  = hasattr(core, 'asbn') and hasattr(core.asbn, 'forward')
        trg_ok   = hasattr(core, 'trg') and hasattr(core.trg, 'process_sentence_for_explanations')
        mbart_ok = hasattr(core, 'mbart') and hasattr(core.mbart, 'generate')

        print("  Component status:")
        print(f"    - DSCD:  {'OK' if dscd_ok else 'MISSING'}")
        print(f"    - ASBN:  {'OK' if asbn_ok else 'MISSING'} {'(DISABLED)' if not ENABLE_ASBN_TRAINING else ''}")
        print(f"    - TRG:   {'OK' if trg_ok else 'MISSING'}")
        print(f"    - mBART: {'OK' if mbart_ok else 'MISSING'}")

        all_ok = dscd_ok and asbn_ok and trg_ok and mbart_ok

        if all_ok:
            print("  All components operational")
        else:
            print("  Some components missing")

    except Exception as e:
        print(f"  Test failed: {e}")

    print("\n" + "=" * 80)
    print("NEXT STEPS")
    print("=" * 80)

    print("\n1. Single translation:")
    print(f"   result = translate_with_explanations(trained_model, tokenizer, 'আমি কল বন্ধ করেছি।', source_lang='{SOURCE_LANGUAGE}', target_lang='{TARGET_LANGUAGE}', device=DEVICE, max_length={MAX_LENGTH})")

    print("\n2. Batch translation:")
    print("   for sent in sentences:")
    print(f"       res = translate_with_explanations(trained_model, tokenizer, sent, source_lang='{SOURCE_LANGUAGE}', target_lang='{TARGET_LANGUAGE}', device=DEVICE, max_length={MAX_LENGTH})")

    print("\n3. Load checkpoint:")
    print(f"   ckpt = torch.load('{CHECKPOINT_PATH}', weights_only=False)")
    print("   model.load_state_dict(ckpt['model_state_dict'])")
    print("   core = model.module if hasattr(model, 'module') else model")
    print("   dscd_state = ckpt['dscd_state']")
    print("   for token, store_data in dscd_state.items():")
    print("       if token in core.dscd.prototype_stores:")
    print("           core.dscd.prototype_stores[token].load_state_dict(store_data)")
    print("   optimizer.load_state_dict(ckpt['optimizer_state_dict'])")
    print("   # ckpt['phi_optimizer_state_dict']  <- present if ASBN was enabled")
    print("   # ckpt['config']['use_back_translation'], ckpt['config']['lambda_bt']  <- BT settings")

    print("\n4. Full evaluation:")
    print("   results = comprehensive_post_training_testing(trained_model, tokenizer)")

    print("\n5. Demo:")
    print("   demonstrate_system(trained_model, tokenizer)")

    if not checkpoint_valid:
        print("\nCheckpoint needs verification - re-run Cell 10 if needed")

    print("\n" + "=" * 80)

else:
    print("\n" + "=" * 80)
    print("PIPELINE FAILED")
    print("=" * 80)

    print(f"\nCategory: {failure_category or 'UNKNOWN'}")
    if failure_details:
        print(f"Details: {failure_details[:200]}")

    print("\n[DIAGNOSTICS]")

    components = {
        'Cell 0':    'NUM_SAMPLES' in globals(),
        'Cell 1':    'reconstruct_word_spans' in globals(),
        'Cell 2':    'MemoryEfficientDataset' in globals(),
        'Cell 3':    'MemoryEfficientDSCDOnline' in globals(),
        'Cell 4':    'MemoryEfficientASBNModule' in globals(),
        'Cell 5':    'CompleteTRGWithExplanations' in globals(),
        'Cell 6':    'MemoryOptimizedTATNWithExplanations' in globals(),
        'Cell 6.5':  'generate_back_translation_pairs' in globals(),
        'bt_train_pairs': 'bt_train_pairs' in globals(),
        'Cell 7':    'train_memory_efficient_tatn' in globals(),
        'Cell 8':    'translate_with_explanations' in globals(),
        'Cell 9':    'comprehensive_post_training_testing' in globals(),
        'Cell 10':   'main_pipeline' in globals(),
    }

    all_present = True
    for comp, present in components.items():
        status = "OK" if present else "MISSING"
        print(f"  {status} {comp}")
        if not present:
            all_present = False

    print("\n[RECOVERY]")

    if failure_category == "MISSING_DEPENDENCY":
        print("\n-> Run Cells 0-10 in sequence, then re-run Cell 11")

    elif failure_category == "TOKENIZER_ERROR":
        print("\n-> Install dependencies:")
        print("  ! pip install transformers==4.30.2 sentencepiece tokenizers")
        print("  Then RESTART kernel and re-run Cells 0-11")

    elif failure_category == "OOM_ERROR":
        print("\n-> Reduce memory in Cell 0:")
        print("  BATCH_SIZE = 2")
        print("  NUM_SAMPLES = 15000")
        print("  ACCUMULATION_STEPS = 32")
        print("  Then re-run Cells 0-11")

    elif failure_category == "RUNTIME_ERROR":
        print("\n-> Enable debug in Cell 0:")
        print("  VERBOSE_LOGGING = True")
        print("  DEBUG_DISCOVERY = True")
        print("  Then re-run Cell 11 for details")

    elif failure_category == "USER_INTERRUPT":
        print("\n-> Check checkpoint exists:")
        print(f"  os.path.exists('{CHECKPOINT_PATH}')")
        print("  If yes, can load and skip training")
        print("  If no, re-run Cell 11")

    else:
        print("\n-> General steps:")
        print("  1. Enable DEBUG in Cell 0")
        print("  2. Re-run Cells 0-11")
        print("  3. Check GPU: torch.cuda.is_available()")
        print("  4. Verify data loaded")

    print("\n" + "=" * 80)

total_duration = time.time() - start_time
end_utc        = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

print("\n" + "=" * 80)
print("EXECUTION SUMMARY")
print("=" * 80)
print(f"User: {user_login}")
print(f"Started: {now_utc}")
print(f"Finished: {end_utc}")
print(f"Duration: {_format_duration(total_duration)}")

if pipeline_success:
    print("Status: SUCCESS")
    if checkpoint_valid:
        print("Checkpoint: VALID")
    else:
        print("Checkpoint: CHECK NEEDED")
else:
    print(f"Status: FAILED ({failure_category or 'UNKNOWN'})")

print("=" * 80)

_safe_cleanup()


print("\n" + "=" * 80)
print("Cell 11: Execution Wrapper (DUAL-PATH COMPATIBLE)")
print("=" * 80)
print("This cell:")
print("  - Loads configuration from Cell 0")
print("  - Executes main_pipeline() from Cell 10")
print("  - Validates checkpoint integrity")
print("  - Tests inference with sample sentences")
print("  - Provides comprehensive diagnostics")
print("  - Shows usage examples for next steps")
print()
print(f"Current config:")
print(f"  - Samples: {NUM_SAMPLES}")
print(f"  - Epochs: {EPOCHS}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Device: {DEVICE}")
print(f"  - Multi-GPU: {USE_MULTI_GPU}")
print(f"  - ASBN Training: {'DISABLED' if not ENABLE_ASBN_TRAINING else 'ENABLED'}")
print(f"  - Discovery Frequency: {PERIODIC_DISCOVERY_FREQUENCY}")
print(f"  - LR Adapter: {LR_ADAPTER}")
print(f"  - Back-translation: {'ENABLED' if USE_BACK_TRANSLATION else 'DISABLED'}, LAMBDA_BT={LAMBDA_BT}")
print("\nDUAL-PATH COMPATIBILITY:")
print("  Uses Cell 8 translate_with_explanations() (already fixed)")
print("  Uses Cell 10 main_pipeline() (already fixed)")
print("  No direct model.forward() calls")
print("  Only inspection functions (no API changes)")
print("  Fully compatible with dual-path system")
print("=" * 80 + "\n")


In [ ]:
# ==============================================================================
# CELL 13: MEMORY CLEANUP + BLEU & CHRF++ & BERTSCORE & COMET EVALUATION
#           + INDICTRANS2 SECOND REFERENCE (CELL 22)
#           + MULTI-REFERENCE BLEU (CELL 23)
#           + MBR DECODING (CELL 24)
# ==============================================================================
import os
import sys
import time
import csv
import gc
from typing import List, Dict, Tuple, Optional, Any
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
from transformers.modeling_outputs import BaseModelOutput


print("\n" + "=" * 80)
print("CELL 13: EVALUATION WITH MEMORY MANAGEMENT (MBART-50)")
print("=" * 80)


# ==============================================================================
# SECTION 1: MEMORY CLEANUP
# ==============================================================================
print("\n[SECTION 1] Memory Cleanup...")
print("-" * 80)


if torch.cuda.is_available():
    try:
        initial_allocated = torch.cuda.memory_allocated(0) / 1024**3
        initial_reserved  = torch.cuda.memory_reserved(0) / 1024**3
        print(f"📊 BEFORE CLEANUP:")
        print(f"   Allocated: {initial_allocated:.2f} GB")
        print(f"   Reserved:  {initial_reserved:.2f} GB")
    except Exception:
        pass


# BUG-13-1 FIX: 'tokenizer' removed from this list.
# Tokenizer is needed all the way through Section 15 (MBR Decoding).
# It is released at the end of Section 15.
variables_to_delete = [
    'model', 'tatn_model',
    'optimizer', 'scheduler',
    'train_dataloader', 'val_dataloader',
    'checkpoint', 'model_state',
    'training_args', 'trainer',
    'dscd_outputs', 'asbn_outputs', 'trg_outputs',
    'encoder_outputs', 'forward_outputs',
    'prototypes_data', 'all_results',
    'result', 'test_case',
    'baseline_model', 'baseline_tokenizer', 'baseline_translations'
]

deleted_count = 0
for var_name in variables_to_delete:
    if var_name in globals():
        try:
            del globals()[var_name]
            deleted_count += 1
        except Exception:
            pass

print(f"✓ Attempted to delete {deleted_count} variables")

gc.collect()
print(f"✓ Python garbage collection invoked")

if torch.cuda.is_available():
    try:
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        print(f"✓ CUDA cache cleared")
        final_allocated = torch.cuda.memory_allocated(0) / 1024**3
        final_reserved  = torch.cuda.memory_reserved(0) / 1024**3
        print(f"\n📊 AFTER CLEANUP:")
        print(f"   Allocated: {final_allocated:.2f} GB")
        print(f"   Reserved:  {final_reserved:.2f} GB")
        try:
            print(f"   Memory freed: {initial_allocated - final_allocated:.2f} GB allocated, "
                  f"{initial_reserved - final_reserved:.2f} GB reserved")
        except Exception:
            pass
    except Exception:
        pass

print("\n✅ Memory cleanup complete - Ready for evaluation")
print("=" * 80)


# ==============================================================================
# SECTION 2: SETUP AND IMPORTS
# ==============================================================================
print("\n[SECTION 2] Setup and Imports...")
print("-" * 80)

try:
    import sacrebleu
    print(f"✅ sacrebleu version: {sacrebleu.__version__}")
except Exception:
    print("⚠️  sacrebleu not available — attempting install...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sacrebleu"])
    import sacrebleu
    print(f"✅ sacrebleu version: {sacrebleu.__version__}")

try:
    from bert_score import BERTScorer
    print(f"✅ bert-score already installed")
except Exception:
    print("⚠️  bert-score not available — installing...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bert-score"])
    from bert_score import BERTScorer
    print(f"✅ bert-score installed successfully")

try:
    from comet import download_model, load_from_checkpoint
    print(f"✅ unbabel-comet already installed")
except Exception:
    print("⚠️  unbabel-comet not available — installing...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "unbabel-comet"])
    from comet import download_model, load_from_checkpoint
    print(f"✅ unbabel-comet installed successfully")

try:
    _DEVICE          = DEVICE if isinstance(DEVICE, torch.device) else torch.device(str(DEVICE)) if isinstance(DEVICE, str) else torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _SOURCE_LANGUAGE = str(SOURCE_LANGUAGE)
    _TARGET_LANGUAGE = str(TARGET_LANGUAGE)
    _MAX_LENGTH      = int(MAX_LENGTH)
    _EVAL_BATCH_SIZE = int(EVAL_BATCH_SIZE) if "EVAL_BATCH_SIZE" in globals() else 4
    _EVAL_NUM_BEAMS  = int(EVAL_NUM_BEAMS)  if "EVAL_NUM_BEAMS"  in globals() else 5
except Exception:
    _DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _SOURCE_LANGUAGE = "bn_IN"
    _TARGET_LANGUAGE = "en_XX"
    _MAX_LENGTH      = 256
    _EVAL_BATCH_SIZE = 4
    _EVAL_NUM_BEAMS  = 8

print(f"✅ Configuration loaded")
print(f"   Device:     {_DEVICE}")
print(f"   Direction:  {_SOURCE_LANGUAGE} → {_TARGET_LANGUAGE}")
print(f"   Max length: {_MAX_LENGTH}")
print(f"   Batch size: {_EVAL_BATCH_SIZE}")
print(f"   Num beams:  {_EVAL_NUM_BEAMS}")
print("=" * 80)


# ==============================================================================
# SECTION 3: LOAD DATASET (5K SAMPLES)
# ==============================================================================
DATASET_PATH     = "/kaggle/input/datasets/manaskumarmanna/samanantar-new/updated_samanantar_homograph.csv"
NUM_EVAL_SAMPLES = 5000

print(f"\n[SECTION 3] Loading Dataset...")
print("-" * 80)
print(f"Path:    {DATASET_PATH}")
print(f"Samples: {NUM_EVAL_SAMPLES}")

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

try:
    df = pd.read_csv(DATASET_PATH, nrows=NUM_EVAL_SAMPLES)
    print(f"✅ Loaded {len(df)} rows")
    print(f"   Columns: {list(df.columns)}")

    if 'src' in df.columns and 'tgt' in df.columns:
        print(f"\n⚠️  SWAPPING COLUMNS:")
        print(f"   Dataset has: src=English, tgt=Bengali")
        print(f"   We need:     src=Bengali, tgt=English")
        df_eval = pd.DataFrame({
            'src': df['tgt'].values,
            'tgt': df['src'].values
        })
        print(f"\n✅ After swap:")
        print(f"   src: {_SOURCE_LANGUAGE} (Bengali)")
        print(f"   tgt: {_TARGET_LANGUAGE} (English)")
        print(f"\n   Sample 1:")
        print(f"      SRC (bn): {df_eval['src'].iloc[0][:80]}...")
        print(f"      TGT (en): {df_eval['tgt'].iloc[0][:80]}...")
    elif 'source' in df.columns and 'target' in df.columns:
        df_eval = df.rename(columns={'source': 'src', 'target': 'tgt'})
        df_eval = pd.DataFrame({
            'src': df_eval['tgt'].values,
            'tgt': df_eval['src'].values
        })
    else:
        raise ValueError(f"Unexpected columns: {list(df.columns)}")

    sources    = df_eval['src'].tolist()
    references = df_eval['tgt'].tolist()

    del df, df_eval
    gc.collect()

    print(f"\n✅ Prepared {len(sources)} samples for evaluation")
    print("=" * 80)

except Exception as e:
    print(f"❌ Failed to load dataset: {e}")
    raise


# ==============================================================================
# SECTION 4: LOAD TRAINED TATN MODEL (MBART-50)
# ==============================================================================
MODEL_CHECKPOINT_PATH = "/kaggle/working/tatn_final.pt"

print(f"\n[SECTION 4] Loading Trained TATN Model (mBART-50)...")
print("-" * 80)
print(f"Path: {MODEL_CHECKPOINT_PATH}")

if not os.path.exists(MODEL_CHECKPOINT_PATH):
    raise FileNotFoundError(f"Model checkpoint not found: {MODEL_CHECKPOINT_PATH}")

try:
    print(f"📂 Loading checkpoint to CPU...")
    checkpoint = torch.load(MODEL_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    print(f"✅ Checkpoint loaded to CPU")

    if "tokenizer" in checkpoint:
        tokenizer = checkpoint["tokenizer"]
        print(f"✅ Tokenizer loaded from checkpoint")
    else:
        from transformers import MBart50Tokenizer
        tokenizer = MBart50Tokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
        print(f"✅ MBart50Tokenizer loaded from pretrained")

    tokenizer.src_lang = _SOURCE_LANGUAGE
    tokenizer.tgt_lang = _TARGET_LANGUAGE
    print(f"✅ Language codes set: {_SOURCE_LANGUAGE} → {_TARGET_LANGUAGE}")

    TATNModelClass = globals().get("MemoryOptimizedTATNWithExplanations") or globals().get("TATNModelWithDSCDAndASBN")
    if TATNModelClass is None:
        raise RuntimeError("TATN model class not found. Run Cell 6 first.")

    print(f"🔧 Initializing TATN model...")
    tatn_model = TATNModelClass(tokenizer)

    # BUG-13-2 FIX: Only use 'model_state_dict' — the correct key saved by Cell 10.
    # 'model' key does not exist in the checkpoint Cell 10 writes.
    if "model_state_dict" in checkpoint:
        model_state = checkpoint["model_state_dict"]
    else:
        raise ValueError(
            f"No 'model_state_dict' key found in checkpoint. "
            f"Available keys: {list(checkpoint.keys())}"
        )

    print(f"🔧 Loading model weights (strict=False)...")
    tatn_model.load_state_dict(model_state, strict=False)

    try:
        del model_state
    except Exception:
        pass
    if 'dscd_state' in checkpoint:
        try:
            del checkpoint['dscd_state']
        except Exception:
            pass
    try:
        del checkpoint
    except Exception:
        pass
    gc.collect()
    torch.cuda.empty_cache()

    print(f"🔧 Moving model to {_DEVICE}...")
    tatn_model.to(_DEVICE)
    tatn_model.eval()

    print(f"✅ TATN model loaded successfully")
    try:
        print(f"   Device: {next(tatn_model.parameters()).device}")
    except Exception:
        pass

    if torch.cuda.is_available():
        try:
            allocated = torch.cuda.memory_allocated(0) / 1024**3
            print(f"   GPU memory: {allocated:.2f} GB")
        except Exception:
            pass

    print("=" * 80)

except Exception as e:
    print(f"❌ Failed to load TATN model: {e}")
    import traceback
    traceback.print_exc()
    raise


# ==============================================================================
# SECTION 5: (RESERVED — NO ADDITIONAL SETUP REQUIRED)
# ==============================================================================


# ==============================================================================
# SECTION 6: TRANSLATION FUNCTION (TATN)
# ==============================================================================
def translate_batch_tatn(
    sentences: List[str],
    model,
    tokenizer,
    max_length: int = 128,
    num_beams: int  = 5,
) -> List[str]:
    try:
        core = model.module if hasattr(model, 'module') else model

        tokenizer.src_lang = _SOURCE_LANGUAGE
        tokenizer.tgt_lang = _TARGET_LANGUAGE

        inputs = tokenizer(
            sentences,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        )

        input_ids      = inputs["input_ids"].to(_DEVICE)
        attention_mask = inputs["attention_mask"].to(_DEVICE)

        # BUG-13-4 FIX: Multi-level fallback for forced_bos_token_id.
        # Handles edge cases where TARGET_LANGUAGE key differs or lang_code_to_id absent.
        en_token_id = None
        if hasattr(tokenizer, 'lang_code_to_id'):
            en_token_id = (
                tokenizer.lang_code_to_id.get(_TARGET_LANGUAGE)
                or tokenizer.lang_code_to_id.get("en_XX")
            )
        if en_token_id is None:
            en_token_id = 250004  # hardcoded mBART-50 en_XX token id

        with torch.no_grad():
            fwd_out = core(
                input_ids=input_ids,
                attention_mask=attention_mask,
                src_texts=sentences,
                token_word_map=None,
                labels=None,
                return_dict=True,
                path=2,
            )

            # BUG-13-3 FIX: Correct key lookup order matching Cell 6's forward() output dict.
            # Cell 6 returns 'sense_augmented_embeddings' as the primary encoder key.
            # 'encoder_last_hidden_state' does NOT exist in Cell 6's output dict.
            # Priority: sense_augmented_embeddings → encoder_outputs (unwrap if BaseModelOutput)
            #           → last_hidden_state → encoder_last_hidden_state (last resort)
            h_sense = fwd_out.get('sense_augmented_embeddings', None)
            if h_sense is None:
                h_sense = fwd_out.get('encoder_outputs', None)
                if h_sense is not None and hasattr(h_sense, 'last_hidden_state'):
                    h_sense = h_sense.last_hidden_state
            if h_sense is None:
                h_sense = fwd_out.get('last_hidden_state', None)
            if h_sense is None:
                h_sense = fwd_out.get('encoder_last_hidden_state', None)

            if h_sense is None:
                raise ValueError(
                    f"No encoder hidden states found in forward output. "
                    f"Available keys: {list(fwd_out.keys()) if isinstance(fwd_out, dict) else type(fwd_out)}"
                )

            encoder_outputs_wrapped = BaseModelOutput(
                last_hidden_state=h_sense,
                hidden_states=None,
                attentions=None,
            )

            generated = core.mbart.generate(
                input_ids=None,
                encoder_outputs=encoder_outputs_wrapped,
                attention_mask=attention_mask,
                max_length=max_length,
                num_beams=num_beams,
                early_stopping=True,
                no_repeat_ngram_size=3,
                length_penalty=1.0,
                repetition_penalty=1.2,
                forced_bos_token_id=en_token_id,
            )

        translations = tokenizer.batch_decode(generated, skip_special_tokens=True)

        try:
            del input_ids, attention_mask, generated, encoder_outputs_wrapped, h_sense, fwd_out
            torch.cuda.empty_cache()
        except Exception:
            pass

        return translations

    except Exception as e:
        print(f"⚠️  Batch translation failed: {type(e).__name__}: {str(e)[:120]}")
        return ["[ERROR]"] * len(sentences)


# ==============================================================================
# SECTION 7: EVALUATE TATN MODEL — TRANSLATION
# ==============================================================================
print(f"\n[SECTION 7] Evaluating TATN Model (Translation)...")
print("-" * 80)
print(f"Translating {len(sources)} samples with batch_size={_EVAL_BATCH_SIZE}...")

# NEW FIX: Initialize before the loop to prevent NameError in Sections 8–17
# if the loop raises before completing.
tatn_translations = []
elapsed_tatn      = 0.0
start_time        = time.time()

for i in range(0, len(sources), _EVAL_BATCH_SIZE):
    batch_sources      = sources[i:i + _EVAL_BATCH_SIZE]
    batch_translations = translate_batch_tatn(
        batch_sources,
        tatn_model,
        tokenizer,
        max_length=_MAX_LENGTH,
        num_beams=_EVAL_NUM_BEAMS,
    )
    tatn_translations.extend(batch_translations)

    processed = min(i + _EVAL_BATCH_SIZE, len(sources))
    if processed % 200 == 0 or processed >= len(sources):
        elapsed = time.time() - start_time
        speed   = processed / elapsed if elapsed > 0 else 0
        eta     = (len(sources) - processed) / speed if speed > 0 else 0
        mem_str = ""
        if torch.cuda.is_available():
            try:
                mem_gb  = torch.cuda.memory_allocated(0) / 1024**3
                mem_str = f" | GPU: {mem_gb:.2f}GB"
            except Exception:
                pass
        print(f"   Progress: {processed}/{len(sources)} ({processed/len(sources)*100:.1f}%) | "
              f"Speed: {speed:.1f} s/s | ETA: {eta/60:.1f} min{mem_str}")

elapsed_tatn = time.time() - start_time

print(f"\n✅ TATN translation complete")
print(f"   Time:  {elapsed_tatn:.1f}s ({elapsed_tatn/60:.2f} min)")
print(f"   Speed: {len(sources)/elapsed_tatn:.2f} samples/s" if elapsed_tatn > 0 else "   Speed: N/A")

# NOTE: tokenizer is NOT deleted here.
# MBR decoding in Section 15 still needs it.
# Final tokenizer cleanup happens at the end of Section 15.
gc.collect()
print("=" * 80)


# ==============================================================================
# SECTION 8: COMPUTE BLEU & CHRF++ SCORES
# ==============================================================================
print(f"\n[SECTION 8] Computing BLEU & ChrF++ Scores...")
print("-" * 80)

tatn_bleu_score = 0.0
tatn_chrf_score = 0.0

try:
    # BUG-13-5 FIX: sacrebleu >= 2.0 requires references as list-of-lists [[ref1, ...]].
    # Passing a flat list gives silently wrong ChrF++ scores.
    tatn_bleu       = sacrebleu.corpus_bleu(tatn_translations, [references])
    tatn_chrf       = sacrebleu.corpus_chrf(tatn_translations, [references])
    tatn_bleu_score = tatn_bleu.score
    tatn_chrf_score = tatn_chrf.score
    print(f"✅ BLEU:   {tatn_bleu_score:.2f}")
    print(f"✅ ChrF++: {tatn_chrf_score:.2f}")
except Exception as e:
    print(f"⚠️  sacrebleu computation failed: {e}")
    tatn_bleu_score = 0.0
    tatn_chrf_score = 0.0

print("=" * 80)


# ==============================================================================
# SECTION 8.5: COMPUTE BERTSCORE
# ==============================================================================
print(f"\n[SECTION 8.5] Computing BERTScore...")
print("-" * 80)

# BUG-13-7 FIX: Initialize ALL bert score variables to safe defaults BEFORE the
# try block. If BERTScorer import failed in Section 2 and the block below also
# raises, Section 10 and 12 would hit NameError on these variables.
tatn_bert_p_score        = 0.0
tatn_bert_r_score        = 0.0
tatn_bert_f1_score       = 0.0
tatn_bert_segment_scores = [0.0] * len(sources)

try:
    print(f"🔧 Initializing BERTScorer (lang=en, rescale_with_baseline=True)...")
    print(f"   Model: microsoft/deberta-xlarge-mnli (default for en)")
    print(f"   Samples: {len(tatn_translations)}")

    bert_scorer = BERTScorer(
        lang="en",
        rescale_with_baseline=True,
        device=str(_DEVICE),
    )

    print(f"🚀 Running BERTScore evaluation...")

    bert_P, bert_R, bert_F1 = bert_scorer.score(
        cands=tatn_translations,
        refs=references,
        verbose=True,
        batch_size=64,
    )

    tatn_bert_p_score        = float(bert_P.mean().item()) * 100
    tatn_bert_r_score        = float(bert_R.mean().item()) * 100
    tatn_bert_f1_score       = float(bert_F1.mean().item()) * 100
    tatn_bert_segment_scores = [float(v) * 100 for v in bert_F1.tolist()]

    print(f"\n✅ BERTScore evaluation complete")
    print(f"   Precision: {tatn_bert_p_score:.2f}")
    print(f"   Recall:    {tatn_bert_r_score:.2f}")
    print(f"   F1:        {tatn_bert_f1_score:.2f}")
    print(f"   Segment score range: [{min(tatn_bert_segment_scores):.2f}, {max(tatn_bert_segment_scores):.2f}]")

    try:
        del bert_scorer, bert_P, bert_R, bert_F1
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"✅ BERTScorer memory freed")
    except Exception:
        pass

except Exception as e:
    print(f"⚠️  BERTScore computation failed: {type(e).__name__}: {str(e)[:200]}")
    import traceback
    traceback.print_exc()
    tatn_bert_p_score        = 0.0
    tatn_bert_r_score        = 0.0
    tatn_bert_f1_score       = 0.0
    tatn_bert_segment_scores = [0.0] * len(sources)

print("=" * 80)


# ==============================================================================
# SECTION 9: COMPUTE COMET SCORE
# ==============================================================================
print(f"\n[SECTION 9] Computing COMET Score...")
print("-" * 80)

tatn_comet_score          = 0.0
tatn_comet_segment_scores = [0.0] * len(sources)

try:
    print(f"📥 Downloading COMET model: Unbabel/wmt22-comet-da...")
    comet_model_path = download_model("Unbabel/wmt22-comet-da")
    print(f"✅ Model downloaded: {comet_model_path}")

    print(f"🔧 Loading COMET model...")
    comet_model = load_from_checkpoint(comet_model_path)
    print(f"✅ COMET model loaded")

    print(f"📋 Preparing {len(sources)} samples for COMET...")
    comet_data = [
        {"src": src, "mt": mt, "ref": ref}
        for src, mt, ref in zip(sources, tatn_translations, references)
    ]

    print(f"🚀 Running COMET evaluation (batch_size=8)...")

    # BUG-13-6 FIX: unbabel-comet >= 2.0 replaced `gpus=int` with `accelerator=`.
    # Try new API first; fall back to legacy API on TypeError.
    try:
        comet_output = comet_model.predict(
            comet_data,
            batch_size=8,
            accelerator="gpu" if torch.cuda.is_available() else "cpu",
        )
        print(f"✅ COMET predict used new API (accelerator=)")
    except TypeError:
        comet_output = comet_model.predict(
            comet_data,
            batch_size=8,
            gpus=1 if torch.cuda.is_available() else 0,
        )
        print(f"✅ COMET predict used legacy API (gpus=)")

    tatn_comet_score          = comet_output.system_score
    tatn_comet_segment_scores = comet_output.scores

    print(f"\n✅ COMET evaluation complete")
    print(f"   System score: {tatn_comet_score:.4f}")
    print(f"   Segment scores: {len(tatn_comet_segment_scores)} samples")
    print(f"   Score range: [{min(tatn_comet_segment_scores):.4f}, {max(tatn_comet_segment_scores):.4f}]")

    try:
        del comet_model, comet_data, comet_output
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"✅ COMET model memory freed")
    except Exception:
        pass

except Exception as e:
    print(f"⚠️  COMET evaluation failed: {type(e).__name__}: {str(e)[:200]}")
    import traceback
    traceback.print_exc()
    tatn_comet_score          = 0.0
    tatn_comet_segment_scores = [0.0] * len(sources)

print("=" * 80)


# ==============================================================================
# SECTION 10: FINAL SUMMARY (STANDARD METRICS)
# ==============================================================================
print(f"\n[SECTION 10] EVALUATION SUMMARY (Standard Metrics)")
print("=" * 80)
print(f"\n📊 TATN MODEL SCORES (5000 samples, bn→en):")
print(f"   BLEU:            {tatn_bleu_score:.2f}")
print(f"   ChrF++:          {tatn_chrf_score:.2f}")
print(f"   BERTScore-P:     {tatn_bert_p_score:.2f}")
print(f"   BERTScore-R:     {tatn_bert_r_score:.2f}")
print(f"   BERTScore-F1:    {tatn_bert_f1_score:.2f}")
print(f"   COMET:           {tatn_comet_score:.4f}")
print(f"\n   Samples evaluated:  {len(sources)}")
print(f"   Translation time:   {elapsed_tatn/60:.2f} minutes" if elapsed_tatn > 0 else "   Translation time:   N/A")
print(f"   Speed:              {len(sources)/elapsed_tatn:.2f} samples/second" if elapsed_tatn > 0 else "   Speed:              N/A")
print("=" * 80)


# ==============================================================================
# SECTION 11: SAMPLE TRANSLATIONS
# ==============================================================================
print(f"\n[SECTION 11] Sample Translations")
print("=" * 80)

num_samples = min(5, len(sources))
for i in range(num_samples):
    print(f"\n{'='*60}")
    print(f"SAMPLE {i+1}/{num_samples}")
    print(f"{'='*60}")
    print(f"\n📝 Source ({_SOURCE_LANGUAGE}):")
    print(f"   {sources[i]}")
    print(f"\n🎯 Reference ({_TARGET_LANGUAGE}):")
    print(f"   {references[i]}")
    print(f"\n🤖 TATN Translation:")
    print(f"   {tatn_translations[i]}")
    print(f"\n📊 Segment Scores:")
    print(f"   BERTScore-F1: {tatn_bert_segment_scores[i]:.2f}")
    print(f"   COMET:        {tatn_comet_segment_scores[i]:.4f}")

print("=" * 80)


# ==============================================================================
# SECTION 12: SAVE INITIAL RESULTS
# ==============================================================================
print(f"\n[SECTION 12] Saving Initial Results...")
print("=" * 80)

results_dir = "/kaggle/working/"
os.makedirs(results_dir, exist_ok=True)

summary_file = os.path.join(results_dir, "evaluation_summary.csv")
summary_data = {
    "Model":        ["TATN"],
    "BLEU":         [tatn_bleu_score],
    "ChrF++":       [tatn_chrf_score],
    "BERTScore_P":  [tatn_bert_p_score],
    "BERTScore_R":  [tatn_bert_r_score],
    "BERTScore_F1": [tatn_bert_f1_score],
    "COMET":        [tatn_comet_score],
    "Num_Samples":  [len(sources)],
}
summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(summary_file, index=False)
print(f"✅ Summary saved: {summary_file}")

# BUG-13-8 FIX: Defensive type+length re-check before building the detailed DataFrame.
# Both initialized as [0.0]*len(sources) earlier; this guards against any unexpected
# reassignment between sections causing a pd.DataFrame ValueError on length mismatch.
if not isinstance(tatn_bert_segment_scores, list) or len(tatn_bert_segment_scores) != len(sources):
    tatn_bert_segment_scores = [0.0] * len(sources)
if not isinstance(tatn_comet_segment_scores, list) or len(tatn_comet_segment_scores) != len(sources):
    tatn_comet_segment_scores = [0.0] * len(sources)

detailed_file = os.path.join(results_dir, "evaluation_detailed.csv")
detailed_data = {
    "source":           sources,
    "reference":        references,
    "tatn_translation": tatn_translations,
    "bertscore_f1":     tatn_bert_segment_scores,
    "comet_score":      tatn_comet_segment_scores,
}
detailed_df = pd.DataFrame(detailed_data)
detailed_df.to_csv(detailed_file, index=False)
print(f"✅ Detailed results saved: {detailed_file}")

print("\n✅ Initial evaluation saved — Proceeding to advanced metrics...")
print("=" * 80)


# ==============================================================================
# SECTION 13: INDICTRANS2 SECOND REFERENCE GENERATION (5000 SAMPLES)
# [CELL 22 EQUIVALENT] — ~30 min on GPU
# Generates a second machine reference via IndicTrans2 (ai4bharat)
# Used in Section 14 for multi-reference BLEU (mathematically cannot decrease score)
# ==============================================================================
print(f"\n[SECTION 13] IndicTrans2 Second Reference Generation (Cell 22)...")
print("-" * 80)
print(f"   Model:   ai4bharat/indictrans2-indic-en-dist-200M")
print(f"   Samples: {len(sources)}")
print(f"   Est. time: ~30 min on GPU")

_IT2_MODEL_NAME = "ai4bharat/indictrans2-indic-en-dist-200M"
_IT2_SRC_LANG   = "ben_Beng"
_IT2_TGT_LANG   = "eng_Latn"
_IT2_BATCH_SIZE = 8
_IT2_MAX_LENGTH = 256
_IT2_NUM_BEAMS  = 5

references_it2  = ["[IT2_UNAVAILABLE]"] * len(sources)
_it2_refs_valid = False

try:
    import subprocess

    try:
        from IndicTransToolkit.processor import IndicProcessor
        print(f"✅ IndicTransToolkit already installed")
    except ImportError:
        print(f"⚠️  IndicTransToolkit not found — installing...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "git+https://github.com/VarunGumma/IndicTransToolkit.git"
        ])
        from IndicTransToolkit.processor import IndicProcessor
        print(f"✅ IndicTransToolkit installed")

    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    print(f"📥 Loading IndicTrans2 tokenizer...")
    it2_tokenizer = AutoTokenizer.from_pretrained(
        _IT2_MODEL_NAME,
        trust_remote_code=True,
    )
    print(f"✅ IndicTrans2 tokenizer loaded")

    print(f"📥 Loading IndicTrans2 model...")
    it2_model = AutoModelForSeq2SeqLM.from_pretrained(
        _IT2_MODEL_NAME,
        trust_remote_code=True,
    )
    it2_model.to(_DEVICE)
    it2_model.eval()
    print(f"✅ IndicTrans2 model loaded → {_DEVICE}")

    if torch.cuda.is_available():
        try:
            print(f"   GPU memory after IT2 load: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")
        except Exception:
            pass

    ip = IndicProcessor(inference=True)
    print(f"✅ IndicProcessor initialized")

    # Robust 3-level forced_bos_token_id resolution for IndicTrans2 AutoTokenizer.
    # lang_code_to_id may not exist on all IndicTrans2 tokenizer versions.
    _it2_bos = None
    if hasattr(it2_tokenizer, 'lang_code_to_id') and it2_tokenizer.lang_code_to_id:
        _it2_bos = it2_tokenizer.lang_code_to_id.get(_IT2_TGT_LANG, None)
    if _it2_bos is None:
        try:
            _it2_bos = it2_tokenizer.convert_tokens_to_ids(_IT2_TGT_LANG)
            if _it2_bos == it2_tokenizer.unk_token_id:
                _it2_bos = None
        except Exception:
            _it2_bos = None
    if _it2_bos is not None:
        print(f"   forced_bos_token_id for {_IT2_TGT_LANG}: {_it2_bos}")
    else:
        print(f"   forced_bos_token_id: None (IndicTrans2 will use default)")

    print(f"\n🚀 Generating IndicTrans2 references ({len(sources)} sentences)...")
    it2_translations = []
    it2_start        = time.time()

    for i in range(0, len(sources), _IT2_BATCH_SIZE):
        batch_src = sources[i:i + _IT2_BATCH_SIZE]
        try:
            preprocessed = ip.preprocess_batch(
                batch_src,
                src_lang=_IT2_SRC_LANG,
                tgt_lang=_IT2_TGT_LANG,
            )

            it2_inputs = it2_tokenizer(
                preprocessed,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=_IT2_MAX_LENGTH,
            )

            it2_input_ids      = it2_inputs["input_ids"].to(_DEVICE)
            it2_attention_mask = it2_inputs["attention_mask"].to(_DEVICE)

            with torch.no_grad():
                generate_kwargs = dict(
                    input_ids=it2_input_ids,
                    attention_mask=it2_attention_mask,
                    num_beams=_IT2_NUM_BEAMS,
                    max_length=_IT2_MAX_LENGTH,
                    early_stopping=True,
                    no_repeat_ngram_size=3,
                )
                if _it2_bos is not None:
                    generate_kwargs["forced_bos_token_id"] = _it2_bos

                it2_generated = it2_model.generate(**generate_kwargs)

            batch_decoded = it2_tokenizer.batch_decode(
                it2_generated,
                skip_special_tokens=True,
            )
            batch_postprocessed = ip.postprocess_batch(
                batch_decoded,
                lang=_IT2_TGT_LANG,
            )
            it2_translations.extend(batch_postprocessed)

            try:
                del it2_input_ids, it2_attention_mask, it2_generated
                torch.cuda.empty_cache()
            except Exception:
                pass

        except Exception as e_batch:
            print(f"⚠️  IT2 batch {i//_IT2_BATCH_SIZE+1} failed: {type(e_batch).__name__}: {str(e_batch)[:100]}")
            it2_translations.extend(["[IT2_ERROR]"] * len(batch_src))

        processed = min(i + _IT2_BATCH_SIZE, len(sources))
        if processed % 500 == 0 or processed >= len(sources):
            elapsed_it2 = time.time() - it2_start
            speed_it2   = processed / elapsed_it2 if elapsed_it2 > 0 else 0
            eta_it2     = (len(sources) - processed) / speed_it2 if speed_it2 > 0 else 0
            mem_str_it2 = ""
            if torch.cuda.is_available():
                try:
                    mem_str_it2 = f" | GPU: {torch.cuda.memory_allocated(0)/1024**3:.2f}GB"
                except Exception:
                    pass
            print(f"   Progress: {processed}/{len(sources)} ({processed/len(sources)*100:.1f}%) | "
                  f"Speed: {speed_it2:.1f} s/s | ETA: {eta_it2/60:.1f} min{mem_str_it2}")

    elapsed_it2_total = time.time() - it2_start

    if len(it2_translations) != len(sources):
        print(f"⚠️  IT2 count mismatch: {len(it2_translations)} vs {len(sources)} — padding")
        it2_translations = (it2_translations + ["[IT2_MISSING]"] * len(sources))[:len(sources)]

    references_it2  = it2_translations
    _it2_refs_valid = not all(
        r in ("[IT2_UNAVAILABLE]", "[IT2_ERROR]", "[IT2_MISSING]")
        for r in references_it2[:20]
    )

    print(f"\n✅ IndicTrans2 reference generation complete")
    print(f"   Time:  {elapsed_it2_total:.1f}s ({elapsed_it2_total/60:.2f} min)")
    print(f"   Speed: {len(sources)/elapsed_it2_total:.2f} samples/s" if elapsed_it2_total > 0 else "   Speed: N/A")
    print(f"   Valid: {_it2_refs_valid}")
    if _it2_refs_valid:
        print(f"   Sample: {references_it2[0][:100]}")

    try:
        del it2_model, it2_tokenizer, ip, it2_translations
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"✅ IndicTrans2 model memory freed")
        if torch.cuda.is_available():
            print(f"   GPU memory after IT2 cleanup: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")
    except Exception:
        pass

except Exception as e:
    print(f"⚠️  IndicTrans2 section failed: {type(e).__name__}: {str(e)[:200]}")
    import traceback
    traceback.print_exc()
    references_it2  = ["[IT2_UNAVAILABLE]"] * len(sources)
    _it2_refs_valid = False
    print(f"   Continuing with fallback: multi-ref BLEU will be skipped")

print("=" * 80)


# ==============================================================================
# SECTION 14: MULTI-REFERENCE BLEU & CHRF++ (2 REFS: HUMAN + INDICTRANS2)
# [CELL 23 EQUIVALENT] — ~2 min
# Multi-reference BLEU is mathematically guaranteed >= single-reference BLEU.
# For each hypothesis, sacrebleu picks the closest reference → score can only rise.
# ==============================================================================
print(f"\n[SECTION 14] Multi-Reference BLEU & ChrF++ (Cell 23)...")
print("-" * 80)
print(f"   Ref 1: Human reference ({len(references)} samples)")
print(f"   Ref 2: IndicTrans2 reference (valid={_it2_refs_valid})")
print(f"   Note: Multi-ref BLEU is mathematically guaranteed >= single-ref BLEU")

multi_ref_bleu_score = tatn_bleu_score
multi_ref_chrf_score = tatn_chrf_score
_multi_ref_ran_ok    = False

try:
    if _it2_refs_valid:
        multi_ref_bleu       = sacrebleu.corpus_bleu(tatn_translations, [references, references_it2])
        multi_ref_chrf       = sacrebleu.corpus_chrf(tatn_translations, [references, references_it2])
        multi_ref_bleu_score = multi_ref_bleu.score
        multi_ref_chrf_score = multi_ref_chrf.score

        bleu_gain = multi_ref_bleu_score - tatn_bleu_score
        chrf_gain = multi_ref_chrf_score - tatn_chrf_score

        print(f"\n✅ Multi-Reference Scores (2 refs):")
        print(f"   BLEU  (2 refs):  {multi_ref_bleu_score:.2f}  (gain vs 1 ref: {bleu_gain:+.2f})")
        print(f"   ChrF++ (2 refs): {multi_ref_chrf_score:.2f}  (gain vs 1 ref: {chrf_gain:+.2f})")
        print(f"\n   Single-ref BLEU:   {tatn_bleu_score:.2f}")
        print(f"   Single-ref ChrF++: {tatn_chrf_score:.2f}")

        if bleu_gain < -0.01:
            print(f"⚠️  WARNING: Multi-ref BLEU decreased — verify IT2 reference quality")
        else:
            print(f"✅ Mathematical guarantee holds: multi-ref BLEU >= single-ref BLEU")

        _multi_ref_ran_ok = True
    else:
        print(f"⚠️  IndicTrans2 references not valid — skipping multi-ref computation")
        print(f"   Multi-ref BLEU/ChrF++ = same as single-ref (IT2 unavailable)")

except Exception as e:
    print(f"⚠️  Multi-reference BLEU failed: {type(e).__name__}: {str(e)[:200]}")
    multi_ref_bleu_score = tatn_bleu_score
    multi_ref_chrf_score = tatn_chrf_score

print("=" * 80)


# ==============================================================================
# SECTION 15: MBR DECODING (1000 SAMPLES)
# [CELL 24 EQUIVALENT] — ~60 min on GPU
# Minimum Bayes Risk (MBR) decoding:
#   1. For each source, generate N=8 candidates via diverse beam search
#   2. Score each candidate pair with sentence-level ChrF++
#   3. Select the candidate with the highest average pairwise ChrF++ score
# ==============================================================================
print(f"\n[SECTION 15] MBR Decoding (Cell 24)...")
print("-" * 80)

_MBR_NUM_SAMPLES    = 1000
_MBR_NUM_CANDIDATES = 8
_MBR_MAX_LENGTH     = _MAX_LENGTH

print(f"   Sentences:    {_MBR_NUM_SAMPLES}")
print(f"   Candidates:   {_MBR_NUM_CANDIDATES} per sentence (diverse beam search)")
print(f"   Scoring:      pairwise sentence-level ChrF++")
print(f"   Est. time:    ~60 min on GPU")

mbr_bleu_score         = 0.0
mbr_chrf_score         = 0.0
mbr_vs_beam_bleu_delta = 0.0
mbr_vs_beam_chrf_delta = 0.0
_mbr_ran_ok            = False

# Initialize before try block so Section 17 can reference them even if Section 15 fails.
mbr_sources      = sources[:_MBR_NUM_SAMPLES]
mbr_references   = references[:_MBR_NUM_SAMPLES]
mbr_translations = []


def _generate_mbr_candidates(
    sentence: str,
    model,
    tokenizer,
    num_candidates: int = 8,
    max_length: int     = 128,
) -> List[str]:
    try:
        core = model.module if hasattr(model, 'module') else model

        tokenizer.src_lang = _SOURCE_LANGUAGE
        tokenizer.tgt_lang = _TARGET_LANGUAGE

        inputs = tokenizer(
            [sentence],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        )

        input_ids      = inputs["input_ids"].to(_DEVICE)
        attention_mask = inputs["attention_mask"].to(_DEVICE)

        en_token_id = None
        if hasattr(tokenizer, 'lang_code_to_id'):
            en_token_id = (
                tokenizer.lang_code_to_id.get(_TARGET_LANGUAGE)
                or tokenizer.lang_code_to_id.get("en_XX")
            )
        if en_token_id is None:
            en_token_id = 250004

        with torch.no_grad():
            fwd_out = core(
                input_ids=input_ids,
                attention_mask=attention_mask,
                src_texts=[sentence],
                token_word_map=None,
                labels=None,
                return_dict=True,
                path=2,
            )

            h_sense = fwd_out.get('sense_augmented_embeddings', None)
            if h_sense is None:
                h_sense = fwd_out.get('encoder_outputs', None)
                if h_sense is not None and hasattr(h_sense, 'last_hidden_state'):
                    h_sense = h_sense.last_hidden_state
            if h_sense is None:
                h_sense = fwd_out.get('last_hidden_state', None)
            if h_sense is None:
                h_sense = fwd_out.get('encoder_last_hidden_state', None)

            if h_sense is None:
                return []

            encoder_outputs_wrapped = BaseModelOutput(
                last_hidden_state=h_sense,
                hidden_states=None,
                attentions=None,
            )

            # Diverse beam search with num_beam_groups = num_candidates for max diversity.
            generated = core.mbart.generate(
                input_ids=None,
                encoder_outputs=encoder_outputs_wrapped,
                attention_mask=attention_mask,
                max_length=max_length,
                num_beams=num_candidates,
                num_beam_groups=num_candidates,
                diversity_penalty=0.5,
                num_return_sequences=num_candidates,
                early_stopping=True,
                no_repeat_ngram_size=3,
                forced_bos_token_id=en_token_id,
            )

        candidates = tokenizer.batch_decode(generated, skip_special_tokens=True)

        try:
            del input_ids, attention_mask, generated, encoder_outputs_wrapped, h_sense, fwd_out
            torch.cuda.empty_cache()
        except Exception:
            pass

        return candidates

    except Exception as e:
        print(f"⚠️  MBR candidate gen failed: {type(e).__name__}: {str(e)[:100]}")
        return []


def _mbr_select(candidates: List[str]) -> str:
    if not candidates:
        return "[MBR_ERROR]"
    if len(candidates) == 1:
        return candidates[0]

    best_score     = -1.0
    best_candidate = candidates[0]

    for i, hyp in enumerate(candidates):
        total_score = 0.0
        count       = 0
        for j, ref in enumerate(candidates):
            if i == j:
                continue
            try:
                score = sacrebleu.sentence_chrf(hyp, [ref]).score
                total_score += score
                count       += 1
            except Exception:
                continue
        avg_score = total_score / count if count > 0 else 0.0
        if avg_score > best_score:
            best_score     = avg_score
            best_candidate = hyp

    return best_candidate


try:
    print(f"\n🚀 Running MBR decoding on {len(mbr_sources)} sentences...")
    mbr_start = time.time()

    for idx, src in enumerate(mbr_sources):
        candidates = _generate_mbr_candidates(
            src,
            tatn_model,
            tokenizer,
            num_candidates=_MBR_NUM_CANDIDATES,
            max_length=_MBR_MAX_LENGTH,
        )

        if candidates:
            best = _mbr_select(candidates)
        else:
            best = tatn_translations[idx] if idx < len(tatn_translations) else "[MBR_ERROR]"

        mbr_translations.append(best)

        if (idx + 1) % 100 == 0 or (idx + 1) >= len(mbr_sources):
            elapsed_mbr = time.time() - mbr_start
            speed_mbr   = (idx + 1) / elapsed_mbr if elapsed_mbr > 0 else 0
            eta_mbr     = (len(mbr_sources) - (idx + 1)) / speed_mbr if speed_mbr > 0 else 0
            mem_str_mbr = ""
            if torch.cuda.is_available():
                try:
                    mem_str_mbr = f" | GPU: {torch.cuda.memory_allocated(0)/1024**3:.2f}GB"
                except Exception:
                    pass
            print(f"   MBR Progress: {idx+1}/{len(mbr_sources)} ({(idx+1)/len(mbr_sources)*100:.1f}%) | "
                  f"Speed: {speed_mbr:.2f} s/s | ETA: {eta_mbr/60:.1f} min{mem_str_mbr}")

    elapsed_mbr_total = time.time() - mbr_start

    print(f"\n✅ MBR decoding complete")
    print(f"   Time:  {elapsed_mbr_total:.1f}s ({elapsed_mbr_total/60:.2f} min)")
    print(f"   Speed: {len(mbr_sources)/elapsed_mbr_total:.4f} samples/s" if elapsed_mbr_total > 0 else "   Speed: N/A")

    mbr_bleu_result = sacrebleu.corpus_bleu(mbr_translations, [mbr_references])
    mbr_chrf_result = sacrebleu.corpus_chrf(mbr_translations, [mbr_references])
    mbr_bleu_score  = mbr_bleu_result.score
    mbr_chrf_score  = mbr_chrf_result.score

    beam_bleu_1k = sacrebleu.corpus_bleu(tatn_translations[:_MBR_NUM_SAMPLES], [mbr_references]).score
    beam_chrf_1k = sacrebleu.corpus_chrf(tatn_translations[:_MBR_NUM_SAMPLES], [mbr_references]).score

    mbr_vs_beam_bleu_delta = mbr_bleu_score - beam_bleu_1k
    mbr_vs_beam_chrf_delta = mbr_chrf_score - beam_chrf_1k

    print(f"\n📊 MBR vs Beam Search (1000 samples):")
    print(f"   BLEU  — MBR: {mbr_bleu_score:.2f}  Beam: {beam_bleu_1k:.2f}  Delta: {mbr_vs_beam_bleu_delta:+.2f}")
    print(f"   ChrF++ — MBR: {mbr_chrf_score:.2f}  Beam: {beam_chrf_1k:.2f}  Delta: {mbr_vs_beam_chrf_delta:+.2f}")

    print(f"\n   MBR sample outputs (first 3):")
    for k in range(min(3, len(mbr_translations))):
        print(f"   [{k+1}] SRC: {mbr_sources[k][:60]}")
        print(f"        MBR: {mbr_translations[k][:80]}")
        print(f"        REF: {mbr_references[k][:80]}")

    _mbr_ran_ok = True

except Exception as e:
    print(f"⚠️  MBR decoding failed: {type(e).__name__}: {str(e)[:200]}")
    import traceback
    traceback.print_exc()
    mbr_bleu_score         = 0.0
    mbr_chrf_score         = 0.0
    mbr_vs_beam_bleu_delta = 0.0
    mbr_vs_beam_chrf_delta = 0.0

# BUG-13-1 FIX (final): Delete tokenizer HERE — after Section 15 (MBR) is done.
# All translation work (Section 7) and MBR (Section 15) are now complete.
for _v in ['tokenizer']:
    if _v in globals():
        try:
            del globals()[_v]
        except Exception:
            pass
gc.collect()
print(f"\n✅ Tokenizer released from memory post-MBR")
print("=" * 80)


# ==============================================================================
# SECTION 16: COMPLETE FINAL SUMMARY (ALL METRICS)
# ==============================================================================
print(f"\n[SECTION 16] COMPLETE FINAL SUMMARY")
print("=" * 80)

print(f"\n📊 TATN MODEL — COMPLETE EVALUATION REPORT")
print(f"   Dataset: bn→en | Samples: {len(sources)}")

print(f"\n{'─'*60}")
print(f"   STANDARD METRICS (5000 samples):")
print(f"{'─'*60}")
print(f"   BLEU:            {tatn_bleu_score:.2f}")
print(f"   ChrF++:          {tatn_chrf_score:.2f}")
print(f"   BERTScore-P:     {tatn_bert_p_score:.2f}")
print(f"   BERTScore-R:     {tatn_bert_r_score:.2f}")
print(f"   BERTScore-F1:    {tatn_bert_f1_score:.2f}")
print(f"   COMET:           {tatn_comet_score:.4f}")

if _multi_ref_ran_ok:
    print(f"\n{'─'*60}")
    print(f"   MULTI-REFERENCE METRICS (5000 samples, 2 refs: human + IT2):")
    print(f"{'─'*60}")
    print(f"   Multi-ref BLEU:   {multi_ref_bleu_score:.2f}  (gain: {multi_ref_bleu_score - tatn_bleu_score:+.2f})")
    print(f"   Multi-ref ChrF++: {multi_ref_chrf_score:.2f}  (gain: {multi_ref_chrf_score - tatn_chrf_score:+.2f})")
else:
    print(f"\n   Multi-ref BLEU/ChrF++: N/A (IT2 references unavailable)")

if _mbr_ran_ok:
    print(f"\n{'─'*60}")
    print(f"   MBR DECODING (1000 samples, {_MBR_NUM_CANDIDATES} candidates/sentence):")
    print(f"{'─'*60}")
    print(f"   MBR BLEU:       {mbr_bleu_score:.2f}  (vs beam: {mbr_vs_beam_bleu_delta:+.2f})")
    print(f"   MBR ChrF++:     {mbr_chrf_score:.2f}  (vs beam: {mbr_vs_beam_chrf_delta:+.2f})")
else:
    print(f"\n   MBR BLEU/ChrF++: N/A (MBR decoding failed)")

print(f"\n{'─'*60}")
print(f"   PERFORMANCE:")
print(f"{'─'*60}")
print(f"   Translation time:   {elapsed_tatn/60:.2f} min" if elapsed_tatn > 0 else "   Translation time:   N/A")
print(f"   Translation speed:  {len(sources)/elapsed_tatn:.2f} samples/s" if elapsed_tatn > 0 else "   Translation speed:  N/A")
print("=" * 80)


# ==============================================================================
# SECTION 17: SAVE COMPLETE RESULTS
# ==============================================================================
print(f"\n[SECTION 17] Saving Complete Results...")
print("=" * 80)

complete_summary_file = os.path.join(results_dir, "evaluation_summary_complete.csv")
complete_summary_data = {
    "Model":               ["TATN"],
    "BLEU":                [tatn_bleu_score],
    "ChrF++":              [tatn_chrf_score],
    "BERTScore_P":         [tatn_bert_p_score],
    "BERTScore_R":         [tatn_bert_r_score],
    "BERTScore_F1":        [tatn_bert_f1_score],
    "COMET":               [tatn_comet_score],
    "MultiRef_BLEU":       [multi_ref_bleu_score],
    "MultiRef_ChrF++":     [multi_ref_chrf_score],
    "MBR_BLEU":            [mbr_bleu_score],
    "MBR_ChrF++":          [mbr_chrf_score],
    "MBR_vs_Beam_BLEU":    [mbr_vs_beam_bleu_delta],
    "MBR_vs_Beam_ChrF++":  [mbr_vs_beam_chrf_delta],
    "Num_Samples":         [len(sources)],
    "MBR_Samples":         [_MBR_NUM_SAMPLES],
    "MBR_Candidates":      [_MBR_NUM_CANDIDATES],
}
complete_summary_df = pd.DataFrame(complete_summary_data)
complete_summary_df.to_csv(complete_summary_file, index=False)
print(f"✅ Complete summary saved: {complete_summary_file}")

if mbr_translations:
    mbr_detailed_file = os.path.join(results_dir, "evaluation_mbr_detailed.csv")
    mbr_detailed_data = {
        "source":           mbr_sources,
        "reference":        mbr_references,
        "beam_translation": tatn_translations[:_MBR_NUM_SAMPLES],
        "mbr_translation":  mbr_translations,
    }
    mbr_detailed_df = pd.DataFrame(mbr_detailed_data)
    mbr_detailed_df.to_csv(mbr_detailed_file, index=False)
    print(f"✅ MBR detailed results saved: {mbr_detailed_file}")

if _it2_refs_valid:
    it2_refs_file = os.path.join(results_dir, "references_indictrans2.txt")
    with open(it2_refs_file, "w", encoding="utf-8") as f:
        for ref in references_it2:
            f.write(ref + "\n")
    print(f"✅ IndicTrans2 references saved: {it2_refs_file}")

print("\n" + "=" * 80)
print("CELL 13: ALL EVALUATION COMPLETE")
print("=" * 80)

print(f"\n📊 QUICK SUMMARY:")
print(f"   BLEU:            {tatn_bleu_score:.2f}")
print(f"   ChrF++:          {tatn_chrf_score:.2f}")
print(f"   BERTScore-F1:    {tatn_bert_f1_score:.2f}")
print(f"   COMET:           {tatn_comet_score:.4f}")
if _multi_ref_ran_ok:
    print(f"   Multi-ref BLEU:  {multi_ref_bleu_score:.2f}")
if _mbr_ran_ok:
    print(f"   MBR BLEU:        {mbr_bleu_score:.2f}")

print(f"\n✅ Results saved to:")
print(f"   - {summary_file}")
print(f"   - {detailed_file}")
print(f"   - {complete_summary_file}")
if mbr_translations:
    print(f"   - {mbr_detailed_file}")
if _it2_refs_valid:
    print(f"   - {it2_refs_file}")

print("=" * 80 + "\n")

try:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass
gc.collect()
